# <span style="color:green"> INDIVIDUAL EXPLORATORY ANALYSIS</span>

## <span style="color:green"> PACKAGES USED </span> ##

In [1]:
import pandas as pd
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io
import base64
from pathlib import Path
import gc
from scipy import stats
from matplotlib.ticker import FuncFormatter

## <span style="color:green"> DATASET IMPORT AND DIRECTION OF ADA STUDIES </span> ##

In [2]:
dataset_final = pd.read_parquet(
    "../../data/final_dataset/dataset_final.parquet"
)

print(dataset_final.shape)

for i, column in enumerate(
    dataset_final.columns,
    start=1
):
    print(f"{i}. {column}")
    

(1852394, 22)
1. NID_ALPHA
2. TRANS_NUM_CARD_FEWF
3. RECEIVE_LOC_FEWF
4. RECEIVE_CATEGORY_OHEWI
5. TRANS_VALUE
6. SEND_GENDER_BE
7. SEND_LAT_REGISTER
8. SEND_LONG_REGISTER
9. SEND_POP_REGISTER
10. SEND_JOB_FEWF
11. RECEIVE_LAT
12. RECEIVE_LONG
13. TRANS_DAY
14. TRANS_WEEK_OHEWI
15. TRANS_YEAR_BE
16. TRANS_MONTH_SIN
17. TRANS_MONTH_COS
18. TRANS_HOUR_SIN
19. TRANS_HOUR_COS
20. SEND_NAME_FEWF
21. SEND_AGE
22. TARGET_OMEGA


## <span style="color:CYAN"> BINARY ENCODING </span> ##

### <span style="color:CYAN"> TARGET_OMEGA </span> ###

In [ ]:
# ============================================================
# 01. ANALYSIS SETTINGS
#
# FOR THE NEXT FEATURES, CHANGE ONLY THESE TWO LINES
# ============================================================

ENCODING_TYPE = "binary_encoding"
FEATURE_NAME = "target_omega"


# Actual column name inside the dataset
FEATURE_COLUMN = FEATURE_NAME.upper()


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / ENCODING_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


LOG_CHART_NAME = (
    f"{FEATURE_NAME}_class_distribution_log"
)


LOG_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{LOG_CHART_NAME}.png"
)


CUMULATIVE_CHART_NAME = (
    f"{FEATURE_NAME}_cumulative_fraud"
)


CUMULATIVE_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{CUMULATIVE_CHART_NAME}.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)

log_chart_exists = (
    LOG_CHART_PATH.exists()
)

cumulative_chart_exists = (
    CUMULATIVE_CHART_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and log_chart_exists
    and cumulative_chart_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )

    print(
        "No analysis or file creation is required."
    )

    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )

    print(
        "\nExisting files:"
    )

    print(
        HTML_PATH
    )

    print(
        LOG_CHART_PATH
    )

    print(
        CUMULATIVE_CHART_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )

    print(
        "=" * 100
    )

    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )

    print(
        "Logarithmic distribution chart:",
        "Already exists"
        if log_chart_exists
        else "Will be created"
    )

    print(
        "Cumulative fraud chart:",
        "Already exists"
        if cumulative_chart_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. VALIDATE THE FEATURE
    # ========================================================

    if feature.isna().any():

        raise ValueError(
            f"{FEATURE_COLUMN} contains missing values."
        )


    found_values = set(
        feature.unique()
    )


    if not found_values.issubset(
        {0, 1}
    ):

        raise ValueError(
            f"{FEATURE_COLUMN} contains values "
            f"different from 0 and 1: "
            f"{found_values}"
        )


    # ========================================================
    # 13. ABSOLUTE COUNT OF EACH CLASS
    # ========================================================

    class_count = (
        feature
        .value_counts()
        .reindex(
            [
                0,
                1
            ],
            fill_value=0
        )
    )


    non_fraud = int(
        class_count.loc[0]
    )


    fraud = int(
        class_count.loc[1]
    )


    total = int(
        class_count.sum()
    )


    # ========================================================
    # 14. PERCENTAGE PROPORTION OF EACH CLASS
    # ========================================================

    non_fraud_percentage = (
        non_fraud
        / total
        * 100
    )


    fraud_percentage = (
        fraud
        / total
        * 100
    )


    # ========================================================
    # 15. NON-FRAUD TO FRAUD RATIO
    # ========================================================

    if fraud > 0:

        non_fraud_fraud_ratio = (
            non_fraud
            / fraud
        )

    else:

        non_fraud_fraud_ratio = np.inf


    # ========================================================
    # 16. IMBALANCE RATIO (IR)
    #
    # Majority class / Minority class
    # ========================================================

    majority_class = int(
        class_count.max()
    )


    minority_class = int(
        class_count.min()
    )


    if minority_class > 0:

        imbalance_ratio = (
            majority_class
            / minority_class
        )

    else:

        imbalance_ratio = np.inf


    # ========================================================
    # 17. SHANNON ENTROPY
    # ========================================================

    proportions = (
        class_count
        / total
    )


    shannon_entropy = -sum(
        proportion
        * np.log2(
            proportion
        )

        for proportion in proportions

        if proportion > 0
    )


    # ========================================================
    # 18. FUNCTION TO CONVERT AN EXISTING PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 19. CREATE THE LOGARITHMIC DISTRIBUTION CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not log_chart_exists:

        labels = [
            "Non-fraud (0)",
            "Fraud (1)"
        ]


        values = [
            non_fraud,
            fraud
        ]


        fig, ax = plt.subplots(
            figsize=(
                8,
                6
            )
        )


        bars = ax.bar(
            labels,
            values
        )


        ax.set_yscale(
            "log"
        )


        ax.set_title(
            f"{FEATURE_COLUMN} class distribution "
            f"on a logarithmic scale"
        )


        ax.set_xlabel(
            "Class"
        )


        ax.set_ylabel(
            "Number of transactions"
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        # ----------------------------------------------------
        # DISPLAY VALUES ABOVE THE BARS
        # WITHOUT THOUSANDS SEPARATORS
        # ----------------------------------------------------

        for bar, value in zip(
            bars,
            values
        ):

            ax.text(
                bar.get_x()
                + bar.get_width() / 2,

                value,

                str(
                    value
                ),

                ha="center",
                va="bottom"
            )


        fig.tight_layout()


        fig.savefig(
            LOG_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nLogarithmic distribution chart created:"
        )

        print(
            LOG_CHART_PATH
        )


    else:

        print(
            "\nLogarithmic distribution chart already exists."
        )

        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 21. IDENTIFY FRAUDULENT TRANSACTION POSITIONS
    # ========================================================

    feature_array = (
        feature.to_numpy()
    )


    fraud_positions = (
        np.flatnonzero(
            feature_array == 1
        )
        + 1
    )


    # ========================================================
    # 21. BUILD THE CUMULATIVE FRAUD CURVE
    # ========================================================

    if len(
        fraud_positions
    ) > 0:

        cumulative_x = np.concatenate(
            (
                [
                    1
                ],

                fraud_positions,

                [
                    total
                ]
            )
        )


        cumulative_y = np.concatenate(
            (
                [
                    0
                ],

                np.arange(
                    1,
                    len(
                        fraud_positions
                    )
                    + 1
                ),

                [
                    len(
                        fraud_positions
                    )
                ]
            )
        )


    else:

        cumulative_x = np.array(
            [
                1,
                total
            ]
        )


        cumulative_y = np.array(
            [
                0,
                0
            ]
        )


    # ========================================================
    # 22. CREATE THE CUMULATIVE FRAUD CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not cumulative_chart_exists:

        fig, ax = plt.subplots(
            figsize=(
                12,
                6
            )
        )


        ax.step(
            cumulative_x,
            cumulative_y,
            where="post"
        )


        ax.set_title(
            "Cumulative fraud by transaction order"
        )


        ax.set_xlabel(
            "Transaction order"
        )


        ax.set_ylabel(
            "Cumulative number of fraudulent transactions"
        )


        ax.set_xlim(
            1,
            total
        )


        ax.set_ylim(
            bottom=0
        )


        ax.grid(
            alpha=0.3
        )


        # ----------------------------------------------------
        # AXIS FORMAT
        #
        # Without thousands separators.
        #
        # Example:
        # 500000
        #
        # Not:
        # 500,000
        # ----------------------------------------------------

        ax.xaxis.set_major_formatter(
            FuncFormatter(
                lambda x, pos:
                str(
                    int(
                        x
                    )
                )
            )
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(
                        y
                    )
                )
            )
        )


        fig.tight_layout()


        fig.savefig(
            CUMULATIVE_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nCumulative fraud chart created:"
        )

        print(
            CUMULATIVE_CHART_PATH
        )


    else:

        print(
            "\nCumulative fraud chart already exists."
        )

        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 23. CREATE THE HTML REPORT
    #
    # Only if the HTML does not already exist.
    # ========================================================

    if not html_exists:

        # ----------------------------------------------------
        # CONVERT THE PNG FILES TO BASE64
        #
        # At this point the PNG files either already existed
        # or were created during this execution.
        # ----------------------------------------------------

        log_chart_base64 = (
            image_to_base64(
                LOG_CHART_PATH
            )
        )


        cumulative_chart_base64 = (
            image_to_base64(
                CUMULATIVE_CHART_PATH
            )
        )


        # ====================================================
        # HTML CONTENT
        # ====================================================

        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1100px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 25px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 10px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 35px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.summary {{
    margin-bottom: 40px;
}}

</style>

</head>


<body>


<!-- ========================================================
     TITLE
========================================================= -->


<h1>
Individual Exploratory Analysis — {FEATURE_COLUMN}
</h1>


<p>

The variable <strong>{FEATURE_COLUMN}</strong>
corresponds to the binary target variable
of the dataset.

</p>


<ul>

<li>
<strong>0:</strong>
non-fraudulent transaction
</li>

<li>
<strong>1:</strong>
fraudulent transaction
</li>

</ul>


<!-- ========================================================
     1. ABSOLUTE CLASS COUNT
========================================================= -->


<h2>
1. Absolute count of each class
</h2>


<table>

<thead>

<tr>

<th>Class</th>

<th>Meaning</th>

<th>Count</th>

</tr>

</thead>


<tbody>


<tr>

<td>0</td>

<td>Non-fraud</td>

<td>{non_fraud}</td>

</tr>


<tr>

<td>1</td>

<td>Fraud</td>

<td>{fraud}</td>

</tr>


<tr>

<td>
<strong>Total</strong>
</td>

<td>-</td>

<td>
<strong>{total}</strong>
</td>

</tr>


</tbody>

</table>


<!-- ========================================================
     2. PERCENTAGE PROPORTION
========================================================= -->


<h2>
2. Percentage proportion of each class
</h2>


<table>

<thead>

<tr>

<th>Class</th>

<th>Meaning</th>

<th>Percentage</th>

</tr>

</thead>


<tbody>


<tr>

<td>0</td>

<td>Non-fraud</td>

<td>
{non_fraud_percentage:.6f}%
</td>

</tr>


<tr>

<td>1</td>

<td>Fraud</td>

<td>
{fraud_percentage:.6f}%
</td>

</tr>


</tbody>

</table>


<!-- ========================================================
     3. NON-FRAUD TO FRAUD RATIO
========================================================= -->


<h2>
3. Non-fraud to fraud ratio
</h2>


<p class="result">

{non_fraud_fraud_ratio:.2f}:1

</p>


<p>

There is approximately

<strong>

1 fraudulent transaction for every
{non_fraud_fraud_ratio:.2f}
non-fraudulent transactions.

</strong>

</p>


<!-- ========================================================
     4. IMBALANCE RATIO
========================================================= -->


<h2>
4. Imbalance Ratio (IR)
</h2>


<p class="result">

IR = {imbalance_ratio:.4f}

</p>


<p>

The Imbalance Ratio represents the ratio
between the number of observations
in the majority class and the number
of observations in the minority class.

</p>


<p>

The larger the IR value,
the greater the imbalance
between the classes.

</p>


<!-- ========================================================
     5. SHANNON ENTROPY
========================================================= -->


<h2>
5. Shannon Entropy
</h2>


<p class="result">

Entropy = {shannon_entropy:.6f} bits

</p>


<p>

For a binary variable,
Shannon Entropy ranges
from 0 to 1 bit.

</p>


<ul>

<li>

Values close to
<strong>0</strong>
indicate a greater concentration
of observations in a single class.

</li>


<li>

Values close to
<strong>1</strong>
indicate a greater balance
between the two classes.

</li>

</ul>


<!-- ========================================================
     6. LOGARITHMIC CLASS DISTRIBUTION
========================================================= -->


<h2>
6. Class distribution on a logarithmic scale
</h2>


<p>

The logarithmic scale facilitates
the visual comparison between the two classes
when there is a large difference
between their absolute frequencies.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{log_chart_base64}"
    alt="{FEATURE_COLUMN} class distribution on a logarithmic scale"
>

</div>


<!-- ========================================================
     7. CUMULATIVE FRAUD
========================================================= -->


<h2>
7. Cumulative fraud by transaction order
</h2>


<p>

The X axis represents the sequential position
of each transaction in the dataset.

Position 1 corresponds to the first transaction,
position 2 to the second transaction,
position 3 to the third transaction,
and so forth.

</p>


<p>

The Y axis represents the cumulative number
of fraudulent transactions.

Whenever an observation has
<strong>{FEATURE_COLUMN} = 1</strong>,
the cumulative value increases by one unit.

When an observation has
<strong>{FEATURE_COLUMN} = 0</strong>,
the cumulative value remains unchanged.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{cumulative_chart_base64}"
    alt="Cumulative fraud by transaction order"
>

</div>


<!-- ========================================================
     8. SUMMARY
========================================================= -->


<h2>
8. Summary of results
</h2>


<div class="summary">

<ul>


<li>

<strong>
Total observations:
</strong>

{total}

</li>


<li>

<strong>
Non-fraudulent transactions:
</strong>

{non_fraud}

</li>


<li>

<strong>
Fraudulent transactions:
</strong>

{fraud}

</li>


<li>

<strong>
Non-fraud percentage:
</strong>

{non_fraud_percentage:.6f}%

</li>


<li>

<strong>
Fraud percentage:
</strong>

{fraud_percentage:.6f}%

</li>


<li>

<strong>
Non-fraud to fraud ratio:
</strong>

{non_fraud_fraud_ratio:.2f}:1

</li>


<li>

<strong>
Imbalance Ratio:
</strong>

{imbalance_ratio:.4f}

</li>


<li>

<strong>
Shannon Entropy:
</strong>

{shannon_entropy:.6f} bits

</li>


</ul>

</div>


</body>

</html>
"""


        # ====================================================
        # 24. SAVE THE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )

        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )

        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 26. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature
    del feature_array
    del fraud_positions
    del cumulative_x
    del cumulative_y

    gc.collect()


    # ========================================================
    # 26. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "ANALYSIS COMPLETED"
    )

    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )

    print(
        HTML_PATH
    )


    print(
        "\nPNG files:"
    )

    print(
        LOG_CHART_PATH
    )

    print(
        CUMULATIVE_CHART_PATH
    )

All analysis files already exist.
No analysis or file creation is required.

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/target_omega

Existing files:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/target_omega/analysis_target_omega.html
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/target_omega/target_omega_class_distribution_log.png
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/target_omega/target_omega_cumulative_fraud.png


### <span style="color:CYAN"> TRANS_YEAR_BE </span> ###

In [7]:
# ============================================================
# 01. ANALYSIS SETTINGS
#
# FOR THE NEXT FEATURES, CHANGE MAINLY THESE TWO LINES
# ============================================================

ENCODING_TYPE = "binary_encoding"
FEATURE_NAME = "trans_year_be"


# Actual column name inside the dataset
FEATURE_COLUMN = FEATURE_NAME.upper()


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / ENCODING_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


DISTRIBUTION_CHART_NAME = (
    f"{FEATURE_NAME}_distribution"
)


DISTRIBUTION_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{DISTRIBUTION_CHART_NAME}.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


distribution_chart_exists = (
    DISTRIBUTION_CHART_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and distribution_chart_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )

    print(
        "No analysis or file creation is required."
    )

    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )

    print(
        "\nExisting files:"
    )

    print(
        HTML_PATH
    )

    print(
        DISTRIBUTION_CHART_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )

    print(
        "=" * 100
    )

    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )

    print(
        "Distribution chart:",
        "Already exists"
        if distribution_chart_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. VALIDATE THE FEATURE
    # ========================================================

    if feature.isna().any():

        raise ValueError(
            f"{FEATURE_COLUMN} contains missing values."
        )


    found_values = set(
        feature.unique()
    )


    if not found_values.issubset(
        {
            2019,
            2020
        }
    ):

        raise ValueError(
            f"{FEATURE_COLUMN} contains unexpected values: "
            f"{found_values}"
        )


    # ========================================================
    # 13. ABSOLUTE COUNT OF EACH YEAR
    # ========================================================

    year_count = (
        feature
        .value_counts()
        .reindex(
            [
                2019,
                2020
            ],
            fill_value=0
        )
    )


    count_2019 = int(
        year_count.loc[2019]
    )


    count_2020 = int(
        year_count.loc[2020]
    )


    total = int(
        year_count.sum()
    )


    if total == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    # ========================================================
    # 14. PERCENTAGE PROPORTION OF EACH YEAR
    # ========================================================

    percentage_2019 = (
        count_2019
        / total
        * 100
    )


    percentage_2020 = (
        count_2020
        / total
        * 100
    )


    # ========================================================
    # 15. IDENTIFY THE MAJORITY AND MINORITY YEARS
    # ========================================================

    if count_2019 >= count_2020:

        majority_year = 2019
        majority_count = count_2019

        minority_year = 2020
        minority_count = count_2020

    else:

        majority_year = 2020
        majority_count = count_2020

        minority_year = 2019
        minority_count = count_2019


    # ========================================================
    # 16. IMBALANCE RATIO
    # ========================================================

    if minority_count > 0:

        imbalance_ratio = (
            majority_count
            / minority_count
        )

    else:

        imbalance_ratio = np.inf


    # ========================================================
    # 17. SHANNON ENTROPY
    # ========================================================

    proportions = (
        year_count
        / total
    )


    shannon_entropy = -sum(
        proportion
        * np.log2(
            proportion
        )

        for proportion in proportions

        if proportion > 0
    )


    # ========================================================
    # 18. FORMAT TEXTUAL RESULTS
    # ========================================================

    if np.isfinite(
        imbalance_ratio
    ):

        imbalance_ratio_text = (
            f"{imbalance_ratio:.2f}:1"
        )


        imbalance_interpretation = (
            f"There is approximately "
            f"1 transaction recorded in {minority_year} "
            f"for every {imbalance_ratio:.2f} transactions "
            f"recorded in {majority_year}."
        )


        ir_text = (
            f"{imbalance_ratio:.4f}"
        )


    else:

        imbalance_ratio_text = (
            "Undefined"
        )


        imbalance_interpretation = (
            "The imbalance ratio could not be calculated "
            "because one of the years contains no observations."
        )


        ir_text = (
            "Undefined"
        )


    # ========================================================
    # 19. DATA USED IN THE DISTRIBUTION CHART
    # ========================================================

    labels = [
        "2019",
        "2020"
    ]


    values = [
        count_2019,
        count_2020
    ]


    # ========================================================
    # 20. CREATE THE DISTRIBUTION CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not distribution_chart_exists:

        fig, ax = plt.subplots(
            figsize=(
                8,
                6
            )
        )


        bars = ax.bar(
            labels,
            values
        )


        ax.set_title(
            "Transaction distribution by year"
        )


        ax.set_xlabel(
            "Year"
        )


        ax.set_ylabel(
            "Number of transactions"
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        # ----------------------------------------------------
        # Y AXIS WITHOUT THOUSANDS SEPARATORS
        #
        # Example:
        # 1000000
        #
        # Not:
        # 1,000,000
        # ----------------------------------------------------

        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(
                        y
                    )
                )
            )
        )


        # ----------------------------------------------------
        # ABSOLUTE COUNT ABOVE EACH BAR
        # ----------------------------------------------------

        for bar, value in zip(
            bars,
            values
        ):

            ax.text(
                bar.get_x()
                + bar.get_width() / 2,

                value,

                str(
                    value
                ),

                ha="center",
                va="bottom"
            )


        fig.tight_layout()


        fig.savefig(
            DISTRIBUTION_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nDistribution chart created:"
        )

        print(
            DISTRIBUTION_CHART_PATH
        )


    else:

        print(
            "\nDistribution chart already exists."
        )

        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 21. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            image_base64 = (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


        return image_base64


    # ========================================================
    # 22. CREATE THE HTML REPORT
    #
    # Only if the HTML does not already exist.
    # ========================================================

    if not html_exists:

        distribution_chart_base64 = (
            image_to_base64(
                DISTRIBUTION_CHART_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1100px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 25px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 10px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 35px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.summary {{
    margin-bottom: 40px;
}}

</style>

</head>


<body>


<!-- ========================================================
     TITLE
========================================================= -->


<h1>
Individual Exploratory Analysis — {FEATURE_COLUMN}
</h1>


<p>

The variable <strong>{FEATURE_COLUMN}</strong>
represents the year in which each transaction
was recorded in the dataset.

</p>


<p>

The values observed in this feature are:

</p>


<ul>

<li>
<strong>2019</strong>
</li>

<li>
<strong>2020</strong>
</li>

</ul>


<!-- ========================================================
     1. ABSOLUTE COUNT
========================================================= -->


<h2>
1. Absolute number of transactions by year
</h2>


<table>

<thead>

<tr>

<th>
Year
</th>

<th>
Count
</th>

</tr>

</thead>


<tbody>


<tr>

<td>
2019
</td>

<td>
{count_2019}
</td>

</tr>


<tr>

<td>
2020
</td>

<td>
{count_2020}
</td>

</tr>


<tr>

<td>
<strong>Total</strong>
</td>

<td>
<strong>{total}</strong>
</td>

</tr>


</tbody>

</table>


<!-- ========================================================
     2. PERCENTAGE PROPORTION
========================================================= -->


<h2>
2. Percentage proportion of transactions by year
</h2>


<table>

<thead>

<tr>

<th>
Year
</th>

<th>
Count
</th>

<th>
Percentage
</th>

</tr>

</thead>


<tbody>


<tr>

<td>
2019
</td>

<td>
{count_2019}
</td>

<td>
{percentage_2019:.6f}%
</td>

</tr>


<tr>

<td>
2020
</td>

<td>
{count_2020}
</td>

<td>
{percentage_2020:.6f}%
</td>

</tr>


</tbody>

</table>


<!-- ========================================================
     3. IMBALANCE RATIO
========================================================= -->


<h2>
3. Imbalance ratio
</h2>


<p class="result">

{imbalance_ratio_text}

</p>


<p>

The year with the largest number
of observations was
<strong>{majority_year}</strong>.

</p>


<p>

<strong>
{imbalance_interpretation}
</strong>

</p>


<!-- ========================================================
     4. IMBALANCE RATIO (IR)
========================================================= -->


<h2>
4. Imbalance Ratio (IR)
</h2>


<p class="result">

IR = {ir_text}

</p>


<p>

The Imbalance Ratio was calculated
as the ratio between the number
of observations in the most frequent year
and the number of observations
in the least frequent year.

</p>


<p>

A value close to <strong>1</strong>
indicates similar frequencies between
the two years.

Progressively larger values indicate
a greater difference between
the number of observations.

</p>


<!-- ========================================================
     5. SHANNON ENTROPY
========================================================= -->


<h2>
5. Shannon Entropy
</h2>


<p class="result">

Entropy = {shannon_entropy:.6f} bits

</p>


<p>

Shannon Entropy provides a measure
of how balanced the distribution
of observations is between the two years.

</p>


<ul>


<li>

Values close to
<strong>0</strong>
indicate a greater concentration
of observations in only one year.

</li>


<li>

Values close to
<strong>1</strong>
indicate a more balanced distribution
between 2019 and 2020.

</li>


</ul>


<!-- ========================================================
     6. ABSOLUTE DISTRIBUTION
========================================================= -->


<h2>
6. Transaction distribution by year
</h2>


<p>

The chart presents the absolute number
of transactions recorded in 2019 and 2020.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{distribution_chart_base64}"
    alt="Transaction distribution between 2019 and 2020"
>

</div>


<!-- ========================================================
     7. SUMMARY
========================================================= -->


<h2>
7. Summary of results
</h2>


<div class="summary">

<ul>


<li>

<strong>
Total observations:
</strong>

{total}

</li>


<li>

<strong>
Transactions in 2019:
</strong>

{count_2019}

</li>


<li>

<strong>
Transactions in 2020:
</strong>

{count_2020}

</li>


<li>

<strong>
Percentage in 2019:
</strong>

{percentage_2019:.6f}%

</li>


<li>

<strong>
Percentage in 2020:
</strong>

{percentage_2020:.6f}%

</li>


<li>

<strong>
Most frequent year:
</strong>

{majority_year}

</li>


<li>

<strong>
Least frequent year:
</strong>

{minority_year}

</li>


<li>

<strong>
Imbalance ratio:
</strong>

{imbalance_ratio_text}

</li>


<li>

<strong>
Imbalance Ratio:
</strong>

{ir_text}

</li>


<li>

<strong>
Shannon Entropy:
</strong>

{shannon_entropy:.6f} bits

</li>


</ul>

</div>


</body>

</html>
"""


        # ====================================================
        # 23. SAVE THE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )

        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )

        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 24. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature

    gc.collect()


    # ========================================================
    # 25. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "ANALYSIS COMPLETED"
    )

    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )

    print(
        HTML_PATH
    )


    print(
        "\nPNG:"
    )

    print(
        DISTRIBUTION_CHART_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Distribution chart: Will be created

Distribution chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/trans_year_be/trans_year_be_distribution.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/trans_year_be/analysis_trans_year_be.html

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/trans_year_be

HTML:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/trans_year_be/analysis_trans_year_be.html

PNG:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/trans_year_be/trans_year_be_distribution.png


### <span style="color:CYAN"> SEND_GENDER_BE </span> ###

In [8]:
# ============================================================
# 01. ANALYSIS SETTINGS
#
# FOR THE NEXT FEATURES, CHANGE MAINLY THESE TWO LINES
# ============================================================

ENCODING_TYPE = "binary_encoding"
FEATURE_NAME = "send_gender_be"


# Actual column name inside the dataset
FEATURE_COLUMN = FEATURE_NAME.upper()


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / ENCODING_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


DISTRIBUTION_CHART_NAME = (
    f"{FEATURE_NAME}_distribution"
)


DISTRIBUTION_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{DISTRIBUTION_CHART_NAME}.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


distribution_chart_exists = (
    DISTRIBUTION_CHART_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and distribution_chart_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )

    print(
        "No analysis or file creation is required."
    )

    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )

    print(
        "\nExisting files:"
    )

    print(
        HTML_PATH
    )

    print(
        DISTRIBUTION_CHART_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )

    print(
        "=" * 100
    )

    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )

    print(
        "Distribution chart:",
        "Already exists"
        if distribution_chart_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. VALIDATE THE FEATURE
    # ========================================================

    if feature.isna().any():

        raise ValueError(
            f"{FEATURE_COLUMN} contains missing values."
        )


    found_values = set(
        feature.unique()
    )


    if not found_values.issubset(
        {
            "F",
            "M"
        }
    ):

        raise ValueError(
            f"{FEATURE_COLUMN} contains unexpected values: "
            f"{found_values}"
        )


    # ========================================================
    # 13. ABSOLUTE COUNT OF EACH CATEGORY
    # ========================================================

    category_count = (
        feature
        .value_counts()
        .reindex(
            [
                "F",
                "M"
            ],
            fill_value=0
        )
    )


    female_count = int(
        category_count.loc["F"]
    )


    male_count = int(
        category_count.loc["M"]
    )


    total = int(
        category_count.sum()
    )


    if total == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    # ========================================================
    # 14. PERCENTAGE PROPORTION OF EACH CATEGORY
    # ========================================================

    female_percentage = (
        female_count
        / total
        * 100
    )


    male_percentage = (
        male_count
        / total
        * 100
    )


    # ========================================================
    # 15. IDENTIFY THE MAJORITY AND MINORITY CATEGORIES
    # ========================================================

    if female_count >= male_count:

        majority_category = "F"

        majority_gender = (
            "Female"
        )

        majority_count = (
            female_count
        )


        minority_category = "M"

        minority_gender = (
            "Male"
        )

        minority_count = (
            male_count
        )


    else:

        majority_category = "M"

        majority_gender = (
            "Male"
        )

        majority_count = (
            male_count
        )


        minority_category = "F"

        minority_gender = (
            "Female"
        )

        minority_count = (
            female_count
        )


    # ========================================================
    # 16. IMBALANCE RATIO
    # ========================================================

    if minority_count > 0:

        imbalance_ratio = (
            majority_count
            / minority_count
        )

    else:

        imbalance_ratio = np.inf


    # ========================================================
    # 17. SHANNON ENTROPY
    # ========================================================

    proportions = (
        category_count
        / total
    )


    shannon_entropy = -sum(
        proportion
        * np.log2(
            proportion
        )

        for proportion in proportions

        if proportion > 0
    )


    # ========================================================
    # 18. FORMAT TEXTUAL RESULTS
    # ========================================================

    if np.isfinite(
        imbalance_ratio
    ):

        imbalance_ratio_text = (
            f"{imbalance_ratio:.2f}:1"
        )


        imbalance_interpretation = (
            f"There is approximately "
            f"1 {minority_gender.lower()} observation "
            f"for every {imbalance_ratio:.2f} "
            f"{majority_gender.lower()} observations."
        )


        ir_text = (
            f"{imbalance_ratio:.4f}"
        )


    else:

        imbalance_ratio_text = (
            "Undefined"
        )


        imbalance_interpretation = (
            "The imbalance ratio could not be calculated "
            "because one of the categories contains "
            "no observations."
        )


        ir_text = (
            "Undefined"
        )


    # ========================================================
    # 19. DATA USED IN THE DISTRIBUTION CHART
    # ========================================================

    labels = [
        "Female (F)",
        "Male (M)"
    ]


    values = [
        female_count,
        male_count
    ]


    # ========================================================
    # 20. CREATE THE DISTRIBUTION CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not distribution_chart_exists:

        fig, ax = plt.subplots(
            figsize=(
                8,
                6
            )
        )


        bars = ax.bar(
            labels,
            values
        )


        ax.set_title(
            "Distribution of observations by gender"
        )


        ax.set_xlabel(
            "Gender"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        # ----------------------------------------------------
        # Y AXIS WITHOUT THOUSANDS SEPARATORS
        #
        # Example:
        # 1000000
        #
        # Not:
        # 1,000,000
        # ----------------------------------------------------

        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(
                        y
                    )
                )
            )
        )


        # ----------------------------------------------------
        # ABSOLUTE COUNT ABOVE EACH BAR
        # ----------------------------------------------------

        for bar, value in zip(
            bars,
            values
        ):

            ax.text(
                bar.get_x()
                + bar.get_width() / 2,

                value,

                str(
                    value
                ),

                ha="center",
                va="bottom"
            )


        fig.tight_layout()


        fig.savefig(
            DISTRIBUTION_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nDistribution chart created:"
        )

        print(
            DISTRIBUTION_CHART_PATH
        )


    else:

        print(
            "\nDistribution chart already exists."
        )

        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 21. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            image_base64 = (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


        return image_base64


    # ========================================================
    # 22. CREATE THE HTML REPORT
    #
    # Only if the HTML does not already exist.
    # ========================================================

    if not html_exists:

        distribution_chart_base64 = (
            image_to_base64(
                DISTRIBUTION_CHART_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1100px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 25px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 10px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 35px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.summary {{
    margin-bottom: 40px;
}}

</style>

</head>


<body>


<!-- ========================================================
     TITLE
========================================================= -->


<h1>
Individual Exploratory Analysis — {FEATURE_COLUMN}
</h1>


<p>

The variable <strong>{FEATURE_COLUMN}</strong>
represents the registered gender of the sender
associated with each transaction.

</p>


<p>

The observed values in this feature are:

</p>


<ul>

<li>
<strong>F:</strong> Female
</li>

<li>
<strong>M:</strong> Male
</li>

</ul>


<!-- ========================================================
     1. ABSOLUTE COUNT
========================================================= -->


<h2>
1. Absolute count by gender
</h2>


<table>

<thead>

<tr>

<th>
Category
</th>

<th>
Gender
</th>

<th>
Count
</th>

</tr>

</thead>


<tbody>


<tr>

<td>
F
</td>

<td>
Female
</td>

<td>
{female_count}
</td>

</tr>


<tr>

<td>
M
</td>

<td>
Male
</td>

<td>
{male_count}
</td>

</tr>


<tr>

<td>
-
</td>

<td>
<strong>Total</strong>
</td>

<td>
<strong>{total}</strong>
</td>

</tr>


</tbody>

</table>


<!-- ========================================================
     2. PERCENTAGE PROPORTION
========================================================= -->


<h2>
2. Percentage proportion by gender
</h2>


<table>

<thead>

<tr>

<th>
Category
</th>

<th>
Gender
</th>

<th>
Count
</th>

<th>
Percentage
</th>

</tr>

</thead>


<tbody>


<tr>

<td>
F
</td>

<td>
Female
</td>

<td>
{female_count}
</td>

<td>
{female_percentage:.6f}%
</td>

</tr>


<tr>

<td>
M
</td>

<td>
Male
</td>

<td>
{male_count}
</td>

<td>
{male_percentage:.6f}%
</td>

</tr>


</tbody>

</table>


<!-- ========================================================
     3. IMBALANCE RATIO
========================================================= -->


<h2>
3. Imbalance ratio
</h2>


<p class="result">

{imbalance_ratio_text}

</p>


<p>

The category with the largest number
of observations was

<strong>
{majority_gender} ({majority_category})
</strong>.

</p>


<p>

<strong>
{imbalance_interpretation}
</strong>

</p>


<!-- ========================================================
     4. IMBALANCE RATIO (IR)
========================================================= -->


<h2>
4. Imbalance Ratio (IR)
</h2>


<p class="result">

IR = {ir_text}

</p>


<p>

The Imbalance Ratio was calculated
as the ratio between the number of observations
in the most frequent category
and the number of observations
in the least frequent category.

</p>


<p>

A value close to <strong>1</strong>
indicates similar frequencies
between the two categories.

Progressively larger values indicate
a greater difference between
the number of observations.

</p>


<!-- ========================================================
     5. SHANNON ENTROPY
========================================================= -->


<h2>
5. Shannon Entropy
</h2>


<p class="result">

Entropy = {shannon_entropy:.6f} bits

</p>


<p>

Shannon Entropy provides a measure
of how balanced the distribution
of observations is between
the Female and Male categories.

</p>


<ul>


<li>

Values close to
<strong>0</strong>
indicate a greater concentration
of observations in a single category.

</li>


<li>

Values close to
<strong>1</strong>
indicate a more balanced distribution
between the two categories.

</li>


</ul>


<!-- ========================================================
     6. ABSOLUTE DISTRIBUTION
========================================================= -->


<h2>
6. Distribution of observations by gender
</h2>


<p>

The chart presents the absolute number
of observations belonging to the
Female and Male categories
in the variable
<strong>{FEATURE_COLUMN}</strong>.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{distribution_chart_base64}"
    alt="Distribution of {FEATURE_COLUMN}"
>

</div>


<!-- ========================================================
     7. SUMMARY
========================================================= -->


<h2>
7. Summary of results
</h2>


<div class="summary">

<ul>


<li>

<strong>
Total observations:
</strong>

{total}

</li>


<li>

<strong>
Female:
</strong>

{female_count}

</li>


<li>

<strong>
Male:
</strong>

{male_count}

</li>


<li>

<strong>
Female percentage:
</strong>

{female_percentage:.6f}%

</li>


<li>

<strong>
Male percentage:
</strong>

{male_percentage:.6f}%

</li>


<li>

<strong>
Most frequent category:
</strong>

{majority_gender}
({majority_category})

</li>


<li>

<strong>
Least frequent category:
</strong>

{minority_gender}
({minority_category})

</li>


<li>

<strong>
Imbalance ratio:
</strong>

{imbalance_ratio_text}

</li>


<li>

<strong>
Imbalance Ratio:
</strong>

{ir_text}

</li>


<li>

<strong>
Shannon Entropy:
</strong>

{shannon_entropy:.6f} bits

</li>


</ul>

</div>


</body>

</html>
"""


        # ====================================================
        # 23. SAVE THE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )

        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )

        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 24. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature

    gc.collect()


    # ========================================================
    # 25. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "ANALYSIS COMPLETED"
    )

    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )

    print(
        HTML_PATH
    )


    print(
        "\nPNG:"
    )

    print(
        DISTRIBUTION_CHART_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Distribution chart: Will be created

Distribution chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/send_gender_be/send_gender_be_distribution.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/send_gender_be/analysis_send_gender_be.html

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/send_gender_be

HTML:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/send_gender_be/analysis_send_gender_be.html

PNG:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/binary_encoding/send_gender_be/send_gender_be_distribution.png


## <span style="color:dodgerblue"> FREQUÊNCY ENCODING WITH FALLBACK (1/N_TOTAL) </span> ##

### <span style="color:dodgerblue"> TRANS_NUM_CARD_FEWF </span> ###

In [20]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ENCODING_TYPE = "frequency_encoding_with_fallback"
FEATURE_NAME = "trans_num_card_fewf"

FEATURE_COLUMN = FEATURE_NAME.upper()

ALPHA = 0.05


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / ENCODING_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


RANK_FREQUENCY_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_rank_frequency.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


rank_frequency_chart_exists = (
    RANK_FREQUENCY_CHART_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and rank_frequency_chart_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )

    print(
        "No analysis or file creation is required."
    )

    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )

    print(
        "\nExisting files:"
    )

    print(
        HTML_PATH
    )

    print(
        RANK_FREQUENCY_CHART_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )

    print(
        "=" * 100
    )

    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )

    print(
        "Rank-frequency chart:",
        "Already exists"
        if rank_frequency_chart_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. BASIC VALIDATION
    # ========================================================

    total = int(
        len(feature)
    )


    if total == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    missing_values = int(
        feature
        .isna()
        .sum()
    )


    if missing_values > 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains "
            f"{missing_values} missing values."
        )


    # ========================================================
    # 13. CATEGORY COUNTS
    # ========================================================

    category_counts = (
        feature
        .value_counts()
        .sort_values(
            ascending=False
        )
    )


    unique_categories = int(
        category_counts.shape[0]
    )


    cardinality_percentage = (
        unique_categories
        / total
        * 100
    )


    # ========================================================
    # 14. FREQUENCY STATISTICS
    # ========================================================

    minimum_count = int(
        category_counts.min()
    )


    maximum_count = int(
        category_counts.max()
    )


    mean_count = float(
        category_counts.mean()
    )


    median_count = float(
        category_counts.median()
    )


    q1_count = float(
        category_counts.quantile(
            0.25
        )
    )


    q3_count = float(
        category_counts.quantile(
            0.75
        )
    )


    # ========================================================
    # 15. FREQUENCY ENCODING VALUES
    #
    # FE(category) = count(category) / N_total
    # ========================================================

    category_fe = (
        category_counts
        / total
    )


    minimum_fe = float(
        category_fe.min()
    )


    maximum_fe = float(
        category_fe.max()
    )


    mean_fe = float(
        category_fe.mean()
    )


    median_fe = float(
        category_fe.median()
    )


    # ========================================================
    # 16. SIMULATE THE TRANSFORMED FEATURE
    #
    # This does not modify the original dataset.
    #
    # Each card identifier is temporarily replaced by its
    # corresponding Frequency Encoding value.
    # ========================================================

    encoded_preview = (
        feature
        .map(
            category_fe
        )
        .astype(
            "float64"
        )
    )


    # ========================================================
    # 17. DESCRIPTIVE STATISTICS AFTER FREQUENCY ENCODING
    # ========================================================

    transformed_minimum = float(
        encoded_preview.min()
    )


    transformed_q1 = float(
        encoded_preview.quantile(
            0.25
        )
    )


    transformed_median = float(
        encoded_preview.median()
    )


    transformed_mean = float(
        encoded_preview.mean()
    )


    transformed_q3 = float(
        encoded_preview.quantile(
            0.75
        )
    )


    transformed_maximum = float(
        encoded_preview.max()
    )


    transformed_standard_deviation = float(
        encoded_preview.std()
    )


    # ========================================================
    # 18. IQR OUTLIER LIMITS AFTER FREQUENCY ENCODING
    # ========================================================

    iqr_fe = (
        transformed_q3
        - transformed_q1
    )


    lower_bound_fe = (
        transformed_q1
        - 1.5 * iqr_fe
    )


    upper_bound_fe = (
        transformed_q3
        + 1.5 * iqr_fe
    )


    # ========================================================
    # 19. IDENTIFY CATEGORIES THAT WOULD BECOME OUTLIERS
    # ========================================================

    outlier_mask = (
        (
            category_fe
            < lower_bound_fe
        )
        |
        (
            category_fe
            > upper_bound_fe
        )
    )


    outlier_categories = (
        category_fe[
            outlier_mask
        ]
    )


    outlier_table = pd.DataFrame({

        "CARD_NUMBER":
            outlier_categories
            .index
            .astype(str),

        "COUNT":
            category_counts.loc[
                outlier_categories.index
            ]
            .values,

        "FE_VALUE":
            outlier_categories.values
    })


    outlier_table[
        "OUTLIER_TYPE"
    ] = np.where(
        outlier_table[
            "FE_VALUE"
        ]
        < lower_bound_fe,

        "Lower outlier",

        "Upper outlier"
    )


    outlier_table = (
        outlier_table
        .sort_values(
            "FE_VALUE",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    number_outlier_categories = int(
        len(
            outlier_table
        )
    )


    outlier_category_percentage = (
        number_outlier_categories
        / unique_categories
        * 100
    )


    if number_outlier_categories > 0:

        outlier_category_values = set(
            outlier_categories.index
        )


        observations_in_outlier_categories = int(
            feature
            .isin(
                outlier_category_values
            )
            .sum()
        )


    else:

        observations_in_outlier_categories = 0


    outlier_observation_percentage = (
        observations_in_outlier_categories
        / total
        * 100
    )


    # ========================================================
    # 20. FREQUENCY COLLISION ANALYSIS
    #
    # Different card categories with the same number of
    # observations receive the same Frequency Encoding value.
    # ========================================================

    unique_fe_values = int(
        category_fe.nunique()
    )


    frequency_groups = (
        category_counts
        .value_counts()
    )


    collision_groups = int(
        (
            frequency_groups
            > 1
        )
        .sum()
    )


    categories_in_collisions = int(
        frequency_groups[
            frequency_groups
            > 1
        ]
        .sum()
    )


    frequency_compression_ratio = (
        unique_fe_values
        / unique_categories
    )


    frequency_reduction_percentage = (
        1
        - frequency_compression_ratio
    ) * 100


    # ========================================================
    # 21. SHAPIRO-WILK NORMALITY TEST
    #
    # H0:
    # The Frequency Encoded feature follows
    # a normal distribution.
    #
    # H1:
    # The Frequency Encoded feature does not follow
    # a normal distribution.
    #
    # Significance level:
    # alpha = 0.05
    #
    # The complete transformed feature is used.
    # ========================================================

    (
        shapiro_statistic,
        shapiro_p_value
    ) = stats.shapiro(
        encoded_preview
    )


    shapiro_statistic = float(
        shapiro_statistic
    )


    shapiro_p_value = float(
        shapiro_p_value
    )


    if shapiro_p_value < ALPHA:

        shapiro_decision = (
            "Reject H0"
        )


        shapiro_interpretation = (
            "There is statistical evidence that "
            "the Frequency Encoded feature does not "
            "follow a normal distribution."
        )


    else:

        shapiro_decision = (
            "Fail to reject H0"
        )


        shapiro_interpretation = (
            "There is insufficient statistical evidence "
            "to conclude that the Frequency Encoded feature "
            "deviates from a normal distribution."
        )


    # ========================================================
    # 22. JARQUE-BERA NORMALITY TEST
    #
    # H0:
    # The Frequency Encoded feature follows
    # a normal distribution.
    #
    # H1:
    # The Frequency Encoded feature does not follow
    # a normal distribution.
    #
    # Significance level:
    # alpha = 0.05
    #
    # The complete transformed feature is used.
    # ========================================================

    jarque_bera_result = (
        stats.jarque_bera(
            encoded_preview
        )
    )


    jarque_bera_statistic = float(
        jarque_bera_result.statistic
    )


    jarque_bera_p_value = float(
        jarque_bera_result.pvalue
    )


    if jarque_bera_p_value < ALPHA:

        jarque_bera_decision = (
            "Reject H0"
        )


        jarque_bera_interpretation = (
            "There is statistical evidence that "
            "the Frequency Encoded feature does not "
            "follow a normal distribution."
        )


    else:

        jarque_bera_decision = (
            "Fail to reject H0"
        )


        jarque_bera_interpretation = (
            "There is insufficient statistical evidence "
            "to conclude that the Frequency Encoded feature "
            "deviates from a normal distribution."
        )


    # ========================================================
    # 23. CREATE THE RANK-FREQUENCY CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not rank_frequency_chart_exists:

        ranked_fe = (
            category_fe
            .sort_values(
                ascending=False
            )
            .reset_index(
                drop=True
            )
        )


        category_rank = np.arange(
            1,
            len(ranked_fe) + 1
        )


        fig, ax = plt.subplots(
            figsize=(
                11,
                6
            )
        )


        ax.plot(
            category_rank,
            ranked_fe.values
        )


        ax.set_title(
            "Rank-frequency curve of card categories"
        )


        ax.set_xlabel(
            "Category rank"
        )


        ax.set_ylabel(
            "Frequency Encoding value"
        )


        ax.xaxis.set_major_formatter(
            FuncFormatter(
                lambda x, pos:
                str(
                    int(x)
                )
            )
        )


        ax.grid(
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            RANK_FREQUENCY_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nRank-frequency chart created:"
        )


        print(
            RANK_FREQUENCY_CHART_PATH
        )


    else:

        print(
            "\nRank-frequency chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 24. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 25. PREPARE OUTLIER TABLE FOR HTML
    # ========================================================

    if number_outlier_categories > 0:

        outlier_html = (
            outlier_table
            .to_html(
                index=False,
                border=0,
                float_format=lambda x:
                f"{x:.12f}"
            )
        )


    else:

        outlier_html = (
            "<p>"
            "No card categories were classified "
            "as outliers by the IQR rule."
            "</p>"
        )


    # ========================================================
    # 26. CREATE THE HTML REPORT
    #
    # Only if the HTML does not already exist.
    # ========================================================

    if not html_exists:

        rank_frequency_chart_base64 = (
            image_to_base64(
                RANK_FREQUENCY_CHART_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Individual Exploratory Analysis — {FEATURE_COLUMN}
</h1>


<p>

The variable <strong>{FEATURE_COLUMN}</strong>
represents the card identifier associated
with each transaction.

Although the original values are numerical,
the card number is treated as a
<strong>categorical identifier</strong>,
not as a continuous numerical variable.

</p>


<!-- ========================================================
     1. FEATURE OVERVIEW
========================================================= -->


<h2>
1. Feature overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total observations</td>
<td>{total}</td>
</tr>

<tr>
<td>Missing values</td>
<td>{missing_values}</td>
</tr>

<tr>
<td>Unique card categories</td>
<td>{unique_categories}</td>
</tr>

<tr>
<td>Cardinality percentage</td>
<td>{cardinality_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     2. FREQUENCY STATISTICS
========================================================= -->


<h2>
2. Frequency statistics
</h2>


<table>

<tr>
<th>Statistic</th>
<th>Transactions per card</th>
</tr>

<tr>
<td>Minimum</td>
<td>{minimum_count}</td>
</tr>

<tr>
<td>Q1</td>
<td>{q1_count:.2f}</td>
</tr>

<tr>
<td>Median</td>
<td>{median_count:.2f}</td>
</tr>

<tr>
<td>Mean</td>
<td>{mean_count:.2f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{q3_count:.2f}</td>
</tr>

<tr>
<td>Maximum</td>
<td>{maximum_count}</td>
</tr>

</table>


<!-- ========================================================
     3. FREQUENCY ENCODING VALUES
========================================================= -->


<h2>
3. Expected Frequency Encoding values
</h2>


<p>

For each card category, the Frequency Encoding
value is calculated as:

</p>


<p class="result">

FE(category) = count(category) / N

</p>


<table>

<tr>
<th>Statistic</th>
<th>FE value across categories</th>
</tr>

<tr>
<td>Minimum</td>
<td>{minimum_fe:.12f}</td>
</tr>

<tr>
<td>Median</td>
<td>{median_fe:.12f}</td>
</tr>

<tr>
<td>Mean</td>
<td>{mean_fe:.12f}</td>
</tr>

<tr>
<td>Maximum</td>
<td>{maximum_fe:.12f}</td>
</tr>

</table>


<h3>
Descriptive statistics after transformation
</h3>


<p>

The following statistics describe the complete
transformed feature across all
<strong>{total}</strong>
observations.

</p>


<table>

<tr>
<th>Statistic</th>
<th>Transformed value</th>
</tr>

<tr>
<td>Minimum</td>
<td>{transformed_minimum:.12f}</td>
</tr>

<tr>
<td>Q1</td>
<td>{transformed_q1:.12f}</td>
</tr>

<tr>
<td>Median</td>
<td>{transformed_median:.12f}</td>
</tr>

<tr>
<td>Mean</td>
<td>{transformed_mean:.12f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{transformed_q3:.12f}</td>
</tr>

<tr>
<td>Maximum</td>
<td>{transformed_maximum:.12f}</td>
</tr>

<tr>
<td>Standard deviation</td>
<td>{transformed_standard_deviation:.12f}</td>
</tr>

</table>


<!-- ========================================================
     4. OUTLIER ANALYSIS
========================================================= -->


<h2>
4. Frequency Encoding outlier analysis
</h2>


<p>

The IQR method was applied to the complete
simulated Frequency Encoded feature.

Categories with encoded values below
the lower bound or above the upper bound
are classified as potential outliers.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Q1</td>
<td>{transformed_q1:.12f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{transformed_q3:.12f}</td>
</tr>

<tr>
<td>IQR</td>
<td>{iqr_fe:.12f}</td>
</tr>

<tr>
<td>Lower bound</td>
<td>{lower_bound_fe:.12f}</td>
</tr>

<tr>
<td>Upper bound</td>
<td>{upper_bound_fe:.12f}</td>
</tr>

<tr>
<td>Outlier card categories</td>
<td>{number_outlier_categories}</td>
</tr>

<tr>
<td>Percentage of card categories</td>
<td>{outlier_category_percentage:.6f}%</td>
</tr>

<tr>
<td>Observations belonging to outlier categories</td>
<td>{observations_in_outlier_categories}</td>
</tr>

<tr>
<td>Percentage of dataset observations</td>
<td>{outlier_observation_percentage:.6f}%</td>
</tr>

</table>


<h3>
Card categories classified as outliers
</h3>


{outlier_html}


<!-- ========================================================
     5. FREQUENCY COLLISIONS
========================================================= -->


<h2>
5. Frequency collisions
</h2>


<p>

Two or more different card identifiers may have
the same number of transactions.

When this occurs, they receive exactly the same
Frequency Encoding value.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Original unique categories</td>
<td>{unique_categories}</td>
</tr>

<tr>
<td>Unique Frequency Encoding values</td>
<td>{unique_fe_values}</td>
</tr>

<tr>
<td>Frequency values shared by multiple cards</td>
<td>{collision_groups}</td>
</tr>

<tr>
<td>Cards involved in collisions</td>
<td>{categories_in_collisions}</td>
</tr>

<tr>
<td>Frequency compression ratio</td>
<td>{frequency_compression_ratio:.6f}</td>
</tr>

<tr>
<td>Reduction in distinct representation</td>
<td>{frequency_reduction_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     6. RANK-FREQUENCY CURVE
========================================================= -->


<h2>
6. Rank-frequency curve
</h2>


<p>

Card categories are ordered from the most frequent
to the least frequent.

The X axis represents the category rank,
while the Y axis represents the expected
Frequency Encoding value.

Horizontal regions indicate card categories
that share identical Frequency Encoding values.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{rank_frequency_chart_base64}"
    alt="Rank-frequency curve of card categories"
>

</div>


<!-- ========================================================
     7. NORMALITY TESTS
========================================================= -->


<h2>
7. Normality tests after Frequency Encoding
</h2>


<p>

The Shapiro-Wilk and Jarque-Bera tests
were applied to the complete transformed feature.

The significance level used for both tests was:

</p>


<p class="result">

α = {ALPHA}

</p>


<ul>

<li>

<strong>H0:</strong>
the Frequency Encoded feature follows
a normal distribution.

</li>

<li>

<strong>H1:</strong>
the Frequency Encoded feature does not
follow a normal distribution.

</li>

</ul>


<!-- ========================================================
     7.1 SHAPIRO-WILK
========================================================= -->


<h3>
7.1 Shapiro-Wilk test
</h3>


<p>

The Shapiro-Wilk test was applied to all
<strong>{total}</strong>
Frequency Encoded observations.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Number of observations</td>
<td>{total}</td>
</tr>

<tr>
<td>Test statistic (W)</td>
<td>{shapiro_statistic:.6f}</td>
</tr>

<tr>
<td>p-value</td>
<td>{shapiro_p_value:.12g}</td>
</tr>

<tr>
<td>Significance level</td>
<td>{ALPHA}</td>
</tr>

<tr>
<td>Decision</td>
<td><strong>{shapiro_decision}</strong></td>
</tr>

</table>


<p>

{shapiro_interpretation}

</p>


<!-- ========================================================
     7.2 JARQUE-BERA
========================================================= -->


<h3>
7.2 Jarque-Bera test
</h3>


<p>

The Jarque-Bera test was applied to all
<strong>{total}</strong>
Frequency Encoded observations.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Number of observations</td>
<td>{total}</td>
</tr>

<tr>
<td>Test statistic</td>
<td>{jarque_bera_statistic:.6f}</td>
</tr>

<tr>
<td>p-value</td>
<td>{jarque_bera_p_value:.12g}</td>
</tr>

<tr>
<td>Significance level</td>
<td>{ALPHA}</td>
</tr>

<tr>
<td>Decision</td>
<td><strong>{jarque_bera_decision}</strong></td>
</tr>

</table>


<p>

{jarque_bera_interpretation}

</p>


<div class="note">

<strong>Decision rule:</strong>

<br><br>

If p-value &lt; 0.05,
H0 is rejected.

<br><br>

If p-value ≥ 0.05,
H0 is not rejected.

<br><br>

<strong>Shapiro-Wilk note:</strong>

For very large sample sizes, the test statistic
can still be calculated using the complete dataset,
but the numerical accuracy of the p-value should
be interpreted with caution.

</div>


<!-- ========================================================
     8. SUMMARY
========================================================= -->


<h2>
8. Summary of results
</h2>


<ul>

<li>
<strong>Total observations:</strong>
{total}
</li>

<li>
<strong>Unique card categories:</strong>
{unique_categories}
</li>

<li>
<strong>Cardinality percentage:</strong>
{cardinality_percentage:.6f}%
</li>

<li>
<strong>Minimum transactions per card:</strong>
{minimum_count}
</li>

<li>
<strong>Median transactions per card:</strong>
{median_count:.2f}
</li>

<li>
<strong>Maximum transactions per card:</strong>
{maximum_count}
</li>

<li>
<strong>Minimum Frequency Encoding value:</strong>
{minimum_fe:.12f}
</li>

<li>
<strong>Maximum Frequency Encoding value:</strong>
{maximum_fe:.12f}
</li>

<li>
<strong>Frequency Encoding outlier categories:</strong>
{number_outlier_categories}
</li>

<li>
<strong>Observations represented by outlier categories:</strong>
{outlier_observation_percentage:.6f}%
</li>

<li>
<strong>Unique Frequency Encoding values:</strong>
{unique_fe_values}
</li>

<li>
<strong>Cards involved in frequency collisions:</strong>
{categories_in_collisions}
</li>

<li>
<strong>Frequency compression ratio:</strong>
{frequency_compression_ratio:.6f}
</li>

<li>
<strong>Shapiro-Wilk observations:</strong>
{total}
</li>

<li>
<strong>Shapiro-Wilk:</strong>
{shapiro_decision}
</li>

<li>
<strong>Jarque-Bera observations:</strong>
{total}
</li>

<li>
<strong>Jarque-Bera:</strong>
{jarque_bera_decision}
</li>

</ul>


</body>

</html>
"""


        # ====================================================
        # 27. SAVE THE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )


        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 28. DISPLAY MAIN RESULTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "TRANS_NUM_CARD_FEWF SUMMARY"
    )


    print(
        "=" * 100
    )


    print(
        "Total observations:",
        total
    )


    print(
        "Unique card categories:",
        unique_categories
    )


    print(
        "Cardinality percentage:",
        f"{cardinality_percentage:.6f}%"
    )


    print(
        "Unique Frequency Encoding values:",
        unique_fe_values
    )


    print(
        "Cards involved in collisions:",
        categories_in_collisions
    )


    print(
        "Frequency Encoding outlier categories:",
        number_outlier_categories
    )


    # ========================================================
    # 29. DISPLAY NORMALITY TESTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "NORMALITY TESTS AFTER FREQUENCY ENCODING"
    )


    print(
        "=" * 100
    )


    print(
        f"\nSignificance level: {ALPHA}"
    )


    print(
        "\nShapiro-Wilk test"
    )


    print(
        "Number of observations:",
        total
    )


    print(
        "Statistic:",
        f"{shapiro_statistic:.6f}"
    )


    print(
        "p-value:",
        f"{shapiro_p_value:.12g}"
    )


    print(
        "Decision:",
        shapiro_decision
    )


    print(
        "\nJarque-Bera test"
    )


    print(
        "Number of observations:",
        total
    )


    print(
        "Statistic:",
        f"{jarque_bera_statistic:.6f}"
    )


    print(
        "p-value:",
        f"{jarque_bera_p_value:.12g}"
    )


    print(
        "Decision:",
        jarque_bera_decision
    )


    # ========================================================
    # 30. DISPLAY OUTLIER CATEGORIES
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "CARD CATEGORIES CLASSIFIED AS OUTLIERS"
    )


    print(
        "=" * 100
    )


    if number_outlier_categories > 0:

        display(
            outlier_table
        )


    else:

        print(
            "No card categories were classified "
            "as outliers by the IQR rule."
        )


    # ========================================================
    # 31. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature
    del encoded_preview

    gc.collect()


    # ========================================================
    # 32. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "ANALYSIS COMPLETED"
    )


    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )


    print(
        HTML_PATH
    )


    print(
        "\nPNG:"
    )


    print(
        RANK_FREQUENCY_CHART_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Rank-frequency chart: Will be created


/usr/local/lib/python3.14/site-packages/scipy/stats/_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 1852394.
  res = hypotest_fun_out(*samples, **kwds)



Rank-frequency chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/frequency_encoding_with_fallback/trans_num_card_fewf/trans_num_card_fewf_rank_frequency.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/frequency_encoding_with_fallback/trans_num_card_fewf/analysis_trans_num_card_fewf.html

TRANS_NUM_CARD_FEWF SUMMARY
Total observations: 1852394
Unique card categories: 999
Cardinality percentage: 0.053930%
Unique Frequency Encoding values: 140
Cards involved in collisions: 970
Frequency Encoding outlier categories: 0

NORMALITY TESTS AFTER FREQUENCY ENCODING

Significance level: 0.05

Shapiro-Wilk test
Number of observations: 1852394
Statistic: 0.936545
p-value: 6.67817223752e-146
Decision: Reject H0

Jarque-Bera test
Number of observations: 1852394
Statistic: 53676.186661
p-value: 0
Decision: Reject H0

CARD CATEGORIES CLASSIFIED AS OUTLIERS
No card categories were classified as outliers by the IQR rule

### <span style="color:dodgerblue"> RECIVE_LOC_FEWF </span> ###

In [25]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ENCODING_TYPE = "onehot_encoding_with_ignore"

FEATURE_NAME = "recive_category_ohewi"

FEATURE_COLUMN = "RECEIVE_CATEGORY_OHEWI"

ALPHA = 0.05


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / ENCODING_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


RANK_FREQUENCY_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_rank_frequency.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


rank_frequency_chart_exists = (
    RANK_FREQUENCY_CHART_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and rank_frequency_chart_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )

    print(
        "No analysis or file creation is required."
    )

    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )

    print(
        "\nExisting files:"
    )

    print(
        HTML_PATH
    )

    print(
        RANK_FREQUENCY_CHART_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )

    print(
        "=" * 100
    )

    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )

    print(
        "Rank-frequency chart:",
        "Already exists"
        if rank_frequency_chart_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. BASIC VALIDATION
    # ========================================================

    total = int(
        len(feature)
    )


    if total == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    missing_values = int(
        feature
        .isna()
        .sum()
    )


    if missing_values > 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains "
            f"{missing_values} missing values."
        )


    # ========================================================
    # 13. CATEGORY COUNTS
    # ========================================================

    category_counts = (
        feature
        .value_counts()
        .sort_values(
            ascending=False
        )
    )


    unique_categories = int(
        category_counts.shape[0]
    )


    cardinality_percentage = (
        unique_categories
        / total
        * 100
    )


    # ========================================================
    # 14. FREQUENCY STATISTICS
    # ========================================================

    minimum_count = int(
        category_counts.min()
    )


    maximum_count = int(
        category_counts.max()
    )


    mean_count = float(
        category_counts.mean()
    )


    median_count = float(
        category_counts.median()
    )


    q1_count = float(
        category_counts.quantile(
            0.25
        )
    )


    q3_count = float(
        category_counts.quantile(
            0.75
        )
    )


    # ========================================================
    # 15. FREQUENCY ENCODING VALUES
    #
    # FE(category) = count(category) / N_total
    # ========================================================

    category_fe = (
        category_counts
        / total
    )


    minimum_fe = float(
        category_fe.min()
    )


    maximum_fe = float(
        category_fe.max()
    )


    mean_fe = float(
        category_fe.mean()
    )


    median_fe = float(
        category_fe.median()
    )


    # ========================================================
    # 16. SIMULATE THE TRANSFORMED FEATURE
    #
    # This does not modify the original dataset.
    #
    # Each receiver location is temporarily replaced
    # by its corresponding Frequency Encoding value.
    # ========================================================

    encoded_preview = (
        feature
        .map(
            category_fe
        )
        .astype(
            "float64"
        )
    )


    # ========================================================
    # 17. DESCRIPTIVE STATISTICS AFTER FREQUENCY ENCODING
    # ========================================================

    transformed_minimum = float(
        encoded_preview.min()
    )


    transformed_q1 = float(
        encoded_preview.quantile(
            0.25
        )
    )


    transformed_median = float(
        encoded_preview.median()
    )


    transformed_mean = float(
        encoded_preview.mean()
    )


    transformed_q3 = float(
        encoded_preview.quantile(
            0.75
        )
    )


    transformed_maximum = float(
        encoded_preview.max()
    )


    transformed_standard_deviation = float(
        encoded_preview.std()
    )


    # ========================================================
    # 18. IQR OUTLIER LIMITS AFTER FREQUENCY ENCODING
    # ========================================================

    iqr_fe = (
        transformed_q3
        - transformed_q1
    )


    lower_bound_fe = (
        transformed_q1
        - 1.5 * iqr_fe
    )


    upper_bound_fe = (
        transformed_q3
        + 1.5 * iqr_fe
    )


    # ========================================================
    # 19. IDENTIFY CATEGORIES THAT WOULD BECOME OUTLIERS
    # ========================================================

    outlier_mask = (
        (
            category_fe
            < lower_bound_fe
        )
        |
        (
            category_fe
            > upper_bound_fe
        )
    )


    outlier_categories = (
        category_fe[
            outlier_mask
        ]
    )


    outlier_table = pd.DataFrame({

        "RECEIVER_LOCATION":
            outlier_categories
            .index
            .astype(str),

        "COUNT":
            category_counts.loc[
                outlier_categories.index
            ]
            .values,

        "FE_VALUE":
            outlier_categories.values
    })


    outlier_table[
        "OUTLIER_TYPE"
    ] = np.where(
        outlier_table[
            "FE_VALUE"
        ]
        < lower_bound_fe,

        "Lower outlier",

        "Upper outlier"
    )


    outlier_table = (
        outlier_table
        .sort_values(
            "FE_VALUE",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    number_outlier_categories = int(
        len(
            outlier_table
        )
    )


    outlier_category_percentage = (
        number_outlier_categories
        / unique_categories
        * 100
    )


    if number_outlier_categories > 0:

        outlier_category_values = set(
            outlier_categories.index
        )


        observations_in_outlier_categories = int(
            feature
            .isin(
                outlier_category_values
            )
            .sum()
        )


    else:

        observations_in_outlier_categories = 0


    outlier_observation_percentage = (
        observations_in_outlier_categories
        / total
        * 100
    )


    # ========================================================
    # 20. FREQUENCY COLLISION ANALYSIS
    #
    # Different receiver locations with the same number
    # of observations receive exactly the same
    # Frequency Encoding value.
    # ========================================================

    unique_fe_values = int(
        category_fe.nunique()
    )


    frequency_groups = (
        category_counts
        .value_counts()
    )


    collision_groups = int(
        (
            frequency_groups
            > 1
        )
        .sum()
    )


    categories_in_collisions = int(
        frequency_groups[
            frequency_groups
            > 1
        ]
        .sum()
    )


    frequency_compression_ratio = (
        unique_fe_values
        / unique_categories
    )


    frequency_reduction_percentage = (
        1
        - frequency_compression_ratio
    ) * 100


    # ========================================================
    # 21. SHAPIRO-WILK NORMALITY TEST
    #
    # H0:
    # The Frequency Encoded feature follows
    # a normal distribution.
    #
    # H1:
    # The Frequency Encoded feature does not follow
    # a normal distribution.
    #
    # Significance level:
    # alpha = 0.05
    #
    # The complete transformed feature is used.
    # ========================================================

    (
        shapiro_statistic,
        shapiro_p_value
    ) = stats.shapiro(
        encoded_preview
    )


    shapiro_statistic = float(
        shapiro_statistic
    )


    shapiro_p_value = float(
        shapiro_p_value
    )


    if shapiro_p_value < ALPHA:

        shapiro_decision = (
            "Reject H0"
        )


        shapiro_interpretation = (
            "There is statistical evidence that "
            "the Frequency Encoded feature does not "
            "follow a normal distribution."
        )


    else:

        shapiro_decision = (
            "Fail to reject H0"
        )


        shapiro_interpretation = (
            "There is insufficient statistical evidence "
            "to conclude that the Frequency Encoded feature "
            "deviates from a normal distribution."
        )


    # ========================================================
    # 22. JARQUE-BERA NORMALITY TEST
    #
    # H0:
    # The Frequency Encoded feature follows
    # a normal distribution.
    #
    # H1:
    # The Frequency Encoded feature does not follow
    # a normal distribution.
    #
    # Significance level:
    # alpha = 0.05
    #
    # The complete transformed feature is used.
    # ========================================================

    jarque_bera_result = (
        stats.jarque_bera(
            encoded_preview
        )
    )


    jarque_bera_statistic = float(
        jarque_bera_result.statistic
    )


    jarque_bera_p_value = float(
        jarque_bera_result.pvalue
    )


    if jarque_bera_p_value < ALPHA:

        jarque_bera_decision = (
            "Reject H0"
        )


        jarque_bera_interpretation = (
            "There is statistical evidence that "
            "the Frequency Encoded feature does not "
            "follow a normal distribution."
        )


    else:

        jarque_bera_decision = (
            "Fail to reject H0"
        )


        jarque_bera_interpretation = (
            "There is insufficient statistical evidence "
            "to conclude that the Frequency Encoded feature "
            "deviates from a normal distribution."
        )


    # ========================================================
    # 23. CREATE THE RANK-FREQUENCY CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not rank_frequency_chart_exists:

        ranked_fe = (
            category_fe
            .sort_values(
                ascending=False
            )
            .reset_index(
                drop=True
            )
        )


        category_rank = np.arange(
            1,
            len(ranked_fe) + 1
        )


        fig, ax = plt.subplots(
            figsize=(
                11,
                6
            )
        )


        ax.plot(
            category_rank,
            ranked_fe.values
        )


        ax.set_title(
            "Rank-frequency curve of receiver locations"
        )


        ax.set_xlabel(
            "Category rank"
        )


        ax.set_ylabel(
            "Frequency Encoding value"
        )


        ax.xaxis.set_major_formatter(
            FuncFormatter(
                lambda x, pos:
                str(
                    int(x)
                )
            )
        )


        ax.grid(
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            RANK_FREQUENCY_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nRank-frequency chart created:"
        )


        print(
            RANK_FREQUENCY_CHART_PATH
        )


    else:

        print(
            "\nRank-frequency chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 24. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 25. PREPARE OUTLIER TABLE FOR HTML
    # ========================================================

    if number_outlier_categories > 0:

        outlier_html = (
            outlier_table
            .to_html(
                index=False,
                border=0,
                float_format=lambda x:
                f"{x:.12f}"
            )
        )


    else:

        outlier_html = (
            "<p>"
            "No receiver location categories were classified "
            "as outliers by the IQR rule."
            "</p>"
        )


    # ========================================================
    # 26. CREATE THE HTML REPORT
    #
    # Only if the HTML does not already exist.
    # ========================================================

    if not html_exists:

        rank_frequency_chart_base64 = (
            image_to_base64(
                RANK_FREQUENCY_CHART_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Individual Exploratory Analysis — {FEATURE_COLUMN}
</h1>


<p>

The variable <strong>{FEATURE_COLUMN}</strong>
represents the merchant or receiver location
associated with each transaction.

The feature is treated as a
<strong>categorical variable</strong>
and is intended for Frequency Encoding
With Fallback.

</p>


<!-- ========================================================
     1. FEATURE OVERVIEW
========================================================= -->


<h2>
1. Feature overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total observations</td>
<td>{total}</td>
</tr>

<tr>
<td>Missing values</td>
<td>{missing_values}</td>
</tr>

<tr>
<td>Unique receiver locations</td>
<td>{unique_categories}</td>
</tr>

<tr>
<td>Cardinality percentage</td>
<td>{cardinality_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     2. FREQUENCY STATISTICS
========================================================= -->


<h2>
2. Frequency statistics
</h2>


<table>

<tr>
<th>Statistic</th>
<th>Transactions per receiver location</th>
</tr>

<tr>
<td>Minimum</td>
<td>{minimum_count}</td>
</tr>

<tr>
<td>Q1</td>
<td>{q1_count:.2f}</td>
</tr>

<tr>
<td>Median</td>
<td>{median_count:.2f}</td>
</tr>

<tr>
<td>Mean</td>
<td>{mean_count:.2f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{q3_count:.2f}</td>
</tr>

<tr>
<td>Maximum</td>
<td>{maximum_count}</td>
</tr>

</table>


<!-- ========================================================
     3. EXPECTED FREQUENCY ENCODING VALUES
========================================================= -->


<h2>
3. Expected Frequency Encoding values
</h2>


<p>

For each receiver location category,
the Frequency Encoding value is calculated as:

</p>


<p class="result">

FE(category) = count(category) / N

</p>


<table>

<tr>
<th>Statistic</th>
<th>FE value across categories</th>
</tr>

<tr>
<td>Minimum</td>
<td>{minimum_fe:.12f}</td>
</tr>

<tr>
<td>Median</td>
<td>{median_fe:.12f}</td>
</tr>

<tr>
<td>Mean</td>
<td>{mean_fe:.12f}</td>
</tr>

<tr>
<td>Maximum</td>
<td>{maximum_fe:.12f}</td>
</tr>

</table>


<h3>
Descriptive statistics after transformation
</h3>


<p>

The following statistics describe the complete
transformed feature across all
<strong>{total}</strong>
observations.

</p>


<table>

<tr>
<th>Statistic</th>
<th>Transformed value</th>
</tr>

<tr>
<td>Minimum</td>
<td>{transformed_minimum:.12f}</td>
</tr>

<tr>
<td>Q1</td>
<td>{transformed_q1:.12f}</td>
</tr>

<tr>
<td>Median</td>
<td>{transformed_median:.12f}</td>
</tr>

<tr>
<td>Mean</td>
<td>{transformed_mean:.12f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{transformed_q3:.12f}</td>
</tr>

<tr>
<td>Maximum</td>
<td>{transformed_maximum:.12f}</td>
</tr>

<tr>
<td>Standard deviation</td>
<td>{transformed_standard_deviation:.12f}</td>
</tr>

</table>


<!-- ========================================================
     4. OUTLIER ANALYSIS
========================================================= -->


<h2>
4. Frequency Encoding outlier analysis
</h2>


<p>

The IQR method was applied to the complete
simulated Frequency Encoded feature.

The detected outliers represent receiver
location categories whose encoded values fall
outside the IQR limits after transformation.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Q1</td>
<td>{transformed_q1:.12f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{transformed_q3:.12f}</td>
</tr>

<tr>
<td>IQR</td>
<td>{iqr_fe:.12f}</td>
</tr>

<tr>
<td>Lower bound</td>
<td>{lower_bound_fe:.12f}</td>
</tr>

<tr>
<td>Upper bound</td>
<td>{upper_bound_fe:.12f}</td>
</tr>

<tr>
<td>Outlier receiver location categories</td>
<td>{number_outlier_categories}</td>
</tr>

<tr>
<td>Percentage of receiver location categories</td>
<td>{outlier_category_percentage:.6f}%</td>
</tr>

<tr>
<td>Observations belonging to outlier categories</td>
<td>{observations_in_outlier_categories}</td>
</tr>

<tr>
<td>Percentage of dataset observations</td>
<td>{outlier_observation_percentage:.6f}%</td>
</tr>

</table>


<h3>
Receiver location categories classified as outliers
</h3>


{outlier_html}


<!-- ========================================================
     5. FREQUENCY COLLISIONS
========================================================= -->


<h2>
5. Frequency collisions
</h2>


<p>

Two or more different receiver locations
may have the same number of observations.

When this occurs, they receive exactly
the same Frequency Encoding value.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Original unique categories</td>
<td>{unique_categories}</td>
</tr>

<tr>
<td>Unique Frequency Encoding values</td>
<td>{unique_fe_values}</td>
</tr>

<tr>
<td>Frequency values shared by multiple receiver locations</td>
<td>{collision_groups}</td>
</tr>

<tr>
<td>Receiver locations involved in collisions</td>
<td>{categories_in_collisions}</td>
</tr>

<tr>
<td>Frequency compression ratio</td>
<td>{frequency_compression_ratio:.6f}</td>
</tr>

<tr>
<td>Reduction in distinct representation</td>
<td>{frequency_reduction_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     6. RANK-FREQUENCY CURVE
========================================================= -->


<h2>
6. Rank-frequency curve
</h2>


<p>

Receiver location categories are ordered
from the most frequent to the least frequent.

The X axis represents the category rank,
while the Y axis represents the expected
Frequency Encoding value.

Horizontal regions indicate categories
that share identical Frequency Encoding values.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{rank_frequency_chart_base64}"
    alt="Rank-frequency curve of receiver locations"
>

</div>


<!-- ========================================================
     7. NORMALITY TESTS
========================================================= -->


<h2>
7. Normality tests after Frequency Encoding
</h2>


<p>

The Shapiro-Wilk and Jarque-Bera tests
were applied to the complete transformed feature.

The significance level used for both tests was:

</p>


<p class="result">

α = {ALPHA}

</p>


<ul>

<li>

<strong>H0:</strong>
the Frequency Encoded feature follows
a normal distribution.

</li>

<li>

<strong>H1:</strong>
the Frequency Encoded feature does not
follow a normal distribution.

</li>

</ul>


<!-- ========================================================
     7.1 SHAPIRO-WILK
========================================================= -->


<h3>
7.1 Shapiro-Wilk test
</h3>


<p>

The Shapiro-Wilk test was applied to all
<strong>{total}</strong>
Frequency Encoded observations.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Number of observations</td>
<td>{total}</td>
</tr>

<tr>
<td>Test statistic (W)</td>
<td>{shapiro_statistic:.6f}</td>
</tr>

<tr>
<td>p-value</td>
<td>{shapiro_p_value:.12g}</td>
</tr>

<tr>
<td>Significance level</td>
<td>{ALPHA}</td>
</tr>

<tr>
<td>Decision</td>
<td><strong>{shapiro_decision}</strong></td>
</tr>

</table>


<p>

{shapiro_interpretation}

</p>


<!-- ========================================================
     7.2 JARQUE-BERA
========================================================= -->


<h3>
7.2 Jarque-Bera test
</h3>


<p>

The Jarque-Bera test was applied to all
<strong>{total}</strong>
Frequency Encoded observations.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Number of observations</td>
<td>{total}</td>
</tr>

<tr>
<td>Test statistic</td>
<td>{jarque_bera_statistic:.6f}</td>
</tr>

<tr>
<td>p-value</td>
<td>{jarque_bera_p_value:.12g}</td>
</tr>

<tr>
<td>Significance level</td>
<td>{ALPHA}</td>
</tr>

<tr>
<td>Decision</td>
<td><strong>{jarque_bera_decision}</strong></td>
</tr>

</table>


<p>

{jarque_bera_interpretation}

</p>


<div class="note">

<strong>Decision rule:</strong>

<br><br>

If p-value &lt; 0.05,
H0 is rejected.

<br><br>

If p-value ≥ 0.05,
H0 is not rejected.

<br><br>

For very large samples, the Shapiro-Wilk
p-value may have limited numerical accuracy
and should be interpreted with caution.

</div>


<!-- ========================================================
     8. SUMMARY
========================================================= -->


<h2>
8. Summary of results
</h2>


<ul>

<li>
<strong>Total observations:</strong>
{total}
</li>

<li>
<strong>Unique receiver locations:</strong>
{unique_categories}
</li>

<li>
<strong>Cardinality percentage:</strong>
{cardinality_percentage:.6f}%
</li>

<li>
<strong>Minimum transactions per receiver location:</strong>
{minimum_count}
</li>

<li>
<strong>Median transactions per receiver location:</strong>
{median_count:.2f}
</li>

<li>
<strong>Maximum transactions per receiver location:</strong>
{maximum_count}
</li>

<li>
<strong>Minimum Frequency Encoding value:</strong>
{minimum_fe:.12f}
</li>

<li>
<strong>Maximum Frequency Encoding value:</strong>
{maximum_fe:.12f}
</li>

<li>
<strong>Frequency Encoding outlier categories:</strong>
{number_outlier_categories}
</li>

<li>
<strong>Observations represented by outlier categories:</strong>
{outlier_observation_percentage:.6f}%
</li>

<li>
<strong>Unique Frequency Encoding values:</strong>
{unique_fe_values}
</li>

<li>
<strong>Receiver locations involved in frequency collisions:</strong>
{categories_in_collisions}
</li>

<li>
<strong>Frequency compression ratio:</strong>
{frequency_compression_ratio:.6f}
</li>

<li>
<strong>Shapiro-Wilk:</strong>
{shapiro_decision}
</li>

<li>
<strong>Jarque-Bera:</strong>
{jarque_bera_decision}
</li>

</ul>


</body>

</html>
"""


        # ====================================================
        # 27. SAVE THE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )


        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 28. DISPLAY MAIN RESULTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "RECEIVE_LOC_FEWF SUMMARY"
    )


    print(
        "=" * 100
    )


    print(
        "Total observations:",
        total
    )


    print(
        "Unique receiver locations:",
        unique_categories
    )


    print(
        "Cardinality percentage:",
        f"{cardinality_percentage:.6f}%"
    )


    print(
        "Unique Frequency Encoding values:",
        unique_fe_values
    )


    print(
        "Receiver locations involved in collisions:",
        categories_in_collisions
    )


    print(
        "Frequency Encoding outlier categories:",
        number_outlier_categories
    )


    # ========================================================
    # 29. DISPLAY NORMALITY TESTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "NORMALITY TESTS AFTER FREQUENCY ENCODING"
    )


    print(
        "=" * 100
    )


    print(
        f"\nSignificance level: {ALPHA}"
    )


    print(
        "\nShapiro-Wilk test"
    )


    print(
        "Number of observations:",
        total
    )


    print(
        "Statistic:",
        f"{shapiro_statistic:.6f}"
    )


    print(
        "p-value:",
        f"{shapiro_p_value:.12g}"
    )


    print(
        "Decision:",
        shapiro_decision
    )


    print(
        "\nJarque-Bera test"
    )


    print(
        "Number of observations:",
        total
    )


    print(
        "Statistic:",
        f"{jarque_bera_statistic:.6f}"
    )


    print(
        "p-value:",
        f"{jarque_bera_p_value:.12g}"
    )


    print(
        "Decision:",
        jarque_bera_decision
    )


    # ========================================================
    # 30. DISPLAY OUTLIER CATEGORIES
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "RECEIVER LOCATION CATEGORIES CLASSIFIED AS OUTLIERS"
    )


    print(
        "=" * 100
    )


    if number_outlier_categories > 0:

        display(
            outlier_table
        )


    else:

        print(
            "No receiver location categories were classified "
            "as outliers by the IQR rule."
        )


    # ========================================================
    # 31. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature
    del encoded_preview

    gc.collect()


    # ========================================================
    # 32. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "ANALYSIS COMPLETED"
    )


    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )


    print(
        HTML_PATH
    )


    print(
        "\nPNG:"
    )


    print(
        RANK_FREQUENCY_CHART_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Rank-frequency chart: Will be created


/usr/local/lib/python3.14/site-packages/scipy/stats/_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 1852394.
  res = hypotest_fun_out(*samples, **kwds)



Rank-frequency chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/onehot_encoding_with_ignore/recive_category_ohewi/recive_category_ohewi_rank_frequency.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/onehot_encoding_with_ignore/recive_category_ohewi/analysis_recive_category_ohewi.html

RECEIVE_LOC_FEWF SUMMARY
Total observations: 1852394
Unique receiver locations: 14
Cardinality percentage: 0.000756%
Unique Frequency Encoding values: 14
Receiver locations involved in collisions: 0
Frequency Encoding outlier categories: 1

NORMALITY TESTS AFTER FREQUENCY ENCODING

Significance level: 0.05

Shapiro-Wilk test
Number of observations: 1852394
Statistic: 0.906890
p-value: 2.60206587163e-158
Decision: Reject H0

Jarque-Bera test
Number of observations: 1852394
Statistic: 176301.045321
p-value: 0
Decision: Reject H0

RECEIVER LOCATION CATEGORIES CLASSIFIED AS OUTLIERS


,RECEIVER_LOCATION,COUNT,FE_VALUE,OUTLIER_TYPE
0,travel,57956,0.031287,Lower outlier



ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/onehot_encoding_with_ignore/recive_category_ohewi

HTML:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/onehot_encoding_with_ignore/recive_category_ohewi/analysis_recive_category_ohewi.html

PNG:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/onehot_encoding_with_ignore/recive_category_ohewi/recive_category_ohewi_rank_frequency.png


### <span style="color:dodgerblue"> SEND_JOB_FEWF </span> ###

In [21]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ENCODING_TYPE = "frequency_encoding_with_fallback"
FEATURE_NAME = "send_job_fewf"

FEATURE_COLUMN = "SEND_JOB_FEWF"

ALPHA = 0.05


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / ENCODING_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


RANK_FREQUENCY_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_rank_frequency.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


rank_frequency_chart_exists = (
    RANK_FREQUENCY_CHART_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and rank_frequency_chart_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )

    print(
        "No analysis or file creation is required."
    )

    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )

    print(
        "\nExisting files:"
    )

    print(
        HTML_PATH
    )

    print(
        RANK_FREQUENCY_CHART_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )

    print(
        "=" * 100
    )

    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )

    print(
        "Rank-frequency chart:",
        "Already exists"
        if rank_frequency_chart_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. BASIC VALIDATION
    # ========================================================

    total = int(
        len(feature)
    )


    if total == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    missing_values = int(
        feature
        .isna()
        .sum()
    )


    if missing_values > 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains "
            f"{missing_values} missing values."
        )


    # ========================================================
    # 13. CATEGORY COUNTS
    # ========================================================

    category_counts = (
        feature
        .value_counts()
        .sort_values(
            ascending=False
        )
    )


    unique_categories = int(
        category_counts.shape[0]
    )


    cardinality_percentage = (
        unique_categories
        / total
        * 100
    )


    # ========================================================
    # 14. FREQUENCY STATISTICS
    # ========================================================

    minimum_count = int(
        category_counts.min()
    )


    maximum_count = int(
        category_counts.max()
    )


    mean_count = float(
        category_counts.mean()
    )


    median_count = float(
        category_counts.median()
    )


    q1_count = float(
        category_counts.quantile(
            0.25
        )
    )


    q3_count = float(
        category_counts.quantile(
            0.75
        )
    )


    # ========================================================
    # 15. FREQUENCY ENCODING VALUES
    #
    # FE(category) = count(category) / N_total
    # ========================================================

    category_fe = (
        category_counts
        / total
    )


    minimum_fe = float(
        category_fe.min()
    )


    maximum_fe = float(
        category_fe.max()
    )


    mean_fe = float(
        category_fe.mean()
    )


    median_fe = float(
        category_fe.median()
    )


    # ========================================================
    # 16. SIMULATE THE TRANSFORMED FEATURE
    #
    # This does not modify the original dataset.
    #
    # Each job category is temporarily replaced by its
    # corresponding Frequency Encoding value.
    # ========================================================

    encoded_preview = (
        feature
        .map(
            category_fe
        )
        .astype(
            "float64"
        )
    )


    # ========================================================
    # 17. DESCRIPTIVE STATISTICS AFTER FREQUENCY ENCODING
    # ========================================================

    transformed_minimum = float(
        encoded_preview.min()
    )


    transformed_q1 = float(
        encoded_preview.quantile(
            0.25
        )
    )


    transformed_median = float(
        encoded_preview.median()
    )


    transformed_mean = float(
        encoded_preview.mean()
    )


    transformed_q3 = float(
        encoded_preview.quantile(
            0.75
        )
    )


    transformed_maximum = float(
        encoded_preview.max()
    )


    transformed_standard_deviation = float(
        encoded_preview.std()
    )


    # ========================================================
    # 18. IQR OUTLIER LIMITS AFTER FREQUENCY ENCODING
    # ========================================================

    iqr_fe = (
        transformed_q3
        - transformed_q1
    )


    lower_bound_fe = (
        transformed_q1
        - 1.5 * iqr_fe
    )


    upper_bound_fe = (
        transformed_q3
        + 1.5 * iqr_fe
    )


    # ========================================================
    # 19. IDENTIFY JOB CATEGORIES THAT WOULD BECOME OUTLIERS
    # ========================================================

    outlier_mask = (
        (
            category_fe
            < lower_bound_fe
        )
        |
        (
            category_fe
            > upper_bound_fe
        )
    )


    outlier_categories = (
        category_fe[
            outlier_mask
        ]
    )


    outlier_table = pd.DataFrame({

        "JOB_CATEGORY":
            outlier_categories
            .index
            .astype(str),

        "COUNT":
            category_counts.loc[
                outlier_categories.index
            ]
            .values,

        "FE_VALUE":
            outlier_categories.values
    })


    outlier_table[
        "OUTLIER_TYPE"
    ] = np.where(
        outlier_table[
            "FE_VALUE"
        ]
        < lower_bound_fe,

        "Lower outlier",

        "Upper outlier"
    )


    outlier_table = (
        outlier_table
        .sort_values(
            "FE_VALUE",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    number_outlier_categories = int(
        len(
            outlier_table
        )
    )


    outlier_category_percentage = (
        number_outlier_categories
        / unique_categories
        * 100
    )


    if number_outlier_categories > 0:

        outlier_category_values = set(
            outlier_categories.index
        )


        observations_in_outlier_categories = int(
            feature
            .isin(
                outlier_category_values
            )
            .sum()
        )


    else:

        observations_in_outlier_categories = 0


    outlier_observation_percentage = (
        observations_in_outlier_categories
        / total
        * 100
    )


    # ========================================================
    # 20. FREQUENCY COLLISION ANALYSIS
    #
    # Different job categories with the same number of
    # observations receive the same Frequency Encoding value.
    # ========================================================

    unique_fe_values = int(
        category_fe.nunique()
    )


    frequency_groups = (
        category_counts
        .value_counts()
    )


    collision_groups = int(
        (
            frequency_groups
            > 1
        )
        .sum()
    )


    categories_in_collisions = int(
        frequency_groups[
            frequency_groups
            > 1
        ]
        .sum()
    )


    frequency_compression_ratio = (
        unique_fe_values
        / unique_categories
    )


    frequency_reduction_percentage = (
        1
        - frequency_compression_ratio
    ) * 100


    # ========================================================
    # 21. SHAPIRO-WILK NORMALITY TEST
    #
    # H0:
    # The Frequency Encoded feature follows
    # a normal distribution.
    #
    # H1:
    # The Frequency Encoded feature does not follow
    # a normal distribution.
    #
    # Significance level:
    # alpha = 0.05
    #
    # The complete transformed feature is used.
    # ========================================================

    (
        shapiro_statistic,
        shapiro_p_value
    ) = stats.shapiro(
        encoded_preview
    )


    shapiro_statistic = float(
        shapiro_statistic
    )


    shapiro_p_value = float(
        shapiro_p_value
    )


    if shapiro_p_value < ALPHA:

        shapiro_decision = (
            "Reject H0"
        )


        shapiro_interpretation = (
            "There is statistical evidence that "
            "the Frequency Encoded feature does not "
            "follow a normal distribution."
        )


    else:

        shapiro_decision = (
            "Fail to reject H0"
        )


        shapiro_interpretation = (
            "There is insufficient statistical evidence "
            "to conclude that the Frequency Encoded feature "
            "deviates from a normal distribution."
        )


    # ========================================================
    # 22. JARQUE-BERA NORMALITY TEST
    #
    # H0:
    # The Frequency Encoded feature follows
    # a normal distribution.
    #
    # H1:
    # The Frequency Encoded feature does not follow
    # a normal distribution.
    #
    # Significance level:
    # alpha = 0.05
    #
    # The complete transformed feature is used.
    # ========================================================

    jarque_bera_result = (
        stats.jarque_bera(
            encoded_preview
        )
    )


    jarque_bera_statistic = float(
        jarque_bera_result.statistic
    )


    jarque_bera_p_value = float(
        jarque_bera_result.pvalue
    )


    if jarque_bera_p_value < ALPHA:

        jarque_bera_decision = (
            "Reject H0"
        )


        jarque_bera_interpretation = (
            "There is statistical evidence that "
            "the Frequency Encoded feature does not "
            "follow a normal distribution."
        )


    else:

        jarque_bera_decision = (
            "Fail to reject H0"
        )


        jarque_bera_interpretation = (
            "There is insufficient statistical evidence "
            "to conclude that the Frequency Encoded feature "
            "deviates from a normal distribution."
        )


    # ========================================================
    # 23. CREATE THE RANK-FREQUENCY CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not rank_frequency_chart_exists:

        ranked_fe = (
            category_fe
            .sort_values(
                ascending=False
            )
            .reset_index(
                drop=True
            )
        )


        category_rank = np.arange(
            1,
            len(ranked_fe) + 1
        )


        fig, ax = plt.subplots(
            figsize=(
                11,
                6
            )
        )


        ax.plot(
            category_rank,
            ranked_fe.values
        )


        ax.set_title(
            "Rank-frequency curve of job categories"
        )


        ax.set_xlabel(
            "Category rank"
        )


        ax.set_ylabel(
            "Frequency Encoding value"
        )


        ax.xaxis.set_major_formatter(
            FuncFormatter(
                lambda x, pos:
                str(
                    int(x)
                )
            )
        )


        ax.grid(
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            RANK_FREQUENCY_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nRank-frequency chart created:"
        )


        print(
            RANK_FREQUENCY_CHART_PATH
        )


    else:

        print(
            "\nRank-frequency chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 24. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 25. PREPARE OUTLIER TABLE FOR HTML
    # ========================================================

    if number_outlier_categories > 0:

        outlier_html = (
            outlier_table
            .to_html(
                index=False,
                border=0,
                float_format=lambda x:
                f"{x:.12f}"
            )
        )


    else:

        outlier_html = (
            "<p>"
            "No job categories were classified "
            "as outliers by the IQR rule."
            "</p>"
        )


    # ========================================================
    # 26. CREATE THE HTML REPORT
    #
    # Only if the HTML does not already exist.
    # ========================================================

    if not html_exists:

        rank_frequency_chart_base64 = (
            image_to_base64(
                RANK_FREQUENCY_CHART_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Individual Exploratory Analysis — {FEATURE_COLUMN}
</h1>


<p>

The variable <strong>{FEATURE_COLUMN}</strong>
represents the job or occupation associated
with the transaction sender.

The feature is treated as a
<strong>categorical variable</strong>
and is intended for Frequency Encoding
With Fallback.

</p>


<!-- ========================================================
     1. FEATURE OVERVIEW
========================================================= -->


<h2>
1. Feature overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total observations</td>
<td>{total}</td>
</tr>

<tr>
<td>Missing values</td>
<td>{missing_values}</td>
</tr>

<tr>
<td>Unique job categories</td>
<td>{unique_categories}</td>
</tr>

<tr>
<td>Cardinality percentage</td>
<td>{cardinality_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     2. FREQUENCY STATISTICS
========================================================= -->


<h2>
2. Frequency statistics
</h2>


<table>

<tr>
<th>Statistic</th>
<th>Observations per job category</th>
</tr>

<tr>
<td>Minimum</td>
<td>{minimum_count}</td>
</tr>

<tr>
<td>Q1</td>
<td>{q1_count:.2f}</td>
</tr>

<tr>
<td>Median</td>
<td>{median_count:.2f}</td>
</tr>

<tr>
<td>Mean</td>
<td>{mean_count:.2f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{q3_count:.2f}</td>
</tr>

<tr>
<td>Maximum</td>
<td>{maximum_count}</td>
</tr>

</table>


<!-- ========================================================
     3. EXPECTED FREQUENCY ENCODING VALUES
========================================================= -->


<h2>
3. Expected Frequency Encoding values
</h2>


<p>

For each job category,
the Frequency Encoding value is calculated as:

</p>


<p class="result">

FE(category) = count(category) / N

</p>


<table>

<tr>
<th>Statistic</th>
<th>FE value across categories</th>
</tr>

<tr>
<td>Minimum</td>
<td>{minimum_fe:.12f}</td>
</tr>

<tr>
<td>Median</td>
<td>{median_fe:.12f}</td>
</tr>

<tr>
<td>Mean</td>
<td>{mean_fe:.12f}</td>
</tr>

<tr>
<td>Maximum</td>
<td>{maximum_fe:.12f}</td>
</tr>

</table>


<h3>
Descriptive statistics after transformation
</h3>


<p>

The following statistics describe the complete
transformed feature across all
<strong>{total}</strong>
observations.

</p>


<table>

<tr>
<th>Statistic</th>
<th>Transformed value</th>
</tr>

<tr>
<td>Minimum</td>
<td>{transformed_minimum:.12f}</td>
</tr>

<tr>
<td>Q1</td>
<td>{transformed_q1:.12f}</td>
</tr>

<tr>
<td>Median</td>
<td>{transformed_median:.12f}</td>
</tr>

<tr>
<td>Mean</td>
<td>{transformed_mean:.12f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{transformed_q3:.12f}</td>
</tr>

<tr>
<td>Maximum</td>
<td>{transformed_maximum:.12f}</td>
</tr>

<tr>
<td>Standard deviation</td>
<td>{transformed_standard_deviation:.12f}</td>
</tr>

</table>


<!-- ========================================================
     4. OUTLIER ANALYSIS
========================================================= -->


<h2>
4. Frequency Encoding outlier analysis
</h2>


<p>

The IQR method was applied to the complete
simulated Frequency Encoded feature.

The detected outliers represent job categories
whose Frequency Encoding values fall outside
the IQR limits after transformation.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Q1</td>
<td>{transformed_q1:.12f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{transformed_q3:.12f}</td>
</tr>

<tr>
<td>IQR</td>
<td>{iqr_fe:.12f}</td>
</tr>

<tr>
<td>Lower bound</td>
<td>{lower_bound_fe:.12f}</td>
</tr>

<tr>
<td>Upper bound</td>
<td>{upper_bound_fe:.12f}</td>
</tr>

<tr>
<td>Outlier job categories</td>
<td>{number_outlier_categories}</td>
</tr>

<tr>
<td>Percentage of job categories</td>
<td>{outlier_category_percentage:.6f}%</td>
</tr>

<tr>
<td>Observations belonging to outlier categories</td>
<td>{observations_in_outlier_categories}</td>
</tr>

<tr>
<td>Percentage of dataset observations</td>
<td>{outlier_observation_percentage:.6f}%</td>
</tr>

</table>


<h3>
Job categories classified as outliers
</h3>


{outlier_html}


<!-- ========================================================
     5. FREQUENCY COLLISIONS
========================================================= -->


<h2>
5. Frequency collisions
</h2>


<p>

Two or more different job categories may have
the same number of observations.

When this occurs, they receive exactly
the same Frequency Encoding value.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Original unique categories</td>
<td>{unique_categories}</td>
</tr>

<tr>
<td>Unique Frequency Encoding values</td>
<td>{unique_fe_values}</td>
</tr>

<tr>
<td>Frequency values shared by multiple job categories</td>
<td>{collision_groups}</td>
</tr>

<tr>
<td>Job categories involved in collisions</td>
<td>{categories_in_collisions}</td>
</tr>

<tr>
<td>Frequency compression ratio</td>
<td>{frequency_compression_ratio:.6f}</td>
</tr>

<tr>
<td>Reduction in distinct representation</td>
<td>{frequency_reduction_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     6. RANK-FREQUENCY CURVE
========================================================= -->


<h2>
6. Rank-frequency curve
</h2>


<p>

Job categories are ordered from the most frequent
to the least frequent.

The X axis represents the category rank,
while the Y axis represents the expected
Frequency Encoding value.

Horizontal regions indicate job categories
that share identical Frequency Encoding values.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{rank_frequency_chart_base64}"
    alt="Rank-frequency curve of job categories"
>

</div>


<!-- ========================================================
     7. NORMALITY TESTS
========================================================= -->


<h2>
7. Normality tests after Frequency Encoding
</h2>


<p>

The Shapiro-Wilk and Jarque-Bera tests
were applied to the complete transformed feature.

The significance level used for both tests was:

</p>


<p class="result">

α = {ALPHA}

</p>


<ul>

<li>

<strong>H0:</strong>
the Frequency Encoded feature follows
a normal distribution.

</li>

<li>

<strong>H1:</strong>
the Frequency Encoded feature does not
follow a normal distribution.

</li>

</ul>


<!-- ========================================================
     7.1 SHAPIRO-WILK
========================================================= -->


<h3>
7.1 Shapiro-Wilk test
</h3>


<p>

The Shapiro-Wilk test was applied to all
<strong>{total}</strong>
Frequency Encoded observations.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Number of observations</td>
<td>{total}</td>
</tr>

<tr>
<td>Test statistic (W)</td>
<td>{shapiro_statistic:.6f}</td>
</tr>

<tr>
<td>p-value</td>
<td>{shapiro_p_value:.12g}</td>
</tr>

<tr>
<td>Significance level</td>
<td>{ALPHA}</td>
</tr>

<tr>
<td>Decision</td>
<td><strong>{shapiro_decision}</strong></td>
</tr>

</table>


<p>

{shapiro_interpretation}

</p>


<!-- ========================================================
     7.2 JARQUE-BERA
========================================================= -->


<h3>
7.2 Jarque-Bera test
</h3>


<p>

The Jarque-Bera test was applied to all
<strong>{total}</strong>
Frequency Encoded observations.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Number of observations</td>
<td>{total}</td>
</tr>

<tr>
<td>Test statistic</td>
<td>{jarque_bera_statistic:.6f}</td>
</tr>

<tr>
<td>p-value</td>
<td>{jarque_bera_p_value:.12g}</td>
</tr>

<tr>
<td>Significance level</td>
<td>{ALPHA}</td>
</tr>

<tr>
<td>Decision</td>
<td><strong>{jarque_bera_decision}</strong></td>
</tr>

</table>


<p>

{jarque_bera_interpretation}

</p>


<div class="note">

<strong>Decision rule:</strong>

<br><br>

If p-value &lt; 0.05,
H0 is rejected.

<br><br>

If p-value ≥ 0.05,
H0 is not rejected.

<br><br>

<strong>Shapiro-Wilk note:</strong>

For very large sample sizes, the test statistic
can still be calculated using the complete dataset,
but the numerical accuracy of the p-value should
be interpreted with caution.

</div>


<!-- ========================================================
     8. SUMMARY
========================================================= -->


<h2>
8. Summary of results
</h2>


<ul>

<li>
<strong>Total observations:</strong>
{total}
</li>

<li>
<strong>Unique job categories:</strong>
{unique_categories}
</li>

<li>
<strong>Cardinality percentage:</strong>
{cardinality_percentage:.6f}%
</li>

<li>
<strong>Minimum observations per job category:</strong>
{minimum_count}
</li>

<li>
<strong>Median observations per job category:</strong>
{median_count:.2f}
</li>

<li>
<strong>Maximum observations per job category:</strong>
{maximum_count}
</li>

<li>
<strong>Minimum Frequency Encoding value:</strong>
{minimum_fe:.12f}
</li>

<li>
<strong>Maximum Frequency Encoding value:</strong>
{maximum_fe:.12f}
</li>

<li>
<strong>Frequency Encoding outlier categories:</strong>
{number_outlier_categories}
</li>

<li>
<strong>Observations represented by outlier categories:</strong>
{outlier_observation_percentage:.6f}%
</li>

<li>
<strong>Unique Frequency Encoding values:</strong>
{unique_fe_values}
</li>

<li>
<strong>Job categories involved in frequency collisions:</strong>
{categories_in_collisions}
</li>

<li>
<strong>Frequency compression ratio:</strong>
{frequency_compression_ratio:.6f}
</li>

<li>
<strong>Shapiro-Wilk observations:</strong>
{total}
</li>

<li>
<strong>Shapiro-Wilk:</strong>
{shapiro_decision}
</li>

<li>
<strong>Jarque-Bera observations:</strong>
{total}
</li>

<li>
<strong>Jarque-Bera:</strong>
{jarque_bera_decision}
</li>

</ul>


</body>

</html>
"""


        # ====================================================
        # 27. SAVE THE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )


        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 28. DISPLAY MAIN RESULTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "SEND_JOB_FEWF SUMMARY"
    )


    print(
        "=" * 100
    )


    print(
        "Total observations:",
        total
    )


    print(
        "Unique job categories:",
        unique_categories
    )


    print(
        "Cardinality percentage:",
        f"{cardinality_percentage:.6f}%"
    )


    print(
        "Unique Frequency Encoding values:",
        unique_fe_values
    )


    print(
        "Job categories involved in collisions:",
        categories_in_collisions
    )


    print(
        "Frequency Encoding outlier categories:",
        number_outlier_categories
    )


    # ========================================================
    # 29. DISPLAY NORMALITY TESTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "NORMALITY TESTS AFTER FREQUENCY ENCODING"
    )


    print(
        "=" * 100
    )


    print(
        f"\nSignificance level: {ALPHA}"
    )


    print(
        "\nShapiro-Wilk test"
    )


    print(
        "Number of observations:",
        total
    )


    print(
        "Statistic:",
        f"{shapiro_statistic:.6f}"
    )


    print(
        "p-value:",
        f"{shapiro_p_value:.12g}"
    )


    print(
        "Decision:",
        shapiro_decision
    )


    print(
        "\nJarque-Bera test"
    )


    print(
        "Number of observations:",
        total
    )


    print(
        "Statistic:",
        f"{jarque_bera_statistic:.6f}"
    )


    print(
        "p-value:",
        f"{jarque_bera_p_value:.12g}"
    )


    print(
        "Decision:",
        jarque_bera_decision
    )


    # ========================================================
    # 30. DISPLAY OUTLIER CATEGORIES
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "JOB CATEGORIES CLASSIFIED AS OUTLIERS"
    )


    print(
        "=" * 100
    )


    if number_outlier_categories > 0:

        display(
            outlier_table
        )


    else:

        print(
            "No job categories were classified "
            "as outliers by the IQR rule."
        )


    # ========================================================
    # 31. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature
    del encoded_preview

    gc.collect()


    # ========================================================
    # 32. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "ANALYSIS COMPLETED"
    )


    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )


    print(
        HTML_PATH
    )


    print(
        "\nPNG:"
    )


    print(
        RANK_FREQUENCY_CHART_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Rank-frequency chart: Will be created


/usr/local/lib/python3.14/site-packages/scipy/stats/_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 1852394.
  res = hypotest_fun_out(*samples, **kwds)



Rank-frequency chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/frequency_encoding_with_fallback/send_job_fewf/send_job_fewf_rank_frequency.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/frequency_encoding_with_fallback/send_job_fewf/analysis_send_job_fewf.html

SEND_JOB_FEWF SUMMARY
Total observations: 1852394
Unique job categories: 497
Cardinality percentage: 0.026830%
Unique Frequency Encoding values: 273
Job categories involved in collisions: 338
Frequency Encoding outlier categories: 0

NORMALITY TESTS AFTER FREQUENCY ENCODING

Significance level: 0.05

Shapiro-Wilk test
Number of observations: 1852394
Statistic: 0.963716
p-value: 9.89072730155e-129
Decision: Reject H0

Jarque-Bera test
Number of observations: 1852394
Statistic: 99988.126362
p-value: 0
Decision: Reject H0

JOB CATEGORIES CLASSIFIED AS OUTLIERS
No job categories were classified as outliers by the IQR rule.

ANALYSIS COMPLETED

R

### <span style="color:dodgerblue"> SEND_NAME_FEWF </span> ###

In [22]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ENCODING_TYPE = "frequency_encoding_with_fallback"
FEATURE_NAME = "send_name_fewf"

FEATURE_COLUMN = "SEND_NAME_FEWF"

ALPHA = 0.05


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / ENCODING_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


RANK_FREQUENCY_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_rank_frequency.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


rank_frequency_chart_exists = (
    RANK_FREQUENCY_CHART_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and rank_frequency_chart_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )

    print(
        "No analysis or file creation is required."
    )

    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )

    print(
        "\nExisting files:"
    )

    print(
        HTML_PATH
    )

    print(
        RANK_FREQUENCY_CHART_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )

    print(
        "=" * 100
    )

    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )

    print(
        "Rank-frequency chart:",
        "Already exists"
        if rank_frequency_chart_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. BASIC VALIDATION
    # ========================================================

    total = int(
        len(feature)
    )


    if total == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    missing_values = int(
        feature
        .isna()
        .sum()
    )


    if missing_values > 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains "
            f"{missing_values} missing values."
        )


    # ========================================================
    # 13. CATEGORY COUNTS
    # ========================================================

    category_counts = (
        feature
        .value_counts()
        .sort_values(
            ascending=False
        )
    )


    unique_categories = int(
        category_counts.shape[0]
    )


    cardinality_percentage = (
        unique_categories
        / total
        * 100
    )


    # ========================================================
    # 14. FREQUENCY STATISTICS
    # ========================================================

    minimum_count = int(
        category_counts.min()
    )


    maximum_count = int(
        category_counts.max()
    )


    mean_count = float(
        category_counts.mean()
    )


    median_count = float(
        category_counts.median()
    )


    q1_count = float(
        category_counts.quantile(
            0.25
        )
    )


    q3_count = float(
        category_counts.quantile(
            0.75
        )
    )


    # ========================================================
    # 15. FREQUENCY ENCODING VALUES
    #
    # FE(category) = count(category) / N_total
    # ========================================================

    category_fe = (
        category_counts
        / total
    )


    minimum_fe = float(
        category_fe.min()
    )


    maximum_fe = float(
        category_fe.max()
    )


    mean_fe = float(
        category_fe.mean()
    )


    median_fe = float(
        category_fe.median()
    )


    # ========================================================
    # 16. SIMULATE THE TRANSFORMED FEATURE
    #
    # This does not modify the original dataset.
    #
    # Each sender name category is temporarily replaced
    # by its corresponding Frequency Encoding value.
    # ========================================================

    encoded_preview = (
        feature
        .map(
            category_fe
        )
        .astype(
            "float64"
        )
    )


    # ========================================================
    # 17. DESCRIPTIVE STATISTICS AFTER FREQUENCY ENCODING
    # ========================================================

    transformed_minimum = float(
        encoded_preview.min()
    )


    transformed_q1 = float(
        encoded_preview.quantile(
            0.25
        )
    )


    transformed_median = float(
        encoded_preview.median()
    )


    transformed_mean = float(
        encoded_preview.mean()
    )


    transformed_q3 = float(
        encoded_preview.quantile(
            0.75
        )
    )


    transformed_maximum = float(
        encoded_preview.max()
    )


    transformed_standard_deviation = float(
        encoded_preview.std()
    )


    # ========================================================
    # 18. IQR OUTLIER LIMITS AFTER FREQUENCY ENCODING
    # ========================================================

    iqr_fe = (
        transformed_q3
        - transformed_q1
    )


    lower_bound_fe = (
        transformed_q1
        - 1.5 * iqr_fe
    )


    upper_bound_fe = (
        transformed_q3
        + 1.5 * iqr_fe
    )


    # ========================================================
    # 19. IDENTIFY SENDER NAME CATEGORIES
    #     THAT WOULD BECOME OUTLIERS
    # ========================================================

    outlier_mask = (
        (
            category_fe
            < lower_bound_fe
        )
        |
        (
            category_fe
            > upper_bound_fe
        )
    )


    outlier_categories = (
        category_fe[
            outlier_mask
        ]
    )


    outlier_table = pd.DataFrame({

        "SENDER_NAME":
            outlier_categories
            .index
            .astype(str),

        "COUNT":
            category_counts.loc[
                outlier_categories.index
            ]
            .values,

        "FE_VALUE":
            outlier_categories.values
    })


    outlier_table[
        "OUTLIER_TYPE"
    ] = np.where(
        outlier_table[
            "FE_VALUE"
        ]
        < lower_bound_fe,

        "Lower outlier",

        "Upper outlier"
    )


    outlier_table = (
        outlier_table
        .sort_values(
            "FE_VALUE",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    number_outlier_categories = int(
        len(
            outlier_table
        )
    )


    outlier_category_percentage = (
        number_outlier_categories
        / unique_categories
        * 100
    )


    if number_outlier_categories > 0:

        outlier_category_values = set(
            outlier_categories.index
        )


        observations_in_outlier_categories = int(
            feature
            .isin(
                outlier_category_values
            )
            .sum()
        )


    else:

        observations_in_outlier_categories = 0


    outlier_observation_percentage = (
        observations_in_outlier_categories
        / total
        * 100
    )


    # ========================================================
    # 20. FREQUENCY COLLISION ANALYSIS
    #
    # Different sender names with the same number of
    # observations receive the same Frequency Encoding value.
    # ========================================================

    unique_fe_values = int(
        category_fe.nunique()
    )


    frequency_groups = (
        category_counts
        .value_counts()
    )


    collision_groups = int(
        (
            frequency_groups
            > 1
        )
        .sum()
    )


    categories_in_collisions = int(
        frequency_groups[
            frequency_groups
            > 1
        ]
        .sum()
    )


    frequency_compression_ratio = (
        unique_fe_values
        / unique_categories
    )


    frequency_reduction_percentage = (
        1
        - frequency_compression_ratio
    ) * 100


    # ========================================================
    # 21. SHAPIRO-WILK NORMALITY TEST
    #
    # H0:
    # The Frequency Encoded feature follows
    # a normal distribution.
    #
    # H1:
    # The Frequency Encoded feature does not follow
    # a normal distribution.
    #
    # Significance level:
    # alpha = 0.05
    #
    # The complete transformed feature is used.
    # ========================================================

    (
        shapiro_statistic,
        shapiro_p_value
    ) = stats.shapiro(
        encoded_preview
    )


    shapiro_statistic = float(
        shapiro_statistic
    )


    shapiro_p_value = float(
        shapiro_p_value
    )


    if shapiro_p_value < ALPHA:

        shapiro_decision = (
            "Reject H0"
        )


        shapiro_interpretation = (
            "There is statistical evidence that "
            "the Frequency Encoded feature does not "
            "follow a normal distribution."
        )


    else:

        shapiro_decision = (
            "Fail to reject H0"
        )


        shapiro_interpretation = (
            "There is insufficient statistical evidence "
            "to conclude that the Frequency Encoded feature "
            "deviates from a normal distribution."
        )


    # ========================================================
    # 22. JARQUE-BERA NORMALITY TEST
    #
    # H0:
    # The Frequency Encoded feature follows
    # a normal distribution.
    #
    # H1:
    # The Frequency Encoded feature does not follow
    # a normal distribution.
    #
    # Significance level:
    # alpha = 0.05
    #
    # The complete transformed feature is used.
    # ========================================================

    jarque_bera_result = (
        stats.jarque_bera(
            encoded_preview
        )
    )


    jarque_bera_statistic = float(
        jarque_bera_result.statistic
    )


    jarque_bera_p_value = float(
        jarque_bera_result.pvalue
    )


    if jarque_bera_p_value < ALPHA:

        jarque_bera_decision = (
            "Reject H0"
        )


        jarque_bera_interpretation = (
            "There is statistical evidence that "
            "the Frequency Encoded feature does not "
            "follow a normal distribution."
        )


    else:

        jarque_bera_decision = (
            "Fail to reject H0"
        )


        jarque_bera_interpretation = (
            "There is insufficient statistical evidence "
            "to conclude that the Frequency Encoded feature "
            "deviates from a normal distribution."
        )


    # ========================================================
    # 23. CREATE THE RANK-FREQUENCY CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not rank_frequency_chart_exists:

        ranked_fe = (
            category_fe
            .sort_values(
                ascending=False
            )
            .reset_index(
                drop=True
            )
        )


        category_rank = np.arange(
            1,
            len(ranked_fe) + 1
        )


        fig, ax = plt.subplots(
            figsize=(
                11,
                6
            )
        )


        ax.plot(
            category_rank,
            ranked_fe.values
        )


        ax.set_title(
            "Rank-frequency curve of sender name categories"
        )


        ax.set_xlabel(
            "Category rank"
        )


        ax.set_ylabel(
            "Frequency Encoding value"
        )


        ax.xaxis.set_major_formatter(
            FuncFormatter(
                lambda x, pos:
                str(
                    int(x)
                )
            )
        )


        ax.grid(
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            RANK_FREQUENCY_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nRank-frequency chart created:"
        )


        print(
            RANK_FREQUENCY_CHART_PATH
        )


    else:

        print(
            "\nRank-frequency chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 24. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 25. PREPARE OUTLIER TABLE FOR HTML
    # ========================================================

    if number_outlier_categories > 0:

        outlier_html = (
            outlier_table
            .to_html(
                index=False,
                border=0,
                float_format=lambda x:
                f"{x:.12f}"
            )
        )


    else:

        outlier_html = (
            "<p>"
            "No sender name categories were classified "
            "as outliers by the IQR rule."
            "</p>"
        )


    # ========================================================
    # 26. CREATE THE HTML REPORT
    #
    # Only if the HTML does not already exist.
    # ========================================================

    if not html_exists:

        rank_frequency_chart_base64 = (
            image_to_base64(
                RANK_FREQUENCY_CHART_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Individual Exploratory Analysis — {FEATURE_COLUMN}
</h1>


<p>

The variable <strong>{FEATURE_COLUMN}</strong>
represents the sender name associated
with each transaction.

The feature is treated as a
<strong>categorical identifier</strong>
and is intended for Frequency Encoding
With Fallback.

</p>


<!-- ========================================================
     1. FEATURE OVERVIEW
========================================================= -->


<h2>
1. Feature overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total observations</td>
<td>{total}</td>
</tr>

<tr>
<td>Missing values</td>
<td>{missing_values}</td>
</tr>

<tr>
<td>Unique sender name categories</td>
<td>{unique_categories}</td>
</tr>

<tr>
<td>Cardinality percentage</td>
<td>{cardinality_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     2. FREQUENCY STATISTICS
========================================================= -->


<h2>
2. Frequency statistics
</h2>


<table>

<tr>
<th>Statistic</th>
<th>Observations per sender name</th>
</tr>

<tr>
<td>Minimum</td>
<td>{minimum_count}</td>
</tr>

<tr>
<td>Q1</td>
<td>{q1_count:.2f}</td>
</tr>

<tr>
<td>Median</td>
<td>{median_count:.2f}</td>
</tr>

<tr>
<td>Mean</td>
<td>{mean_count:.2f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{q3_count:.2f}</td>
</tr>

<tr>
<td>Maximum</td>
<td>{maximum_count}</td>
</tr>

</table>


<!-- ========================================================
     3. EXPECTED FREQUENCY ENCODING VALUES
========================================================= -->


<h2>
3. Expected Frequency Encoding values
</h2>


<p>

For each sender name category,
the Frequency Encoding value is calculated as:

</p>


<p class="result">

FE(category) = count(category) / N

</p>


<table>

<tr>
<th>Statistic</th>
<th>FE value across categories</th>
</tr>

<tr>
<td>Minimum</td>
<td>{minimum_fe:.12f}</td>
</tr>

<tr>
<td>Median</td>
<td>{median_fe:.12f}</td>
</tr>

<tr>
<td>Mean</td>
<td>{mean_fe:.12f}</td>
</tr>

<tr>
<td>Maximum</td>
<td>{maximum_fe:.12f}</td>
</tr>

</table>


<h3>
Descriptive statistics after transformation
</h3>


<p>

The following statistics describe the complete
transformed feature across all
<strong>{total}</strong>
observations.

</p>


<table>

<tr>
<th>Statistic</th>
<th>Transformed value</th>
</tr>

<tr>
<td>Minimum</td>
<td>{transformed_minimum:.12f}</td>
</tr>

<tr>
<td>Q1</td>
<td>{transformed_q1:.12f}</td>
</tr>

<tr>
<td>Median</td>
<td>{transformed_median:.12f}</td>
</tr>

<tr>
<td>Mean</td>
<td>{transformed_mean:.12f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{transformed_q3:.12f}</td>
</tr>

<tr>
<td>Maximum</td>
<td>{transformed_maximum:.12f}</td>
</tr>

<tr>
<td>Standard deviation</td>
<td>{transformed_standard_deviation:.12f}</td>
</tr>

</table>


<!-- ========================================================
     4. OUTLIER ANALYSIS
========================================================= -->


<h2>
4. Frequency Encoding outlier analysis
</h2>


<p>

The IQR method was applied to the complete
simulated Frequency Encoded feature.

The detected outliers represent sender name
categories whose Frequency Encoding values
fall outside the IQR limits after transformation.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Q1</td>
<td>{transformed_q1:.12f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{transformed_q3:.12f}</td>
</tr>

<tr>
<td>IQR</td>
<td>{iqr_fe:.12f}</td>
</tr>

<tr>
<td>Lower bound</td>
<td>{lower_bound_fe:.12f}</td>
</tr>

<tr>
<td>Upper bound</td>
<td>{upper_bound_fe:.12f}</td>
</tr>

<tr>
<td>Outlier sender name categories</td>
<td>{number_outlier_categories}</td>
</tr>

<tr>
<td>Percentage of sender name categories</td>
<td>{outlier_category_percentage:.6f}%</td>
</tr>

<tr>
<td>Observations belonging to outlier categories</td>
<td>{observations_in_outlier_categories}</td>
</tr>

<tr>
<td>Percentage of dataset observations</td>
<td>{outlier_observation_percentage:.6f}%</td>
</tr>

</table>


<h3>
Sender name categories classified as outliers
</h3>


{outlier_html}


<!-- ========================================================
     5. FREQUENCY COLLISIONS
========================================================= -->


<h2>
5. Frequency collisions
</h2>


<p>

Two or more different sender names may have
the same number of observations.

When this occurs, they receive exactly
the same Frequency Encoding value.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Original unique categories</td>
<td>{unique_categories}</td>
</tr>

<tr>
<td>Unique Frequency Encoding values</td>
<td>{unique_fe_values}</td>
</tr>

<tr>
<td>Frequency values shared by multiple sender names</td>
<td>{collision_groups}</td>
</tr>

<tr>
<td>Sender names involved in collisions</td>
<td>{categories_in_collisions}</td>
</tr>

<tr>
<td>Frequency compression ratio</td>
<td>{frequency_compression_ratio:.6f}</td>
</tr>

<tr>
<td>Reduction in distinct representation</td>
<td>{frequency_reduction_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     6. RANK-FREQUENCY CURVE
========================================================= -->


<h2>
6. Rank-frequency curve
</h2>


<p>

Sender name categories are ordered
from the most frequent to the least frequent.

The X axis represents the category rank,
while the Y axis represents the expected
Frequency Encoding value.

Horizontal regions indicate sender name
categories that share identical
Frequency Encoding values.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{rank_frequency_chart_base64}"
    alt="Rank-frequency curve of sender name categories"
>

</div>


<!-- ========================================================
     7. NORMALITY TESTS
========================================================= -->


<h2>
7. Normality tests after Frequency Encoding
</h2>


<p>

The Shapiro-Wilk and Jarque-Bera tests
were applied to the complete transformed feature.

The significance level used for both tests was:

</p>


<p class="result">

α = {ALPHA}

</p>


<ul>

<li>

<strong>H0:</strong>
the Frequency Encoded feature follows
a normal distribution.

</li>

<li>

<strong>H1:</strong>
the Frequency Encoded feature does not
follow a normal distribution.

</li>

</ul>


<!-- ========================================================
     7.1 SHAPIRO-WILK
========================================================= -->


<h3>
7.1 Shapiro-Wilk test
</h3>


<p>

The Shapiro-Wilk test was applied to all
<strong>{total}</strong>
Frequency Encoded observations.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Number of observations</td>
<td>{total}</td>
</tr>

<tr>
<td>Test statistic (W)</td>
<td>{shapiro_statistic:.6f}</td>
</tr>

<tr>
<td>p-value</td>
<td>{shapiro_p_value:.12g}</td>
</tr>

<tr>
<td>Significance level</td>
<td>{ALPHA}</td>
</tr>

<tr>
<td>Decision</td>
<td><strong>{shapiro_decision}</strong></td>
</tr>

</table>


<p>

{shapiro_interpretation}

</p>


<!-- ========================================================
     7.2 JARQUE-BERA
========================================================= -->


<h3>
7.2 Jarque-Bera test
</h3>


<p>

The Jarque-Bera test was applied to all
<strong>{total}</strong>
Frequency Encoded observations.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Number of observations</td>
<td>{total}</td>
</tr>

<tr>
<td>Test statistic</td>
<td>{jarque_bera_statistic:.6f}</td>
</tr>

<tr>
<td>p-value</td>
<td>{jarque_bera_p_value:.12g}</td>
</tr>

<tr>
<td>Significance level</td>
<td>{ALPHA}</td>
</tr>

<tr>
<td>Decision</td>
<td><strong>{jarque_bera_decision}</strong></td>
</tr>

</table>


<p>

{jarque_bera_interpretation}

</p>


<div class="note">

<strong>Decision rule:</strong>

<br><br>

If p-value &lt; 0.05,
H0 is rejected.

<br><br>

If p-value ≥ 0.05,
H0 is not rejected.

<br><br>

<strong>Shapiro-Wilk note:</strong>

For very large sample sizes, the test statistic
can still be calculated using the complete dataset,
but the numerical accuracy of the p-value should
be interpreted with caution.

</div>


<!-- ========================================================
     8. SUMMARY
========================================================= -->


<h2>
8. Summary of results
</h2>


<ul>

<li>
<strong>Total observations:</strong>
{total}
</li>

<li>
<strong>Unique sender name categories:</strong>
{unique_categories}
</li>

<li>
<strong>Cardinality percentage:</strong>
{cardinality_percentage:.6f}%
</li>

<li>
<strong>Minimum observations per sender name:</strong>
{minimum_count}
</li>

<li>
<strong>Median observations per sender name:</strong>
{median_count:.2f}
</li>

<li>
<strong>Maximum observations per sender name:</strong>
{maximum_count}
</li>

<li>
<strong>Minimum Frequency Encoding value:</strong>
{minimum_fe:.12f}
</li>

<li>
<strong>Maximum Frequency Encoding value:</strong>
{maximum_fe:.12f}
</li>

<li>
<strong>Frequency Encoding outlier categories:</strong>
{number_outlier_categories}
</li>

<li>
<strong>Observations represented by outlier categories:</strong>
{outlier_observation_percentage:.6f}%
</li>

<li>
<strong>Unique Frequency Encoding values:</strong>
{unique_fe_values}
</li>

<li>
<strong>Sender names involved in frequency collisions:</strong>
{categories_in_collisions}
</li>

<li>
<strong>Frequency compression ratio:</strong>
{frequency_compression_ratio:.6f}
</li>

<li>
<strong>Shapiro-Wilk observations:</strong>
{total}
</li>

<li>
<strong>Shapiro-Wilk:</strong>
{shapiro_decision}
</li>

<li>
<strong>Jarque-Bera observations:</strong>
{total}
</li>

<li>
<strong>Jarque-Bera:</strong>
{jarque_bera_decision}
</li>

</ul>


</body>

</html>
"""


        # ====================================================
        # 27. SAVE THE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )


        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 28. DISPLAY MAIN RESULTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "SEND_NAME_FEWF SUMMARY"
    )


    print(
        "=" * 100
    )


    print(
        "Total observations:",
        total
    )


    print(
        "Unique sender name categories:",
        unique_categories
    )


    print(
        "Cardinality percentage:",
        f"{cardinality_percentage:.6f}%"
    )


    print(
        "Unique Frequency Encoding values:",
        unique_fe_values
    )


    print(
        "Sender names involved in collisions:",
        categories_in_collisions
    )


    print(
        "Frequency Encoding outlier categories:",
        number_outlier_categories
    )


    # ========================================================
    # 29. DISPLAY NORMALITY TESTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "NORMALITY TESTS AFTER FREQUENCY ENCODING"
    )


    print(
        "=" * 100
    )


    print(
        f"\nSignificance level: {ALPHA}"
    )


    print(
        "\nShapiro-Wilk test"
    )


    print(
        "Number of observations:",
        total
    )


    print(
        "Statistic:",
        f"{shapiro_statistic:.6f}"
    )


    print(
        "p-value:",
        f"{shapiro_p_value:.12g}"
    )


    print(
        "Decision:",
        shapiro_decision
    )


    print(
        "\nJarque-Bera test"
    )


    print(
        "Number of observations:",
        total
    )


    print(
        "Statistic:",
        f"{jarque_bera_statistic:.6f}"
    )


    print(
        "p-value:",
        f"{jarque_bera_p_value:.12g}"
    )


    print(
        "Decision:",
        jarque_bera_decision
    )


    # ========================================================
    # 30. DISPLAY OUTLIER CATEGORIES
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "SENDER NAME CATEGORIES CLASSIFIED AS OUTLIERS"
    )


    print(
        "=" * 100
    )


    if number_outlier_categories > 0:

        display(
            outlier_table
        )


    else:

        print(
            "No sender name categories were classified "
            "as outliers by the IQR rule."
        )


    # ========================================================
    # 31. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature
    del encoded_preview

    gc.collect()


    # ========================================================
    # 32. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "ANALYSIS COMPLETED"
    )


    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )


    print(
        HTML_PATH
    )


    print(
        "\nPNG:"
    )


    print(
        RANK_FREQUENCY_CHART_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Rank-frequency chart: Will be created


/usr/local/lib/python3.14/site-packages/scipy/stats/_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 1852394.
  res = hypotest_fun_out(*samples, **kwds)



Rank-frequency chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/frequency_encoding_with_fallback/send_name_fewf/send_name_fewf_rank_frequency.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/frequency_encoding_with_fallback/send_name_fewf/analysis_send_name_fewf.html

SEND_NAME_FEWF SUMMARY
Total observations: 1852394
Unique sender name categories: 989
Cardinality percentage: 0.053390%
Unique Frequency Encoding values: 149
Sender names involved in collisions: 951
Frequency Encoding outlier categories: 1

NORMALITY TESTS AFTER FREQUENCY ENCODING

Significance level: 0.05

Shapiro-Wilk test
Number of observations: 1852394
Statistic: 0.943326
p-value: 2.42129536868e-142
Decision: Reject H0

Jarque-Bera test
Number of observations: 1852394
Statistic: 23975.876206
p-value: 0
Decision: Reject H0

SENDER NAME CATEGORIES CLASSIFIED AS OUTLIERS


,SENDER_NAME,COUNT,FE_VALUE,OUTLIER_TYPE
0,Scott Martin,6583,0.003554,Upper outlier



ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/frequency_encoding_with_fallback/send_name_fewf

HTML:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/frequency_encoding_with_fallback/send_name_fewf/analysis_send_name_fewf.html

PNG:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/frequency_encoding_with_fallback/send_name_fewf/send_name_fewf_rank_frequency.png


## <span style="color:crimson"> ONE-HOT ENCODING WITH IGNORE </span> ##

### <span style="color:crimson"> RECIVE_CATEGORY_OHEWI </span> ###

In [4]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ENCODING_TYPE = "onehot_encoding_with_ignore"

FEATURE_NAME = "receive_category_ohewi"

FEATURE_COLUMN = "RECEIVE_CATEGORY_OHEWI"


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / ENCODING_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


CATEGORY_DISTRIBUTION_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_category_distribution.png"
)


MATRIX_COMPOSITION_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_matrix_composition.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


category_distribution_chart_exists = (
    CATEGORY_DISTRIBUTION_CHART_PATH.exists()
)


matrix_composition_chart_exists = (
    MATRIX_COMPOSITION_CHART_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and category_distribution_chart_exists
    and matrix_composition_chart_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )

    print(
        "No analysis or file creation is required."
    )

    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )

    print(
        "\nExisting files:"
    )

    print(
        HTML_PATH
    )

    print(
        CATEGORY_DISTRIBUTION_CHART_PATH
    )

    print(
        MATRIX_COMPOSITION_CHART_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )

    print(
        "=" * 100
    )


    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )


    print(
        "Category distribution chart:",
        "Already exists"
        if category_distribution_chart_exists
        else "Will be created"
    )


    print(
        "Matrix composition chart:",
        "Already exists"
        if matrix_composition_chart_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. BASIC VALIDATION
    # ========================================================

    total_observations = int(
        len(feature)
    )


    if total_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    missing_values = int(
        feature
        .isna()
        .sum()
    )


    if missing_values > 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains "
            f"{missing_values} missing values."
        )


    # ========================================================
    # 13. CATEGORY COUNTS
    # ========================================================

    category_counts = (
        feature
        .value_counts()
        .sort_values(
            ascending=False
        )
    )


    unique_categories = int(
        category_counts.shape[0]
    )


    cardinality_percentage = (
        unique_categories
        / total_observations
        * 100
    )


    # ========================================================
    # 14. CATEGORY DISTRIBUTION
    # ========================================================

    category_distribution = (
        category_counts
        .rename_axis(
            "CATEGORY"
        )
        .reset_index(
            name="COUNT"
        )
    )


    category_distribution[
        "PERCENTAGE"
    ] = (
        category_distribution[
            "COUNT"
        ]
        / total_observations
        * 100
    )


    # ========================================================
    # 15. EXPECTED ONE-HOT ENCODING STRUCTURE
    #
    # Assuming:
    #
    # OneHotEncoder(
    #     handle_unknown="ignore",
    #     drop=None
    # )
    #
    # Each known category generates one binary column.
    # ========================================================

    original_feature_count = 1


    expected_ohe_columns = (
        unique_categories
    )


    added_columns = (
        expected_ohe_columns
        - original_feature_count
    )


    dimensional_expansion_factor = (
        expected_ohe_columns
        / original_feature_count
    )


    # ========================================================
    # 16. EXPECTED ONE-HOT MATRIX SIZE
    #
    # The complete dense matrix is not created.
    #
    # The matrix characteristics are calculated analytically
    # to avoid unnecessary memory consumption.
    # ========================================================

    matrix_rows = (
        total_observations
    )


    matrix_columns = (
        expected_ohe_columns
    )


    total_matrix_cells = (
        matrix_rows
        * matrix_columns
    )


    # Every known category activates exactly one OHE column.
    total_ones = (
        total_observations
    )


    total_zeros = (
        total_matrix_cells
        - total_ones
    )


    # ========================================================
    # 17. MATRIX DENSITY AND SPARSITY
    # ========================================================

    if total_matrix_cells > 0:

        matrix_density = (
            total_ones
            / total_matrix_cells
        )


        matrix_sparsity = (
            total_zeros
            / total_matrix_cells
        )


    else:

        matrix_density = 0.0

        matrix_sparsity = 0.0


    density_percentage = (
        matrix_density
        * 100
    )


    sparsity_percentage = (
        matrix_sparsity
        * 100
    )


    # ========================================================
    # 18. ACTIVE COLUMNS PER OBSERVATION
    #
    # For categories known during encoder fitting,
    # exactly one binary column is active.
    # ========================================================

    rows_with_zero_active_columns = 0


    rows_with_one_active_column = (
        total_observations
    )


    rows_with_more_than_one_active_column = 0


    zero_active_percentage = (
        rows_with_zero_active_columns
        / total_observations
        * 100
    )


    one_active_percentage = (
        rows_with_one_active_column
        / total_observations
        * 100
    )


    more_than_one_active_percentage = (
        rows_with_more_than_one_active_column
        / total_observations
        * 100
    )


    # ========================================================
    # 19. HANDLE_UNKNOWN="IGNORE" BEHAVIOR
    #
    # Known category:
    # exactly one active OHE column.
    #
    # Unknown category:
    # all OHE columns are zero.
    #
    # Unknown categories are not measured empirically here
    # because this analysis does not perform a train/test split.
    # ========================================================

    known_category_active_columns = 1


    unknown_category_active_columns = 0


    known_category_representation = (
        "One active binary column"
    )


    unknown_category_representation = (
        "All-zero vector"
    )


    # ========================================================
    # 20. OHE REPRESENTATION UNIQUENESS
    #
    # Each known category receives its own binary vector.
    #
    # Therefore, known categories do not generate
    # representation collisions.
    # ========================================================

    unique_known_ohe_vectors = (
        unique_categories
    )


    known_category_collisions = 0


    known_representation_ratio = (
        unique_known_ohe_vectors
        / unique_categories
        if unique_categories > 0
        else 0.0
    )


    # ========================================================
    # 21. CREATE CATEGORY DISTRIBUTION CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not category_distribution_chart_exists:

        chart_data = (
            category_distribution
            .sort_values(
                "COUNT",
                ascending=True
            )
        )


        fig, ax = plt.subplots(
            figsize=(
                11,
                7
            )
        )


        ax.barh(
            chart_data[
                "CATEGORY"
            ].astype(str),
            chart_data[
                "COUNT"
            ]
        )


        ax.set_title(
            "Distribution of receiver categories"
        )


        ax.set_xlabel(
            "Number of observations"
        )


        ax.set_ylabel(
            "Receiver category"
        )


        ax.xaxis.set_major_formatter(
            FuncFormatter(
                lambda x, pos:
                str(
                    int(x)
                )
            )
        )


        ax.grid(
            axis="x",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            CATEGORY_DISTRIBUTION_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nCategory distribution chart created:"
        )


        print(
            CATEGORY_DISTRIBUTION_CHART_PATH
        )


    else:

        print(
            "\nCategory distribution chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 22. CREATE MATRIX COMPOSITION CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not matrix_composition_chart_exists:

        matrix_labels = [
            "Zeros",
            "Ones"
        ]


        matrix_percentages = [
            sparsity_percentage,
            density_percentage
        ]


        fig, ax = plt.subplots(
            figsize=(
                8,
                6
            )
        )


        ax.bar(
            matrix_labels,
            matrix_percentages
        )


        ax.set_title(
            "Expected One-Hot matrix composition"
        )


        ax.set_xlabel(
            "Binary value"
        )


        ax.set_ylabel(
            "Percentage of matrix cells"
        )


        ax.set_ylim(
            0,
            100
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                f"{y:.0f}%"
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            MATRIX_COMPOSITION_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nMatrix composition chart created:"
        )


        print(
            MATRIX_COMPOSITION_CHART_PATH
        )


    else:

        print(
            "\nMatrix composition chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 23. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 24. PREPARE CATEGORY DISTRIBUTION TABLE FOR HTML
    # ========================================================

    category_distribution_html = (
        category_distribution
        .to_html(
            index=False,
            border=0,
            formatters={
                "PERCENTAGE":
                    lambda x:
                    f"{x:.6f}%"
            }
        )
    )


    # ========================================================
    # 25. CREATE THE HTML REPORT
    #
    # Only if the HTML does not already exist.
    # ========================================================

    if not html_exists:

        category_distribution_chart_base64 = (
            image_to_base64(
                CATEGORY_DISTRIBUTION_CHART_PATH
            )
        )


        matrix_composition_chart_base64 = (
            image_to_base64(
                MATRIX_COMPOSITION_CHART_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Individual Exploratory Analysis — {FEATURE_COLUMN}
</h1>


<p>

The variable <strong>{FEATURE_COLUMN}</strong>
represents the category associated with
the transaction receiver or merchant.

The feature is treated as a
<strong>categorical variable</strong>
and is intended for
<strong>One-Hot Encoding With Ignore</strong>.

</p>


<!-- ========================================================
     1. FEATURE OVERVIEW
========================================================= -->


<h2>
1. Feature overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>Missing values</td>
<td>{missing_values}</td>
</tr>

<tr>
<td>Unique receiver categories</td>
<td>{unique_categories}</td>
</tr>

<tr>
<td>Cardinality percentage</td>
<td>{cardinality_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     2. CATEGORY DISTRIBUTION
========================================================= -->


<h2>
2. Category distribution
</h2>


<p>

The table below presents the absolute
and relative frequency of each receiver
category before One-Hot Encoding.

</p>


{category_distribution_html}


<div class="chart">

<img
    src="data:image/png;base64,{category_distribution_chart_base64}"
    alt="Distribution of receiver categories"
>

</div>


<!-- ========================================================
     3. EXPECTED ONE-HOT ENCODING STRUCTURE
========================================================= -->


<h2>
3. Expected One-Hot Encoding structure
</h2>


<p>

With
<code>handle_unknown="ignore"</code>
and
<code>drop=None</code>,
each category known during fitting generates
one independent binary feature.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Original categorical features</td>
<td>{original_feature_count}</td>
</tr>

<tr>
<td>Known categories</td>
<td>{unique_categories}</td>
</tr>

<tr>
<td>Expected OHE columns</td>
<td>{expected_ohe_columns}</td>
</tr>

<tr>
<td>Additional columns created</td>
<td>{added_columns}</td>
</tr>

<tr>
<td>Dimensional expansion factor</td>
<td>{dimensional_expansion_factor:.6f}</td>
</tr>

</table>


<p class="result">

1 categorical feature
→
{expected_ohe_columns} binary features

</p>


<!-- ========================================================
     4. OHE MATRIX CHARACTERISTICS
========================================================= -->


<h2>
4. Expected OHE matrix characteristics
</h2>


<p>

The expected matrix characteristics are calculated
analytically.

The complete dense One-Hot matrix is not created
in memory.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Matrix rows</td>
<td>{matrix_rows}</td>
</tr>

<tr>
<td>Matrix columns</td>
<td>{matrix_columns}</td>
</tr>

<tr>
<td>Total matrix cells</td>
<td>{total_matrix_cells}</td>
</tr>

<tr>
<td>Expected ones</td>
<td>{total_ones}</td>
</tr>

<tr>
<td>Expected zeros</td>
<td>{total_zeros}</td>
</tr>

<tr>
<td>Density</td>
<td>{density_percentage:.6f}%</td>
</tr>

<tr>
<td>Sparsity</td>
<td>{sparsity_percentage:.6f}%</td>
</tr>

</table>


<div class="chart">

<img
    src="data:image/png;base64,{matrix_composition_chart_base64}"
    alt="Expected One-Hot matrix composition"
>

</div>


<!-- ========================================================
     5. ACTIVE COLUMNS PER OBSERVATION
========================================================= -->


<h2>
5. Active columns per observation
</h2>


<p>

For observations containing categories known
during encoder fitting, exactly one binary
One-Hot column is active.

</p>


<table>

<tr>
<th>Metric</th>
<th>Rows</th>
<th>Percentage</th>
</tr>

<tr>
<td>0 active columns</td>
<td>{rows_with_zero_active_columns}</td>
<td>{zero_active_percentage:.6f}%</td>
</tr>

<tr>
<td>1 active column</td>
<td>{rows_with_one_active_column}</td>
<td>{one_active_percentage:.6f}%</td>
</tr>

<tr>
<td>More than 1 active column</td>
<td>{rows_with_more_than_one_active_column}</td>
<td>{more_than_one_active_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     6. UNKNOWN-CATEGORY BEHAVIOR
========================================================= -->


<h2>
6. Unknown-category behavior
</h2>


<p>

The encoding strategy uses:

</p>


<p class="result">

handle_unknown="ignore"

</p>


<p>

A category known during encoder fitting activates
exactly one of the existing binary features.

</p>


<p>

A category not observed during encoder fitting
does not generate a new feature.

Instead, all One-Hot columns associated with this
original variable receive zero.

</p>


<table>

<tr>
<th>Category type</th>
<th>Active columns</th>
<th>Expected representation</th>
</tr>

<tr>
<td>Known category</td>
<td>{known_category_active_columns}</td>
<td>{known_category_representation}</td>
</tr>

<tr>
<td>Unknown category</td>
<td>{unknown_category_active_columns}</td>
<td>{unknown_category_representation}</td>
</tr>

</table>


<div class="note">

<strong>Important:</strong>

<br><br>

The unknown-category rate cannot be empirically
calculated in this individual full-dataset analysis.

To measure it correctly, the encoder must later
be fitted only on the training data and then
applied to separate test data.

Categories present only in the test data will
then be represented by an all-zero vector.

</div>


<!-- ========================================================
     7. OHE REPRESENTATION UNIQUENESS
========================================================= -->


<h2>
7. OHE representation uniqueness
</h2>


<p>

Unlike Frequency Encoding,
One-Hot Encoding preserves a unique representation
for every category known during encoder fitting.

Each known category activates a different
binary column.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Known categories</td>
<td>{unique_categories}</td>
</tr>

<tr>
<td>Unique known OHE vectors</td>
<td>{unique_known_ohe_vectors}</td>
</tr>

<tr>
<td>Known-category collisions</td>
<td>{known_category_collisions}</td>
</tr>

<tr>
<td>Known representation ratio</td>
<td>{known_representation_ratio:.6f}</td>
</tr>

</table>


<p>

Different unknown categories would all receive
the same all-zero representation when
<code>handle_unknown="ignore"</code>
is applied.

</p>


<!-- ========================================================
     8. SUMMARY
========================================================= -->


<h2>
8. Summary of results
</h2>


<ul>

<li>
<strong>Total observations:</strong>
{total_observations}
</li>

<li>
<strong>Missing values:</strong>
{missing_values}
</li>

<li>
<strong>Unique receiver categories:</strong>
{unique_categories}
</li>

<li>
<strong>Cardinality percentage:</strong>
{cardinality_percentage:.6f}%
</li>

<li>
<strong>Expected OHE columns:</strong>
{expected_ohe_columns}
</li>

<li>
<strong>Dimensional expansion:</strong>
1 → {expected_ohe_columns}
</li>

<li>
<strong>Expected matrix density:</strong>
{density_percentage:.6f}%
</li>

<li>
<strong>Expected matrix sparsity:</strong>
{sparsity_percentage:.6f}%
</li>

<li>
<strong>Rows with exactly one active column:</strong>
{rows_with_one_active_column}
</li>

<li>
<strong>Known-category collisions:</strong>
{known_category_collisions}
</li>

<li>
<strong>Unknown-category representation:</strong>
All-zero vector
</li>

</ul>


</body>

</html>
"""


        # ====================================================
        # 26. SAVE THE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )


        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 27. DISPLAY MAIN RESULTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "RECEIVE_CATEGORY_OHEWI SUMMARY"
    )


    print(
        "=" * 100
    )


    print(
        "Total observations:",
        total_observations
    )


    print(
        "Missing values:",
        missing_values
    )


    print(
        "Unique receiver categories:",
        unique_categories
    )


    print(
        "Cardinality percentage:",
        f"{cardinality_percentage:.6f}%"
    )


    print(
        "Expected OHE columns:",
        expected_ohe_columns
    )


    print(
        "Expected matrix density:",
        f"{density_percentage:.6f}%"
    )


    print(
        "Expected matrix sparsity:",
        f"{sparsity_percentage:.6f}%"
    )


    # ========================================================
    # 28. DISPLAY CATEGORY DISTRIBUTION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "CATEGORY DISTRIBUTION"
    )


    print(
        "=" * 100
    )


    display(
        category_distribution
    )


    # ========================================================
    # 29. DISPLAY OHE STRUCTURE
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "EXPECTED ONE-HOT ENCODING STRUCTURE"
    )


    print(
        "=" * 100
    )


    print(
        "Original feature count:",
        original_feature_count
    )


    print(
        "Known categories:",
        unique_categories
    )


    print(
        "Expected OHE columns:",
        expected_ohe_columns
    )


    print(
        "Additional columns created:",
        added_columns
    )


    print(
        "Dimensional expansion factor:",
        f"{dimensional_expansion_factor:.6f}"
    )


    print(
        "Total matrix cells:",
        total_matrix_cells
    )


    print(
        "Expected ones:",
        total_ones
    )


    print(
        "Expected zeros:",
        total_zeros
    )


    print(
        "Density:",
        f"{density_percentage:.6f}%"
    )


    print(
        "Sparsity:",
        f"{sparsity_percentage:.6f}%"
    )


    # ========================================================
    # 30. DISPLAY UNKNOWN-CATEGORY BEHAVIOR
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        'HANDLE_UNKNOWN="IGNORE" BEHAVIOR'
    )


    print(
        "=" * 100
    )


    print(
        "Known category:"
    )


    print(
        "Exactly one active OHE column."
    )


    print(
        "\nUnknown category:"
    )


    print(
        "All OHE columns are zero."
    )


    print(
        "\nUnknown-category rate cannot be empirically "
        "calculated without fitting the encoder on training "
        "data and transforming separate test data."
    )


    # ========================================================
    # 31. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature

    gc.collect()


    # ========================================================
    # 32. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "ANALYSIS COMPLETED"
    )


    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )


    print(
        HTML_PATH
    )


    print(
        "\nPNGs:"
    )


    print(
        CATEGORY_DISTRIBUTION_CHART_PATH
    )


    print(
        MATRIX_COMPOSITION_CHART_PATH
    )

All analysis files already exist.
No analysis or file creation is required.

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/onehot_encoding_with_ignore/receive_category_ohewi

Existing files:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/onehot_encoding_with_ignore/receive_category_ohewi/analysis_receive_category_ohewi.html
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/onehot_encoding_with_ignore/receive_category_ohewi/receive_category_ohewi_category_distribution.png
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/onehot_encoding_with_ignore/receive_category_ohewi/receive_category_ohewi_matrix_composition.png


### <span style="color:crimson"> TRANS_WEEK_OHEWI </span> ###

In [5]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ENCODING_TYPE = "onehot_encoding_with_ignore"

FEATURE_NAME = "trans_week_ohewi"

FEATURE_COLUMN = "TRANS_WEEK_OHEWI"


# ============================================================
# 02. EXPECTED WEEKDAY CATEGORIES
# ============================================================

EXPECTED_WEEKDAYS = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]


# ============================================================
# 03. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 04. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 05. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / ENCODING_TYPE
    / FEATURE_NAME
)


# ============================================================
# 06. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 07. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


CATEGORY_DISTRIBUTION_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_category_distribution.png"
)


MATRIX_COMPOSITION_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_matrix_composition.png"
)


# ============================================================
# 08. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


category_distribution_chart_exists = (
    CATEGORY_DISTRIBUTION_CHART_PATH.exists()
)


matrix_composition_chart_exists = (
    MATRIX_COMPOSITION_CHART_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and category_distribution_chart_exists
    and matrix_composition_chart_exists
)


# ============================================================
# 09. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )

    print(
        "No analysis or file creation is required."
    )

    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )

    print(
        "\nExisting files:"
    )

    print(
        HTML_PATH
    )

    print(
        CATEGORY_DISTRIBUTION_CHART_PATH
    )

    print(
        MATRIX_COMPOSITION_CHART_PATH
    )


else:

    # ========================================================
    # 10. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 11. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )

    print(
        "=" * 100
    )


    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )


    print(
        "Category distribution chart:",
        "Already exists"
        if category_distribution_chart_exists
        else "Will be created"
    )


    print(
        "Matrix composition chart:",
        "Already exists"
        if matrix_composition_chart_exists
        else "Will be created"
    )


    # ========================================================
    # 12. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 13. BASIC VALIDATION
    # ========================================================

    total_observations = int(
        len(feature)
    )


    if total_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    missing_values = int(
        feature
        .isna()
        .sum()
    )


    if missing_values > 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains "
            f"{missing_values} missing values."
        )


    # ========================================================
    # 14. IDENTIFY OBSERVED CATEGORIES
    # ========================================================

    observed_categories = (
        feature
        .astype(str)
        .unique()
        .tolist()
    )


    unique_categories = int(
        feature
        .nunique()
    )


    unexpected_categories = sorted(
        set(
            observed_categories
        )
        - set(
            EXPECTED_WEEKDAYS
        )
    )


    missing_expected_categories = [
        weekday
        for weekday in EXPECTED_WEEKDAYS
        if weekday not in observed_categories
    ]


    # ========================================================
    # 15. WEEKDAY VALIDATION
    # ========================================================

    if unexpected_categories:

        raise ValueError(
            "Unexpected categories were found in "
            f"{FEATURE_COLUMN}:\n"
            + "\n".join(
                unexpected_categories
            )
        )


    # ========================================================
    # 16. CARDINALITY
    # ========================================================

    cardinality_percentage = (
        unique_categories
        / total_observations
        * 100
    )


    # ========================================================
    # 17. CATEGORY DISTRIBUTION
    #
    # The natural chronological weekday order is preserved.
    # ========================================================

    category_counts = (
        feature
        .value_counts()
        .reindex(
            EXPECTED_WEEKDAYS,
            fill_value=0
        )
    )


    category_distribution = (
        category_counts
        .rename_axis(
            "WEEKDAY"
        )
        .reset_index(
            name="COUNT"
        )
    )


    category_distribution[
        "PERCENTAGE"
    ] = (
        category_distribution[
            "COUNT"
        ]
        / total_observations
        * 100
    )


    # ========================================================
    # 18. EXPECTED ONE-HOT ENCODING STRUCTURE
    #
    # Assuming:
    #
    # OneHotEncoder(
    #     handle_unknown="ignore",
    #     drop=None
    # )
    #
    # Each known weekday generates one binary column.
    # ========================================================

    original_feature_count = 1


    expected_ohe_columns = (
        unique_categories
    )


    added_columns = (
        expected_ohe_columns
        - original_feature_count
    )


    dimensional_expansion_factor = (
        expected_ohe_columns
        / original_feature_count
    )


    # ========================================================
    # 19. EXPECTED ONE-HOT MATRIX SIZE
    #
    # The complete dense matrix is not created.
    #
    # Matrix characteristics are calculated analytically.
    # ========================================================

    matrix_rows = (
        total_observations
    )


    matrix_columns = (
        expected_ohe_columns
    )


    total_matrix_cells = (
        matrix_rows
        * matrix_columns
    )


    # Every known weekday activates exactly one OHE column.
    total_ones = (
        total_observations
    )


    total_zeros = (
        total_matrix_cells
        - total_ones
    )


    # ========================================================
    # 20. MATRIX DENSITY AND SPARSITY
    # ========================================================

    if total_matrix_cells > 0:

        matrix_density = (
            total_ones
            / total_matrix_cells
        )


        matrix_sparsity = (
            total_zeros
            / total_matrix_cells
        )


    else:

        matrix_density = 0.0

        matrix_sparsity = 0.0


    density_percentage = (
        matrix_density
        * 100
    )


    sparsity_percentage = (
        matrix_sparsity
        * 100
    )


    # ========================================================
    # 21. ACTIVE COLUMNS PER OBSERVATION
    #
    # Every known weekday activates exactly one OHE column.
    # ========================================================

    rows_with_zero_active_columns = 0


    rows_with_one_active_column = (
        total_observations
    )


    rows_with_more_than_one_active_column = 0


    zero_active_percentage = (
        rows_with_zero_active_columns
        / total_observations
        * 100
    )


    one_active_percentage = (
        rows_with_one_active_column
        / total_observations
        * 100
    )


    more_than_one_active_percentage = (
        rows_with_more_than_one_active_column
        / total_observations
        * 100
    )


    # ========================================================
    # 22. HANDLE_UNKNOWN="IGNORE" BEHAVIOR
    #
    # Known weekday:
    # exactly one active OHE column.
    #
    # Unknown weekday/category:
    # all OHE columns are zero.
    #
    # Unknown categories are not measured empirically here
    # because this analysis does not perform a train/test split.
    # ========================================================

    known_category_active_columns = 1


    unknown_category_active_columns = 0


    known_category_representation = (
        "One active binary column"
    )


    unknown_category_representation = (
        "All-zero vector"
    )


    # ========================================================
    # 23. OHE REPRESENTATION UNIQUENESS
    #
    # Every known weekday receives its own binary vector.
    # ========================================================

    unique_known_ohe_vectors = (
        unique_categories
    )


    known_category_collisions = 0


    known_representation_ratio = (
        unique_known_ohe_vectors
        / unique_categories
        if unique_categories > 0
        else 0.0
    )


    # ========================================================
    # 24. CREATE CATEGORY DISTRIBUTION CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not category_distribution_chart_exists:

        fig, ax = plt.subplots(
            figsize=(
                11,
                6
            )
        )


        ax.bar(
            category_distribution[
                "WEEKDAY"
            ],
            category_distribution[
                "COUNT"
            ]
        )


        ax.set_title(
            "Distribution of transactions by weekday"
        )


        ax.set_xlabel(
            "Weekday"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(y)
                )
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            CATEGORY_DISTRIBUTION_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nCategory distribution chart created:"
        )


        print(
            CATEGORY_DISTRIBUTION_CHART_PATH
        )


    else:

        print(
            "\nCategory distribution chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 25. CREATE MATRIX COMPOSITION CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not matrix_composition_chart_exists:

        matrix_labels = [
            "Zeros",
            "Ones"
        ]


        matrix_percentages = [
            sparsity_percentage,
            density_percentage
        ]


        fig, ax = plt.subplots(
            figsize=(
                8,
                6
            )
        )


        ax.bar(
            matrix_labels,
            matrix_percentages
        )


        ax.set_title(
            "Expected One-Hot matrix composition"
        )


        ax.set_xlabel(
            "Binary value"
        )


        ax.set_ylabel(
            "Percentage of matrix cells"
        )


        ax.set_ylim(
            0,
            100
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                f"{y:.0f}%"
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            MATRIX_COMPOSITION_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nMatrix composition chart created:"
        )


        print(
            MATRIX_COMPOSITION_CHART_PATH
        )


    else:

        print(
            "\nMatrix composition chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 26. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 27. PREPARE CATEGORY DISTRIBUTION TABLE FOR HTML
    # ========================================================

    category_distribution_html = (
        category_distribution
        .to_html(
            index=False,
            border=0,
            formatters={
                "PERCENTAGE":
                    lambda x:
                    f"{x:.6f}%"
            }
        )
    )


    # ========================================================
    # 28. PREPARE WEEKDAY VALIDATION INFORMATION
    # ========================================================

    if len(
        missing_expected_categories
    ) == 0:

        expected_category_status = (
            "All seven expected weekday categories are present."
        )


    else:

        expected_category_status = (
            "The following expected weekday categories "
            "were not observed: "
            + ", ".join(
                missing_expected_categories
            )
        )


    # ========================================================
    # 29. CREATE THE HTML REPORT
    #
    # Only if the HTML does not already exist.
    # ========================================================

    if not html_exists:

        category_distribution_chart_base64 = (
            image_to_base64(
                CATEGORY_DISTRIBUTION_CHART_PATH
            )
        )


        matrix_composition_chart_base64 = (
            image_to_base64(
                MATRIX_COMPOSITION_CHART_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Individual Exploratory Analysis — {FEATURE_COLUMN}
</h1>


<p>

The variable <strong>{FEATURE_COLUMN}</strong>
represents the weekday on which each transaction occurred.

The feature is treated as a
<strong>categorical temporal variable</strong>
and is intended for
<strong>One-Hot Encoding With Ignore</strong>.

</p>


<!-- ========================================================
     1. FEATURE OVERVIEW
========================================================= -->


<h2>
1. Feature overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>Missing values</td>
<td>{missing_values}</td>
</tr>

<tr>
<td>Unique weekday categories</td>
<td>{unique_categories}</td>
</tr>

<tr>
<td>Cardinality percentage</td>
<td>{cardinality_percentage:.6f}%</td>
</tr>

</table>


<div class="note">

<strong>Weekday validation:</strong>

<br><br>

{expected_category_status}

</div>


<!-- ========================================================
     2. CATEGORY DISTRIBUTION
========================================================= -->


<h2>
2. Weekday distribution
</h2>


<p>

The table below presents the absolute and
relative frequency of transactions for each weekday.

The natural chronological weekday order is preserved.

</p>


{category_distribution_html}


<div class="chart">

<img
    src="data:image/png;base64,{category_distribution_chart_base64}"
    alt="Distribution of transactions by weekday"
>

</div>


<!-- ========================================================
     3. EXPECTED ONE-HOT ENCODING STRUCTURE
========================================================= -->


<h2>
3. Expected One-Hot Encoding structure
</h2>


<p>

With
<code>handle_unknown="ignore"</code>
and
<code>drop=None</code>,
each weekday category known during encoder fitting
generates one independent binary feature.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Original categorical features</td>
<td>{original_feature_count}</td>
</tr>

<tr>
<td>Known weekday categories</td>
<td>{unique_categories}</td>
</tr>

<tr>
<td>Expected OHE columns</td>
<td>{expected_ohe_columns}</td>
</tr>

<tr>
<td>Additional columns created</td>
<td>{added_columns}</td>
</tr>

<tr>
<td>Dimensional expansion factor</td>
<td>{dimensional_expansion_factor:.6f}</td>
</tr>

</table>


<p class="result">

1 categorical feature
→
{expected_ohe_columns} binary features

</p>


<!-- ========================================================
     4. OHE MATRIX CHARACTERISTICS
========================================================= -->


<h2>
4. Expected OHE matrix characteristics
</h2>


<p>

The expected matrix characteristics are calculated
analytically.

The complete dense One-Hot matrix is not created
in memory.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Matrix rows</td>
<td>{matrix_rows}</td>
</tr>

<tr>
<td>Matrix columns</td>
<td>{matrix_columns}</td>
</tr>

<tr>
<td>Total matrix cells</td>
<td>{total_matrix_cells}</td>
</tr>

<tr>
<td>Expected ones</td>
<td>{total_ones}</td>
</tr>

<tr>
<td>Expected zeros</td>
<td>{total_zeros}</td>
</tr>

<tr>
<td>Density</td>
<td>{density_percentage:.6f}%</td>
</tr>

<tr>
<td>Sparsity</td>
<td>{sparsity_percentage:.6f}%</td>
</tr>

</table>


<div class="chart">

<img
    src="data:image/png;base64,{matrix_composition_chart_base64}"
    alt="Expected One-Hot matrix composition"
>

</div>


<!-- ========================================================
     5. ACTIVE COLUMNS PER OBSERVATION
========================================================= -->


<h2>
5. Active columns per observation
</h2>


<p>

For every weekday category known during
encoder fitting, exactly one One-Hot column
is active in each observation.

</p>


<table>

<tr>
<th>Metric</th>
<th>Rows</th>
<th>Percentage</th>
</tr>

<tr>
<td>0 active columns</td>
<td>{rows_with_zero_active_columns}</td>
<td>{zero_active_percentage:.6f}%</td>
</tr>

<tr>
<td>1 active column</td>
<td>{rows_with_one_active_column}</td>
<td>{one_active_percentage:.6f}%</td>
</tr>

<tr>
<td>More than 1 active column</td>
<td>{rows_with_more_than_one_active_column}</td>
<td>{more_than_one_active_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     6. UNKNOWN-CATEGORY BEHAVIOR
========================================================= -->


<h2>
6. Unknown-category behavior
</h2>


<p>

The encoding strategy uses:

</p>


<p class="result">

handle_unknown="ignore"

</p>


<p>

A weekday category known during encoder fitting
activates exactly one binary feature.

</p>


<p>

An unknown category does not create a new feature.

Instead, all One-Hot columns associated with
the weekday variable receive zero.

</p>


<table>

<tr>
<th>Category type</th>
<th>Active columns</th>
<th>Expected representation</th>
</tr>

<tr>
<td>Known weekday</td>
<td>{known_category_active_columns}</td>
<td>{known_category_representation}</td>
</tr>

<tr>
<td>Unknown category</td>
<td>{unknown_category_active_columns}</td>
<td>{unknown_category_representation}</td>
</tr>

</table>


<div class="note">

<strong>Important:</strong>

<br><br>

The unknown-category rate cannot be empirically
calculated in this individual full-dataset analysis.

To measure it correctly, the encoder must later
be fitted only on the training data and then
applied to separate test data.

A category present only in the test data would
be represented by an all-zero vector.

</div>


<!-- ========================================================
     7. OHE REPRESENTATION UNIQUENESS
========================================================= -->


<h2>
7. OHE representation uniqueness
</h2>


<p>

Each known weekday receives its own
exclusive binary representation.

Therefore, different known weekday categories
do not collide after One-Hot Encoding.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Known weekday categories</td>
<td>{unique_categories}</td>
</tr>

<tr>
<td>Unique known OHE vectors</td>
<td>{unique_known_ohe_vectors}</td>
</tr>

<tr>
<td>Known-category collisions</td>
<td>{known_category_collisions}</td>
</tr>

<tr>
<td>Known representation ratio</td>
<td>{known_representation_ratio:.6f}</td>
</tr>

</table>


<p>

Different unknown categories would all receive
the same all-zero representation when
<code>handle_unknown="ignore"</code>
is applied.

</p>


<!-- ========================================================
     8. SUMMARY
========================================================= -->


<h2>
8. Summary of results
</h2>


<ul>

<li>
<strong>Total observations:</strong>
{total_observations}
</li>

<li>
<strong>Missing values:</strong>
{missing_values}
</li>

<li>
<strong>Unique weekday categories:</strong>
{unique_categories}
</li>

<li>
<strong>Expected weekday categories:</strong>
7
</li>

<li>
<strong>Cardinality percentage:</strong>
{cardinality_percentage:.6f}%
</li>

<li>
<strong>Expected OHE columns:</strong>
{expected_ohe_columns}
</li>

<li>
<strong>Dimensional expansion:</strong>
1 → {expected_ohe_columns}
</li>

<li>
<strong>Expected matrix density:</strong>
{density_percentage:.6f}%
</li>

<li>
<strong>Expected matrix sparsity:</strong>
{sparsity_percentage:.6f}%
</li>

<li>
<strong>Rows with exactly one active column:</strong>
{rows_with_one_active_column}
</li>

<li>
<strong>Known-category collisions:</strong>
{known_category_collisions}
</li>

<li>
<strong>Unknown-category representation:</strong>
All-zero vector
</li>

</ul>


</body>

</html>
"""


        # ====================================================
        # 30. SAVE THE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )


        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 31. DISPLAY MAIN RESULTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "TRANS_WEEK_OHEWI SUMMARY"
    )


    print(
        "=" * 100
    )


    print(
        "Total observations:",
        total_observations
    )


    print(
        "Missing values:",
        missing_values
    )


    print(
        "Unique weekday categories:",
        unique_categories
    )


    print(
        "Cardinality percentage:",
        f"{cardinality_percentage:.6f}%"
    )


    print(
        "Expected OHE columns:",
        expected_ohe_columns
    )


    print(
        "Expected matrix density:",
        f"{density_percentage:.6f}%"
    )


    print(
        "Expected matrix sparsity:",
        f"{sparsity_percentage:.6f}%"
    )


    print(
        "Weekday validation:",
        expected_category_status
    )


    # ========================================================
    # 32. DISPLAY WEEKDAY DISTRIBUTION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "WEEKDAY DISTRIBUTION"
    )


    print(
        "=" * 100
    )


    display(
        category_distribution
    )


    # ========================================================
    # 33. DISPLAY OHE STRUCTURE
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "EXPECTED ONE-HOT ENCODING STRUCTURE"
    )


    print(
        "=" * 100
    )


    print(
        "Original feature count:",
        original_feature_count
    )


    print(
        "Known weekday categories:",
        unique_categories
    )


    print(
        "Expected OHE columns:",
        expected_ohe_columns
    )


    print(
        "Additional columns created:",
        added_columns
    )


    print(
        "Dimensional expansion factor:",
        f"{dimensional_expansion_factor:.6f}"
    )


    print(
        "Total matrix cells:",
        total_matrix_cells
    )


    print(
        "Expected ones:",
        total_ones
    )


    print(
        "Expected zeros:",
        total_zeros
    )


    print(
        "Density:",
        f"{density_percentage:.6f}%"
    )


    print(
        "Sparsity:",
        f"{sparsity_percentage:.6f}%"
    )


    # ========================================================
    # 34. DISPLAY UNKNOWN-CATEGORY BEHAVIOR
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        'HANDLE_UNKNOWN="IGNORE" BEHAVIOR'
    )


    print(
        "=" * 100
    )


    print(
        "Known weekday:"
    )


    print(
        "Exactly one active OHE column."
    )


    print(
        "\nUnknown category:"
    )


    print(
        "All OHE columns are zero."
    )


    print(
        "\nUnknown-category rate cannot be empirically "
        "calculated without fitting the encoder on training "
        "data and transforming separate test data."
    )


    # ========================================================
    # 35. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature

    gc.collect()


    # ========================================================
    # 36. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "ANALYSIS COMPLETED"
    )


    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )


    print(
        HTML_PATH
    )


    print(
        "\nPNGs:"
    )


    print(
        CATEGORY_DISTRIBUTION_CHART_PATH
    )


    print(
        MATRIX_COMPOSITION_CHART_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Category distribution chart: Will be created
Matrix composition chart: Will be created

Category distribution chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/onehot_encoding_with_ignore/trans_week_ohewi/trans_week_ohewi_category_distribution.png

Matrix composition chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/onehot_encoding_with_ignore/trans_week_ohewi/trans_week_ohewi_matrix_composition.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/onehot_encoding_with_ignore/trans_week_ohewi/analysis_trans_week_ohewi.html

TRANS_WEEK_OHEWI SUMMARY
Total observations: 1852394
Missing values: 0
Unique weekday categories: 7
Cardinality percentage: 0.000378%
Expected OHE columns: 7
Expected matrix density: 14.285714%
Expected matrix sparsity: 85.714286%
Weekday validation: All seven expected weekday categories are present.

WEEKDAY DI

,WEEKDAY,COUNT,PERCENTAGE
0,Monday,369418,19.942734
1,Tuesday,270340,14.594087
2,Wednesday,183913,9.928395
3,Thursday,206741,11.160747
4,Friday,215078,11.610813
5,Saturday,263227,14.210098
6,Sunday,343677,18.553126



EXPECTED ONE-HOT ENCODING STRUCTURE
Original feature count: 1
Known weekday categories: 7
Expected OHE columns: 7
Additional columns created: 6
Dimensional expansion factor: 7.000000
Total matrix cells: 12966758
Expected ones: 1852394
Expected zeros: 11114364
Density: 14.285714%
Sparsity: 85.714286%

HANDLE_UNKNOWN="IGNORE" BEHAVIOR
Known weekday:
Exactly one active OHE column.

Unknown category:
All OHE columns are zero.

Unknown-category rate cannot be empirically calculated without fitting the encoder on training data and transforming separate test data.

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/onehot_encoding_with_ignore/trans_week_ohewi

HTML:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/onehot_encoding_with_ignore/trans_week_ohewi/analysis_trans_week_ohewi.html

PNGs:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/onehot_encoding_with_ignore/trans_week_ohewi/tr

## <span style="color:olive"> CYCLICAL ENCODING USING SINE AND COSINE </span> ##

### <span style="color:olive"> TRANS_MONTH_SEN + TRANS_MONTH_COS </span> ###

In [2]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ENCODING_TYPE = "cyclical_encoding_using_sine_and_cosine"

FEATURE_NAME = "trans_month_sin_cos"

SIN_COLUMN = "TRANS_MONTH_SIN"

COS_COLUMN = "TRANS_MONTH_COS"

PERIOD = 12

UNIT_CIRCLE_TOLERANCE = 1e-6

ALIGNMENT_TOLERANCE = 1e-6


# ============================================================
# 02. MONTH SETTINGS
# ============================================================

MONTH_NUMBERS = list(
    range(
        1,
        13
    )
)


MONTH_NAMES = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}


MONTH_ABBREVIATIONS = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun",
    7: "Jul",
    8: "Aug",
    9: "Sep",
    10: "Oct",
    11: "Nov",
    12: "Dec"
}


# ============================================================
# 03. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 04. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 05. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / ENCODING_TYPE
    / FEATURE_NAME
)


# ============================================================
# 06. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 07. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


MONTH_DISTRIBUTION_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_month_distribution.png"
)


UNIT_CIRCLE_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_unit_circle.png"
)


CONSECUTIVE_DISTANCE_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_consecutive_distance.png"
)


# ============================================================
# 08. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


month_distribution_chart_exists = (
    MONTH_DISTRIBUTION_CHART_PATH.exists()
)


unit_circle_chart_exists = (
    UNIT_CIRCLE_CHART_PATH.exists()
)


consecutive_distance_chart_exists = (
    CONSECUTIVE_DISTANCE_CHART_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and month_distribution_chart_exists
    and unit_circle_chart_exists
    and consecutive_distance_chart_exists
)


# ============================================================
# 09. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )

    print(
        "No analysis or file creation is required."
    )

    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )

    print(
        "\nExisting files:"
    )

    print(
        HTML_PATH
    )

    print(
        MONTH_DISTRIBUTION_CHART_PATH
    )

    print(
        UNIT_CIRCLE_CHART_PATH
    )

    print(
        CONSECUTIVE_DISTANCE_CHART_PATH
    )


else:

    # ========================================================
    # 10. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 11. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )

    print(
        "=" * 100
    )


    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )


    print(
        "Month distribution chart:",
        "Already exists"
        if month_distribution_chart_exists
        else "Will be created"
    )


    print(
        "Unit-circle chart:",
        "Already exists"
        if unit_circle_chart_exists
        else "Will be created"
    )


    print(
        "Consecutive-distance chart:",
        "Already exists"
        if consecutive_distance_chart_exists
        else "Will be created"
    )


    # ========================================================
    # 12. LOAD ONLY THE TWO FEATURES BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            SIN_COLUMN,
            COS_COLUMN
        ]
    )


    sin_feature = (
        dataset_feature[
            SIN_COLUMN
        ]
    )


    cos_feature = (
        dataset_feature[
            COS_COLUMN
        ]
    )


    # ========================================================
    # 13. BASIC VALIDATION
    # ========================================================

    total_observations = int(
        len(
            dataset_feature
        )
    )


    if total_observations == 0:

        raise ValueError(
            "The selected cyclical features "
            "contain no observations."
        )


    sin_missing_values = int(
        sin_feature
        .isna()
        .sum()
    )


    cos_missing_values = int(
        cos_feature
        .isna()
        .sum()
    )


    pair_missing_values = int(
        (
            sin_feature.isna()
            |
            cos_feature.isna()
        )
        .sum()
    )


    valid_mask = (
        sin_feature.notna()
        &
        cos_feature.notna()
    )


    valid_observations = int(
        valid_mask.sum()
    )


    if valid_observations == 0:

        raise ValueError(
            "There are no complete SIN/COS pairs "
            "available for analysis."
        )


    sin_valid = (
        sin_feature[
            valid_mask
        ]
        .astype(
            "float64"
        )
    )


    cos_valid = (
        cos_feature[
            valid_mask
        ]
        .astype(
            "float64"
        )
    )


    # ========================================================
    # 14. UNIQUE VALUES AND UNIQUE PAIRS
    # ========================================================

    unique_sin_values = int(
        sin_valid.nunique()
    )


    unique_cos_values = int(
        cos_valid.nunique()
    )


    unique_pairs = int(
        pd.DataFrame({
            "SIN": sin_valid.values,
            "COS": cos_valid.values
        })
        .drop_duplicates()
        .shape[0]
    )


    # ========================================================
    # 15. DESCRIPTIVE STATISTICS
    # ========================================================

    component_statistics = pd.DataFrame({

        "STATISTIC": [
            "Minimum",
            "Q1",
            "Median",
            "Mean",
            "Q3",
            "Maximum",
            "Standard deviation"
        ],

        "SIN": [
            float(
                sin_valid.min()
            ),
            float(
                sin_valid.quantile(
                    0.25
                )
            ),
            float(
                sin_valid.median()
            ),
            float(
                sin_valid.mean()
            ),
            float(
                sin_valid.quantile(
                    0.75
                )
            ),
            float(
                sin_valid.max()
            ),
            float(
                sin_valid.std()
            )
        ],

        "COS": [
            float(
                cos_valid.min()
            ),
            float(
                cos_valid.quantile(
                    0.25
                )
            ),
            float(
                cos_valid.median()
            ),
            float(
                cos_valid.mean()
            ),
            float(
                cos_valid.quantile(
                    0.75
                )
            ),
            float(
                cos_valid.max()
            ),
            float(
                cos_valid.std()
            )
        ]
    })


    # ========================================================
    # 16. RANGE VALIDATION
    #
    # Both sine and cosine must remain inside [-1, 1].
    # ========================================================

    sin_below_minus_one = int(
        (
            sin_valid < -1
        )
        .sum()
    )


    sin_above_one = int(
        (
            sin_valid > 1
        )
        .sum()
    )


    cos_below_minus_one = int(
        (
            cos_valid < -1
        )
        .sum()
    )


    cos_above_one = int(
        (
            cos_valid > 1
        )
        .sum()
    )


    sin_outside_range = (
        sin_below_minus_one
        + sin_above_one
    )


    cos_outside_range = (
        cos_below_minus_one
        + cos_above_one
    )


    # ========================================================
    # 17. UNIT-CIRCLE VALIDATION
    #
    # For a valid cyclical encoding:
    #
    # SIN² + COS² ≈ 1
    # ========================================================

    radius_squared = (
        sin_valid.pow(2)
        +
        cos_valid.pow(2)
    )


    radius_squared_minimum = float(
        radius_squared.min()
    )


    radius_squared_mean = float(
        radius_squared.mean()
    )


    radius_squared_maximum = float(
        radius_squared.max()
    )


    radius_squared_error = (
        radius_squared
        - 1.0
    ).abs()


    mean_absolute_radius_squared_error = float(
        radius_squared_error.mean()
    )


    maximum_absolute_radius_squared_error = float(
        radius_squared_error.max()
    )


    observations_outside_unit_circle_tolerance = int(
        (
            radius_squared_error
            > UNIT_CIRCLE_TOLERANCE
        )
        .sum()
    )


    unit_circle_valid_percentage = (
        (
            valid_observations
            - observations_outside_unit_circle_tolerance
        )
        / valid_observations
        * 100
    )


    # ========================================================
    # 18. RECONSTRUCT ANGLES
    #
    # atan2(SIN, COS) reconstructs the cyclical angle.
    # ========================================================

    angles = np.mod(
        np.arctan2(
            sin_valid.to_numpy(),
            cos_valid.to_numpy()
        ),
        2 * np.pi
    )


    angles_degrees = (
        np.degrees(
            angles
        )
    )


    # ========================================================
    # 19. RECONSTRUCT MONTH CATEGORIES
    #
    # Encoding assumed:
    #
    # angle = 2π × month / 12
    #
    # Month 12 corresponds to the end/start of the cycle.
    # ========================================================

    reconstructed_raw = np.rint(
        angles
        * PERIOD
        / (
            2 * np.pi
        )
    ).astype(int)


    reconstructed_month = np.where(
        reconstructed_raw == 0,
        PERIOD,
        reconstructed_raw
    )


    reconstructed_month = np.where(
        reconstructed_month > PERIOD,
        PERIOD,
        reconstructed_month
    )


    reconstructed_month = np.where(
        reconstructed_month < 1,
        1,
        reconstructed_month
    )


    # ========================================================
    # 20. EXPECTED COORDINATES FOR RECONSTRUCTED MONTH
    # ========================================================

    expected_angles = (
        2
        * np.pi
        * reconstructed_month
        / PERIOD
    )


    expected_sin = np.sin(
        expected_angles
    )


    expected_cos = np.cos(
        expected_angles
    )


    # ========================================================
    # 21. ENCODING ALIGNMENT ERROR
    #
    # Euclidean distance between each stored pair
    # and its theoretical month coordinate.
    # ========================================================

    alignment_error = np.sqrt(
        (
            sin_valid.to_numpy()
            - expected_sin
        ) ** 2
        +
        (
            cos_valid.to_numpy()
            - expected_cos
        ) ** 2
    )


    minimum_alignment_error = float(
        alignment_error.min()
    )


    mean_alignment_error = float(
        alignment_error.mean()
    )


    maximum_alignment_error = float(
        alignment_error.max()
    )


    aligned_observations = int(
        (
            alignment_error
            <= ALIGNMENT_TOLERANCE
        )
        .sum()
    )


    misaligned_observations = int(
        valid_observations
        - aligned_observations
    )


    alignment_percentage = (
        aligned_observations
        / valid_observations
        * 100
    )


    # ========================================================
    # 22. RECONSTRUCTED MONTH DISTRIBUTION
    # ========================================================

    reconstructed_month_series = pd.Series(
        reconstructed_month,
        name="MONTH_NUMBER"
    )


    month_counts = (
        reconstructed_month_series
        .value_counts()
        .reindex(
            MONTH_NUMBERS,
            fill_value=0
        )
    )


    month_distribution = pd.DataFrame({

        "MONTH_NUMBER":
            MONTH_NUMBERS,

        "MONTH":
            [
                MONTH_NAMES[
                    month
                ]
                for month in MONTH_NUMBERS
            ],

        "COUNT":
            [
                int(
                    month_counts.loc[
                        month
                    ]
                )
                for month in MONTH_NUMBERS
            ]
    })


    month_distribution[
        "PERCENTAGE"
    ] = (
        month_distribution[
            "COUNT"
        ]
        / valid_observations
        * 100
    )


    observed_months = (
        month_distribution.loc[
            month_distribution[
                "COUNT"
            ] > 0,
            "MONTH_NUMBER"
        ]
        .tolist()
    )


    missing_months = [
        month
        for month in MONTH_NUMBERS
        if month not in observed_months
    ]


    # ========================================================
    # 23. MONTH REPRESENTATIVE COORDINATES
    #
    # Mean SIN/COS coordinates are calculated for each
    # reconstructed month.
    # ========================================================

    cyclical_coordinates = pd.DataFrame({

        "MONTH_NUMBER":
            reconstructed_month,

        "SIN":
            sin_valid.to_numpy(),

        "COS":
            cos_valid.to_numpy()
    })


    month_representatives = (
        cyclical_coordinates
        .groupby(
            "MONTH_NUMBER"
        )[
            [
                "SIN",
                "COS"
            ]
        ]
        .mean()
        .reindex(
            MONTH_NUMBERS
        )
    )


    # ========================================================
    # 24. CONSECUTIVE MONTH DISTANCES
    #
    # January → February
    # ...
    # November → December
    # December → January
    # ========================================================

    expected_consecutive_distance = float(
        2
        * np.sin(
            np.pi
            / PERIOD
        )
    )


    consecutive_distance_rows = []


    for current_month in MONTH_NUMBERS:

        next_month = (
            1
            if current_month == PERIOD
            else current_month + 1
        )


        current_coordinates = (
            month_representatives.loc[
                current_month
            ]
        )


        next_coordinates = (
            month_representatives.loc[
                next_month
            ]
        )


        if (
            current_coordinates.isna().any()
            or
            next_coordinates.isna().any()
        ):

            observed_distance = np.nan

            absolute_error = np.nan


        else:

            observed_distance = float(
                np.sqrt(
                    (
                        current_coordinates[
                            "SIN"
                        ]
                        -
                        next_coordinates[
                            "SIN"
                        ]
                    ) ** 2
                    +
                    (
                        current_coordinates[
                            "COS"
                        ]
                        -
                        next_coordinates[
                            "COS"
                        ]
                    ) ** 2
                )
            )


            absolute_error = float(
                abs(
                    observed_distance
                    - expected_consecutive_distance
                )
            )


        consecutive_distance_rows.append({

            "FROM_MONTH":
                MONTH_NAMES[
                    current_month
                ],

            "TO_MONTH":
                MONTH_NAMES[
                    next_month
                ],

            "TRANSITION":
                (
                    MONTH_ABBREVIATIONS[
                        current_month
                    ]
                    +
                    " → "
                    +
                    MONTH_ABBREVIATIONS[
                        next_month
                    ]
                ),

            "OBSERVED_DISTANCE":
                observed_distance,

            "EXPECTED_DISTANCE":
                expected_consecutive_distance,

            "ABSOLUTE_ERROR":
                absolute_error
        })


    consecutive_distances = pd.DataFrame(
        consecutive_distance_rows
    )


    valid_consecutive_distances = (
        consecutive_distances[
            "OBSERVED_DISTANCE"
        ]
        .dropna()
    )


    if len(
        valid_consecutive_distances
    ) > 0:

        minimum_consecutive_distance = float(
            valid_consecutive_distances.min()
        )


        mean_consecutive_distance = float(
            valid_consecutive_distances.mean()
        )


        maximum_consecutive_distance = float(
            valid_consecutive_distances.max()
        )


    else:

        minimum_consecutive_distance = np.nan

        mean_consecutive_distance = np.nan

        maximum_consecutive_distance = np.nan


    december_january_row = (
        consecutive_distances[
            consecutive_distances[
                "FROM_MONTH"
            ]
            == "December"
        ]
    )


    if not december_january_row.empty:

        december_january_distance = float(
            december_january_row[
                "OBSERVED_DISTANCE"
            ]
            .iloc[0]
        )


    else:

        december_january_distance = np.nan


    # ========================================================
    # 25. CIRCULAR STATISTICS
    # ========================================================

    mean_sin = float(
        sin_valid.mean()
    )


    mean_cos = float(
        cos_valid.mean()
    )


    mean_resultant_length = float(
        np.sqrt(
            mean_sin ** 2
            +
            mean_cos ** 2
        )
    )


    circular_variance = float(
        1
        - mean_resultant_length
    )


    circular_mean_angle = float(
        np.mod(
            np.arctan2(
                mean_sin,
                mean_cos
            ),
            2 * np.pi
        )
    )


    circular_mean_degrees = float(
        np.degrees(
            circular_mean_angle
        )
    )


    circular_mean_raw = int(
        np.rint(
            circular_mean_angle
            * PERIOD
            / (
                2 * np.pi
            )
        )
    )


    if circular_mean_raw == 0:

        circular_mean_month = 12

    elif circular_mean_raw > 12:

        circular_mean_month = 12

    else:

        circular_mean_month = (
            circular_mean_raw
        )


    circular_mean_month_name = (
        MONTH_NAMES[
            circular_mean_month
        ]
    )


    # ========================================================
    # 26. LINEAR CORRELATION BETWEEN COMPONENTS
    #
    # This is complementary information only.
    # SIN and COS should be interpreted jointly.
    # ========================================================

    sin_cos_correlation = float(
        sin_valid.corr(
            cos_valid
        )
    )


    # ========================================================
    # 27. CREATE MONTH DISTRIBUTION CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not month_distribution_chart_exists:

        fig, ax = plt.subplots(
            figsize=(
                11,
                6
            )
        )


        ax.bar(
            month_distribution[
                "MONTH"
            ],
            month_distribution[
                "COUNT"
            ]
        )


        ax.set_title(
            "Reconstructed transaction distribution by month"
        )


        ax.set_xlabel(
            "Month"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.tick_params(
            axis="x",
            rotation=45
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(y)
                )
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            MONTH_DISTRIBUTION_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nMonth distribution chart created:"
        )


        print(
            MONTH_DISTRIBUTION_CHART_PATH
        )


    else:

        print(
            "\nMonth distribution chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 28. CREATE UNIT-CIRCLE CHART
    #
    # The chart uses representative coordinates instead
    # of plotting all observations.
    # ========================================================

    if not unit_circle_chart_exists:

        circle_angle = np.linspace(
            0,
            2 * np.pi,
            500
        )


        circle_x = np.cos(
            circle_angle
        )


        circle_y = np.sin(
            circle_angle
        )


        fig, ax = plt.subplots(
            figsize=(
                8,
                8
            )
        )


        ax.plot(
            circle_x,
            circle_y
        )


        valid_representatives = (
            month_representatives
            .dropna()
        )


        ax.scatter(
            valid_representatives[
                "COS"
            ],
            valid_representatives[
                "SIN"
            ]
        )


        for month in MONTH_NUMBERS:

            if month in valid_representatives.index:

                month_cos = float(
                    valid_representatives.loc[
                        month,
                        "COS"
                    ]
                )


                month_sin = float(
                    valid_representatives.loc[
                        month,
                        "SIN"
                    ]
                )


                ax.annotate(
                    MONTH_ABBREVIATIONS[
                        month
                    ],
                    (
                        month_cos,
                        month_sin
                    ),
                    xytext=(
                        5,
                        5
                    ),
                    textcoords="offset points"
                )


        ax.axhline(
            0,
            linewidth=0.8
        )


        ax.axvline(
            0,
            linewidth=0.8
        )


        ax.set_title(
            "Cyclical representation of transaction month"
        )


        ax.set_xlabel(
            "Cosine component"
        )


        ax.set_ylabel(
            "Sine component"
        )


        ax.set_xlim(
            -1.15,
            1.15
        )


        ax.set_ylim(
            -1.15,
            1.15
        )


        ax.set_aspect(
            "equal",
            adjustable="box"
        )


        ax.grid(
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            UNIT_CIRCLE_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nUnit-circle chart created:"
        )


        print(
            UNIT_CIRCLE_CHART_PATH
        )


    else:

        print(
            "\nUnit-circle chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 29. CREATE CONSECUTIVE-DISTANCE CHART
    #
    # Only if the PNG does not already exist.
    # ========================================================

    if not consecutive_distance_chart_exists:

        fig, ax = plt.subplots(
            figsize=(
                12,
                6
            )
        )


        ax.plot(
            consecutive_distances[
                "TRANSITION"
            ],
            consecutive_distances[
                "OBSERVED_DISTANCE"
            ],
            marker="o"
        )


        ax.axhline(
            expected_consecutive_distance,
            linestyle="--"
        )


        ax.set_title(
            "Distance between consecutive month representations"
        )


        ax.set_xlabel(
            "Month transition"
        )


        ax.set_ylabel(
            "Euclidean distance"
        )


        ax.tick_params(
            axis="x",
            rotation=45
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            CONSECUTIVE_DISTANCE_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nConsecutive-distance chart created:"
        )


        print(
            CONSECUTIVE_DISTANCE_CHART_PATH
        )


    else:

        print(
            "\nConsecutive-distance chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 30. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 31. PREPARE TABLES FOR HTML
    # ========================================================

    component_statistics_html = (
        component_statistics
        .to_html(
            index=False,
            border=0,
            formatters={
                "SIN":
                    lambda x:
                    f"{x:.12f}",

                "COS":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    month_distribution_html = (
        month_distribution
        .to_html(
            index=False,
            border=0,
            formatters={
                "PERCENTAGE":
                    lambda x:
                    f"{x:.6f}%"
            }
        )
    )


    consecutive_distances_html = (
        consecutive_distances
        .to_html(
            index=False,
            border=0,
            columns=[
                "TRANSITION",
                "OBSERVED_DISTANCE",
                "EXPECTED_DISTANCE",
                "ABSOLUTE_ERROR"
            ],
            formatters={
                "OBSERVED_DISTANCE":
                    lambda x:
                    (
                        ""
                        if pd.isna(x)
                        else f"{x:.12f}"
                    ),

                "EXPECTED_DISTANCE":
                    lambda x:
                    f"{x:.12f}",

                "ABSOLUTE_ERROR":
                    lambda x:
                    (
                        ""
                        if pd.isna(x)
                        else f"{x:.12f}"
                    )
            }
        )
    )


    # ========================================================
    # 32. MONTH VALIDATION TEXT
    # ========================================================

    if len(
        missing_months
    ) == 0:

        month_validation_text = (
            "All 12 expected month positions "
            "were identified."
        )


    else:

        month_validation_text = (
            "The following reconstructed month "
            "positions were not observed: "
            +
            ", ".join(
                [
                    MONTH_NAMES[
                        month
                    ]
                    for month in missing_months
                ]
            )
        )


    # ========================================================
    # 33. CREATE THE HTML REPORT
    #
    # Only if the HTML does not already exist.
    # ========================================================

    if not html_exists:

        month_distribution_chart_base64 = (
            image_to_base64(
                MONTH_DISTRIBUTION_CHART_PATH
            )
        )


        unit_circle_chart_base64 = (
            image_to_base64(
                UNIT_CIRCLE_CHART_PATH
            )
        )


        consecutive_distance_chart_base64 = (
            image_to_base64(
                CONSECUTIVE_DISTANCE_CHART_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_NAME}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Individual Exploratory Analysis —
TRANS_MONTH_SIN and TRANS_MONTH_COS
</h1>


<p>

The variables
<strong>{SIN_COLUMN}</strong>
and
<strong>{COS_COLUMN}</strong>
jointly represent the transaction month
using cyclical encoding based on sine and cosine.

</p>


<p>

The encoding preserves the circular structure
of the calendar, meaning that December and January
remain adjacent in the transformed space.

</p>


<!-- ========================================================
     1. FEATURE PAIR OVERVIEW
========================================================= -->


<h2>
1. Feature pair overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>Valid SIN/COS pairs</td>
<td>{valid_observations}</td>
</tr>

<tr>
<td>Missing SIN values</td>
<td>{sin_missing_values}</td>
</tr>

<tr>
<td>Missing COS values</td>
<td>{cos_missing_values}</td>
</tr>

<tr>
<td>Rows with incomplete pairs</td>
<td>{pair_missing_values}</td>
</tr>

<tr>
<td>Unique SIN values</td>
<td>{unique_sin_values}</td>
</tr>

<tr>
<td>Unique COS values</td>
<td>{unique_cos_values}</td>
</tr>

<tr>
<td>Unique SIN/COS pairs</td>
<td>{unique_pairs}</td>
</tr>

</table>


<!-- ========================================================
     2. COMPONENT DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
2. Component descriptive statistics
</h2>


<p>

The sine and cosine components are bounded
numerical representations of the original
cyclical month variable.

</p>


{component_statistics_html}


<!-- ========================================================
     3. RANGE VALIDATION
========================================================= -->


<h2>
3. Range validation
</h2>


<p>

For valid sine and cosine transformations,
both components must remain within:

</p>


<p class="result">

-1 ≤ value ≤ 1

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>SIN values below -1</td>
<td>{sin_below_minus_one}</td>
</tr>

<tr>
<td>SIN values above 1</td>
<td>{sin_above_one}</td>
</tr>

<tr>
<td>Total SIN values outside range</td>
<td>{sin_outside_range}</td>
</tr>

<tr>
<td>COS values below -1</td>
<td>{cos_below_minus_one}</td>
</tr>

<tr>
<td>COS values above 1</td>
<td>{cos_above_one}</td>
</tr>

<tr>
<td>Total COS values outside range</td>
<td>{cos_outside_range}</td>
</tr>

</table>


<!-- ========================================================
     4. UNIT-CIRCLE VALIDATION
========================================================= -->


<h2>
4. Unit-circle validation
</h2>


<p>

A correctly encoded sine/cosine pair should satisfy:

</p>


<p class="result">

SIN² + COS² ≈ 1

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Minimum SIN² + COS²</td>
<td>{radius_squared_minimum:.12f}</td>
</tr>

<tr>
<td>Mean SIN² + COS²</td>
<td>{radius_squared_mean:.12f}</td>
</tr>

<tr>
<td>Maximum SIN² + COS²</td>
<td>{radius_squared_maximum:.12f}</td>
</tr>

<tr>
<td>Mean absolute deviation from 1</td>
<td>{mean_absolute_radius_squared_error:.12f}</td>
</tr>

<tr>
<td>Maximum absolute deviation from 1</td>
<td>{maximum_absolute_radius_squared_error:.12f}</td>
</tr>

<tr>
<td>Validation tolerance</td>
<td>{UNIT_CIRCLE_TOLERANCE}</td>
</tr>

<tr>
<td>Observations outside tolerance</td>
<td>{observations_outside_unit_circle_tolerance}</td>
</tr>

<tr>
<td>Observations satisfying tolerance</td>
<td>{unit_circle_valid_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     5. RECONSTRUCTED MONTH DISTRIBUTION
========================================================= -->


<h2>
5. Reconstructed month distribution
</h2>


<p>

The original month position was reconstructed
from the sine and cosine coordinates using
the cyclical angle.

</p>


<div class="note">

<strong>Month validation:</strong>

<br><br>

{month_validation_text}

</div>


{month_distribution_html}


<div class="chart">

<img
    src="data:image/png;base64,{month_distribution_chart_base64}"
    alt="Reconstructed transaction distribution by month"
>

</div>


<!-- ========================================================
     6. CYCLICAL REPRESENTATION
========================================================= -->


<h2>
6. Cyclical representation
</h2>


<p>

The chart below represents the mean coordinate
of each reconstructed month on the unit circle.

The cosine component is shown on the horizontal axis,
and the sine component is shown on the vertical axis.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{unit_circle_chart_base64}"
    alt="Cyclical representation of transaction month"
>

</div>


<!-- ========================================================
     7. CONSECUTIVE MONTH DISTANCES
========================================================= -->


<h2>
7. Consecutive month distances
</h2>


<p>

The Euclidean distance between consecutive
month representations is evaluated to verify
whether equal temporal steps remain equally
spaced in the cyclical representation.

</p>


<p>

The closing transition
<strong>December → January</strong>
is included explicitly because it represents
the circular boundary of the calendar.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Theoretical consecutive distance</td>
<td>{expected_consecutive_distance:.12f}</td>
</tr>

<tr>
<td>Minimum observed distance</td>
<td>{minimum_consecutive_distance:.12f}</td>
</tr>

<tr>
<td>Mean observed distance</td>
<td>{mean_consecutive_distance:.12f}</td>
</tr>

<tr>
<td>Maximum observed distance</td>
<td>{maximum_consecutive_distance:.12f}</td>
</tr>

<tr>
<td>December → January distance</td>
<td>{december_january_distance:.12f}</td>
</tr>

</table>


{consecutive_distances_html}


<div class="chart">

<img
    src="data:image/png;base64,{consecutive_distance_chart_base64}"
    alt="Distance between consecutive month representations"
>

</div>


<!-- ========================================================
     8. ANGULAR RECONSTRUCTION VALIDATION
========================================================= -->


<h2>
8. Angular reconstruction validation
</h2>


<p>

Each stored sine/cosine pair was converted back
to a cyclical angle and assigned to its nearest
theoretical month position.

The observed coordinate was then compared with
the exact theoretical coordinate of that month.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Alignment tolerance</td>
<td>{ALIGNMENT_TOLERANCE}</td>
</tr>

<tr>
<td>Minimum alignment error</td>
<td>{minimum_alignment_error:.12f}</td>
</tr>

<tr>
<td>Mean alignment error</td>
<td>{mean_alignment_error:.12f}</td>
</tr>

<tr>
<td>Maximum alignment error</td>
<td>{maximum_alignment_error:.12f}</td>
</tr>

<tr>
<td>Aligned observations</td>
<td>{aligned_observations}</td>
</tr>

<tr>
<td>Misaligned observations</td>
<td>{misaligned_observations}</td>
</tr>

<tr>
<td>Alignment percentage</td>
<td>{alignment_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     9. CIRCULAR STATISTICS
========================================================= -->


<h2>
9. Circular statistics
</h2>


<p>

Circular statistics summarize the distribution
while respecting the cyclical nature of the month.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Mean sine component</td>
<td>{mean_sin:.12f}</td>
</tr>

<tr>
<td>Mean cosine component</td>
<td>{mean_cos:.12f}</td>
</tr>

<tr>
<td>Circular mean angle</td>
<td>{circular_mean_degrees:.6f}°</td>
</tr>

<tr>
<td>Nearest circular mean month</td>
<td>{circular_mean_month_name}</td>
</tr>

<tr>
<td>Mean resultant length</td>
<td>{mean_resultant_length:.12f}</td>
</tr>

<tr>
<td>Circular variance</td>
<td>{circular_variance:.12f}</td>
</tr>

<tr>
<td>Pearson correlation between SIN and COS</td>
<td>{sin_cos_correlation:.12f}</td>
</tr>

</table>


<div class="note">

<strong>Mean resultant length interpretation:</strong>

<br><br>

Values closer to 0 indicate that observations
are more dispersed around the cycle.

<br><br>

Values closer to 1 indicate stronger concentration
around a particular region of the cycle.

<br><br>

Circular variance is calculated as
1 - mean resultant length.

</div>


<!-- ========================================================
     10. SUMMARY
========================================================= -->


<h2>
10. Summary of results
</h2>


<ul>

<li>
<strong>Total observations:</strong>
{total_observations}
</li>

<li>
<strong>Valid SIN/COS pairs:</strong>
{valid_observations}
</li>

<li>
<strong>Unique cyclical coordinates:</strong>
{unique_pairs}
</li>

<li>
<strong>SIN values outside [-1, 1]:</strong>
{sin_outside_range}
</li>

<li>
<strong>COS values outside [-1, 1]:</strong>
{cos_outside_range}
</li>

<li>
<strong>Maximum unit-circle error:</strong>
{maximum_absolute_radius_squared_error:.12f}
</li>

<li>
<strong>Unit-circle validation percentage:</strong>
{unit_circle_valid_percentage:.6f}%
</li>

<li>
<strong>Reconstructed months observed:</strong>
{len(observed_months)}
</li>

<li>
<strong>Alignment percentage:</strong>
{alignment_percentage:.6f}%
</li>

<li>
<strong>Theoretical consecutive-month distance:</strong>
{expected_consecutive_distance:.12f}
</li>

<li>
<strong>December → January distance:</strong>
{december_january_distance:.12f}
</li>

<li>
<strong>Mean resultant length:</strong>
{mean_resultant_length:.12f}
</li>

<li>
<strong>Circular variance:</strong>
{circular_variance:.12f}
</li>

<li>
<strong>Nearest circular mean month:</strong>
{circular_mean_month_name}
</li>

</ul>


</body>

</html>
"""


        # ====================================================
        # 34. SAVE THE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )


        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 35. DISPLAY MAIN RESULTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "TRANS_MONTH CYCLICAL ENCODING SUMMARY"
    )


    print(
        "=" * 100
    )


    print(
        "Total observations:",
        total_observations
    )


    print(
        "Valid SIN/COS pairs:",
        valid_observations
    )


    print(
        "Unique SIN values:",
        unique_sin_values
    )


    print(
        "Unique COS values:",
        unique_cos_values
    )


    print(
        "Unique SIN/COS pairs:",
        unique_pairs
    )


    print(
        "SIN values outside [-1, 1]:",
        sin_outside_range
    )


    print(
        "COS values outside [-1, 1]:",
        cos_outside_range
    )


    # ========================================================
    # 36. DISPLAY UNIT-CIRCLE VALIDATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "UNIT-CIRCLE VALIDATION"
    )


    print(
        "=" * 100
    )


    print(
        "Mean SIN² + COS²:",
        f"{radius_squared_mean:.12f}"
    )


    print(
        "Maximum absolute deviation from 1:",
        f"{maximum_absolute_radius_squared_error:.12f}"
    )


    print(
        "Observations outside tolerance:",
        observations_outside_unit_circle_tolerance
    )


    print(
        "Validation percentage:",
        f"{unit_circle_valid_percentage:.6f}%"
    )


    # ========================================================
    # 37. DISPLAY MONTH DISTRIBUTION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "RECONSTRUCTED MONTH DISTRIBUTION"
    )


    print(
        "=" * 100
    )


    display(
        month_distribution
    )


    # ========================================================
    # 38. DISPLAY CONSECUTIVE DISTANCES
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "CONSECUTIVE MONTH DISTANCES"
    )


    print(
        "=" * 100
    )


    display(
        consecutive_distances
    )


    # ========================================================
    # 39. DISPLAY RECONSTRUCTION VALIDATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "CYCLICAL ALIGNMENT VALIDATION"
    )


    print(
        "=" * 100
    )


    print(
        "Aligned observations:",
        aligned_observations
    )


    print(
        "Misaligned observations:",
        misaligned_observations
    )


    print(
        "Alignment percentage:",
        f"{alignment_percentage:.6f}%"
    )


    print(
        "Maximum alignment error:",
        f"{maximum_alignment_error:.12f}"
    )


    # ========================================================
    # 40. DISPLAY CIRCULAR STATISTICS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "CIRCULAR STATISTICS"
    )


    print(
        "=" * 100
    )


    print(
        "Circular mean angle:",
        f"{circular_mean_degrees:.6f} degrees"
    )


    print(
        "Nearest circular mean month:",
        circular_mean_month_name
    )


    print(
        "Mean resultant length:",
        f"{mean_resultant_length:.12f}"
    )


    print(
        "Circular variance:",
        f"{circular_variance:.12f}"
    )


    print(
        "SIN/COS Pearson correlation:",
        f"{sin_cos_correlation:.12f}"
    )


    # ========================================================
    # 41. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del sin_feature
    del cos_feature
    del sin_valid
    del cos_valid
    del radius_squared
    del radius_squared_error
    del cyclical_coordinates

    gc.collect()


    # ========================================================
    # 42. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "ANALYSIS COMPLETED"
    )


    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )


    print(
        HTML_PATH
    )


    print(
        "\nPNGs:"
    )


    print(
        MONTH_DISTRIBUTION_CHART_PATH
    )


    print(
        UNIT_CIRCLE_CHART_PATH
    )


    print(
        CONSECUTIVE_DISTANCE_CHART_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Month distribution chart: Will be created
Unit-circle chart: Will be created
Consecutive-distance chart: Will be created

Month distribution chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/cyclical_encoding_using_sine_and_cosine/trans_month_sin_cos/trans_month_sin_cos_month_distribution.png

Unit-circle chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/cyclical_encoding_using_sine_and_cosine/trans_month_sin_cos/trans_month_sin_cos_unit_circle.png

Consecutive-distance chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/cyclical_encoding_using_sine_and_cosine/trans_month_sin_cos/trans_month_sin_cos_consecutive_distance.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/cyclical_encoding_using_sine_and_cosine/trans_month_sin_cos/analysis_trans_month_sin_cos.html

TRANS_MONTH CYCLICAL ENCODING SU

,MONTH_NUMBER,MONTH,COUNT,PERCENTAGE
0,1,January,97657,5.271935
1,2,February,143789,7.762333
2,3,March,134970,7.286247
3,4,April,146875,7.928929
4,5,May,173869,9.386178
5,6,June,172444,9.309251
6,7,July,176118,9.507589
7,8,August,140185,7.567774
8,9,September,138106,7.455541
9,10,October,143056,7.722763



CONSECUTIVE MONTH DISTANCES


,FROM_MONTH,TO_MONTH,TRANSITION,OBSERVED_DISTANCE,EXPECTED_DISTANCE,ABSOLUTE_ERROR
0,January,February,Jan → Feb,0.517638,0.517638,2.198200e-08
1,February,March,Feb → Mar,0.517638,0.517638,4.022986e-09
2,March,April,Mar → Apr,0.517638,0.517638,4.022986e-09
3,April,May,Apr → May,0.517638,0.517638,2.198200e-08
4,May,June,May → Jun,0.517638,0.517638,4.022986e-09
5,June,July,Jun → Jul,0.517638,0.517638,4.022986e-09
6,July,August,Jul → Aug,0.517638,0.517638,2.198200e-08
7,August,September,Aug → Sep,0.517638,0.517638,4.022986e-09
8,September,October,Sep → Oct,0.517638,0.517638,4.022986e-09
9,October,November,Oct → Nov,0.517638,0.517638,2.198200e-08



CYCLICAL ALIGNMENT VALIDATION
Aligned observations: 1852394
Misaligned observations: 0
Alignment percentage: 100.000000%
Maximum alignment error: 0.000000015544

CIRCULAR STATISTICS
Circular mean angle: 244.110004 degrees
Nearest circular mean month: August
Mean resultant length: 0.053587604719
Circular variance: 0.946412395281
SIN/COS Pearson correlation: -0.089869639247

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/cyclical_encoding_using_sine_and_cosine/trans_month_sin_cos

HTML:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/cyclical_encoding_using_sine_and_cosine/trans_month_sin_cos/analysis_trans_month_sin_cos.html

PNGs:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/cyclical_encoding_using_sine_and_cosine/trans_month_sin_cos/trans_month_sin_cos_month_distribution.png
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/cyclical_encoding_using_sine_

### <span style="color:olive"> TRANS_HOUR_SEN + TRANS_HOUR_COS  </span> ###

In [9]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ENCODING_TYPE = "cyclical_encoding_using_sine_and_cosine"

FEATURE_NAME = "trans_hour_sin_cos"

SIN_COLUMN = "TRANS_HOUR_SIN"

COS_COLUMN = "TRANS_HOUR_COS"

HOURS_PER_DAY = 24

MINUTES_PER_HOUR = 60

SECONDS_PER_MINUTE = 60

SECONDS_PER_HOUR = (
    MINUTES_PER_HOUR
    * SECONDS_PER_MINUTE
)

SECONDS_PER_DAY = (
    HOURS_PER_DAY
    * SECONDS_PER_HOUR
)

UNIT_CIRCLE_TOLERANCE = 1e-6

TIME_RECONSTRUCTION_TOLERANCE_SECONDS = 0.01


# ============================================================
# 02. TIME SETTINGS
# ============================================================

HOURS = list(
    range(
        0,
        24
    )
)


MINUTES = list(
    range(
        0,
        60
    )
)


SECONDS = list(
    range(
        0,
        60
    )
)


HOUR_LABELS = {
    hour: f"{hour:02d}:00"
    for hour in HOURS
}


MINUTE_LABELS = {
    minute: f"{minute:02d}"
    for minute in MINUTES
}


SECOND_LABELS = {
    second: f"{second:02d}"
    for second in SECONDS
}


# ============================================================
# 03. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 04. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 05. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / ENCODING_TYPE
    / FEATURE_NAME
)


# ============================================================
# 06. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 07. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


HOUR_DISTRIBUTION_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_hour_distribution.png"
)


MINUTE_DISTRIBUTION_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_minute_distribution.png"
)


SECOND_DISTRIBUTION_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_second_distribution.png"
)


UNIT_CIRCLE_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_unit_circle.png"
)


TIME_RECONSTRUCTION_ERROR_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_time_reconstruction_error.png"
)


# ============================================================
# 08. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


hour_distribution_chart_exists = (
    HOUR_DISTRIBUTION_CHART_PATH.exists()
)


minute_distribution_chart_exists = (
    MINUTE_DISTRIBUTION_CHART_PATH.exists()
)


second_distribution_chart_exists = (
    SECOND_DISTRIBUTION_CHART_PATH.exists()
)


unit_circle_chart_exists = (
    UNIT_CIRCLE_CHART_PATH.exists()
)


time_reconstruction_error_chart_exists = (
    TIME_RECONSTRUCTION_ERROR_CHART_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and hour_distribution_chart_exists
    and minute_distribution_chart_exists
    and second_distribution_chart_exists
    and unit_circle_chart_exists
    and time_reconstruction_error_chart_exists
)


# ============================================================
# 09. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )

    print(
        "No analysis or file creation is required."
    )

    print(
        "\nResults directory:"
    )

    print(
        RESULTS_DIRECTORY
    )

    print(
        "\nExisting files:"
    )

    print(
        HTML_PATH
    )

    print(
        HOUR_DISTRIBUTION_CHART_PATH
    )

    print(
        MINUTE_DISTRIBUTION_CHART_PATH
    )

    print(
        SECOND_DISTRIBUTION_CHART_PATH
    )

    print(
        UNIT_CIRCLE_CHART_PATH
    )

    print(
        TIME_RECONSTRUCTION_ERROR_CHART_PATH
    )


else:

    # ========================================================
    # 10. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 11. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )

    print(
        "=" * 100
    )


    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )


    print(
        "Hour distribution chart:",
        "Already exists"
        if hour_distribution_chart_exists
        else "Will be created"
    )


    print(
        "Minute distribution chart:",
        "Already exists"
        if minute_distribution_chart_exists
        else "Will be created"
    )


    print(
        "Second distribution chart:",
        "Already exists"
        if second_distribution_chart_exists
        else "Will be created"
    )


    print(
        "Unit-circle chart:",
        "Already exists"
        if unit_circle_chart_exists
        else "Will be created"
    )


    print(
        "Time reconstruction error chart:",
        "Already exists"
        if time_reconstruction_error_chart_exists
        else "Will be created"
    )


    # ========================================================
    # 12. LOAD ONLY THE TWO FEATURES BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            SIN_COLUMN,
            COS_COLUMN
        ]
    )


    sin_feature = (
        dataset_feature[
            SIN_COLUMN
        ]
    )


    cos_feature = (
        dataset_feature[
            COS_COLUMN
        ]
    )


    # ========================================================
    # 13. BASIC VALIDATION
    # ========================================================

    total_observations = int(
        len(
            dataset_feature
        )
    )


    if total_observations == 0:

        raise ValueError(
            "The selected cyclical features "
            "contain no observations."
        )


    sin_missing_values = int(
        sin_feature
        .isna()
        .sum()
    )


    cos_missing_values = int(
        cos_feature
        .isna()
        .sum()
    )


    pair_missing_values = int(
        (
            sin_feature.isna()
            |
            cos_feature.isna()
        )
        .sum()
    )


    valid_mask = (
        sin_feature.notna()
        &
        cos_feature.notna()
    )


    valid_observations = int(
        valid_mask.sum()
    )


    if valid_observations == 0:

        raise ValueError(
            "There are no complete SIN/COS pairs "
            "available for analysis."
        )


    sin_valid = (
        sin_feature[
            valid_mask
        ]
        .astype(
            "float64"
        )
    )


    cos_valid = (
        cos_feature[
            valid_mask
        ]
        .astype(
            "float64"
        )
    )


    # ========================================================
    # 14. UNIQUE COMPONENT VALUES AND PAIRS
    # ========================================================

    unique_sin_values = int(
        sin_valid.nunique()
    )


    unique_cos_values = int(
        cos_valid.nunique()
    )


    unique_cyclical_points = (
        pd.DataFrame({

            "SIN":
                sin_valid.to_numpy(),

            "COS":
                cos_valid.to_numpy()

        })
        .drop_duplicates()
        .reset_index(
            drop=True
        )
    )


    unique_pairs = int(
        len(
            unique_cyclical_points
        )
    )


    number_unique_cyclical_points = (
        unique_pairs
    )


    # ========================================================
    # 15. COMPONENT DESCRIPTIVE STATISTICS
    # ========================================================

    component_statistics = pd.DataFrame({

        "STATISTIC": [
            "Minimum",
            "Q1",
            "Median",
            "Mean",
            "Q3",
            "Maximum",
            "Standard deviation"
        ],

        "SIN": [
            float(
                sin_valid.min()
            ),
            float(
                sin_valid.quantile(
                    0.25
                )
            ),
            float(
                sin_valid.median()
            ),
            float(
                sin_valid.mean()
            ),
            float(
                sin_valid.quantile(
                    0.75
                )
            ),
            float(
                sin_valid.max()
            ),
            float(
                sin_valid.std()
            )
        ],

        "COS": [
            float(
                cos_valid.min()
            ),
            float(
                cos_valid.quantile(
                    0.25
                )
            ),
            float(
                cos_valid.median()
            ),
            float(
                cos_valid.mean()
            ),
            float(
                cos_valid.quantile(
                    0.75
                )
            ),
            float(
                cos_valid.max()
            ),
            float(
                cos_valid.std()
            )
        ]
    })


    # ========================================================
    # 16. RANGE VALIDATION
    # ========================================================

    sin_below_minus_one = int(
        (
            sin_valid < -1
        )
        .sum()
    )


    sin_above_one = int(
        (
            sin_valid > 1
        )
        .sum()
    )


    cos_below_minus_one = int(
        (
            cos_valid < -1
        )
        .sum()
    )


    cos_above_one = int(
        (
            cos_valid > 1
        )
        .sum()
    )


    sin_outside_range = (
        sin_below_minus_one
        + sin_above_one
    )


    cos_outside_range = (
        cos_below_minus_one
        + cos_above_one
    )


    # ========================================================
    # 17. UNIT-CIRCLE VALIDATION
    #
    # For a valid cyclical encoding:
    #
    # SIN² + COS² ≈ 1
    # ========================================================

    radius_squared = (
        sin_valid.pow(2)
        +
        cos_valid.pow(2)
    )


    radius_squared_minimum = float(
        radius_squared.min()
    )


    radius_squared_mean = float(
        radius_squared.mean()
    )


    radius_squared_maximum = float(
        radius_squared.max()
    )


    radius_squared_error = (
        radius_squared
        - 1.0
    ).abs()


    mean_absolute_radius_squared_error = float(
        radius_squared_error.mean()
    )


    maximum_absolute_radius_squared_error = float(
        radius_squared_error.max()
    )


    observations_outside_unit_circle_tolerance = int(
        (
            radius_squared_error
            > UNIT_CIRCLE_TOLERANCE
        )
        .sum()
    )


    observations_inside_unit_circle_tolerance = int(
        valid_observations
        - observations_outside_unit_circle_tolerance
    )


    unit_circle_valid_percentage = (
        observations_inside_unit_circle_tolerance
        / valid_observations
        * 100
    )


    # ========================================================
    # 18. RECONSTRUCT THE CYCLICAL ANGLE
    #
    # atan2(SIN, COS) reconstructs the original angle.
    # ========================================================

    angles = np.mod(
        np.arctan2(
            sin_valid.to_numpy(),
            cos_valid.to_numpy()
        ),
        2 * np.pi
    )


    # ========================================================
    # 19. RECONSTRUCT CONTINUOUS TIME OF DAY
    #
    # Original preprocessing:
    #
    # decimal_hour =
    #     hour
    #     + minute / 60
    #     + second / 3600
    #
    # Therefore:
    #
    # decimal_hour =
    #     angle × 24 / (2π)
    # ========================================================

    reconstructed_decimal_hour = (
        angles
        * HOURS_PER_DAY
        / (
            2 * np.pi
        )
    )


    reconstructed_continuous_seconds = (
        reconstructed_decimal_hour
        * SECONDS_PER_HOUR
    )


    # ========================================================
    # 20. RECONSTRUCT THE NEAREST EXACT SECOND
    # ========================================================

    reconstructed_second_of_day = (
        np.rint(
            reconstructed_continuous_seconds
        )
        .astype(
            np.int64
        )
        % SECONDS_PER_DAY
    )


    # ========================================================
    # 21. RECONSTRUCT HOUR, MINUTE, AND SECOND
    # ========================================================

    reconstructed_hour = (
        reconstructed_second_of_day
        // SECONDS_PER_HOUR
    )


    reconstructed_minute = (
        (
            reconstructed_second_of_day
            % SECONDS_PER_HOUR
        )
        // SECONDS_PER_MINUTE
    )


    reconstructed_second = (
        reconstructed_second_of_day
        % SECONDS_PER_MINUTE
    )


    # ========================================================
    # 22. TEMPORAL RECONSTRUCTION ERROR
    #
    # Circular distance is used so midnight is handled
    # correctly.
    # ========================================================

    raw_time_error_seconds = np.abs(
        reconstructed_continuous_seconds
        - reconstructed_second_of_day
    )


    temporal_reconstruction_error_seconds = np.minimum(
        raw_time_error_seconds,
        SECONDS_PER_DAY
        - raw_time_error_seconds
    )


    minimum_time_error_seconds = float(
        temporal_reconstruction_error_seconds.min()
    )


    q1_time_error_seconds = float(
        np.quantile(
            temporal_reconstruction_error_seconds,
            0.25
        )
    )


    median_time_error_seconds = float(
        np.median(
            temporal_reconstruction_error_seconds
        )
    )


    mean_time_error_seconds = float(
        temporal_reconstruction_error_seconds.mean()
    )


    q3_time_error_seconds = float(
        np.quantile(
            temporal_reconstruction_error_seconds,
            0.75
        )
    )


    maximum_time_error_seconds = float(
        temporal_reconstruction_error_seconds.max()
    )


    observations_within_time_tolerance = int(
        (
            temporal_reconstruction_error_seconds
            <= TIME_RECONSTRUCTION_TOLERANCE_SECONDS
        )
        .sum()
    )


    observations_outside_time_tolerance = int(
        valid_observations
        - observations_within_time_tolerance
    )


    time_reconstruction_valid_percentage = (
        observations_within_time_tolerance
        / valid_observations
        * 100
    )


    # ========================================================
    # 23. UNIQUE RECONSTRUCTED TIME POSITIONS
    #
    # There are 86,400 possible second-level positions
    # in a 24-hour day.
    # ========================================================

    unique_reconstructed_seconds = int(
        np.unique(
            reconstructed_second_of_day
        )
        .size
    )


    time_position_coverage_percentage = (
        unique_reconstructed_seconds
        / SECONDS_PER_DAY
        * 100
    )


    # ========================================================
    # 24. THEORETICAL COORDINATES
    #
    # Each reconstructed exact second is converted back
    # into its theoretical cyclical coordinate.
    # ========================================================

    theoretical_angles = (
        2
        * np.pi
        * reconstructed_second_of_day
        / SECONDS_PER_DAY
    )


    theoretical_sin = np.sin(
        theoretical_angles
    )


    theoretical_cos = np.cos(
        theoretical_angles
    )


    coordinate_alignment_error = np.sqrt(
        (
            sin_valid.to_numpy()
            - theoretical_sin
        ) ** 2
        +
        (
            cos_valid.to_numpy()
            - theoretical_cos
        ) ** 2
    )


    minimum_coordinate_error = float(
        coordinate_alignment_error.min()
    )


    mean_coordinate_error = float(
        coordinate_alignment_error.mean()
    )


    maximum_coordinate_error = float(
        coordinate_alignment_error.max()
    )


    # ========================================================
    # 25. DISTRIBUTION BY HOUR
    #
    # Hours are aggregated only for descriptive analysis.
    #
    # Minute and second information remains preserved in
    # the original cyclical coordinates.
    # ========================================================

    hour_counts = (
        pd.Series(
            reconstructed_hour
        )
        .value_counts()
        .reindex(
            HOURS,
            fill_value=0
        )
    )


    hour_distribution = pd.DataFrame({

        "HOUR":
            HOURS,

        "HOUR_LABEL":
            [
                HOUR_LABELS[
                    hour
                ]
                for hour in HOURS
            ],

        "COUNT":
            [
                int(
                    hour_counts.loc[
                        hour
                    ]
                )
                for hour in HOURS
            ]
    })


    hour_distribution[
        "PERCENTAGE"
    ] = (
        hour_distribution[
            "COUNT"
        ]
        / valid_observations
        * 100
    )


    # ========================================================
    # 26. DISTRIBUTION BY MINUTE
    #
    # Minute position inside each hour:
    #
    # 00, 01, ..., 59
    # ========================================================

    minute_counts = (
        pd.Series(
            reconstructed_minute
        )
        .value_counts()
        .reindex(
            MINUTES,
            fill_value=0
        )
    )


    minute_distribution = pd.DataFrame({

        "MINUTE":
            MINUTES,

        "MINUTE_LABEL":
            [
                MINUTE_LABELS[
                    minute
                ]
                for minute in MINUTES
            ],

        "COUNT":
            [
                int(
                    minute_counts.loc[
                        minute
                    ]
                )
                for minute in MINUTES
            ]
    })


    minute_distribution[
        "PERCENTAGE"
    ] = (
        minute_distribution[
            "COUNT"
        ]
        / valid_observations
        * 100
    )


    # ========================================================
    # 27. DISTRIBUTION BY SECOND
    #
    # Second position inside each minute:
    #
    # 00, 01, ..., 59
    # ========================================================

    second_counts = (
        pd.Series(
            reconstructed_second
        )
        .value_counts()
        .reindex(
            SECONDS,
            fill_value=0
        )
    )


    second_distribution = pd.DataFrame({

        "SECOND":
            SECONDS,

        "SECOND_LABEL":
            [
                SECOND_LABELS[
                    second
                ]
                for second in SECONDS
            ],

        "COUNT":
            [
                int(
                    second_counts.loc[
                        second
                    ]
                )
                for second in SECONDS
            ]
    })


    second_distribution[
        "PERCENTAGE"
    ] = (
        second_distribution[
            "COUNT"
        ]
        / valid_observations
        * 100
    )


    # ========================================================
    # 28. TEMPORAL STEP DISTANCES
    #
    # Chord distance:
    #
    # d = 2 × sin(π × step_seconds / 86400)
    # ========================================================

    one_second_distance = float(
        2
        * np.sin(
            np.pi
            / SECONDS_PER_DAY
        )
    )


    one_minute_distance = float(
        2
        * np.sin(
            np.pi
            * SECONDS_PER_MINUTE
            / SECONDS_PER_DAY
        )
    )


    one_hour_distance = float(
        2
        * np.sin(
            np.pi
            * SECONDS_PER_HOUR
            / SECONDS_PER_DAY
        )
    )


    # ========================================================
    # 29. MIDNIGHT BOUNDARY VALIDATION
    #
    # 23:59:59 → 00:00:00
    # ========================================================

    last_second_angle = (
        2
        * np.pi
        * (
            SECONDS_PER_DAY
            - 1
        )
        / SECONDS_PER_DAY
    )


    last_second_sin = np.sin(
        last_second_angle
    )


    last_second_cos = np.cos(
        last_second_angle
    )


    midnight_sin = 0.0

    midnight_cos = 1.0


    boundary_distance = float(
        np.sqrt(
            (
                last_second_sin
                - midnight_sin
            ) ** 2
            +
            (
                last_second_cos
                - midnight_cos
            ) ** 2
        )
    )


    boundary_distance_error = float(
        abs(
            boundary_distance
            - one_second_distance
        )
    )


    # ========================================================
    # 30. CIRCULAR STATISTICS
    # ========================================================

    mean_sin = float(
        sin_valid.mean()
    )


    mean_cos = float(
        cos_valid.mean()
    )


    mean_resultant_length = float(
        np.sqrt(
            mean_sin ** 2
            +
            mean_cos ** 2
        )
    )


    circular_variance = float(
        1
        - mean_resultant_length
    )


    circular_mean_angle = float(
        np.mod(
            np.arctan2(
                mean_sin,
                mean_cos
            ),
            2 * np.pi
        )
    )


    circular_mean_degrees = float(
        np.degrees(
            circular_mean_angle
        )
    )


    circular_mean_second_of_day = int(
        np.rint(
            circular_mean_angle
            * SECONDS_PER_DAY
            / (
                2 * np.pi
            )
        )
        % SECONDS_PER_DAY
    )


    circular_mean_hour = (
        circular_mean_second_of_day
        // SECONDS_PER_HOUR
    )


    circular_mean_minute = (
        (
            circular_mean_second_of_day
            % SECONDS_PER_HOUR
        )
        // SECONDS_PER_MINUTE
    )


    circular_mean_second = (
        circular_mean_second_of_day
        % SECONDS_PER_MINUTE
    )


    circular_mean_time_label = (
        f"{circular_mean_hour:02d}:"
        f"{circular_mean_minute:02d}:"
        f"{circular_mean_second:02d}"
    )


    # ========================================================
    # 31. LINEAR CORRELATION BETWEEN COMPONENTS
    #
    # Complementary information only.
    # ========================================================

    sin_cos_correlation = float(
        sin_valid.corr(
            cos_valid
        )
    )


    # ========================================================
    # 32. TIME RECONSTRUCTION STATISTICS TABLE
    # ========================================================

    time_reconstruction_statistics = pd.DataFrame({

        "STATISTIC": [
            "Minimum",
            "Q1",
            "Median",
            "Mean",
            "Q3",
            "Maximum"
        ],

        "ERROR_SECONDS": [
            minimum_time_error_seconds,
            q1_time_error_seconds,
            median_time_error_seconds,
            mean_time_error_seconds,
            q3_time_error_seconds,
            maximum_time_error_seconds
        ]
    })


    # ========================================================
    # 33. CREATE HOUR DISTRIBUTION CHART
    # ========================================================

    if not hour_distribution_chart_exists:

        fig, ax = plt.subplots(
            figsize=(
                13,
                6
            )
        )


        ax.bar(
            hour_distribution[
                "HOUR_LABEL"
            ],
            hour_distribution[
                "COUNT"
            ]
        )


        ax.set_title(
            "Transaction distribution by hour"
        )


        ax.set_xlabel(
            "Hour"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.tick_params(
            axis="x",
            rotation=45
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(y)
                )
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            HOUR_DISTRIBUTION_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nHour distribution chart created:"
        )


        print(
            HOUR_DISTRIBUTION_CHART_PATH
        )


    else:

        print(
            "\nHour distribution chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 34. CREATE MINUTE DISTRIBUTION CHART
    # ========================================================

    if not minute_distribution_chart_exists:

        fig, ax = plt.subplots(
            figsize=(
                14,
                6
            )
        )


        ax.bar(
            minute_distribution[
                "MINUTE_LABEL"
            ],
            minute_distribution[
                "COUNT"
            ]
        )


        ax.set_title(
            "Transaction distribution by minute"
        )


        ax.set_xlabel(
            "Minute within the hour"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.tick_params(
            axis="x",
            rotation=90
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(y)
                )
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            MINUTE_DISTRIBUTION_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nMinute distribution chart created:"
        )


        print(
            MINUTE_DISTRIBUTION_CHART_PATH
        )


    else:

        print(
            "\nMinute distribution chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 35. CREATE SECOND DISTRIBUTION CHART
    # ========================================================

    if not second_distribution_chart_exists:

        fig, ax = plt.subplots(
            figsize=(
                14,
                6
            )
        )


        ax.bar(
            second_distribution[
                "SECOND_LABEL"
            ],
            second_distribution[
                "COUNT"
            ]
        )


        ax.set_title(
            "Transaction distribution by second"
        )


        ax.set_xlabel(
            "Second within the minute"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.tick_params(
            axis="x",
            rotation=90
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(y)
                )
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            SECOND_DISTRIBUTION_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nSecond distribution chart created:"
        )


        print(
            SECOND_DISTRIBUTION_CHART_PATH
        )


    else:

        print(
            "\nSecond distribution chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 36. CREATE UNIT-CIRCLE CHART
    #
    # One point is plotted for every distinct temporal
    # position actually represented in the dataset.
    #
    # No averaging is performed.
    # No random sampling is performed.
    # ========================================================

    if not unit_circle_chart_exists:

        # ----------------------------------------------------
        # REFERENCE UNIT CIRCLE
        # ----------------------------------------------------

        circle_angle = np.linspace(
            0,
            2 * np.pi,
            2000
        )


        circle_x = np.cos(
            circle_angle
        )


        circle_y = np.sin(
            circle_angle
        )


        # ----------------------------------------------------
        # FIGURE
        # ----------------------------------------------------

        fig, ax = plt.subplots(
            figsize=(
                10,
                10
            )
        )


        # ----------------------------------------------------
        # UNIT-CIRCLE REFERENCE LINE
        # ----------------------------------------------------

        ax.plot(
            circle_x,
            circle_y,
            linewidth=0.8
        )


        # ----------------------------------------------------
        # ALL DISTINCT TEMPORAL POSITIONS OBSERVED
        #
        # COS is plotted on X.
        # SIN is plotted on Y.
        # ----------------------------------------------------

        ax.scatter(
            unique_cyclical_points[
                "COS"
            ],
            unique_cyclical_points[
                "SIN"
            ],
            s=3,
            alpha=0.35,
            rasterized=True
        )


        # ----------------------------------------------------
        # EXACT INTEGER-HOUR REFERENCE POSITIONS
        # ----------------------------------------------------

        hour_angles = (
            2
            * np.pi
            * np.array(
                HOURS
            )
            / HOURS_PER_DAY
        )


        hour_sin = np.sin(
            hour_angles
        )


        hour_cos = np.cos(
            hour_angles
        )


        ax.scatter(
            hour_cos,
            hour_sin,
            s=25
        )


        # ----------------------------------------------------
        # HOUR LABELS
        # ----------------------------------------------------

        for hour in HOURS:

            ax.annotate(
                f"{hour:02d}",
                (
                    hour_cos[
                        hour
                    ],
                    hour_sin[
                        hour
                    ]
                ),
                xytext=(
                    6,
                    6
                ),
                textcoords="offset points"
            )


        # ----------------------------------------------------
        # REFERENCE AXES
        # ----------------------------------------------------

        ax.axhline(
            0,
            linewidth=0.8
        )


        ax.axvline(
            0,
            linewidth=0.8
        )


        # ----------------------------------------------------
        # TITLES AND LABELS
        # ----------------------------------------------------

        ax.set_title(
            "Observed second-level cyclical positions "
            "of transaction time"
        )


        ax.set_xlabel(
            "Cosine component"
        )


        ax.set_ylabel(
            "Sine component"
        )


        # ----------------------------------------------------
        # AXIS LIMITS
        # ----------------------------------------------------

        ax.set_xlim(
            -1.15,
            1.15
        )


        ax.set_ylim(
            -1.15,
            1.15
        )


        # ----------------------------------------------------
        # PRESERVE CIRCLE GEOMETRY
        # ----------------------------------------------------

        ax.set_aspect(
            "equal",
            adjustable="box"
        )


        ax.grid(
            alpha=0.3
        )


        # ----------------------------------------------------
        # NUMBER OF DISTINCT POSITIONS
        # ----------------------------------------------------

        ax.text(
            0.02,
            0.02,
            (
                "Distinct temporal positions: "
                f"{number_unique_cyclical_points}"
            ),
            transform=ax.transAxes,
            verticalalignment="bottom"
        )


        fig.tight_layout()


        # ----------------------------------------------------
        # SAVE FIGURE
        # ----------------------------------------------------

        fig.savefig(
            UNIT_CIRCLE_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nUnit-circle chart created:"
        )


        print(
            UNIT_CIRCLE_CHART_PATH
        )


        print(
            "Distinct temporal positions plotted:",
            number_unique_cyclical_points
        )


    else:

        print(
            "\nUnit-circle chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 37. CREATE TIME RECONSTRUCTION ERROR CHART
    # ========================================================

    if not time_reconstruction_error_chart_exists:

        fig, ax = plt.subplots(
            figsize=(
                10,
                6
            )
        )


        ax.hist(
            temporal_reconstruction_error_seconds,
            bins=50
        )


        ax.set_title(
            "Temporal reconstruction error"
        )


        ax.set_xlabel(
            "Absolute reconstruction error in seconds"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(y)
                )
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            TIME_RECONSTRUCTION_ERROR_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nTime reconstruction error chart created:"
        )


        print(
            TIME_RECONSTRUCTION_ERROR_CHART_PATH
        )


    else:

        print(
            "\nTime reconstruction error chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 38. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 39. PREPARE HTML TABLES
    # ========================================================

    component_statistics_html = (
        component_statistics
        .to_html(
            index=False,
            border=0,
            formatters={
                "SIN":
                    lambda x:
                    f"{x:.12f}",

                "COS":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    time_reconstruction_statistics_html = (
        time_reconstruction_statistics
        .to_html(
            index=False,
            border=0,
            formatters={
                "ERROR_SECONDS":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    hour_distribution_html = (
        hour_distribution
        .to_html(
            index=False,
            border=0,
            formatters={
                "PERCENTAGE":
                    lambda x:
                    f"{x:.6f}%"
            }
        )
    )


    minute_distribution_html = (
        minute_distribution
        .to_html(
            index=False,
            border=0,
            formatters={
                "PERCENTAGE":
                    lambda x:
                    f"{x:.6f}%"
            }
        )
    )


    second_distribution_html = (
        second_distribution
        .to_html(
            index=False,
            border=0,
            formatters={
                "PERCENTAGE":
                    lambda x:
                    f"{x:.6f}%"
            }
        )
    )


    # ========================================================
    # 40. CREATE HTML REPORT
    # ========================================================

    if not html_exists:

        hour_distribution_chart_base64 = (
            image_to_base64(
                HOUR_DISTRIBUTION_CHART_PATH
            )
        )


        minute_distribution_chart_base64 = (
            image_to_base64(
                MINUTE_DISTRIBUTION_CHART_PATH
            )
        )


        second_distribution_chart_base64 = (
            image_to_base64(
                SECOND_DISTRIBUTION_CHART_PATH
            )
        )


        unit_circle_chart_base64 = (
            image_to_base64(
                UNIT_CIRCLE_CHART_PATH
            )
        )


        time_reconstruction_error_chart_base64 = (
            image_to_base64(
                TIME_RECONSTRUCTION_ERROR_CHART_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_NAME}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Individual Exploratory Analysis —
TRANS_HOUR_SIN and TRANS_HOUR_COS
</h1>


<p>

The variables
<strong>{SIN_COLUMN}</strong>
and
<strong>{COS_COLUMN}</strong>
jointly represent the complete transaction time
using cyclical encoding based on sine and cosine.

The original transformation preserves
<strong>hour, minute, and second</strong>
information.

</p>


<p class="result">

decimal hour =
hour + minute / 60 + second / 3600

</p>


<!-- ========================================================
     1. FEATURE PAIR OVERVIEW
========================================================= -->


<h2>
1. Feature pair overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>Valid SIN/COS pairs</td>
<td>{valid_observations}</td>
</tr>

<tr>
<td>Missing SIN values</td>
<td>{sin_missing_values}</td>
</tr>

<tr>
<td>Missing COS values</td>
<td>{cos_missing_values}</td>
</tr>

<tr>
<td>Rows with incomplete pairs</td>
<td>{pair_missing_values}</td>
</tr>

<tr>
<td>Unique SIN values</td>
<td>{unique_sin_values}</td>
</tr>

<tr>
<td>Unique COS values</td>
<td>{unique_cos_values}</td>
</tr>

<tr>
<td>Unique stored SIN/COS pairs</td>
<td>{unique_pairs}</td>
</tr>

<tr>
<td>Distinct temporal positions plotted</td>
<td>{number_unique_cyclical_points}</td>
</tr>

<tr>
<td>Unique reconstructed second-level times</td>
<td>{unique_reconstructed_seconds}</td>
</tr>

<tr>
<td>Possible second-level positions per day</td>
<td>{SECONDS_PER_DAY}</td>
</tr>

<tr>
<td>Observed time-position coverage</td>
<td>{time_position_coverage_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     2. COMPONENT DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
2. Component descriptive statistics
</h2>


<p>

The sine and cosine components are bounded
numerical representations of continuous
time throughout the 24-hour daily cycle.

</p>


{component_statistics_html}


<!-- ========================================================
     3. RANGE VALIDATION
========================================================= -->


<h2>
3. Range validation
</h2>


<p>

For valid sine and cosine transformations,
both components must remain within:

</p>


<p class="result">

-1 ≤ SIN ≤ 1

<br>

-1 ≤ COS ≤ 1

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>SIN values below -1</td>
<td>{sin_below_minus_one}</td>
</tr>

<tr>
<td>SIN values above 1</td>
<td>{sin_above_one}</td>
</tr>

<tr>
<td>Total SIN values outside range</td>
<td>{sin_outside_range}</td>
</tr>

<tr>
<td>COS values below -1</td>
<td>{cos_below_minus_one}</td>
</tr>

<tr>
<td>COS values above 1</td>
<td>{cos_above_one}</td>
</tr>

<tr>
<td>Total COS values outside range</td>
<td>{cos_outside_range}</td>
</tr>

</table>


<!-- ========================================================
     4. UNIT-CIRCLE VALIDATION
========================================================= -->


<h2>
4. Unit-circle validation
</h2>


<p>

A correctly encoded sine/cosine pair should satisfy:

</p>


<p class="result">

SIN² + COS² ≈ 1

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Minimum SIN² + COS²</td>
<td>{radius_squared_minimum:.12f}</td>
</tr>

<tr>
<td>Mean SIN² + COS²</td>
<td>{radius_squared_mean:.12f}</td>
</tr>

<tr>
<td>Maximum SIN² + COS²</td>
<td>{radius_squared_maximum:.12f}</td>
</tr>

<tr>
<td>Mean absolute deviation from 1</td>
<td>{mean_absolute_radius_squared_error:.12f}</td>
</tr>

<tr>
<td>Maximum absolute deviation from 1</td>
<td>{maximum_absolute_radius_squared_error:.12f}</td>
</tr>

<tr>
<td>Validation tolerance</td>
<td>{UNIT_CIRCLE_TOLERANCE}</td>
</tr>

<tr>
<td>Observations outside tolerance</td>
<td>{observations_outside_unit_circle_tolerance}</td>
</tr>

<tr>
<td>Validation percentage</td>
<td>{unit_circle_valid_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     5. TIME RECONSTRUCTION VALIDATION
========================================================= -->


<h2>
5. Time reconstruction validation
</h2>


<p>

The original time position was reconstructed
from the stored sine and cosine coordinates.

The reconstruction preserves the
<strong>HH:MM:SS</strong>
temporal resolution.

</p>


<p class="result">

angle = atan2(SIN, COS)

</p>


<p class="result">

decimal hour =
angle × 24 / (2π)

</p>


{time_reconstruction_statistics_html}


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Unique reconstructed times</td>
<td>{unique_reconstructed_seconds}</td>
</tr>

<tr>
<td>Possible second-level positions</td>
<td>{SECONDS_PER_DAY}</td>
</tr>

<tr>
<td>Observed time-position coverage</td>
<td>{time_position_coverage_percentage:.6f}%</td>
</tr>

<tr>
<td>Time reconstruction tolerance</td>
<td>{TIME_RECONSTRUCTION_TOLERANCE_SECONDS:.6f} seconds</td>
</tr>

<tr>
<td>Observations within tolerance</td>
<td>{observations_within_time_tolerance}</td>
</tr>

<tr>
<td>Observations outside tolerance</td>
<td>{observations_outside_time_tolerance}</td>
</tr>

<tr>
<td>Validation percentage</td>
<td>{time_reconstruction_valid_percentage:.6f}%</td>
</tr>

<tr>
<td>Minimum coordinate alignment error</td>
<td>{minimum_coordinate_error:.12f}</td>
</tr>

<tr>
<td>Mean coordinate alignment error</td>
<td>{mean_coordinate_error:.12f}</td>
</tr>

<tr>
<td>Maximum coordinate alignment error</td>
<td>{maximum_coordinate_error:.12f}</td>
</tr>

</table>


<div class="chart">

<img
    src="data:image/png;base64,{time_reconstruction_error_chart_base64}"
    alt="Temporal reconstruction error"
>

</div>


<!-- ========================================================
     6. DISTRIBUTION BY HOUR
========================================================= -->


<h2>
6. Distribution by hour
</h2>


<p>

The reconstructed observations are grouped
according to the hour of the day.

Minute and second information remains preserved
in the original cyclical representation.

</p>


{hour_distribution_html}


<div class="chart">

<img
    src="data:image/png;base64,{hour_distribution_chart_base64}"
    alt="Transaction distribution by hour"
>

</div>


<!-- ========================================================
     7. DISTRIBUTION BY MINUTE
========================================================= -->


<h2>
7. Distribution by minute
</h2>


<p>

The reconstructed observations are grouped according
to the minute position within each hour.

All minute values from
<strong>00 to 59</strong>
are evaluated independently of the hour.

</p>


{minute_distribution_html}


<div class="chart">

<img
    src="data:image/png;base64,{minute_distribution_chart_base64}"
    alt="Transaction distribution by minute"
>

</div>


<!-- ========================================================
     8. DISTRIBUTION BY SECOND
========================================================= -->


<h2>
8. Distribution by second
</h2>


<p>

The reconstructed observations are grouped according
to the second position within each minute.

All second values from
<strong>00 to 59</strong>
are evaluated independently of the minute and hour.

</p>


{second_distribution_html}


<div class="chart">

<img
    src="data:image/png;base64,{second_distribution_chart_base64}"
    alt="Transaction distribution by second"
>

</div>


<!-- ========================================================
     9. CONTINUOUS CYCLICAL REPRESENTATION
========================================================= -->


<h2>
9. Continuous cyclical representation
</h2>


<p>

Each point in the chart represents one distinct
time position actually observed in the dataset.

The complete transaction time is preserved,
including
<strong>hour, minute, and second</strong>.

Therefore, observations are not restricted
to the 24 integer-hour positions.

</p>


<p>

Repeated transactions occurring at the same exact
HH:MM:SS position share the same cyclical coordinate
and therefore overlap at the same point.

</p>


<p>

A total of
<strong>{number_unique_cyclical_points}</strong>
distinct temporal positions are represented
in the chart.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{unit_circle_chart_base64}"
    alt="Observed second-level cyclical positions of transaction time"
>

</div>


<!-- ========================================================
     10. TEMPORAL STEP AND MIDNIGHT VALIDATION
========================================================= -->


<h2>
10. Temporal step and midnight-boundary validation
</h2>


<p>

Cyclical encoding should preserve temporal proximity
across the midnight boundary.

The transition from
<strong>23:59:59 to 00:00:00</strong>
should therefore have the same geometric distance
as any other one-second transition.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Theoretical distance for 1 second</td>
<td>{one_second_distance:.12f}</td>
</tr>

<tr>
<td>23:59:59 → 00:00:00 distance</td>
<td>{boundary_distance:.12f}</td>
</tr>

<tr>
<td>Absolute boundary-distance error</td>
<td>{boundary_distance_error:.12f}</td>
</tr>

<tr>
<td>Theoretical distance for 1 minute</td>
<td>{one_minute_distance:.12f}</td>
</tr>

<tr>
<td>Theoretical distance for 1 hour</td>
<td>{one_hour_distance:.12f}</td>
</tr>

</table>


<!-- ========================================================
     11. CIRCULAR STATISTICS
========================================================= -->


<h2>
11. Circular statistics
</h2>


<p>

Circular statistics summarize the transaction-time
distribution while preserving the periodic structure
of the 24-hour clock.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Mean sine component</td>
<td>{mean_sin:.12f}</td>
</tr>

<tr>
<td>Mean cosine component</td>
<td>{mean_cos:.12f}</td>
</tr>

<tr>
<td>Circular mean angle</td>
<td>{circular_mean_degrees:.6f}°</td>
</tr>

<tr>
<td>Circular mean time</td>
<td>{circular_mean_time_label}</td>
</tr>

<tr>
<td>Mean resultant length</td>
<td>{mean_resultant_length:.12f}</td>
</tr>

<tr>
<td>Circular variance</td>
<td>{circular_variance:.12f}</td>
</tr>

<tr>
<td>Pearson correlation between SIN and COS</td>
<td>{sin_cos_correlation:.12f}</td>
</tr>

</table>


<div class="note">

<strong>Mean resultant length interpretation:</strong>

<br><br>

Values closer to 0 indicate greater dispersion
of observations throughout the daily cycle.

<br><br>

Values closer to 1 indicate stronger concentration
around a particular region of the daily cycle.

<br><br>

Circular variance is calculated as:

<br><br>

<strong>
1 - mean resultant length
</strong>

</div>


<!-- ========================================================
     12. SUMMARY
========================================================= -->


<h2>
12. Summary of results
</h2>


<ul>

<li>
<strong>Total observations:</strong>
{total_observations}
</li>

<li>
<strong>Valid SIN/COS pairs:</strong>
{valid_observations}
</li>

<li>
<strong>Temporal resolution:</strong>
Hour, minute, and second
</li>

<li>
<strong>Distinct cyclical positions observed:</strong>
{number_unique_cyclical_points}
</li>

<li>
<strong>Unique reconstructed second-level times:</strong>
{unique_reconstructed_seconds}
</li>

<li>
<strong>Observed second-level coverage:</strong>
{time_position_coverage_percentage:.6f}%
</li>

<li>
<strong>SIN values outside [-1, 1]:</strong>
{sin_outside_range}
</li>

<li>
<strong>COS values outside [-1, 1]:</strong>
{cos_outside_range}
</li>

<li>
<strong>Maximum unit-circle error:</strong>
{maximum_absolute_radius_squared_error:.12f}
</li>

<li>
<strong>Unit-circle validation percentage:</strong>
{unit_circle_valid_percentage:.6f}%
</li>

<li>
<strong>Maximum temporal reconstruction error:</strong>
{maximum_time_error_seconds:.12f} seconds
</li>

<li>
<strong>Temporal reconstruction validation:</strong>
{time_reconstruction_valid_percentage:.6f}%
</li>

<li>
<strong>23:59:59 → 00:00:00 distance:</strong>
{boundary_distance:.12f}
</li>

<li>
<strong>Theoretical one-second distance:</strong>
{one_second_distance:.12f}
</li>

<li>
<strong>Circular mean time:</strong>
{circular_mean_time_label}
</li>

<li>
<strong>Mean resultant length:</strong>
{mean_resultant_length:.12f}
</li>

<li>
<strong>Circular variance:</strong>
{circular_variance:.12f}
</li>

</ul>


</body>

</html>
"""


        # ====================================================
        # 41. SAVE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )


        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 42. DISPLAY HOURLY DISTRIBUTION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "HOURLY DISTRIBUTION"
    )


    print(
        "=" * 100
    )


    display(
        hour_distribution
    )


    # ========================================================
    # 43. DISPLAY MINUTE DISTRIBUTION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "MINUTE DISTRIBUTION"
    )


    print(
        "=" * 100
    )


    display(
        minute_distribution
    )


    # ========================================================
    # 44. DISPLAY SECOND DISTRIBUTION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "SECOND DISTRIBUTION"
    )


    print(
        "=" * 100
    )


    display(
        second_distribution
    )


    # ========================================================
    # 45. DISPLAY MAIN VALIDATION RESULTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "TRANS_HOUR CYCLICAL ENCODING SUMMARY"
    )


    print(
        "=" * 100
    )


    print(
        "Total observations:",
        total_observations
    )


    print(
        "Valid SIN/COS pairs:",
        valid_observations
    )


    print(
        "Distinct cyclical positions plotted:",
        number_unique_cyclical_points
    )


    print(
        "Unique reconstructed second-level times:",
        unique_reconstructed_seconds
    )


    print(
        "Observed second-level coverage:",
        f"{time_position_coverage_percentage:.6f}%"
    )


    print(
        "Unit-circle validation:",
        f"{unit_circle_valid_percentage:.6f}%"
    )


    print(
        "Time reconstruction validation:",
        f"{time_reconstruction_valid_percentage:.6f}%"
    )


    print(
        "Maximum time reconstruction error:",
        f"{maximum_time_error_seconds:.12f}",
        "seconds"
    )


    print(
        "Circular mean time:",
        circular_mean_time_label
    )


    # ========================================================
    # 46. DISPLAY MIDNIGHT-BOUNDARY VALIDATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "MIDNIGHT-BOUNDARY VALIDATION"
    )


    print(
        "=" * 100
    )


    print(
        "Theoretical one-second distance:",
        f"{one_second_distance:.12f}"
    )


    print(
        "23:59:59 -> 00:00:00 distance:",
        f"{boundary_distance:.12f}"
    )


    print(
        "Absolute difference:",
        f"{boundary_distance_error:.12f}"
    )


    print(
        "Theoretical one-minute distance:",
        f"{one_minute_distance:.12f}"
    )


    print(
        "Theoretical one-hour distance:",
        f"{one_hour_distance:.12f}"
    )


    # ========================================================
    # 47. DISPLAY CIRCULAR STATISTICS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "CIRCULAR STATISTICS"
    )


    print(
        "=" * 100
    )


    print(
        "Circular mean angle:",
        f"{circular_mean_degrees:.6f} degrees"
    )


    print(
        "Circular mean time:",
        circular_mean_time_label
    )


    print(
        "Mean resultant length:",
        f"{mean_resultant_length:.12f}"
    )


    print(
        "Circular variance:",
        f"{circular_variance:.12f}"
    )


    print(
        "SIN/COS Pearson correlation:",
        f"{sin_cos_correlation:.12f}"
    )


    # ========================================================
    # 48. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del sin_feature
    del cos_feature
    del sin_valid
    del cos_valid
    del radius_squared
    del radius_squared_error
    del unique_cyclical_points
    del reconstructed_decimal_hour
    del reconstructed_continuous_seconds
    del temporal_reconstruction_error_seconds
    del coordinate_alignment_error
    del theoretical_sin
    del theoretical_cos
    del theoretical_angles

    gc.collect()


    # ========================================================
    # 49. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "ANALYSIS COMPLETED"
    )


    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )


    print(
        HTML_PATH
    )


    print(
        "\nPNGs:"
    )


    print(
        HOUR_DISTRIBUTION_CHART_PATH
    )


    print(
        MINUTE_DISTRIBUTION_CHART_PATH
    )


    print(
        SECOND_DISTRIBUTION_CHART_PATH
    )


    print(
        UNIT_CIRCLE_CHART_PATH
    )


    print(
        TIME_RECONSTRUCTION_ERROR_CHART_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Hour distribution chart: Will be created
Minute distribution chart: Will be created
Second distribution chart: Will be created
Unit-circle chart: Will be created
Time reconstruction error chart: Will be created

Hour distribution chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/cyclical_encoding_using_sine_and_cosine/trans_hour_sin_cos/trans_hour_sin_cos_hour_distribution.png

Minute distribution chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/cyclical_encoding_using_sine_and_cosine/trans_hour_sin_cos/trans_hour_sin_cos_minute_distribution.png

Second distribution chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/cyclical_encoding_using_sine_and_cosine/trans_hour_sin_cos/trans_hour_sin_cos_second_distribution.png

Unit-circle chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/cyclical_encoding_using_s

,HOUR,HOUR_LABEL,COUNT,PERCENTAGE
0,0,00:00,60655,3.274411
1,1,01:00,61330,3.310851
2,2,02:00,60796,3.282023
3,3,03:00,60968,3.291308
4,4,04:00,59938,3.235705
5,5,05:00,60088,3.243802
6,6,06:00,60406,3.260969
7,7,07:00,60301,3.255301
8,8,08:00,60498,3.265936
9,9,09:00,60231,3.251522



MINUTE DISTRIBUTION


,MINUTE,MINUTE_LABEL,COUNT,PERCENTAGE
0,0,00,30578,1.650729
1,1,01,31096,1.678693
2,2,02,30928,1.669623
3,3,03,31303,1.689867
4,4,04,31093,1.678531
5,5,05,30986,1.672754
6,6,06,30723,1.658556
7,7,07,31080,1.677829
8,8,08,30812,1.663361
9,9,09,30815,1.663523



SECOND DISTRIBUTION


,SECOND,SECOND_LABEL,COUNT,PERCENTAGE
0,0,00,30563,1.649919
1,1,01,31077,1.677667
2,2,02,30877,1.666870
3,3,03,31047,1.676047
4,4,04,31097,1.678747
5,5,05,30803,1.662875
6,6,06,30905,1.668382
7,7,07,31171,1.682741
8,8,08,30754,1.660230
9,9,09,30586,1.651161



TRANS_HOUR CYCLICAL ENCODING SUMMARY
Total observations: 1852394
Valid SIN/COS pairs: 1852394
Distinct cyclical positions plotted: 86400
Unique reconstructed second-level times: 86400
Observed second-level coverage: 100.000000%
Unit-circle validation: 100.000000%
Time reconstruction validation: 100.000000%
Maximum time reconstruction error: 0.000569171461 seconds
Circular mean time: 18:05:38

MIDNIGHT-BOUNDARY VALIDATION
Theoretical one-second distance: 0.000072722052
23:59:59 -> 00:00:00 distance: 0.000072722052
Absolute difference: 0.000000000000
Theoretical one-minute distance: 0.004363319669
Theoretical one-hour distance: 0.261052384440

CIRCULAR STATISTICS
Circular mean angle: 271.408449 degrees
Circular mean time: 18:05:38
Mean resultant length: 0.137382278464
Circular variance: 0.862617721536
SIN/COS Pearson correlation: 0.000774869279

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/cyclical_encoding_using_sine_and_

## <span style="color:grey"> CONTINUOUS GEOGRAPHIC</span> ##

### <span style="color:grey"> RECIVE_LONG </span> ###

In [10]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

FEATURE_GROUP = "comum_features"

FEATURE_TYPE = "continuous_geographic"

FEATURE_NAME = "recive_long"

FEATURE_COLUMN = "RECEIVE_LONG"

VALID_MINIMUM = -180.0

VALID_MAXIMUM = 180.0

ALPHA = 0.05


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / FEATURE_GROUP
    / FEATURE_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


HISTOGRAM_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_histogram.png"
)


BOXPLOT_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_boxplot.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


histogram_exists = (
    HISTOGRAM_PATH.exists()
)


boxplot_exists = (
    BOXPLOT_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and histogram_exists
    and boxplot_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )


    print(
        "No analysis or file creation is required."
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nExisting files:"
    )


    print(
        HTML_PATH
    )


    print(
        HISTOGRAM_PATH
    )


    print(
        BOXPLOT_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )


    print(
        "=" * 100
    )


    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )


    print(
        "Histogram:",
        "Already exists"
        if histogram_exists
        else "Will be created"
    )


    print(
        "Boxplot:",
        "Already exists"
        if boxplot_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. BASIC FEATURE OVERVIEW
    # ========================================================

    total_observations = int(
        len(
            feature
        )
    )


    if total_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    missing_values = int(
        feature
        .isna()
        .sum()
    )


    missing_percentage = (
        missing_values
        / total_observations
        * 100
    )


    valid_feature = (
        feature
        .dropna()
        .astype(
            "float64"
        )
    )


    valid_observations = int(
        len(
            valid_feature
        )
    )


    if valid_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no valid observations."
        )


    unique_values = int(
        valid_feature
        .nunique()
    )


    unique_percentage = (
        unique_values
        / valid_observations
        * 100
    )


    # ========================================================
    # 13. DESCRIPTIVE STATISTICS
    # ========================================================

    minimum_value = float(
        valid_feature.min()
    )


    q1_value = float(
        valid_feature.quantile(
            0.25
        )
    )


    median_value = float(
        valid_feature.median()
    )


    mean_value = float(
        valid_feature.mean()
    )


    q3_value = float(
        valid_feature.quantile(
            0.75
        )
    )


    maximum_value = float(
        valid_feature.max()
    )


    standard_deviation = float(
        valid_feature.std()
    )


    variance = float(
        valid_feature.var()
    )


    data_range = float(
        maximum_value
        - minimum_value
    )


    descriptive_statistics = pd.DataFrame({

        "STATISTIC": [
            "Minimum",
            "Q1",
            "Median",
            "Mean",
            "Q3",
            "Maximum",
            "Standard deviation",
            "Variance",
            "Range"
        ],

        "VALUE": [
            minimum_value,
            q1_value,
            median_value,
            mean_value,
            q3_value,
            maximum_value,
            standard_deviation,
            variance,
            data_range
        ]
    })


    # ========================================================
    # 14. PERCENTILES
    # ========================================================

    percentile_levels = [
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]


    percentile_values = (
        valid_feature
        .quantile(
            percentile_levels
        )
    )


    percentiles_table = pd.DataFrame({

        "PERCENTILE": [
            "P1",
            "P5",
            "P10",
            "P25",
            "P50",
            "P75",
            "P90",
            "P95",
            "P99"
        ],

        "VALUE": [
            float(
                percentile_values.loc[
                    percentile
                ]
            )
            for percentile in percentile_levels
        ]
    })


    # ========================================================
    # 15. GEOGRAPHIC RANGE VALIDATION
    #
    # Longitude must remain within:
    #
    # -180 <= longitude <= 180
    # ========================================================

    values_below_valid_range = int(
        (
            valid_feature
            < VALID_MINIMUM
        )
        .sum()
    )


    values_above_valid_range = int(
        (
            valid_feature
            > VALID_MAXIMUM
        )
        .sum()
    )


    invalid_geographic_values = (
        values_below_valid_range
        + values_above_valid_range
    )


    invalid_geographic_percentage = (
        invalid_geographic_values
        / valid_observations
        * 100
    )


    valid_geographic_values = (
        valid_observations
        - invalid_geographic_values
    )


    valid_geographic_percentage = (
        valid_geographic_values
        / valid_observations
        * 100
    )


    # ========================================================
    # 16. LONGITUDE ORIENTATION
    #
    # Negative longitude:
    # West of the Greenwich meridian.
    #
    # Positive longitude:
    # East of the Greenwich meridian.
    # ========================================================

    negative_longitudes = int(
        (
            valid_feature
            < 0
        )
        .sum()
    )


    zero_longitudes = int(
        (
            valid_feature
            == 0
        )
        .sum()
    )


    positive_longitudes = int(
        (
            valid_feature
            > 0
        )
        .sum()
    )


    negative_longitude_percentage = (
        negative_longitudes
        / valid_observations
        * 100
    )


    zero_longitude_percentage = (
        zero_longitudes
        / valid_observations
        * 100
    )


    positive_longitude_percentage = (
        positive_longitudes
        / valid_observations
        * 100
    )


    # ========================================================
    # 17. DISTRIBUTION SHAPE
    # ========================================================

    skewness = float(
        valid_feature.skew()
    )


    kurtosis = float(
        valid_feature.kurt()
    )


    distribution_shape_table = pd.DataFrame({

        "METRIC": [
            "Skewness",
            "Kurtosis"
        ],

        "VALUE": [
            skewness,
            kurtosis
        ]
    })


    # ========================================================
    # 18. SHAPIRO-WILK NORMALITY TEST
    #
    # H0:
    # The feature follows a normal distribution.
    #
    # H1:
    # The feature does not follow a normal distribution.
    #
    # The complete valid feature is used.
    # No random sampling is performed.
    # ========================================================

    (
        shapiro_statistic,
        shapiro_p_value
    ) = stats.shapiro(
        valid_feature.to_numpy()
    )


    shapiro_statistic = float(
        shapiro_statistic
    )


    shapiro_p_value = float(
        shapiro_p_value
    )


    # ========================================================
    # 19. SHAPIRO-WILK DECISION
    # ========================================================

    if shapiro_p_value < ALPHA:

        shapiro_decision = (
            "Reject H0"
        )


        shapiro_interpretation = (
            "The data provide statistical evidence "
            "against a normal distribution."
        )


    else:

        shapiro_decision = (
            "Fail to reject H0"
        )


        shapiro_interpretation = (
            "The data do not provide sufficient "
            "statistical evidence against a normal distribution."
        )


    # ========================================================
    # 20. JARQUE-BERA NORMALITY TEST
    #
    # H0:
    # The feature follows a normal distribution.
    #
    # H1:
    # The feature does not follow a normal distribution.
    # ========================================================

    jarque_bera_result = stats.jarque_bera(
        valid_feature.to_numpy()
    )


    jarque_bera_statistic = float(
        jarque_bera_result.statistic
    )


    jarque_bera_p_value = float(
        jarque_bera_result.pvalue
    )


    # ========================================================
    # 21. JARQUE-BERA DECISION
    # ========================================================

    if jarque_bera_p_value < ALPHA:

        jarque_bera_decision = (
            "Reject H0"
        )


        jarque_bera_interpretation = (
            "The data provide statistical evidence "
            "against a normal distribution."
        )


    else:

        jarque_bera_decision = (
            "Fail to reject H0"
        )


        jarque_bera_interpretation = (
            "The data do not provide sufficient "
            "statistical evidence against a normal distribution."
        )


    # ========================================================
    # 22. NORMALITY TEST RESULTS TABLE
    # ========================================================

    normality_tests_table = pd.DataFrame({

        "TEST": [
            "Shapiro-Wilk",
            "Jarque-Bera"
        ],

        "STATISTIC": [
            shapiro_statistic,
            jarque_bera_statistic
        ],

        "P_VALUE": [
            shapiro_p_value,
            jarque_bera_p_value
        ],

        "ALPHA": [
            ALPHA,
            ALPHA
        ],

        "DECISION": [
            shapiro_decision,
            jarque_bera_decision
        ]
    })


    # ========================================================
    # 23. IQR OUTLIER ANALYSIS
    #
    # IMPORTANT:
    #
    # IQR outliers are statistical extremes only.
    #
    # For geographic coordinates, they should not
    # automatically be interpreted as invalid observations.
    # ========================================================

    iqr = (
        q3_value
        - q1_value
    )


    lower_iqr_bound = (
        q1_value
        - 1.5
        * iqr
    )


    upper_iqr_bound = (
        q3_value
        + 1.5
        * iqr
    )


    lower_iqr_outliers = int(
        (
            valid_feature
            < lower_iqr_bound
        )
        .sum()
    )


    upper_iqr_outliers = int(
        (
            valid_feature
            > upper_iqr_bound
        )
        .sum()
    )


    total_iqr_outliers = (
        lower_iqr_outliers
        + upper_iqr_outliers
    )


    iqr_outlier_percentage = (
        total_iqr_outliers
        / valid_observations
        * 100
    )


    observations_inside_iqr_limits = (
        valid_observations
        - total_iqr_outliers
    )


    observations_inside_iqr_percentage = (
        observations_inside_iqr_limits
        / valid_observations
        * 100
    )


    iqr_table = pd.DataFrame({

        "METRIC": [
            "Q1",
            "Q3",
            "IQR",
            "Lower IQR bound",
            "Upper IQR bound",
            "Lower statistical outliers",
            "Upper statistical outliers",
            "Total statistical outliers",
            "Statistical outlier percentage",
            "Observations inside IQR limits",
            "Percentage inside IQR limits"
        ],

        "VALUE": [
            q1_value,
            q3_value,
            iqr,
            lower_iqr_bound,
            upper_iqr_bound,
            lower_iqr_outliers,
            upper_iqr_outliers,
            total_iqr_outliers,
            iqr_outlier_percentage,
            observations_inside_iqr_limits,
            observations_inside_iqr_percentage
        ]
    })


    # ========================================================
    # 24. CREATE HISTOGRAM
    # ========================================================

    if not histogram_exists:

        fig, ax = plt.subplots(
            figsize=(
                11,
                6
            )
        )


        ax.hist(
            valid_feature,
            bins=60
        )


        ax.set_title(
            "Distribution of receiver longitude"
        )


        ax.set_xlabel(
            "Longitude"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(y)
                )
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            HISTOGRAM_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nHistogram created:"
        )


        print(
            HISTOGRAM_PATH
        )


    else:

        print(
            "\nHistogram already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 25. CREATE BOXPLOT
    # ========================================================

    if not boxplot_exists:

        fig, ax = plt.subplots(
            figsize=(
                11,
                4
            )
        )


        ax.boxplot(
            valid_feature,
            vert=False
        )


        ax.set_title(
            "Boxplot of receiver longitude"
        )


        ax.set_xlabel(
            "Longitude"
        )


        ax.set_yticks(
            []
        )


        ax.grid(
            axis="x",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            BOXPLOT_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nBoxplot created:"
        )


        print(
            BOXPLOT_PATH
        )


    else:

        print(
            "\nBoxplot already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 26. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 27. PREPARE DESCRIPTIVE STATISTICS TABLE FOR HTML
    # ========================================================

    descriptive_statistics_html = (
        descriptive_statistics
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    # ========================================================
    # 28. PREPARE PERCENTILES TABLE FOR HTML
    # ========================================================

    percentiles_html = (
        percentiles_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    # ========================================================
    # 29. PREPARE DISTRIBUTION SHAPE TABLE FOR HTML
    # ========================================================

    distribution_shape_html = (
        distribution_shape_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    # ========================================================
    # 30. PREPARE NORMALITY TEST TABLE FOR HTML
    # ========================================================

    normality_tests_html = (
        normality_tests_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "STATISTIC":
                    lambda x:
                    f"{x:.12f}",

                "P_VALUE":
                    lambda x:
                    f"{x:.12e}",

                "ALPHA":
                    lambda x:
                    f"{x:.2f}"
            }
        )
    )


    # ========================================================
    # 31. CREATE HTML REPORT
    # ========================================================

    if not html_exists:

        histogram_base64 = (
            image_to_base64(
                HISTOGRAM_PATH
            )
        )


        boxplot_base64 = (
            image_to_base64(
                BOXPLOT_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Individual Exploratory Analysis —
{FEATURE_COLUMN}
</h1>


<p>

The variable
<strong>{FEATURE_COLUMN}</strong>
represents the geographic longitude
of the transaction receiver location.

Longitude is a continuous geographic coordinate
measured in degrees.

</p>


<p class="result">

-180° ≤ longitude ≤ 180°

</p>


<!-- ========================================================
     1. FEATURE OVERVIEW
========================================================= -->


<h2>
1. Feature overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>Valid observations</td>
<td>{valid_observations}</td>
</tr>

<tr>
<td>Missing values</td>
<td>{missing_values}</td>
</tr>

<tr>
<td>Missing percentage</td>
<td>{missing_percentage:.6f}%</td>
</tr>

<tr>
<td>Unique values</td>
<td>{unique_values}</td>
</tr>

<tr>
<td>Unique-value percentage</td>
<td>{unique_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     2. DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
2. Descriptive statistics
</h2>


{descriptive_statistics_html}


<!-- ========================================================
     3. PERCENTILES
========================================================= -->


<h2>
3. Percentiles
</h2>


<p>

Percentiles provide a detailed description
of the empirical distribution of receiver
longitude values.

</p>


{percentiles_html}


<!-- ========================================================
     4. GEOGRAPHIC RANGE VALIDATION
========================================================= -->


<h2>
4. Geographic range validation
</h2>


<p>

A valid geographic longitude must remain
within the interval:

</p>


<p class="result">

-180° ≤ longitude ≤ 180°

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Values below -180°</td>
<td>{values_below_valid_range}</td>
</tr>

<tr>
<td>Values above 180°</td>
<td>{values_above_valid_range}</td>
</tr>

<tr>
<td>Total invalid geographic values</td>
<td>{invalid_geographic_values}</td>
</tr>

<tr>
<td>Invalid geographic percentage</td>
<td>{invalid_geographic_percentage:.6f}%</td>
</tr>

<tr>
<td>Valid geographic values</td>
<td>{valid_geographic_values}</td>
</tr>

<tr>
<td>Valid geographic percentage</td>
<td>{valid_geographic_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     5. LONGITUDE ORIENTATION
========================================================= -->


<h2>
5. Longitude orientation
</h2>


<p>

Negative longitude values indicate locations
west of the Greenwich meridian.

Positive longitude values indicate locations
east of the Greenwich meridian.

</p>


<table>

<tr>
<th>Longitude group</th>
<th>Observations</th>
<th>Percentage</th>
</tr>

<tr>
<td>Western longitude (&lt; 0)</td>
<td>{negative_longitudes}</td>
<td>{negative_longitude_percentage:.6f}%</td>
</tr>

<tr>
<td>Greenwich meridian (= 0)</td>
<td>{zero_longitudes}</td>
<td>{zero_longitude_percentage:.6f}%</td>
</tr>

<tr>
<td>Eastern longitude (&gt; 0)</td>
<td>{positive_longitudes}</td>
<td>{positive_longitude_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     6. DISTRIBUTION SHAPE
========================================================= -->


<h2>
6. Distribution shape
</h2>


<p>

Skewness and kurtosis are used to characterize
the empirical shape of the longitude distribution.

</p>


{distribution_shape_html}


<div class="note">

<strong>Interpretation:</strong>

<br><br>

Skewness measures asymmetry in the observed
distribution.

<br><br>

Kurtosis provides information about the shape
and tail behavior of the distribution.

<br><br>

These statistics describe the empirical longitude
distribution and do not imply that geographic
coordinates are expected to follow a normal
distribution.

</div>


<!-- ========================================================
     7. NORMALITY TESTS
========================================================= -->


<h2>
7. Normality tests
</h2>


<p>

The Shapiro-Wilk and Jarque-Bera tests were
applied to evaluate whether the empirical
distribution of
<strong>{FEATURE_COLUMN}</strong>
is statistically compatible with a normal
distribution.

</p>


<p>

The complete set of valid observations was used.

No random sampling was performed.

</p>


<p class="result">

H0: The feature follows a normal distribution.

<br><br>

H1: The feature does not follow a normal distribution.

<br><br>

Significance level: α = {ALPHA}

</p>


{normality_tests_html}


<h3>
Shapiro-Wilk interpretation
</h3>


<p>

<strong>Decision:</strong>
{shapiro_decision}

</p>


<p>

{shapiro_interpretation}

</p>


<h3>
Jarque-Bera interpretation
</h3>


<p>

<strong>Decision:</strong>
{jarque_bera_decision}

</p>


<p>

{jarque_bera_interpretation}

</p>


<div class="note">

<strong>Important:</strong>

<br><br>

Normality tests characterize the statistical
shape of the longitude distribution.

Rejecting the null hypothesis of normality
does not indicate that the geographic coordinates
are invalid.

<br><br>

Geographic validity is evaluated separately
using the valid longitude range from
-180° to 180°.

<br><br>

Because the dataset contains a very large number
of observations, normality tests may detect
small deviations from a theoretical normal
distribution.

Therefore, the test results should be interpreted
together with the histogram, skewness, kurtosis,
and descriptive statistics.

</div>


<!-- ========================================================
     8. IQR OUTLIER ANALYSIS
========================================================= -->


<h2>
8. IQR outlier analysis
</h2>


<p>

The interquartile range method identifies
longitude values located far from the central
portion of the empirical distribution.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Q1</td>
<td>{q1_value:.12f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{q3_value:.12f}</td>
</tr>

<tr>
<td>IQR</td>
<td>{iqr:.12f}</td>
</tr>

<tr>
<td>Lower IQR bound</td>
<td>{lower_iqr_bound:.12f}</td>
</tr>

<tr>
<td>Upper IQR bound</td>
<td>{upper_iqr_bound:.12f}</td>
</tr>

<tr>
<td>Lower statistical outliers</td>
<td>{lower_iqr_outliers}</td>
</tr>

<tr>
<td>Upper statistical outliers</td>
<td>{upper_iqr_outliers}</td>
</tr>

<tr>
<td>Total statistical outliers</td>
<td>{total_iqr_outliers}</td>
</tr>

<tr>
<td>Statistical outlier percentage</td>
<td>{iqr_outlier_percentage:.6f}%</td>
</tr>

<tr>
<td>Observations inside IQR limits</td>
<td>{observations_inside_iqr_limits}</td>
</tr>

<tr>
<td>Percentage inside IQR limits</td>
<td>{observations_inside_iqr_percentage:.6f}%</td>
</tr>

</table>


<div class="note">

<strong>Important:</strong>

<br><br>

A geographic coordinate classified as an IQR
outlier is not necessarily invalid.

It only represents a longitude value that is
statistically distant from the central region
of the observed longitude distribution.

<br><br>

Geographic validity should primarily be evaluated
using the valid interval from
-180° to 180°.

</div>


<!-- ========================================================
     9. HISTOGRAM
========================================================= -->


<h2>
9. Longitude distribution
</h2>


<p>

The histogram presents the empirical distribution
of receiver longitude values across the dataset.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{histogram_base64}"
    alt="Distribution of receiver longitude"
>

</div>


<!-- ========================================================
     10. BOXPLOT
========================================================= -->


<h2>
10. Longitude boxplot
</h2>


<p>

The boxplot summarizes the central distribution
of receiver longitude and highlights observations
classified as statistical extremes according to
the IQR rule.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{boxplot_base64}"
    alt="Boxplot of receiver longitude"
>

</div>


<!-- ========================================================
     11. SUMMARY
========================================================= -->


<h2>
11. Summary of results
</h2>


<ul>

<li>
<strong>Total observations:</strong>
{total_observations}
</li>

<li>
<strong>Valid observations:</strong>
{valid_observations}
</li>

<li>
<strong>Missing values:</strong>
{missing_values}
</li>

<li>
<strong>Missing percentage:</strong>
{missing_percentage:.6f}%
</li>

<li>
<strong>Unique longitude values:</strong>
{unique_values}
</li>

<li>
<strong>Minimum longitude:</strong>
{minimum_value:.12f}
</li>

<li>
<strong>Median longitude:</strong>
{median_value:.12f}
</li>

<li>
<strong>Mean longitude:</strong>
{mean_value:.12f}
</li>

<li>
<strong>Maximum longitude:</strong>
{maximum_value:.12f}
</li>

<li>
<strong>Standard deviation:</strong>
{standard_deviation:.12f}
</li>

<li>
<strong>Skewness:</strong>
{skewness:.12f}
</li>

<li>
<strong>Kurtosis:</strong>
{kurtosis:.12f}
</li>

<li>
<strong>Invalid geographic values:</strong>
{invalid_geographic_values}
</li>

<li>
<strong>Geographically valid observations:</strong>
{valid_geographic_percentage:.6f}%
</li>

<li>
<strong>Shapiro-Wilk statistic:</strong>
{shapiro_statistic:.12f}
</li>

<li>
<strong>Shapiro-Wilk p-value:</strong>
{shapiro_p_value:.12e}
</li>

<li>
<strong>Shapiro-Wilk decision:</strong>
{shapiro_decision}
</li>

<li>
<strong>Jarque-Bera statistic:</strong>
{jarque_bera_statistic:.12f}
</li>

<li>
<strong>Jarque-Bera p-value:</strong>
{jarque_bera_p_value:.12e}
</li>

<li>
<strong>Jarque-Bera decision:</strong>
{jarque_bera_decision}
</li>

<li>
<strong>IQR statistical outliers:</strong>
{total_iqr_outliers}
</li>

<li>
<strong>IQR outlier percentage:</strong>
{iqr_outlier_percentage:.6f}%
</li>

</ul>


</body>

</html>
"""


        # ====================================================
        # 32. SAVE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )


        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 33. DISPLAY FEATURE OVERVIEW
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "RECEIVE_LONG SUMMARY"
    )


    print(
        "=" * 100
    )


    print(
        "Total observations:",
        total_observations
    )


    print(
        "Valid observations:",
        valid_observations
    )


    print(
        "Missing values:",
        missing_values
    )


    print(
        "Missing percentage:",
        f"{missing_percentage:.6f}%"
    )


    print(
        "Unique values:",
        unique_values
    )


    print(
        "Unique-value percentage:",
        f"{unique_percentage:.6f}%"
    )


    # ========================================================
    # 34. DISPLAY DESCRIPTIVE STATISTICS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DESCRIPTIVE STATISTICS"
    )


    print(
        "=" * 100
    )


    display(
        descriptive_statistics
    )


    # ========================================================
    # 35. DISPLAY PERCENTILES
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "PERCENTILES"
    )


    print(
        "=" * 100
    )


    display(
        percentiles_table
    )


    # ========================================================
    # 36. DISPLAY GEOGRAPHIC RANGE VALIDATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "GEOGRAPHIC RANGE VALIDATION"
    )


    print(
        "=" * 100
    )


    print(
        "Valid longitude interval:",
        f"[{VALID_MINIMUM}, {VALID_MAXIMUM}]"
    )


    print(
        "Values below valid range:",
        values_below_valid_range
    )


    print(
        "Values above valid range:",
        values_above_valid_range
    )


    print(
        "Total invalid geographic values:",
        invalid_geographic_values
    )


    print(
        "Invalid geographic percentage:",
        f"{invalid_geographic_percentage:.6f}%"
    )


    print(
        "Geographic validation percentage:",
        f"{valid_geographic_percentage:.6f}%"
    )


    # ========================================================
    # 37. DISPLAY LONGITUDE ORIENTATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "LONGITUDE ORIENTATION"
    )


    print(
        "=" * 100
    )


    print(
        "Western longitudes:",
        negative_longitudes,
        f"({negative_longitude_percentage:.6f}%)"
    )


    print(
        "Greenwich meridian:",
        zero_longitudes,
        f"({zero_longitude_percentage:.6f}%)"
    )


    print(
        "Eastern longitudes:",
        positive_longitudes,
        f"({positive_longitude_percentage:.6f}%)"
    )


    # ========================================================
    # 38. DISPLAY DISTRIBUTION SHAPE
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DISTRIBUTION SHAPE"
    )


    print(
        "=" * 100
    )


    print(
        "Skewness:",
        f"{skewness:.12f}"
    )


    print(
        "Kurtosis:",
        f"{kurtosis:.12f}"
    )


    # ========================================================
    # 39. DISPLAY NORMALITY TESTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "NORMALITY TESTS"
    )


    print(
        "=" * 100
    )


    print(
        "Significance level:",
        ALPHA
    )


    print(
        "\nShapiro-Wilk statistic:",
        f"{shapiro_statistic:.12f}"
    )


    print(
        "Shapiro-Wilk p-value:",
        f"{shapiro_p_value:.12e}"
    )


    print(
        "Shapiro-Wilk decision:",
        shapiro_decision
    )


    print(
        "Shapiro-Wilk interpretation:",
        shapiro_interpretation
    )


    print(
        "\nJarque-Bera statistic:",
        f"{jarque_bera_statistic:.12f}"
    )


    print(
        "Jarque-Bera p-value:",
        f"{jarque_bera_p_value:.12e}"
    )


    print(
        "Jarque-Bera decision:",
        jarque_bera_decision
    )


    print(
        "Jarque-Bera interpretation:",
        jarque_bera_interpretation
    )


    # ========================================================
    # 40. DISPLAY IQR OUTLIER ANALYSIS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "IQR OUTLIER ANALYSIS"
    )


    print(
        "=" * 100
    )


    print(
        "Q1:",
        f"{q1_value:.12f}"
    )


    print(
        "Q3:",
        f"{q3_value:.12f}"
    )


    print(
        "IQR:",
        f"{iqr:.12f}"
    )


    print(
        "Lower bound:",
        f"{lower_iqr_bound:.12f}"
    )


    print(
        "Upper bound:",
        f"{upper_iqr_bound:.12f}"
    )


    print(
        "Lower statistical outliers:",
        lower_iqr_outliers
    )


    print(
        "Upper statistical outliers:",
        upper_iqr_outliers
    )


    print(
        "Total statistical outliers:",
        total_iqr_outliers
    )


    print(
        "Outlier percentage:",
        f"{iqr_outlier_percentage:.6f}%"
    )


    # ========================================================
    # 41. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature
    del valid_feature
    del percentile_values

    gc.collect()


    # ========================================================
    # 42. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "ANALYSIS COMPLETED"
    )


    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )


    print(
        HTML_PATH
    )


    print(
        "\nPNGs:"
    )


    print(
        HISTOGRAM_PATH
    )


    print(
        BOXPLOT_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Histogram: Will be created
Boxplot: Will be created


/usr/local/lib/python3.14/site-packages/scipy/stats/_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 1852394.
  res = hypotest_fun_out(*samples, **kwds)



Histogram created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_geographic/recive_long/recive_long_histogram.png


/tmp/ipykernel_2304/2059015986.py:927: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(



Boxplot created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_geographic/recive_long/recive_long_boxplot.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_geographic/recive_long/analysis_recive_long.html

RECEIVE_LONG SUMMARY
Total observations: 1852394
Valid observations: 1852394
Missing values: 0
Missing percentage: 0.000000%
Unique values: 1559837
Unique-value percentage: 84.206546%

DESCRIPTIVE STATISTICS


,STATISTIC,VALUE
0,Minimum,-166.671570
1,Q1,-96.899443
2,Median,-87.440693
3,Mean,-90.227940
4,Q3,-80.245106
5,Maximum,-66.950905
6,Standard deviation,13.759692
7,Variance,189.329127
8,Range,99.720665



PERCENTILES


,PERCENTILE,VALUE
0,P1,-123.557883
1,P5,-119.309277
2,P10,-111.244813
3,P25,-96.899443
4,P50,-87.440693
5,P75,-80.245106
6,P90,-74.937711
7,P95,-73.365168
8,P99,-70.397263



GEOGRAPHIC RANGE VALIDATION
Valid longitude interval: [-180.0, 180.0]
Values below valid range: 0
Values above valid range: 0
Total invalid geographic values: 0
Invalid geographic percentage: 0.000000%
Geographic validation percentage: 100.000000%

LONGITUDE ORIENTATION
Western longitudes: 1852394 (100.000000%)
Greenwich meridian: 0 (0.000000%)
Eastern longitudes: 0 (0.000000%)

DISTRIBUTION SHAPE
Skewness: -1.143933016854
Kurtosis: 1.831258362841

NORMALITY TESTS
Significance level: 0.05

Shapiro-Wilk statistic: 0.918653855982
Shapiro-Wilk p-value: 6.997694565841e-154
Shapiro-Wilk decision: Reject H0
Shapiro-Wilk interpretation: The data provide statistical evidence against a normal distribution.

Jarque-Bera statistic: 662832.862666248227
Jarque-Bera p-value: 0.000000000000e+00
Jarque-Bera decision: Reject H0
Jarque-Bera interpretation: The data provide statistical evidence against a normal distribution.

IQR OUTLIER ANALYSIS
Q1: -96.899442672729
Q3: -80.245105743408
IQR: 16.6543369

#### <span style="color:grey"> RECIVE_LAT </span> ####

In [11]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

FEATURE_GROUP = "comum_features"

FEATURE_TYPE = "continuous_geographic"

FEATURE_NAME = "recive_lat"

FEATURE_COLUMN = "RECEIVE_LAT"

VALID_MINIMUM = -90.0

VALID_MAXIMUM = 90.0

ALPHA = 0.05


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / FEATURE_GROUP
    / FEATURE_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


HISTOGRAM_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_histogram.png"
)


BOXPLOT_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_boxplot.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


histogram_exists = (
    HISTOGRAM_PATH.exists()
)


boxplot_exists = (
    BOXPLOT_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and histogram_exists
    and boxplot_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )


    print(
        "No analysis or file creation is required."
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nExisting files:"
    )


    print(
        HTML_PATH
    )


    print(
        HISTOGRAM_PATH
    )


    print(
        BOXPLOT_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )


    print(
        "=" * 100
    )


    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )


    print(
        "Histogram:",
        "Already exists"
        if histogram_exists
        else "Will be created"
    )


    print(
        "Boxplot:",
        "Already exists"
        if boxplot_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. BASIC FEATURE OVERVIEW
    # ========================================================

    total_observations = int(
        len(
            feature
        )
    )


    if total_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    missing_values = int(
        feature
        .isna()
        .sum()
    )


    missing_percentage = (
        missing_values
        / total_observations
        * 100
    )


    valid_feature = (
        feature
        .dropna()
        .astype(
            "float64"
        )
    )


    valid_observations = int(
        len(
            valid_feature
        )
    )


    if valid_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no valid observations."
        )


    unique_values = int(
        valid_feature
        .nunique()
    )


    unique_percentage = (
        unique_values
        / valid_observations
        * 100
    )


    # ========================================================
    # 13. DESCRIPTIVE STATISTICS
    # ========================================================

    minimum_value = float(
        valid_feature.min()
    )


    q1_value = float(
        valid_feature.quantile(
            0.25
        )
    )


    median_value = float(
        valid_feature.median()
    )


    mean_value = float(
        valid_feature.mean()
    )


    q3_value = float(
        valid_feature.quantile(
            0.75
        )
    )


    maximum_value = float(
        valid_feature.max()
    )


    standard_deviation = float(
        valid_feature.std()
    )


    variance = float(
        valid_feature.var()
    )


    data_range = float(
        maximum_value
        - minimum_value
    )


    descriptive_statistics = pd.DataFrame({

        "STATISTIC": [
            "Minimum",
            "Q1",
            "Median",
            "Mean",
            "Q3",
            "Maximum",
            "Standard deviation",
            "Variance",
            "Range"
        ],

        "VALUE": [
            minimum_value,
            q1_value,
            median_value,
            mean_value,
            q3_value,
            maximum_value,
            standard_deviation,
            variance,
            data_range
        ]
    })


    # ========================================================
    # 14. PERCENTILES
    # ========================================================

    percentile_levels = [
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]


    percentile_values = (
        valid_feature
        .quantile(
            percentile_levels
        )
    )


    percentiles_table = pd.DataFrame({

        "PERCENTILE": [
            "P1",
            "P5",
            "P10",
            "P25",
            "P50",
            "P75",
            "P90",
            "P95",
            "P99"
        ],

        "VALUE": [
            float(
                percentile_values.loc[
                    percentile
                ]
            )
            for percentile in percentile_levels
        ]
    })


    # ========================================================
    # 15. GEOGRAPHIC RANGE VALIDATION
    #
    # Latitude must remain within:
    #
    # -90 <= latitude <= 90
    # ========================================================

    values_below_valid_range = int(
        (
            valid_feature
            < VALID_MINIMUM
        )
        .sum()
    )


    values_above_valid_range = int(
        (
            valid_feature
            > VALID_MAXIMUM
        )
        .sum()
    )


    invalid_geographic_values = (
        values_below_valid_range
        + values_above_valid_range
    )


    invalid_geographic_percentage = (
        invalid_geographic_values
        / valid_observations
        * 100
    )


    valid_geographic_values = (
        valid_observations
        - invalid_geographic_values
    )


    valid_geographic_percentage = (
        valid_geographic_values
        / valid_observations
        * 100
    )


    # ========================================================
    # 16. LATITUDE ORIENTATION
    #
    # Negative latitude:
    # Southern Hemisphere.
    #
    # Zero latitude:
    # Equator.
    #
    # Positive latitude:
    # Northern Hemisphere.
    # ========================================================

    southern_latitudes = int(
        (
            valid_feature
            < 0
        )
        .sum()
    )


    equator_latitudes = int(
        (
            valid_feature
            == 0
        )
        .sum()
    )


    northern_latitudes = int(
        (
            valid_feature
            > 0
        )
        .sum()
    )


    southern_latitude_percentage = (
        southern_latitudes
        / valid_observations
        * 100
    )


    equator_latitude_percentage = (
        equator_latitudes
        / valid_observations
        * 100
    )


    northern_latitude_percentage = (
        northern_latitudes
        / valid_observations
        * 100
    )


    # ========================================================
    # 17. DISTRIBUTION SHAPE
    # ========================================================

    skewness = float(
        valid_feature.skew()
    )


    kurtosis = float(
        valid_feature.kurt()
    )


    distribution_shape_table = pd.DataFrame({

        "METRIC": [
            "Skewness",
            "Kurtosis"
        ],

        "VALUE": [
            skewness,
            kurtosis
        ]
    })


    # ========================================================
    # 18. SHAPIRO-WILK NORMALITY TEST
    #
    # H0:
    # The feature follows a normal distribution.
    #
    # H1:
    # The feature does not follow a normal distribution.
    #
    # The complete valid feature is used.
    # No random sampling is performed.
    # ========================================================

    (
        shapiro_statistic,
        shapiro_p_value
    ) = stats.shapiro(
        valid_feature.to_numpy()
    )


    shapiro_statistic = float(
        shapiro_statistic
    )


    shapiro_p_value = float(
        shapiro_p_value
    )


    # ========================================================
    # 19. SHAPIRO-WILK DECISION
    # ========================================================

    if shapiro_p_value < ALPHA:

        shapiro_decision = (
            "Reject H0"
        )


        shapiro_interpretation = (
            "The data provide statistical evidence "
            "against a normal distribution."
        )


    else:

        shapiro_decision = (
            "Fail to reject H0"
        )


        shapiro_interpretation = (
            "The data do not provide sufficient "
            "statistical evidence against a normal distribution."
        )


    # ========================================================
    # 20. JARQUE-BERA NORMALITY TEST
    #
    # H0:
    # The feature follows a normal distribution.
    #
    # H1:
    # The feature does not follow a normal distribution.
    # ========================================================

    jarque_bera_result = stats.jarque_bera(
        valid_feature.to_numpy()
    )


    jarque_bera_statistic = float(
        jarque_bera_result.statistic
    )


    jarque_bera_p_value = float(
        jarque_bera_result.pvalue
    )


    # ========================================================
    # 21. JARQUE-BERA DECISION
    # ========================================================

    if jarque_bera_p_value < ALPHA:

        jarque_bera_decision = (
            "Reject H0"
        )


        jarque_bera_interpretation = (
            "The data provide statistical evidence "
            "against a normal distribution."
        )


    else:

        jarque_bera_decision = (
            "Fail to reject H0"
        )


        jarque_bera_interpretation = (
            "The data do not provide sufficient "
            "statistical evidence against a normal distribution."
        )


    # ========================================================
    # 22. NORMALITY TEST RESULTS TABLE
    # ========================================================

    normality_tests_table = pd.DataFrame({

        "TEST": [
            "Shapiro-Wilk",
            "Jarque-Bera"
        ],

        "STATISTIC": [
            shapiro_statistic,
            jarque_bera_statistic
        ],

        "P_VALUE": [
            shapiro_p_value,
            jarque_bera_p_value
        ],

        "ALPHA": [
            ALPHA,
            ALPHA
        ],

        "DECISION": [
            shapiro_decision,
            jarque_bera_decision
        ]
    })


    # ========================================================
    # 23. IQR OUTLIER ANALYSIS
    #
    # IMPORTANT:
    #
    # IQR outliers are statistical extremes only.
    #
    # For geographic coordinates, they should not
    # automatically be interpreted as invalid observations.
    # ========================================================

    iqr = (
        q3_value
        - q1_value
    )


    lower_iqr_bound = (
        q1_value
        - 1.5
        * iqr
    )


    upper_iqr_bound = (
        q3_value
        + 1.5
        * iqr
    )


    lower_iqr_outliers = int(
        (
            valid_feature
            < lower_iqr_bound
        )
        .sum()
    )


    upper_iqr_outliers = int(
        (
            valid_feature
            > upper_iqr_bound
        )
        .sum()
    )


    total_iqr_outliers = (
        lower_iqr_outliers
        + upper_iqr_outliers
    )


    iqr_outlier_percentage = (
        total_iqr_outliers
        / valid_observations
        * 100
    )


    observations_inside_iqr_limits = (
        valid_observations
        - total_iqr_outliers
    )


    observations_inside_iqr_percentage = (
        observations_inside_iqr_limits
        / valid_observations
        * 100
    )


    # ========================================================
    # 24. CREATE HISTOGRAM
    # ========================================================

    if not histogram_exists:

        fig, ax = plt.subplots(
            figsize=(
                11,
                6
            )
        )


        ax.hist(
            valid_feature,
            bins=60
        )


        ax.set_title(
            "Distribution of receiver latitude"
        )


        ax.set_xlabel(
            "Latitude"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(y)
                )
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            HISTOGRAM_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nHistogram created:"
        )


        print(
            HISTOGRAM_PATH
        )


    else:

        print(
            "\nHistogram already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 25. CREATE BOXPLOT
    # ========================================================

    if not boxplot_exists:

        fig, ax = plt.subplots(
            figsize=(
                11,
                4
            )
        )


        ax.boxplot(
            valid_feature,
            vert=False
        )


        ax.set_title(
            "Boxplot of receiver latitude"
        )


        ax.set_xlabel(
            "Latitude"
        )


        ax.set_yticks(
            []
        )


        ax.grid(
            axis="x",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            BOXPLOT_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nBoxplot created:"
        )


        print(
            BOXPLOT_PATH
        )


    else:

        print(
            "\nBoxplot already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 26. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 27. PREPARE DESCRIPTIVE STATISTICS TABLE FOR HTML
    # ========================================================

    descriptive_statistics_html = (
        descriptive_statistics
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    # ========================================================
    # 28. PREPARE PERCENTILES TABLE FOR HTML
    # ========================================================

    percentiles_html = (
        percentiles_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    # ========================================================
    # 29. PREPARE DISTRIBUTION SHAPE TABLE FOR HTML
    # ========================================================

    distribution_shape_html = (
        distribution_shape_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    # ========================================================
    # 30. PREPARE NORMALITY TEST TABLE FOR HTML
    # ========================================================

    normality_tests_html = (
        normality_tests_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "STATISTIC":
                    lambda x:
                    f"{x:.12f}",

                "P_VALUE":
                    lambda x:
                    f"{x:.12e}",

                "ALPHA":
                    lambda x:
                    f"{x:.2f}"
            }
        )
    )


    # ========================================================
    # 31. CREATE HTML REPORT
    # ========================================================

    if not html_exists:

        histogram_base64 = (
            image_to_base64(
                HISTOGRAM_PATH
            )
        )


        boxplot_base64 = (
            image_to_base64(
                BOXPLOT_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Individual Exploratory Analysis —
{FEATURE_COLUMN}
</h1>


<p>

The variable
<strong>{FEATURE_COLUMN}</strong>
represents the geographic latitude
of the transaction receiver location.

Latitude is a continuous geographic coordinate
measured in degrees.

</p>


<p class="result">

-90° ≤ latitude ≤ 90°

</p>


<!-- ========================================================
     1. FEATURE OVERVIEW
========================================================= -->


<h2>
1. Feature overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>Valid observations</td>
<td>{valid_observations}</td>
</tr>

<tr>
<td>Missing values</td>
<td>{missing_values}</td>
</tr>

<tr>
<td>Missing percentage</td>
<td>{missing_percentage:.6f}%</td>
</tr>

<tr>
<td>Unique values</td>
<td>{unique_values}</td>
</tr>

<tr>
<td>Unique-value percentage</td>
<td>{unique_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     2. DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
2. Descriptive statistics
</h2>


{descriptive_statistics_html}


<!-- ========================================================
     3. PERCENTILES
========================================================= -->


<h2>
3. Percentiles
</h2>


<p>

Percentiles provide a detailed description
of the empirical distribution of receiver
latitude values.

</p>


{percentiles_html}


<!-- ========================================================
     4. GEOGRAPHIC RANGE VALIDATION
========================================================= -->


<h2>
4. Geographic range validation
</h2>


<p>

A valid geographic latitude must remain
within the interval:

</p>


<p class="result">

-90° ≤ latitude ≤ 90°

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Values below -90°</td>
<td>{values_below_valid_range}</td>
</tr>

<tr>
<td>Values above 90°</td>
<td>{values_above_valid_range}</td>
</tr>

<tr>
<td>Total invalid geographic values</td>
<td>{invalid_geographic_values}</td>
</tr>

<tr>
<td>Invalid geographic percentage</td>
<td>{invalid_geographic_percentage:.6f}%</td>
</tr>

<tr>
<td>Valid geographic values</td>
<td>{valid_geographic_values}</td>
</tr>

<tr>
<td>Valid geographic percentage</td>
<td>{valid_geographic_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     5. LATITUDE ORIENTATION
========================================================= -->


<h2>
5. Latitude orientation
</h2>


<p>

Negative latitude values indicate locations
in the Southern Hemisphere.

Positive latitude values indicate locations
in the Northern Hemisphere.

A latitude equal to zero lies on the Equator.

</p>


<table>

<tr>
<th>Latitude group</th>
<th>Observations</th>
<th>Percentage</th>
</tr>

<tr>
<td>Southern Hemisphere (&lt; 0)</td>
<td>{southern_latitudes}</td>
<td>{southern_latitude_percentage:.6f}%</td>
</tr>

<tr>
<td>Equator (= 0)</td>
<td>{equator_latitudes}</td>
<td>{equator_latitude_percentage:.6f}%</td>
</tr>

<tr>
<td>Northern Hemisphere (&gt; 0)</td>
<td>{northern_latitudes}</td>
<td>{northern_latitude_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     6. DISTRIBUTION SHAPE
========================================================= -->


<h2>
6. Distribution shape
</h2>


<p>

Skewness and kurtosis are used to characterize
the empirical shape of the latitude distribution.

</p>


{distribution_shape_html}


<div class="note">

<strong>Interpretation:</strong>

<br><br>

Skewness measures asymmetry in the observed
distribution.

<br><br>

Kurtosis provides information about the shape
and tail behavior of the distribution.

<br><br>

These statistics describe the empirical latitude
distribution and do not imply that geographic
coordinates are expected to follow a normal
distribution.

</div>


<!-- ========================================================
     7. NORMALITY TESTS
========================================================= -->


<h2>
7. Normality tests
</h2>


<p>

The Shapiro-Wilk and Jarque-Bera tests were
applied to evaluate whether the empirical
distribution of
<strong>{FEATURE_COLUMN}</strong>
is statistically compatible with a normal
distribution.

</p>


<p>

The complete set of valid observations was used.

No random sampling was performed.

</p>


<p class="result">

H0: The feature follows a normal distribution.

<br><br>

H1: The feature does not follow a normal distribution.

<br><br>

Significance level: α = {ALPHA}

</p>


{normality_tests_html}


<h3>
Shapiro-Wilk interpretation
</h3>


<p>

<strong>Decision:</strong>
{shapiro_decision}

</p>


<p>

{shapiro_interpretation}

</p>


<h3>
Jarque-Bera interpretation
</h3>


<p>

<strong>Decision:</strong>
{jarque_bera_decision}

</p>


<p>

{jarque_bera_interpretation}

</p>


<div class="note">

<strong>Important:</strong>

<br><br>

Normality tests characterize the statistical
shape of the latitude distribution.

Rejecting the null hypothesis of normality
does not indicate that the geographic coordinates
are invalid.

<br><br>

Geographic validity is evaluated separately
using the valid latitude range from
-90° to 90°.

<br><br>

Because the dataset contains a very large number
of observations, normality tests may detect
small deviations from a theoretical normal
distribution.

Therefore, the test results should be interpreted
together with the histogram, skewness, kurtosis,
and descriptive statistics.

</div>


<!-- ========================================================
     8. IQR OUTLIER ANALYSIS
========================================================= -->


<h2>
8. IQR outlier analysis
</h2>


<p>

The interquartile range method identifies
latitude values located far from the central
portion of the empirical distribution.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Q1</td>
<td>{q1_value:.12f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{q3_value:.12f}</td>
</tr>

<tr>
<td>IQR</td>
<td>{iqr:.12f}</td>
</tr>

<tr>
<td>Lower IQR bound</td>
<td>{lower_iqr_bound:.12f}</td>
</tr>

<tr>
<td>Upper IQR bound</td>
<td>{upper_iqr_bound:.12f}</td>
</tr>

<tr>
<td>Lower statistical outliers</td>
<td>{lower_iqr_outliers}</td>
</tr>

<tr>
<td>Upper statistical outliers</td>
<td>{upper_iqr_outliers}</td>
</tr>

<tr>
<td>Total statistical outliers</td>
<td>{total_iqr_outliers}</td>
</tr>

<tr>
<td>Statistical outlier percentage</td>
<td>{iqr_outlier_percentage:.6f}%</td>
</tr>

<tr>
<td>Observations inside IQR limits</td>
<td>{observations_inside_iqr_limits}</td>
</tr>

<tr>
<td>Percentage inside IQR limits</td>
<td>{observations_inside_iqr_percentage:.6f}%</td>
</tr>

</table>


<div class="note">

<strong>Important:</strong>

<br><br>

A geographic coordinate classified as an IQR
outlier is not necessarily invalid.

It only represents a latitude value that is
statistically distant from the central region
of the observed latitude distribution.

<br><br>

Geographic validity should primarily be evaluated
using the valid interval from
-90° to 90°.

</div>


<!-- ========================================================
     9. HISTOGRAM
========================================================= -->


<h2>
9. Latitude distribution
</h2>


<p>

The histogram presents the empirical distribution
of receiver latitude values across the dataset.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{histogram_base64}"
    alt="Distribution of receiver latitude"
>

</div>


<!-- ========================================================
     10. BOXPLOT
========================================================= -->


<h2>
10. Latitude boxplot
</h2>


<p>

The boxplot summarizes the central distribution
of receiver latitude and highlights observations
classified as statistical extremes according to
the IQR rule.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{boxplot_base64}"
    alt="Boxplot of receiver latitude"
>

</div>


<!-- ========================================================
     11. SUMMARY
========================================================= -->


<h2>
11. Summary of results
</h2>


<ul>

<li>
<strong>Total observations:</strong>
{total_observations}
</li>

<li>
<strong>Valid observations:</strong>
{valid_observations}
</li>

<li>
<strong>Missing values:</strong>
{missing_values}
</li>

<li>
<strong>Missing percentage:</strong>
{missing_percentage:.6f}%
</li>

<li>
<strong>Unique latitude values:</strong>
{unique_values}
</li>

<li>
<strong>Minimum latitude:</strong>
{minimum_value:.12f}
</li>

<li>
<strong>Median latitude:</strong>
{median_value:.12f}
</li>

<li>
<strong>Mean latitude:</strong>
{mean_value:.12f}
</li>

<li>
<strong>Maximum latitude:</strong>
{maximum_value:.12f}
</li>

<li>
<strong>Standard deviation:</strong>
{standard_deviation:.12f}
</li>

<li>
<strong>Skewness:</strong>
{skewness:.12f}
</li>

<li>
<strong>Kurtosis:</strong>
{kurtosis:.12f}
</li>

<li>
<strong>Invalid geographic values:</strong>
{invalid_geographic_values}
</li>

<li>
<strong>Geographically valid observations:</strong>
{valid_geographic_percentage:.6f}%
</li>

<li>
<strong>Southern Hemisphere:</strong>
{southern_latitude_percentage:.6f}%
</li>

<li>
<strong>Equator:</strong>
{equator_latitude_percentage:.6f}%
</li>

<li>
<strong>Northern Hemisphere:</strong>
{northern_latitude_percentage:.6f}%
</li>

<li>
<strong>Shapiro-Wilk statistic:</strong>
{shapiro_statistic:.12f}
</li>

<li>
<strong>Shapiro-Wilk p-value:</strong>
{shapiro_p_value:.12e}
</li>

<li>
<strong>Shapiro-Wilk decision:</strong>
{shapiro_decision}
</li>

<li>
<strong>Jarque-Bera statistic:</strong>
{jarque_bera_statistic:.12f}
</li>

<li>
<strong>Jarque-Bera p-value:</strong>
{jarque_bera_p_value:.12e}
</li>

<li>
<strong>Jarque-Bera decision:</strong>
{jarque_bera_decision}
</li>

<li>
<strong>IQR statistical outliers:</strong>
{total_iqr_outliers}
</li>

<li>
<strong>IQR outlier percentage:</strong>
{iqr_outlier_percentage:.6f}%
</li>

</ul>


</body>

</html>
"""


        # ====================================================
        # 32. SAVE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )


        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 33. DISPLAY FEATURE OVERVIEW
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "RECEIVE_LAT SUMMARY"
    )


    print(
        "=" * 100
    )


    print(
        "Total observations:",
        total_observations
    )


    print(
        "Valid observations:",
        valid_observations
    )


    print(
        "Missing values:",
        missing_values
    )


    print(
        "Missing percentage:",
        f"{missing_percentage:.6f}%"
    )


    print(
        "Unique values:",
        unique_values
    )


    print(
        "Unique-value percentage:",
        f"{unique_percentage:.6f}%"
    )


    # ========================================================
    # 34. DISPLAY DESCRIPTIVE STATISTICS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DESCRIPTIVE STATISTICS"
    )


    print(
        "=" * 100
    )


    display(
        descriptive_statistics
    )


    # ========================================================
    # 35. DISPLAY PERCENTILES
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "PERCENTILES"
    )


    print(
        "=" * 100
    )


    display(
        percentiles_table
    )


    # ========================================================
    # 36. DISPLAY GEOGRAPHIC RANGE VALIDATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "GEOGRAPHIC RANGE VALIDATION"
    )


    print(
        "=" * 100
    )


    print(
        "Valid latitude interval:",
        f"[{VALID_MINIMUM}, {VALID_MAXIMUM}]"
    )


    print(
        "Values below valid range:",
        values_below_valid_range
    )


    print(
        "Values above valid range:",
        values_above_valid_range
    )


    print(
        "Total invalid geographic values:",
        invalid_geographic_values
    )


    print(
        "Invalid geographic percentage:",
        f"{invalid_geographic_percentage:.6f}%"
    )


    print(
        "Geographic validation percentage:",
        f"{valid_geographic_percentage:.6f}%"
    )


    # ========================================================
    # 37. DISPLAY LATITUDE ORIENTATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "LATITUDE ORIENTATION"
    )


    print(
        "=" * 100
    )


    print(
        "Southern Hemisphere:",
        southern_latitudes,
        f"({southern_latitude_percentage:.6f}%)"
    )


    print(
        "Equator:",
        equator_latitudes,
        f"({equator_latitude_percentage:.6f}%)"
    )


    print(
        "Northern Hemisphere:",
        northern_latitudes,
        f"({northern_latitude_percentage:.6f}%)"
    )


    # ========================================================
    # 38. DISPLAY DISTRIBUTION SHAPE
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DISTRIBUTION SHAPE"
    )


    print(
        "=" * 100
    )


    print(
        "Skewness:",
        f"{skewness:.12f}"
    )


    print(
        "Kurtosis:",
        f"{kurtosis:.12f}"
    )


    # ========================================================
    # 39. DISPLAY NORMALITY TESTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "NORMALITY TESTS"
    )


    print(
        "=" * 100
    )


    print(
        "Significance level:",
        ALPHA
    )


    print(
        "\nShapiro-Wilk statistic:",
        f"{shapiro_statistic:.12f}"
    )


    print(
        "Shapiro-Wilk p-value:",
        f"{shapiro_p_value:.12e}"
    )


    print(
        "Shapiro-Wilk decision:",
        shapiro_decision
    )


    print(
        "Shapiro-Wilk interpretation:",
        shapiro_interpretation
    )


    print(
        "\nJarque-Bera statistic:",
        f"{jarque_bera_statistic:.12f}"
    )


    print(
        "Jarque-Bera p-value:",
        f"{jarque_bera_p_value:.12e}"
    )


    print(
        "Jarque-Bera decision:",
        jarque_bera_decision
    )


    print(
        "Jarque-Bera interpretation:",
        jarque_bera_interpretation
    )


    # ========================================================
    # 40. DISPLAY IQR OUTLIER ANALYSIS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "IQR OUTLIER ANALYSIS"
    )


    print(
        "=" * 100
    )


    print(
        "Q1:",
        f"{q1_value:.12f}"
    )


    print(
        "Q3:",
        f"{q3_value:.12f}"
    )


    print(
        "IQR:",
        f"{iqr:.12f}"
    )


    print(
        "Lower bound:",
        f"{lower_iqr_bound:.12f}"
    )


    print(
        "Upper bound:",
        f"{upper_iqr_bound:.12f}"
    )


    print(
        "Lower statistical outliers:",
        lower_iqr_outliers
    )


    print(
        "Upper statistical outliers:",
        upper_iqr_outliers
    )


    print(
        "Total statistical outliers:",
        total_iqr_outliers
    )


    print(
        "Outlier percentage:",
        f"{iqr_outlier_percentage:.6f}%"
    )


    # ========================================================
    # 41. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature
    del valid_feature
    del percentile_values

    gc.collect()


    # ========================================================
    # 42. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "ANALYSIS COMPLETED"
    )


    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )


    print(
        HTML_PATH
    )


    print(
        "\nPNGs:"
    )


    print(
        HISTOGRAM_PATH
    )


    print(
        BOXPLOT_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Histogram: Will be created
Boxplot: Will be created


/usr/local/lib/python3.14/site-packages/scipy/stats/_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 1852394.
  res = hypotest_fun_out(*samples, **kwds)



Histogram created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_geographic/recive_lat/recive_lat_histogram.png


/tmp/ipykernel_2304/2169045064.py:898: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(



Boxplot created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_geographic/recive_lat/recive_lat_boxplot.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_geographic/recive_lat/analysis_recive_lat.html

RECEIVE_LAT SUMMARY
Total observations: 1852394
Valid observations: 1852394
Missing values: 0
Missing percentage: 0.000000%
Unique values: 1524018
Unique-value percentage: 82.272886%

DESCRIPTIVE STATISTICS


,STATISTIC,VALUE
0,Minimum,19.027422
1,Q1,34.740122
2,Median,39.368900
3,Mean,38.538976
4,Q3,41.956264
5,Maximum,67.510269
6,Standard deviation,5.105604
7,Variance,26.067191
8,Range,48.482847



PERCENTILES


,PERCENTILE,VALUE
0,P1,26.393135
1,P5,29.753794
2,P10,31.637751
3,P25,34.740122
4,P50,39.368900
5,P75,41.956264
6,P90,44.492343
7,P95,46.002012
8,P99,48.577546



GEOGRAPHIC RANGE VALIDATION
Valid latitude interval: [-90.0, 90.0]
Values below valid range: 0
Values above valid range: 0
Total invalid geographic values: 0
Invalid geographic percentage: 0.000000%
Geographic validation percentage: 100.000000%

LATITUDE ORIENTATION
Southern Hemisphere: 0 (0.000000%)
Equator: 0 (0.000000%)
Northern Hemisphere: 1852394 (100.000000%)

DISTRIBUTION SHAPE
Skewness: -0.188096900292
Kurtosis: 0.774233620140

NORMALITY TESTS
Significance level: 0.05

Shapiro-Wilk statistic: 0.979520031811
Shapiro-Wilk p-value: 2.731848696155e-112
Shapiro-Wilk decision: Reject H0
Shapiro-Wilk interpretation: The data provide statistical evidence against a normal distribution.

Jarque-Bera statistic: 57188.882270067974
Jarque-Bera p-value: 0.000000000000e+00
Jarque-Bera decision: Reject H0
Jarque-Bera interpretation: The data provide statistical evidence against a normal distribution.

IQR OUTLIER ANALYSIS
Q1: 34.740121841431
Q3: 41.956263542175
IQR: 7.216141700745
Lower bound

#### <span style="color:grey"> SEND_LAT_REGISTER </span> ####

In [12]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

FEATURE_GROUP = "comum_features"

FEATURE_TYPE = "continuous_geographic"

FEATURE_NAME = "send_lat_register"

FEATURE_COLUMN = "SEND_LAT_REGISTER"

VALID_MINIMUM = -90.0

VALID_MAXIMUM = 90.0

ALPHA = 0.05


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / FEATURE_GROUP
    / FEATURE_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


HISTOGRAM_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_histogram.png"
)


BOXPLOT_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_boxplot.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


histogram_exists = (
    HISTOGRAM_PATH.exists()
)


boxplot_exists = (
    BOXPLOT_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and histogram_exists
    and boxplot_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )


    print(
        "No analysis or file creation is required."
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nExisting files:"
    )


    print(
        HTML_PATH
    )


    print(
        HISTOGRAM_PATH
    )


    print(
        BOXPLOT_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )


    print(
        "=" * 100
    )


    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )


    print(
        "Histogram:",
        "Already exists"
        if histogram_exists
        else "Will be created"
    )


    print(
        "Boxplot:",
        "Already exists"
        if boxplot_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. BASIC FEATURE OVERVIEW
    # ========================================================

    total_observations = int(
        len(
            feature
        )
    )


    if total_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    missing_values = int(
        feature
        .isna()
        .sum()
    )


    missing_percentage = (
        missing_values
        / total_observations
        * 100
    )


    valid_feature = (
        feature
        .dropna()
        .astype(
            "float64"
        )
    )


    valid_observations = int(
        len(
            valid_feature
        )
    )


    if valid_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no valid observations."
        )


    unique_values = int(
        valid_feature
        .nunique()
    )


    unique_percentage = (
        unique_values
        / valid_observations
        * 100
    )


    # ========================================================
    # 13. DESCRIPTIVE STATISTICS
    # ========================================================

    minimum_value = float(
        valid_feature.min()
    )


    q1_value = float(
        valid_feature.quantile(
            0.25
        )
    )


    median_value = float(
        valid_feature.median()
    )


    mean_value = float(
        valid_feature.mean()
    )


    q3_value = float(
        valid_feature.quantile(
            0.75
        )
    )


    maximum_value = float(
        valid_feature.max()
    )


    standard_deviation = float(
        valid_feature.std()
    )


    variance = float(
        valid_feature.var()
    )


    data_range = float(
        maximum_value
        - minimum_value
    )


    descriptive_statistics = pd.DataFrame({

        "STATISTIC": [
            "Minimum",
            "Q1",
            "Median",
            "Mean",
            "Q3",
            "Maximum",
            "Standard deviation",
            "Variance",
            "Range"
        ],

        "VALUE": [
            minimum_value,
            q1_value,
            median_value,
            mean_value,
            q3_value,
            maximum_value,
            standard_deviation,
            variance,
            data_range
        ]
    })


    # ========================================================
    # 14. PERCENTILES
    # ========================================================

    percentile_levels = [
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]


    percentile_values = (
        valid_feature
        .quantile(
            percentile_levels
        )
    )


    percentiles_table = pd.DataFrame({

        "PERCENTILE": [
            "P1",
            "P5",
            "P10",
            "P25",
            "P50",
            "P75",
            "P90",
            "P95",
            "P99"
        ],

        "VALUE": [
            float(
                percentile_values.loc[
                    percentile
                ]
            )
            for percentile in percentile_levels
        ]
    })


    # ========================================================
    # 15. GEOGRAPHIC RANGE VALIDATION
    #
    # Latitude must remain within:
    #
    # -90 <= latitude <= 90
    # ========================================================

    values_below_valid_range = int(
        (
            valid_feature
            < VALID_MINIMUM
        )
        .sum()
    )


    values_above_valid_range = int(
        (
            valid_feature
            > VALID_MAXIMUM
        )
        .sum()
    )


    invalid_geographic_values = (
        values_below_valid_range
        + values_above_valid_range
    )


    invalid_geographic_percentage = (
        invalid_geographic_values
        / valid_observations
        * 100
    )


    valid_geographic_values = (
        valid_observations
        - invalid_geographic_values
    )


    valid_geographic_percentage = (
        valid_geographic_values
        / valid_observations
        * 100
    )


    # ========================================================
    # 16. LATITUDE ORIENTATION
    #
    # Negative latitude:
    # Southern Hemisphere.
    #
    # Zero latitude:
    # Equator.
    #
    # Positive latitude:
    # Northern Hemisphere.
    # ========================================================

    southern_latitudes = int(
        (
            valid_feature
            < 0
        )
        .sum()
    )


    equator_latitudes = int(
        (
            valid_feature
            == 0
        )
        .sum()
    )


    northern_latitudes = int(
        (
            valid_feature
            > 0
        )
        .sum()
    )


    southern_latitude_percentage = (
        southern_latitudes
        / valid_observations
        * 100
    )


    equator_latitude_percentage = (
        equator_latitudes
        / valid_observations
        * 100
    )


    northern_latitude_percentage = (
        northern_latitudes
        / valid_observations
        * 100
    )


    # ========================================================
    # 17. DISTRIBUTION SHAPE
    # ========================================================

    skewness = float(
        valid_feature.skew()
    )


    kurtosis = float(
        valid_feature.kurt()
    )


    distribution_shape_table = pd.DataFrame({

        "METRIC": [
            "Skewness",
            "Kurtosis"
        ],

        "VALUE": [
            skewness,
            kurtosis
        ]
    })


    # ========================================================
    # 18. SHAPIRO-WILK NORMALITY TEST
    #
    # H0:
    # The feature follows a normal distribution.
    #
    # H1:
    # The feature does not follow a normal distribution.
    #
    # The complete valid feature is used.
    # No random sampling is performed.
    # ========================================================

    (
        shapiro_statistic,
        shapiro_p_value
    ) = stats.shapiro(
        valid_feature.to_numpy()
    )


    shapiro_statistic = float(
        shapiro_statistic
    )


    shapiro_p_value = float(
        shapiro_p_value
    )


    # ========================================================
    # 19. SHAPIRO-WILK DECISION
    # ========================================================

    if shapiro_p_value < ALPHA:

        shapiro_decision = (
            "Reject H0"
        )


        shapiro_interpretation = (
            "The data provide statistical evidence "
            "against a normal distribution."
        )


    else:

        shapiro_decision = (
            "Fail to reject H0"
        )


        shapiro_interpretation = (
            "The data do not provide sufficient "
            "statistical evidence against a normal distribution."
        )


    # ========================================================
    # 20. JARQUE-BERA NORMALITY TEST
    #
    # H0:
    # The feature follows a normal distribution.
    #
    # H1:
    # The feature does not follow a normal distribution.
    # ========================================================

    jarque_bera_result = stats.jarque_bera(
        valid_feature.to_numpy()
    )


    jarque_bera_statistic = float(
        jarque_bera_result.statistic
    )


    jarque_bera_p_value = float(
        jarque_bera_result.pvalue
    )


    # ========================================================
    # 21. JARQUE-BERA DECISION
    # ========================================================

    if jarque_bera_p_value < ALPHA:

        jarque_bera_decision = (
            "Reject H0"
        )


        jarque_bera_interpretation = (
            "The data provide statistical evidence "
            "against a normal distribution."
        )


    else:

        jarque_bera_decision = (
            "Fail to reject H0"
        )


        jarque_bera_interpretation = (
            "The data do not provide sufficient "
            "statistical evidence against a normal distribution."
        )


    # ========================================================
    # 22. NORMALITY TEST RESULTS TABLE
    # ========================================================

    normality_tests_table = pd.DataFrame({

        "TEST": [
            "Shapiro-Wilk",
            "Jarque-Bera"
        ],

        "STATISTIC": [
            shapiro_statistic,
            jarque_bera_statistic
        ],

        "P_VALUE": [
            shapiro_p_value,
            jarque_bera_p_value
        ],

        "ALPHA": [
            ALPHA,
            ALPHA
        ],

        "DECISION": [
            shapiro_decision,
            jarque_bera_decision
        ]
    })


    # ========================================================
    # 23. IQR OUTLIER ANALYSIS
    #
    # IQR outliers represent statistical extremes.
    #
    # They are not automatically invalid geographic
    # coordinates.
    # ========================================================

    iqr = (
        q3_value
        - q1_value
    )


    lower_iqr_bound = (
        q1_value
        - 1.5
        * iqr
    )


    upper_iqr_bound = (
        q3_value
        + 1.5
        * iqr
    )


    lower_iqr_outliers = int(
        (
            valid_feature
            < lower_iqr_bound
        )
        .sum()
    )


    upper_iqr_outliers = int(
        (
            valid_feature
            > upper_iqr_bound
        )
        .sum()
    )


    total_iqr_outliers = (
        lower_iqr_outliers
        + upper_iqr_outliers
    )


    iqr_outlier_percentage = (
        total_iqr_outliers
        / valid_observations
        * 100
    )


    observations_inside_iqr_limits = (
        valid_observations
        - total_iqr_outliers
    )


    observations_inside_iqr_percentage = (
        observations_inside_iqr_limits
        / valid_observations
        * 100
    )


    # ========================================================
    # 24. CREATE HISTOGRAM
    # ========================================================

    if not histogram_exists:

        fig, ax = plt.subplots(
            figsize=(
                11,
                6
            )
        )


        ax.hist(
            valid_feature,
            bins=60
        )


        ax.set_title(
            "Distribution of sender registered latitude"
        )


        ax.set_xlabel(
            "Latitude"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(y)
                )
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            HISTOGRAM_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nHistogram created:"
        )


        print(
            HISTOGRAM_PATH
        )


    else:

        print(
            "\nHistogram already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 25. CREATE BOXPLOT
    # ========================================================

    if not boxplot_exists:

        fig, ax = plt.subplots(
            figsize=(
                11,
                4
            )
        )


        ax.boxplot(
            valid_feature,
            vert=False
        )


        ax.set_title(
            "Boxplot of sender registered latitude"
        )


        ax.set_xlabel(
            "Latitude"
        )


        ax.set_yticks(
            []
        )


        ax.grid(
            axis="x",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            BOXPLOT_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nBoxplot created:"
        )


        print(
            BOXPLOT_PATH
        )


    else:

        print(
            "\nBoxplot already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 26. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 27. PREPARE DESCRIPTIVE STATISTICS TABLE FOR HTML
    # ========================================================

    descriptive_statistics_html = (
        descriptive_statistics
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    # ========================================================
    # 28. PREPARE PERCENTILES TABLE FOR HTML
    # ========================================================

    percentiles_html = (
        percentiles_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    # ========================================================
    # 29. PREPARE DISTRIBUTION SHAPE TABLE FOR HTML
    # ========================================================

    distribution_shape_html = (
        distribution_shape_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    # ========================================================
    # 30. PREPARE NORMALITY TEST TABLE FOR HTML
    # ========================================================

    normality_tests_html = (
        normality_tests_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "STATISTIC":
                    lambda x:
                    f"{x:.12f}",

                "P_VALUE":
                    lambda x:
                    f"{x:.12e}",

                "ALPHA":
                    lambda x:
                    f"{x:.2f}"
            }
        )
    )


    # ========================================================
    # 31. CREATE HTML REPORT
    # ========================================================

    if not html_exists:

        histogram_base64 = (
            image_to_base64(
                HISTOGRAM_PATH
            )
        )


        boxplot_base64 = (
            image_to_base64(
                BOXPLOT_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Individual Exploratory Analysis —
{FEATURE_COLUMN}
</h1>


<p>

The variable
<strong>{FEATURE_COLUMN}</strong>
represents the registered geographic latitude
associated with the transaction sender.

Latitude is a continuous geographic coordinate
measured in degrees.

</p>


<p class="result">

-90° ≤ latitude ≤ 90°

</p>


<!-- ========================================================
     1. FEATURE OVERVIEW
========================================================= -->


<h2>
1. Feature overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>Valid observations</td>
<td>{valid_observations}</td>
</tr>

<tr>
<td>Missing values</td>
<td>{missing_values}</td>
</tr>

<tr>
<td>Missing percentage</td>
<td>{missing_percentage:.6f}%</td>
</tr>

<tr>
<td>Unique values</td>
<td>{unique_values}</td>
</tr>

<tr>
<td>Unique-value percentage</td>
<td>{unique_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     2. DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
2. Descriptive statistics
</h2>


{descriptive_statistics_html}


<!-- ========================================================
     3. PERCENTILES
========================================================= -->


<h2>
3. Percentiles
</h2>


<p>

Percentiles provide a detailed description
of the empirical distribution of sender
registered latitude values.

</p>


{percentiles_html}


<!-- ========================================================
     4. GEOGRAPHIC RANGE VALIDATION
========================================================= -->


<h2>
4. Geographic range validation
</h2>


<p>

A valid geographic latitude must remain
within the interval:

</p>


<p class="result">

-90° ≤ latitude ≤ 90°

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Values below -90°</td>
<td>{values_below_valid_range}</td>
</tr>

<tr>
<td>Values above 90°</td>
<td>{values_above_valid_range}</td>
</tr>

<tr>
<td>Total invalid geographic values</td>
<td>{invalid_geographic_values}</td>
</tr>

<tr>
<td>Invalid geographic percentage</td>
<td>{invalid_geographic_percentage:.6f}%</td>
</tr>

<tr>
<td>Valid geographic values</td>
<td>{valid_geographic_values}</td>
</tr>

<tr>
<td>Valid geographic percentage</td>
<td>{valid_geographic_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     5. LATITUDE ORIENTATION
========================================================= -->


<h2>
5. Latitude orientation
</h2>


<p>

Negative latitude values indicate locations
in the Southern Hemisphere.

Positive latitude values indicate locations
in the Northern Hemisphere.

A latitude equal to zero lies on the Equator.

</p>


<table>

<tr>
<th>Latitude group</th>
<th>Observations</th>
<th>Percentage</th>
</tr>

<tr>
<td>Southern Hemisphere (&lt; 0)</td>
<td>{southern_latitudes}</td>
<td>{southern_latitude_percentage:.6f}%</td>
</tr>

<tr>
<td>Equator (= 0)</td>
<td>{equator_latitudes}</td>
<td>{equator_latitude_percentage:.6f}%</td>
</tr>

<tr>
<td>Northern Hemisphere (&gt; 0)</td>
<td>{northern_latitudes}</td>
<td>{northern_latitude_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     6. DISTRIBUTION SHAPE
========================================================= -->


<h2>
6. Distribution shape
</h2>


<p>

Skewness and kurtosis characterize the empirical
shape of the sender registered latitude
distribution.

</p>


{distribution_shape_html}


<div class="note">

<strong>Interpretation:</strong>

<br><br>

Skewness measures asymmetry in the observed
distribution.

<br><br>

Kurtosis provides information about the shape
and tail behavior of the distribution.

<br><br>

These statistics describe the empirical latitude
distribution and do not imply that geographic
coordinates are expected to follow a normal
distribution.

</div>


<!-- ========================================================
     7. NORMALITY TESTS
========================================================= -->


<h2>
7. Normality tests
</h2>


<p>

The Shapiro-Wilk and Jarque-Bera tests were
applied to evaluate whether the empirical
distribution of
<strong>{FEATURE_COLUMN}</strong>
is statistically compatible with a normal
distribution.

</p>


<p>

The complete set of valid observations was used.

No random sampling was performed.

</p>


<p class="result">

H0: The feature follows a normal distribution.

<br><br>

H1: The feature does not follow a normal distribution.

<br><br>

Significance level: α = {ALPHA}

</p>


{normality_tests_html}


<h3>
Shapiro-Wilk interpretation
</h3>


<p>

<strong>Decision:</strong>
{shapiro_decision}

</p>


<p>

{shapiro_interpretation}

</p>


<h3>
Jarque-Bera interpretation
</h3>


<p>

<strong>Decision:</strong>
{jarque_bera_decision}

</p>


<p>

{jarque_bera_interpretation}

</p>


<div class="note">

<strong>Important:</strong>

<br><br>

Normality tests characterize the statistical
shape of the latitude distribution.

Rejecting the null hypothesis of normality
does not indicate that the geographic coordinates
are invalid.

<br><br>

Geographic validity is evaluated separately
using the valid latitude range from
-90° to 90°.

<br><br>

Because the dataset contains a very large number
of observations, normality tests may detect
small deviations from a theoretical normal
distribution.

Therefore, the test results should be interpreted
together with the histogram, skewness, kurtosis,
and descriptive statistics.

</div>


<!-- ========================================================
     8. IQR OUTLIER ANALYSIS
========================================================= -->


<h2>
8. IQR outlier analysis
</h2>


<p>

The interquartile range method identifies
latitude values located far from the central
portion of the empirical distribution.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Q1</td>
<td>{q1_value:.12f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{q3_value:.12f}</td>
</tr>

<tr>
<td>IQR</td>
<td>{iqr:.12f}</td>
</tr>

<tr>
<td>Lower IQR bound</td>
<td>{lower_iqr_bound:.12f}</td>
</tr>

<tr>
<td>Upper IQR bound</td>
<td>{upper_iqr_bound:.12f}</td>
</tr>

<tr>
<td>Lower statistical outliers</td>
<td>{lower_iqr_outliers}</td>
</tr>

<tr>
<td>Upper statistical outliers</td>
<td>{upper_iqr_outliers}</td>
</tr>

<tr>
<td>Total statistical outliers</td>
<td>{total_iqr_outliers}</td>
</tr>

<tr>
<td>Statistical outlier percentage</td>
<td>{iqr_outlier_percentage:.6f}%</td>
</tr>

<tr>
<td>Observations inside IQR limits</td>
<td>{observations_inside_iqr_limits}</td>
</tr>

<tr>
<td>Percentage inside IQR limits</td>
<td>{observations_inside_iqr_percentage:.6f}%</td>
</tr>

</table>


<div class="note">

<strong>Important:</strong>

<br><br>

A geographic coordinate classified as an IQR
outlier is not necessarily invalid.

It only represents a latitude value statistically
distant from the central region of the observed
distribution.

<br><br>

Geographic validity should primarily be evaluated
using the valid interval from
-90° to 90°.

</div>


<!-- ========================================================
     9. HISTOGRAM
========================================================= -->


<h2>
9. Latitude distribution
</h2>


<p>

The histogram presents the empirical distribution
of sender registered latitude values.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{histogram_base64}"
    alt="Distribution of sender registered latitude"
>

</div>


<!-- ========================================================
     10. BOXPLOT
========================================================= -->


<h2>
10. Latitude boxplot
</h2>


<p>

The boxplot summarizes the central distribution
of sender registered latitude and highlights
observations classified as statistical extremes
according to the IQR rule.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{boxplot_base64}"
    alt="Boxplot of sender registered latitude"
>

</div>


<!-- ========================================================
     11. SUMMARY
========================================================= -->


<h2>
11. Summary of results
</h2>


<ul>

<li>
<strong>Total observations:</strong>
{total_observations}
</li>

<li>
<strong>Valid observations:</strong>
{valid_observations}
</li>

<li>
<strong>Missing values:</strong>
{missing_values}
</li>

<li>
<strong>Missing percentage:</strong>
{missing_percentage:.6f}%
</li>

<li>
<strong>Unique latitude values:</strong>
{unique_values}
</li>

<li>
<strong>Minimum latitude:</strong>
{minimum_value:.12f}
</li>

<li>
<strong>Median latitude:</strong>
{median_value:.12f}
</li>

<li>
<strong>Mean latitude:</strong>
{mean_value:.12f}
</li>

<li>
<strong>Maximum latitude:</strong>
{maximum_value:.12f}
</li>

<li>
<strong>Standard deviation:</strong>
{standard_deviation:.12f}
</li>

<li>
<strong>Skewness:</strong>
{skewness:.12f}
</li>

<li>
<strong>Kurtosis:</strong>
{kurtosis:.12f}
</li>

<li>
<strong>Invalid geographic values:</strong>
{invalid_geographic_values}
</li>

<li>
<strong>Geographically valid observations:</strong>
{valid_geographic_percentage:.6f}%
</li>

<li>
<strong>Southern Hemisphere:</strong>
{southern_latitude_percentage:.6f}%
</li>

<li>
<strong>Equator:</strong>
{equator_latitude_percentage:.6f}%
</li>

<li>
<strong>Northern Hemisphere:</strong>
{northern_latitude_percentage:.6f}%
</li>

<li>
<strong>Shapiro-Wilk statistic:</strong>
{shapiro_statistic:.12f}
</li>

<li>
<strong>Shapiro-Wilk p-value:</strong>
{shapiro_p_value:.12e}
</li>

<li>
<strong>Shapiro-Wilk decision:</strong>
{shapiro_decision}
</li>

<li>
<strong>Jarque-Bera statistic:</strong>
{jarque_bera_statistic:.12f}
</li>

<li>
<strong>Jarque-Bera p-value:</strong>
{jarque_bera_p_value:.12e}
</li>

<li>
<strong>Jarque-Bera decision:</strong>
{jarque_bera_decision}
</li>

<li>
<strong>IQR statistical outliers:</strong>
{total_iqr_outliers}
</li>

<li>
<strong>IQR outlier percentage:</strong>
{iqr_outlier_percentage:.6f}%
</li>

</ul>


</body>

</html>
"""


        # ====================================================
        # 32. SAVE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )


        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 33. DISPLAY FEATURE OVERVIEW
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "SEND_LAT_REGISTER SUMMARY"
    )


    print(
        "=" * 100
    )


    print(
        "Total observations:",
        total_observations
    )


    print(
        "Valid observations:",
        valid_observations
    )


    print(
        "Missing values:",
        missing_values
    )


    print(
        "Missing percentage:",
        f"{missing_percentage:.6f}%"
    )


    print(
        "Unique values:",
        unique_values
    )


    print(
        "Unique-value percentage:",
        f"{unique_percentage:.6f}%"
    )


    # ========================================================
    # 34. DISPLAY DESCRIPTIVE STATISTICS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DESCRIPTIVE STATISTICS"
    )


    print(
        "=" * 100
    )


    display(
        descriptive_statistics
    )


    # ========================================================
    # 35. DISPLAY PERCENTILES
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "PERCENTILES"
    )


    print(
        "=" * 100
    )


    display(
        percentiles_table
    )


    # ========================================================
    # 36. DISPLAY GEOGRAPHIC RANGE VALIDATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "GEOGRAPHIC RANGE VALIDATION"
    )


    print(
        "=" * 100
    )


    print(
        "Valid latitude interval:",
        f"[{VALID_MINIMUM}, {VALID_MAXIMUM}]"
    )


    print(
        "Values below valid range:",
        values_below_valid_range
    )


    print(
        "Values above valid range:",
        values_above_valid_range
    )


    print(
        "Total invalid geographic values:",
        invalid_geographic_values
    )


    print(
        "Invalid geographic percentage:",
        f"{invalid_geographic_percentage:.6f}%"
    )


    print(
        "Geographic validation percentage:",
        f"{valid_geographic_percentage:.6f}%"
    )


    # ========================================================
    # 37. DISPLAY LATITUDE ORIENTATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "LATITUDE ORIENTATION"
    )


    print(
        "=" * 100
    )


    print(
        "Southern Hemisphere:",
        southern_latitudes,
        f"({southern_latitude_percentage:.6f}%)"
    )


    print(
        "Equator:",
        equator_latitudes,
        f"({equator_latitude_percentage:.6f}%)"
    )


    print(
        "Northern Hemisphere:",
        northern_latitudes,
        f"({northern_latitude_percentage:.6f}%)"
    )


    # ========================================================
    # 38. DISPLAY DISTRIBUTION SHAPE
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DISTRIBUTION SHAPE"
    )


    print(
        "=" * 100
    )


    print(
        "Skewness:",
        f"{skewness:.12f}"
    )


    print(
        "Kurtosis:",
        f"{kurtosis:.12f}"
    )


    # ========================================================
    # 39. DISPLAY NORMALITY TESTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "NORMALITY TESTS"
    )


    print(
        "=" * 100
    )


    print(
        "Significance level:",
        ALPHA
    )


    print(
        "\nShapiro-Wilk statistic:",
        f"{shapiro_statistic:.12f}"
    )


    print(
        "Shapiro-Wilk p-value:",
        f"{shapiro_p_value:.12e}"
    )


    print(
        "Shapiro-Wilk decision:",
        shapiro_decision
    )


    print(
        "Shapiro-Wilk interpretation:",
        shapiro_interpretation
    )


    print(
        "\nJarque-Bera statistic:",
        f"{jarque_bera_statistic:.12f}"
    )


    print(
        "Jarque-Bera p-value:",
        f"{jarque_bera_p_value:.12e}"
    )


    print(
        "Jarque-Bera decision:",
        jarque_bera_decision
    )


    print(
        "Jarque-Bera interpretation:",
        jarque_bera_interpretation
    )


    # ========================================================
    # 40. DISPLAY IQR OUTLIER ANALYSIS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "IQR OUTLIER ANALYSIS"
    )


    print(
        "=" * 100
    )


    print(
        "Q1:",
        f"{q1_value:.12f}"
    )


    print(
        "Q3:",
        f"{q3_value:.12f}"
    )


    print(
        "IQR:",
        f"{iqr:.12f}"
    )


    print(
        "Lower bound:",
        f"{lower_iqr_bound:.12f}"
    )


    print(
        "Upper bound:",
        f"{upper_iqr_bound:.12f}"
    )


    print(
        "Lower statistical outliers:",
        lower_iqr_outliers
    )


    print(
        "Upper statistical outliers:",
        upper_iqr_outliers
    )


    print(
        "Total statistical outliers:",
        total_iqr_outliers
    )


    print(
        "Outlier percentage:",
        f"{iqr_outlier_percentage:.6f}%"
    )


    # ========================================================
    # 41. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature
    del valid_feature
    del percentile_values

    gc.collect()


    # ========================================================
    # 42. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "ANALYSIS COMPLETED"
    )


    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )


    print(
        HTML_PATH
    )


    print(
        "\nPNGs:"
    )


    print(
        HISTOGRAM_PATH
    )


    print(
        BOXPLOT_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Histogram: Will be created
Boxplot: Will be created


/usr/local/lib/python3.14/site-packages/scipy/stats/_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 1852394.
  res = hypotest_fun_out(*samples, **kwds)



Histogram created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_geographic/send_lat_register/send_lat_register_histogram.png


/tmp/ipykernel_2304/3806257272.py:896: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(



Boxplot created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_geographic/send_lat_register/send_lat_register_boxplot.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_geographic/send_lat_register/analysis_send_lat_register.html

SEND_LAT_REGISTER SUMMARY
Total observations: 1852394
Valid observations: 1852394
Missing values: 0
Missing percentage: 0.000000%
Unique values: 983
Unique-value percentage: 0.053066%

DESCRIPTIVE STATISTICS


,STATISTIC,VALUE
0,Minimum,20.027100
1,Q1,34.668900
2,Median,39.354301
3,Mean,38.539311
4,Q3,41.940399
5,Maximum,66.693298
6,Standard deviation,5.071470
7,Variance,25.719812
8,Range,46.666199



PERCENTILES


,PERCENTILE,VALUE
0,P1,26.472200
1,P5,29.882601
2,P10,31.770599
3,P25,34.668900
4,P50,39.354301
5,P75,41.940399
6,P90,44.447701
7,P95,45.843300
8,P99,48.478600



GEOGRAPHIC RANGE VALIDATION
Valid latitude interval: [-90.0, 90.0]
Values below valid range: 0
Values above valid range: 0
Total invalid geographic values: 0
Invalid geographic percentage: 0.000000%
Geographic validation percentage: 100.000000%

LATITUDE ORIENTATION
Southern Hemisphere: 0 (0.000000%)
Equator: 0 (0.000000%)
Northern Hemisphere: 1852394 (100.000000%)

DISTRIBUTION SHAPE
Skewness: -0.191998963028
Kurtosis: 0.791077135058

NORMALITY TESTS
Significance level: 0.05

Shapiro-Wilk statistic: 0.977471188260
Shapiro-Wilk p-value: 5.952124839052e-115
Shapiro-Wilk decision: Reject H0
Shapiro-Wilk interpretation: The data provide statistical evidence against a normal distribution.

Jarque-Bera statistic: 59681.718774445399
Jarque-Bera p-value: 0.000000000000e+00
Jarque-Bera decision: Reject H0
Jarque-Bera interpretation: The data provide statistical evidence against a normal distribution.

IQR OUTLIER ANALYSIS
Q1: 34.668899536133
Q3: 41.940399169922
IQR: 7.271499633789
Lower bound

#### <span style="color:grey"> SEND_LONG_REGISTER </span> ####

In [13]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

FEATURE_GROUP = "comum_features"

FEATURE_TYPE = "continuous_geographic"

FEATURE_NAME = "send_long_register"

FEATURE_COLUMN = "SEND_LONG_REGISTER"

VALID_MINIMUM = -180.0

VALID_MAXIMUM = 180.0

ALPHA = 0.05


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / FEATURE_GROUP
    / FEATURE_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


HISTOGRAM_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_histogram.png"
)


BOXPLOT_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_boxplot.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


histogram_exists = (
    HISTOGRAM_PATH.exists()
)


boxplot_exists = (
    BOXPLOT_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and histogram_exists
    and boxplot_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )


    print(
        "No analysis or file creation is required."
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nExisting files:"
    )


    print(
        HTML_PATH
    )


    print(
        HISTOGRAM_PATH
    )


    print(
        BOXPLOT_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )


    print(
        "=" * 100
    )


    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )


    print(
        "Histogram:",
        "Already exists"
        if histogram_exists
        else "Will be created"
    )


    print(
        "Boxplot:",
        "Already exists"
        if boxplot_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. BASIC FEATURE OVERVIEW
    # ========================================================

    total_observations = int(
        len(
            feature
        )
    )


    if total_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    missing_values = int(
        feature
        .isna()
        .sum()
    )


    missing_percentage = (
        missing_values
        / total_observations
        * 100
    )


    valid_feature = (
        feature
        .dropna()
        .astype(
            "float64"
        )
    )


    valid_observations = int(
        len(
            valid_feature
        )
    )


    if valid_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no valid observations."
        )


    unique_values = int(
        valid_feature
        .nunique()
    )


    unique_percentage = (
        unique_values
        / valid_observations
        * 100
    )


    # ========================================================
    # 13. DESCRIPTIVE STATISTICS
    # ========================================================

    minimum_value = float(
        valid_feature.min()
    )


    q1_value = float(
        valid_feature.quantile(
            0.25
        )
    )


    median_value = float(
        valid_feature.median()
    )


    mean_value = float(
        valid_feature.mean()
    )


    q3_value = float(
        valid_feature.quantile(
            0.75
        )
    )


    maximum_value = float(
        valid_feature.max()
    )


    standard_deviation = float(
        valid_feature.std()
    )


    variance = float(
        valid_feature.var()
    )


    data_range = float(
        maximum_value
        - minimum_value
    )


    descriptive_statistics = pd.DataFrame({

        "STATISTIC": [
            "Minimum",
            "Q1",
            "Median",
            "Mean",
            "Q3",
            "Maximum",
            "Standard deviation",
            "Variance",
            "Range"
        ],

        "VALUE": [
            minimum_value,
            q1_value,
            median_value,
            mean_value,
            q3_value,
            maximum_value,
            standard_deviation,
            variance,
            data_range
        ]
    })


    # ========================================================
    # 14. PERCENTILES
    # ========================================================

    percentile_levels = [
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]


    percentile_values = (
        valid_feature
        .quantile(
            percentile_levels
        )
    )


    percentiles_table = pd.DataFrame({

        "PERCENTILE": [
            "P1",
            "P5",
            "P10",
            "P25",
            "P50",
            "P75",
            "P90",
            "P95",
            "P99"
        ],

        "VALUE": [
            float(
                percentile_values.loc[
                    percentile
                ]
            )
            for percentile in percentile_levels
        ]
    })


    # ========================================================
    # 15. GEOGRAPHIC RANGE VALIDATION
    #
    # Longitude must remain within:
    #
    # -180 <= longitude <= 180
    # ========================================================

    values_below_valid_range = int(
        (
            valid_feature
            < VALID_MINIMUM
        )
        .sum()
    )


    values_above_valid_range = int(
        (
            valid_feature
            > VALID_MAXIMUM
        )
        .sum()
    )


    invalid_geographic_values = (
        values_below_valid_range
        + values_above_valid_range
    )


    invalid_geographic_percentage = (
        invalid_geographic_values
        / valid_observations
        * 100
    )


    valid_geographic_values = (
        valid_observations
        - invalid_geographic_values
    )


    valid_geographic_percentage = (
        valid_geographic_values
        / valid_observations
        * 100
    )


    # ========================================================
    # 16. LONGITUDE ORIENTATION
    #
    # Negative longitude:
    # West of the Greenwich meridian.
    #
    # Zero longitude:
    # Greenwich meridian.
    #
    # Positive longitude:
    # East of the Greenwich meridian.
    # ========================================================

    western_longitudes = int(
        (
            valid_feature
            < 0
        )
        .sum()
    )


    greenwich_longitudes = int(
        (
            valid_feature
            == 0
        )
        .sum()
    )


    eastern_longitudes = int(
        (
            valid_feature
            > 0
        )
        .sum()
    )


    western_longitude_percentage = (
        western_longitudes
        / valid_observations
        * 100
    )


    greenwich_longitude_percentage = (
        greenwich_longitudes
        / valid_observations
        * 100
    )


    eastern_longitude_percentage = (
        eastern_longitudes
        / valid_observations
        * 100
    )


    # ========================================================
    # 17. DISTRIBUTION SHAPE
    # ========================================================

    skewness = float(
        valid_feature.skew()
    )


    kurtosis = float(
        valid_feature.kurt()
    )


    distribution_shape_table = pd.DataFrame({

        "METRIC": [
            "Skewness",
            "Kurtosis"
        ],

        "VALUE": [
            skewness,
            kurtosis
        ]
    })


    # ========================================================
    # 18. SHAPIRO-WILK NORMALITY TEST
    #
    # H0:
    # The feature follows a normal distribution.
    #
    # H1:
    # The feature does not follow a normal distribution.
    #
    # The complete valid feature is used.
    # No random sampling is performed.
    # ========================================================

    (
        shapiro_statistic,
        shapiro_p_value
    ) = stats.shapiro(
        valid_feature.to_numpy()
    )


    shapiro_statistic = float(
        shapiro_statistic
    )


    shapiro_p_value = float(
        shapiro_p_value
    )


    # ========================================================
    # 19. SHAPIRO-WILK DECISION
    # ========================================================

    if shapiro_p_value < ALPHA:

        shapiro_decision = (
            "Reject H0"
        )


        shapiro_interpretation = (
            "The data provide statistical evidence "
            "against a normal distribution."
        )


    else:

        shapiro_decision = (
            "Fail to reject H0"
        )


        shapiro_interpretation = (
            "The data do not provide sufficient "
            "statistical evidence against a normal distribution."
        )


    # ========================================================
    # 20. JARQUE-BERA NORMALITY TEST
    #
    # H0:
    # The feature follows a normal distribution.
    #
    # H1:
    # The feature does not follow a normal distribution.
    #
    # The complete valid feature is used.
    # ========================================================

    jarque_bera_result = stats.jarque_bera(
        valid_feature.to_numpy()
    )


    jarque_bera_statistic = float(
        jarque_bera_result.statistic
    )


    jarque_bera_p_value = float(
        jarque_bera_result.pvalue
    )


    # ========================================================
    # 21. JARQUE-BERA DECISION
    # ========================================================

    if jarque_bera_p_value < ALPHA:

        jarque_bera_decision = (
            "Reject H0"
        )


        jarque_bera_interpretation = (
            "The data provide statistical evidence "
            "against a normal distribution."
        )


    else:

        jarque_bera_decision = (
            "Fail to reject H0"
        )


        jarque_bera_interpretation = (
            "The data do not provide sufficient "
            "statistical evidence against a normal distribution."
        )


    # ========================================================
    # 22. NORMALITY TEST RESULTS TABLE
    # ========================================================

    normality_tests_table = pd.DataFrame({

        "TEST": [
            "Shapiro-Wilk",
            "Jarque-Bera"
        ],

        "STATISTIC": [
            shapiro_statistic,
            jarque_bera_statistic
        ],

        "P_VALUE": [
            shapiro_p_value,
            jarque_bera_p_value
        ],

        "ALPHA": [
            ALPHA,
            ALPHA
        ],

        "DECISION": [
            shapiro_decision,
            jarque_bera_decision
        ]
    })


    # ========================================================
    # 23. IQR OUTLIER ANALYSIS
    #
    # IQR outliers represent statistical extremes only.
    #
    # They should not automatically be interpreted as
    # invalid geographic coordinates.
    # ========================================================

    iqr = (
        q3_value
        - q1_value
    )


    lower_iqr_bound = (
        q1_value
        - 1.5
        * iqr
    )


    upper_iqr_bound = (
        q3_value
        + 1.5
        * iqr
    )


    lower_iqr_outliers = int(
        (
            valid_feature
            < lower_iqr_bound
        )
        .sum()
    )


    upper_iqr_outliers = int(
        (
            valid_feature
            > upper_iqr_bound
        )
        .sum()
    )


    total_iqr_outliers = (
        lower_iqr_outliers
        + upper_iqr_outliers
    )


    iqr_outlier_percentage = (
        total_iqr_outliers
        / valid_observations
        * 100
    )


    observations_inside_iqr_limits = (
        valid_observations
        - total_iqr_outliers
    )


    observations_inside_iqr_percentage = (
        observations_inside_iqr_limits
        / valid_observations
        * 100
    )


    # ========================================================
    # 24. CREATE HISTOGRAM
    # ========================================================

    if not histogram_exists:

        fig, ax = plt.subplots(
            figsize=(
                11,
                6
            )
        )


        ax.hist(
            valid_feature,
            bins=60
        )


        ax.set_title(
            "Distribution of sender registered longitude"
        )


        ax.set_xlabel(
            "Longitude"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(y)
                )
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            HISTOGRAM_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nHistogram created:"
        )


        print(
            HISTOGRAM_PATH
        )


    else:

        print(
            "\nHistogram already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 25. CREATE BOXPLOT
    # ========================================================

    if not boxplot_exists:

        fig, ax = plt.subplots(
            figsize=(
                11,
                4
            )
        )


        ax.boxplot(
            valid_feature,
            vert=False
        )


        ax.set_title(
            "Boxplot of sender registered longitude"
        )


        ax.set_xlabel(
            "Longitude"
        )


        ax.set_yticks(
            []
        )


        ax.grid(
            axis="x",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            BOXPLOT_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nBoxplot created:"
        )


        print(
            BOXPLOT_PATH
        )


    else:

        print(
            "\nBoxplot already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 26. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 27. PREPARE DESCRIPTIVE STATISTICS TABLE FOR HTML
    # ========================================================

    descriptive_statistics_html = (
        descriptive_statistics
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    # ========================================================
    # 28. PREPARE PERCENTILES TABLE FOR HTML
    # ========================================================

    percentiles_html = (
        percentiles_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    # ========================================================
    # 29. PREPARE DISTRIBUTION SHAPE TABLE FOR HTML
    # ========================================================

    distribution_shape_html = (
        distribution_shape_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    # ========================================================
    # 30. PREPARE NORMALITY TEST TABLE FOR HTML
    # ========================================================

    normality_tests_html = (
        normality_tests_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "STATISTIC":
                    lambda x:
                    f"{x:.12f}",

                "P_VALUE":
                    lambda x:
                    f"{x:.12e}",

                "ALPHA":
                    lambda x:
                    f"{x:.2f}"
            }
        )
    )


    # ========================================================
    # 31. CREATE HTML REPORT
    # ========================================================

    if not html_exists:

        histogram_base64 = (
            image_to_base64(
                HISTOGRAM_PATH
            )
        )


        boxplot_base64 = (
            image_to_base64(
                BOXPLOT_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Individual Exploratory Analysis —
{FEATURE_COLUMN}
</h1>


<p>

The variable
<strong>{FEATURE_COLUMN}</strong>
represents the registered geographic longitude
associated with the transaction sender.

Longitude is a continuous geographic coordinate
measured in degrees.

</p>


<p class="result">

-180° ≤ longitude ≤ 180°

</p>


<!-- ========================================================
     1. FEATURE OVERVIEW
========================================================= -->


<h2>
1. Feature overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>Valid observations</td>
<td>{valid_observations}</td>
</tr>

<tr>
<td>Missing values</td>
<td>{missing_values}</td>
</tr>

<tr>
<td>Missing percentage</td>
<td>{missing_percentage:.6f}%</td>
</tr>

<tr>
<td>Unique values</td>
<td>{unique_values}</td>
</tr>

<tr>
<td>Unique-value percentage</td>
<td>{unique_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     2. DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
2. Descriptive statistics
</h2>


{descriptive_statistics_html}


<!-- ========================================================
     3. PERCENTILES
========================================================= -->


<h2>
3. Percentiles
</h2>


<p>

Percentiles provide a detailed description
of the empirical distribution of sender
registered longitude values.

</p>


{percentiles_html}


<!-- ========================================================
     4. GEOGRAPHIC RANGE VALIDATION
========================================================= -->


<h2>
4. Geographic range validation
</h2>


<p>

A valid geographic longitude must remain
within the interval:

</p>


<p class="result">

-180° ≤ longitude ≤ 180°

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Values below -180°</td>
<td>{values_below_valid_range}</td>
</tr>

<tr>
<td>Values above 180°</td>
<td>{values_above_valid_range}</td>
</tr>

<tr>
<td>Total invalid geographic values</td>
<td>{invalid_geographic_values}</td>
</tr>

<tr>
<td>Invalid geographic percentage</td>
<td>{invalid_geographic_percentage:.6f}%</td>
</tr>

<tr>
<td>Valid geographic values</td>
<td>{valid_geographic_values}</td>
</tr>

<tr>
<td>Valid geographic percentage</td>
<td>{valid_geographic_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     5. LONGITUDE ORIENTATION
========================================================= -->


<h2>
5. Longitude orientation
</h2>


<p>

Negative longitude values indicate locations
west of the Greenwich meridian.

Positive longitude values indicate locations
east of the Greenwich meridian.

A longitude equal to zero lies on the
Greenwich meridian.

</p>


<table>

<tr>
<th>Longitude group</th>
<th>Observations</th>
<th>Percentage</th>
</tr>

<tr>
<td>Western longitude (&lt; 0)</td>
<td>{western_longitudes}</td>
<td>{western_longitude_percentage:.6f}%</td>
</tr>

<tr>
<td>Greenwich meridian (= 0)</td>
<td>{greenwich_longitudes}</td>
<td>{greenwich_longitude_percentage:.6f}%</td>
</tr>

<tr>
<td>Eastern longitude (&gt; 0)</td>
<td>{eastern_longitudes}</td>
<td>{eastern_longitude_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     6. DISTRIBUTION SHAPE
========================================================= -->


<h2>
6. Distribution shape
</h2>


<p>

Skewness and kurtosis characterize the empirical
shape of the sender registered longitude
distribution.

</p>


{distribution_shape_html}


<div class="note">

<strong>Interpretation:</strong>

<br><br>

Skewness measures asymmetry in the observed
distribution.

<br><br>

Kurtosis provides information about the shape
and tail behavior of the distribution.

<br><br>

These statistics describe the empirical longitude
distribution and do not imply that geographic
coordinates are expected to follow a normal
distribution.

</div>


<!-- ========================================================
     7. NORMALITY TESTS
========================================================= -->


<h2>
7. Normality tests
</h2>


<p>

The Shapiro-Wilk and Jarque-Bera tests were
applied to evaluate whether the empirical
distribution of
<strong>{FEATURE_COLUMN}</strong>
is statistically compatible with a normal
distribution.

</p>


<p>

The complete set of valid observations was used.

No random sampling was performed.

</p>


<p class="result">

H0: The feature follows a normal distribution.

<br><br>

H1: The feature does not follow a normal distribution.

<br><br>

Significance level: α = {ALPHA}

</p>


{normality_tests_html}


<h3>
Shapiro-Wilk interpretation
</h3>


<p>

<strong>Decision:</strong>
{shapiro_decision}

</p>


<p>

{shapiro_interpretation}

</p>


<h3>
Jarque-Bera interpretation
</h3>


<p>

<strong>Decision:</strong>
{jarque_bera_decision}

</p>


<p>

{jarque_bera_interpretation}

</p>


<div class="note">

<strong>Important:</strong>

<br><br>

Normality tests characterize the statistical
shape of the longitude distribution.

Rejecting the null hypothesis of normality
does not indicate that the geographic coordinates
are invalid.

<br><br>

Geographic validity is evaluated separately
using the valid longitude range from
-180° to 180°.

<br><br>

Because the dataset contains a very large number
of observations, normality tests may detect
small deviations from a theoretical normal
distribution.

Therefore, the test results should be interpreted
together with the histogram, skewness, kurtosis,
and descriptive statistics.

</div>


<!-- ========================================================
     8. IQR OUTLIER ANALYSIS
========================================================= -->


<h2>
8. IQR outlier analysis
</h2>


<p>

The interquartile range method identifies
longitude values located far from the central
portion of the empirical distribution.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Q1</td>
<td>{q1_value:.12f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{q3_value:.12f}</td>
</tr>

<tr>
<td>IQR</td>
<td>{iqr:.12f}</td>
</tr>

<tr>
<td>Lower IQR bound</td>
<td>{lower_iqr_bound:.12f}</td>
</tr>

<tr>
<td>Upper IQR bound</td>
<td>{upper_iqr_bound:.12f}</td>
</tr>

<tr>
<td>Lower statistical outliers</td>
<td>{lower_iqr_outliers}</td>
</tr>

<tr>
<td>Upper statistical outliers</td>
<td>{upper_iqr_outliers}</td>
</tr>

<tr>
<td>Total statistical outliers</td>
<td>{total_iqr_outliers}</td>
</tr>

<tr>
<td>Statistical outlier percentage</td>
<td>{iqr_outlier_percentage:.6f}%</td>
</tr>

<tr>
<td>Observations inside IQR limits</td>
<td>{observations_inside_iqr_limits}</td>
</tr>

<tr>
<td>Percentage inside IQR limits</td>
<td>{observations_inside_iqr_percentage:.6f}%</td>
</tr>

</table>


<div class="note">

<strong>Important:</strong>

<br><br>

A geographic coordinate classified as an IQR
outlier is not necessarily invalid.

It only represents a longitude value statistically
distant from the central region of the observed
distribution.

<br><br>

Geographic validity should primarily be evaluated
using the valid interval from
-180° to 180°.

</div>


<!-- ========================================================
     9. HISTOGRAM
========================================================= -->


<h2>
9. Longitude distribution
</h2>


<p>

The histogram presents the empirical distribution
of sender registered longitude values.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{histogram_base64}"
    alt="Distribution of sender registered longitude"
>

</div>


<!-- ========================================================
     10. BOXPLOT
========================================================= -->


<h2>
10. Longitude boxplot
</h2>


<p>

The boxplot summarizes the central distribution
of sender registered longitude and highlights
observations classified as statistical extremes
according to the IQR rule.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{boxplot_base64}"
    alt="Boxplot of sender registered longitude"
>

</div>


<!-- ========================================================
     11. SUMMARY
========================================================= -->


<h2>
11. Summary of results
</h2>


<ul>

<li>
<strong>Total observations:</strong>
{total_observations}
</li>

<li>
<strong>Valid observations:</strong>
{valid_observations}
</li>

<li>
<strong>Missing values:</strong>
{missing_values}
</li>

<li>
<strong>Missing percentage:</strong>
{missing_percentage:.6f}%
</li>

<li>
<strong>Unique longitude values:</strong>
{unique_values}
</li>

<li>
<strong>Minimum longitude:</strong>
{minimum_value:.12f}
</li>

<li>
<strong>Median longitude:</strong>
{median_value:.12f}
</li>

<li>
<strong>Mean longitude:</strong>
{mean_value:.12f}
</li>

<li>
<strong>Maximum longitude:</strong>
{maximum_value:.12f}
</li>

<li>
<strong>Standard deviation:</strong>
{standard_deviation:.12f}
</li>

<li>
<strong>Skewness:</strong>
{skewness:.12f}
</li>

<li>
<strong>Kurtosis:</strong>
{kurtosis:.12f}
</li>

<li>
<strong>Invalid geographic values:</strong>
{invalid_geographic_values}
</li>

<li>
<strong>Geographically valid observations:</strong>
{valid_geographic_percentage:.6f}%
</li>

<li>
<strong>Western longitudes:</strong>
{western_longitude_percentage:.6f}%
</li>

<li>
<strong>Greenwich meridian:</strong>
{greenwich_longitude_percentage:.6f}%
</li>

<li>
<strong>Eastern longitudes:</strong>
{eastern_longitude_percentage:.6f}%
</li>

<li>
<strong>Shapiro-Wilk statistic:</strong>
{shapiro_statistic:.12f}
</li>

<li>
<strong>Shapiro-Wilk p-value:</strong>
{shapiro_p_value:.12e}
</li>

<li>
<strong>Shapiro-Wilk decision:</strong>
{shapiro_decision}
</li>

<li>
<strong>Jarque-Bera statistic:</strong>
{jarque_bera_statistic:.12f}
</li>

<li>
<strong>Jarque-Bera p-value:</strong>
{jarque_bera_p_value:.12e}
</li>

<li>
<strong>Jarque-Bera decision:</strong>
{jarque_bera_decision}
</li>

<li>
<strong>IQR statistical outliers:</strong>
{total_iqr_outliers}
</li>

<li>
<strong>IQR outlier percentage:</strong>
{iqr_outlier_percentage:.6f}%
</li>

</ul>


</body>

</html>
"""


        # ====================================================
        # 32. SAVE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )


        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 33. DISPLAY FEATURE OVERVIEW
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "SEND_LONG_REGISTER SUMMARY"
    )


    print(
        "=" * 100
    )


    print(
        "Total observations:",
        total_observations
    )


    print(
        "Valid observations:",
        valid_observations
    )


    print(
        "Missing values:",
        missing_values
    )


    print(
        "Missing percentage:",
        f"{missing_percentage:.6f}%"
    )


    print(
        "Unique values:",
        unique_values
    )


    print(
        "Unique-value percentage:",
        f"{unique_percentage:.6f}%"
    )


    # ========================================================
    # 34. DISPLAY DESCRIPTIVE STATISTICS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DESCRIPTIVE STATISTICS"
    )


    print(
        "=" * 100
    )


    display(
        descriptive_statistics
    )


    # ========================================================
    # 35. DISPLAY PERCENTILES
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "PERCENTILES"
    )


    print(
        "=" * 100
    )


    display(
        percentiles_table
    )


    # ========================================================
    # 36. DISPLAY GEOGRAPHIC RANGE VALIDATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "GEOGRAPHIC RANGE VALIDATION"
    )


    print(
        "=" * 100
    )


    print(
        "Valid longitude interval:",
        f"[{VALID_MINIMUM}, {VALID_MAXIMUM}]"
    )


    print(
        "Values below valid range:",
        values_below_valid_range
    )


    print(
        "Values above valid range:",
        values_above_valid_range
    )


    print(
        "Total invalid geographic values:",
        invalid_geographic_values
    )


    print(
        "Invalid geographic percentage:",
        f"{invalid_geographic_percentage:.6f}%"
    )


    print(
        "Geographic validation percentage:",
        f"{valid_geographic_percentage:.6f}%"
    )


    # ========================================================
    # 37. DISPLAY LONGITUDE ORIENTATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "LONGITUDE ORIENTATION"
    )


    print(
        "=" * 100
    )


    print(
        "Western longitudes:",
        western_longitudes,
        f"({western_longitude_percentage:.6f}%)"
    )


    print(
        "Greenwich meridian:",
        greenwich_longitudes,
        f"({greenwich_longitude_percentage:.6f}%)"
    )


    print(
        "Eastern longitudes:",
        eastern_longitudes,
        f"({eastern_longitude_percentage:.6f}%)"
    )


    # ========================================================
    # 38. DISPLAY DISTRIBUTION SHAPE
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DISTRIBUTION SHAPE"
    )


    print(
        "=" * 100
    )


    print(
        "Skewness:",
        f"{skewness:.12f}"
    )


    print(
        "Kurtosis:",
        f"{kurtosis:.12f}"
    )


    # ========================================================
    # 39. DISPLAY NORMALITY TESTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "NORMALITY TESTS"
    )


    print(
        "=" * 100
    )


    print(
        "Significance level:",
        ALPHA
    )


    print(
        "\nShapiro-Wilk statistic:",
        f"{shapiro_statistic:.12f}"
    )


    print(
        "Shapiro-Wilk p-value:",
        f"{shapiro_p_value:.12e}"
    )


    print(
        "Shapiro-Wilk decision:",
        shapiro_decision
    )


    print(
        "Shapiro-Wilk interpretation:",
        shapiro_interpretation
    )


    print(
        "\nJarque-Bera statistic:",
        f"{jarque_bera_statistic:.12f}"
    )


    print(
        "Jarque-Bera p-value:",
        f"{jarque_bera_p_value:.12e}"
    )


    print(
        "Jarque-Bera decision:",
        jarque_bera_decision
    )


    print(
        "Jarque-Bera interpretation:",
        jarque_bera_interpretation
    )


    # ========================================================
    # 40. DISPLAY IQR OUTLIER ANALYSIS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "IQR OUTLIER ANALYSIS"
    )


    print(
        "=" * 100
    )


    print(
        "Q1:",
        f"{q1_value:.12f}"
    )


    print(
        "Q3:",
        f"{q3_value:.12f}"
    )


    print(
        "IQR:",
        f"{iqr:.12f}"
    )


    print(
        "Lower bound:",
        f"{lower_iqr_bound:.12f}"
    )


    print(
        "Upper bound:",
        f"{upper_iqr_bound:.12f}"
    )


    print(
        "Lower statistical outliers:",
        lower_iqr_outliers
    )


    print(
        "Upper statistical outliers:",
        upper_iqr_outliers
    )


    print(
        "Total statistical outliers:",
        total_iqr_outliers
    )


    print(
        "Outlier percentage:",
        f"{iqr_outlier_percentage:.6f}%"
    )


    # ========================================================
    # 41. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature
    del valid_feature
    del percentile_values

    gc.collect()


    # ========================================================
    # 42. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "ANALYSIS COMPLETED"
    )


    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )


    print(
        HTML_PATH
    )


    print(
        "\nPNGs:"
    )


    print(
        HISTOGRAM_PATH
    )


    print(
        BOXPLOT_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Histogram: Will be created
Boxplot: Will be created


/usr/local/lib/python3.14/site-packages/scipy/stats/_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 1852394.
  res = hypotest_fun_out(*samples, **kwds)



Histogram created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_geographic/send_long_register/send_long_register_histogram.png


/tmp/ipykernel_2304/3543040620.py:898: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(



Boxplot created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_geographic/send_long_register/send_long_register_boxplot.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_geographic/send_long_register/analysis_send_long_register.html

SEND_LONG_REGISTER SUMMARY
Total observations: 1852394
Valid observations: 1852394
Missing values: 0
Missing percentage: 0.000000%
Unique values: 983
Unique-value percentage: 0.053066%

DESCRIPTIVE STATISTICS


,STATISTIC,VALUE
0,Minimum,-165.672302
1,Q1,-96.797997
2,Median,-87.476898
3,Mean,-90.227832
4,Q3,-80.157997
5,Maximum,-67.950302
6,Standard deviation,13.747895
7,Variance,189.004615
8,Range,97.722000



PERCENTILES


,PERCENTILE,VALUE
0,P1,-123.061401
1,P5,-119.082497
2,P10,-111.098503
3,P25,-96.797997
4,P50,-87.476898
5,P75,-80.157997
6,P90,-74.978104
7,P95,-73.536499
8,P99,-70.345703



GEOGRAPHIC RANGE VALIDATION
Valid longitude interval: [-180.0, 180.0]
Values below valid range: 0
Values above valid range: 0
Total invalid geographic values: 0
Invalid geographic percentage: 0.000000%
Geographic validation percentage: 100.000000%

LONGITUDE ORIENTATION
Western longitudes: 1852394 (100.000000%)
Greenwich meridian: 0 (0.000000%)
Eastern longitudes: 0 (0.000000%)

DISTRIBUTION SHAPE
Skewness: -1.146918908606
Kurtosis: 1.837558986782

NORMALITY TESTS
Significance level: 0.05

Shapiro-Wilk statistic: 0.917662243410
Shapiro-Wilk p-value: 2.819652355796e-154
Shapiro-Wilk decision: Reject H0
Shapiro-Wilk interpretation: The data provide statistical evidence against a normal distribution.

Jarque-Bera statistic: 666728.800128663308
Jarque-Bera p-value: 0.000000000000e+00
Jarque-Bera decision: Reject H0
Jarque-Bera interpretation: The data provide statistical evidence against a normal distribution.

IQR OUTLIER ANALYSIS
Q1: -96.797996520996
Q3: -80.157997131348
IQR: 16.6399993

## <span style="color:maroon"> DISCRETE NUMERICAL </span> ##

### <span style="color:maroon"> TRANS_DAY </span> ###

In [14]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

FEATURE_GROUP = "comum_features"

FEATURE_TYPE = "discrete_numeral"

FEATURE_NAME = "trans_day"

FEATURE_COLUMN = "TRANS_DAY"

VALID_MINIMUM = 1

VALID_MAXIMUM = 31


# ============================================================
# 02. DAY SETTINGS
# ============================================================

EXPECTED_DAYS = list(
    range(
        VALID_MINIMUM,
        VALID_MAXIMUM + 1
    )
)


# ============================================================
# 03. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 04. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 05. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / FEATURE_GROUP
    / FEATURE_TYPE
    / FEATURE_NAME
)


# ============================================================
# 06. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 07. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


DISTRIBUTION_CHART_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_distribution.png"
)


# ============================================================
# 08. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


distribution_chart_exists = (
    DISTRIBUTION_CHART_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and distribution_chart_exists
)


# ============================================================
# 09. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )


    print(
        "No analysis or file creation is required."
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nExisting files:"
    )


    print(
        HTML_PATH
    )


    print(
        DISTRIBUTION_CHART_PATH
    )


else:

    # ========================================================
    # 10. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 11. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )


    print(
        "=" * 100
    )


    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )


    print(
        "Distribution chart:",
        "Already exists"
        if distribution_chart_exists
        else "Will be created"
    )


    # ========================================================
    # 12. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 13. BASIC FEATURE OVERVIEW
    # ========================================================

    total_observations = int(
        len(
            feature
        )
    )


    if total_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    missing_values = int(
        feature
        .isna()
        .sum()
    )


    missing_percentage = (
        missing_values
        / total_observations
        * 100
    )


    valid_feature = (
        feature
        .dropna()
        .astype(
            "float64"
        )
    )


    valid_observations = int(
        len(
            valid_feature
        )
    )


    if valid_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no valid observations."
        )


    unique_values = int(
        valid_feature
        .nunique()
    )


    unique_percentage = (
        unique_values
        / valid_observations
        * 100
    )


    # ========================================================
    # 14. DESCRIPTIVE STATISTICS
    # ========================================================

    minimum_value = float(
        valid_feature.min()
    )


    q1_value = float(
        valid_feature.quantile(
            0.25
        )
    )


    median_value = float(
        valid_feature.median()
    )


    mean_value = float(
        valid_feature.mean()
    )


    q3_value = float(
        valid_feature.quantile(
            0.75
        )
    )


    maximum_value = float(
        valid_feature.max()
    )


    standard_deviation = float(
        valid_feature.std()
    )


    variance = float(
        valid_feature.var()
    )


    data_range = float(
        maximum_value
        - minimum_value
    )


    skewness = float(
        valid_feature.skew()
    )


    kurtosis = float(
        valid_feature.kurt()
    )


    descriptive_statistics = pd.DataFrame({

        "STATISTIC": [
            "Minimum",
            "Q1",
            "Median",
            "Mean",
            "Q3",
            "Maximum",
            "Standard deviation",
            "Variance",
            "Range",
            "Skewness",
            "Kurtosis"
        ],

        "VALUE": [
            minimum_value,
            q1_value,
            median_value,
            mean_value,
            q3_value,
            maximum_value,
            standard_deviation,
            variance,
            data_range,
            skewness,
            kurtosis
        ]
    })


    # ========================================================
    # 15. DAY-VARIABLE VALIDATION
    #
    # Valid day-of-month values:
    #
    # 1 <= day <= 31
    #
    # Values must also be integer values.
    # ========================================================

    values_below_valid_range = int(
        (
            valid_feature
            < VALID_MINIMUM
        )
        .sum()
    )


    values_above_valid_range = int(
        (
            valid_feature
            > VALID_MAXIMUM
        )
        .sum()
    )


    non_integer_values = int(
        (
            ~np.isclose(
                valid_feature.to_numpy(),
                np.rint(
                    valid_feature.to_numpy()
                )
            )
        )
        .sum()
    )


    invalid_range_values = (
        values_below_valid_range
        + values_above_valid_range
    )


    invalid_day_values = int(
        (
            (
                valid_feature
                < VALID_MINIMUM
            )
            |
            (
                valid_feature
                > VALID_MAXIMUM
            )
            |
            (
                ~np.isclose(
                    valid_feature.to_numpy(),
                    np.rint(
                        valid_feature.to_numpy()
                    )
                )
            )
        )
        .sum()
    )


    invalid_day_percentage = (
        invalid_day_values
        / valid_observations
        * 100
    )


    valid_day_values = (
        valid_observations
        - invalid_day_values
    )


    valid_day_percentage = (
        valid_day_values
        / valid_observations
        * 100
    )


    # ========================================================
    # 16. DAYS PRESENT IN THE DATASET
    # ========================================================

    observed_days = sorted(
        valid_feature
        .astype(
            "int64"
        )
        .unique()
        .tolist()
    )


    expected_day_set = set(
        EXPECTED_DAYS
    )


    observed_day_set = set(
        observed_days
    )


    missing_expected_days = sorted(
        expected_day_set
        - observed_day_set
    )


    unexpected_days = sorted(
        observed_day_set
        - expected_day_set
    )


    number_expected_days = int(
        len(
            EXPECTED_DAYS
        )
    )


    number_observed_valid_days = int(
        len(
            observed_day_set
            &
            expected_day_set
        )
    )


    day_coverage_percentage = (
        number_observed_valid_days
        / number_expected_days
        * 100
    )


    # ========================================================
    # 17. DAY FREQUENCY DISTRIBUTION
    # ========================================================

    day_counts = (
        valid_feature
        .astype(
            "int64"
        )
        .value_counts()
        .reindex(
            EXPECTED_DAYS,
            fill_value=0
        )
    )


    day_distribution = pd.DataFrame({

        "DAY":
            EXPECTED_DAYS,

        "COUNT":
            [
                int(
                    day_counts.loc[
                        day
                    ]
                )
                for day in EXPECTED_DAYS
            ]
    })


    day_distribution[
        "PERCENTAGE"
    ] = (
        day_distribution[
            "COUNT"
        ]
        / valid_observations
        * 100
    )


    # ========================================================
    # 18. MOST REPRESENTED DAY
    # ========================================================

    maximum_day_count = int(
        day_distribution[
            "COUNT"
        ]
        .max()
    )


    most_represented_days = (
        day_distribution.loc[
            day_distribution[
                "COUNT"
            ]
            == maximum_day_count,
            "DAY"
        ]
        .astype(
            int
        )
        .tolist()
    )


    most_represented_percentage = (
        maximum_day_count
        / valid_observations
        * 100
    )


    # ========================================================
    # 19. LEAST REPRESENTED OBSERVED DAY
    #
    # Only days that actually occur are considered.
    # ========================================================

    observed_day_distribution = (
        day_distribution[
            day_distribution[
                "COUNT"
            ]
            > 0
        ]
    )


    minimum_day_count = int(
        observed_day_distribution[
            "COUNT"
        ]
        .min()
    )


    least_represented_days = (
        observed_day_distribution.loc[
            observed_day_distribution[
                "COUNT"
            ]
            == minimum_day_count,
            "DAY"
        ]
        .astype(
            int
        )
        .tolist()
    )


    least_represented_percentage = (
        minimum_day_count
        / valid_observations
        * 100
    )


    # ========================================================
    # 20. FREQUENCY DISPERSION
    #
    # These statistics describe variation in the number
    # of observations across days 1 through 31.
    # ========================================================

    mean_day_frequency = float(
        day_distribution[
            "COUNT"
        ]
        .mean()
    )


    median_day_frequency = float(
        day_distribution[
            "COUNT"
        ]
        .median()
    )


    standard_deviation_day_frequency = float(
        day_distribution[
            "COUNT"
        ]
        .std()
    )


    variance_day_frequency = float(
        day_distribution[
            "COUNT"
        ]
        .var()
    )


    coefficient_of_variation_day_frequency = (
        standard_deviation_day_frequency
        / mean_day_frequency
        * 100
        if mean_day_frequency != 0
        else np.nan
    )


    minimum_frequency = int(
        day_distribution[
            "COUNT"
        ]
        .min()
    )


    maximum_frequency = int(
        day_distribution[
            "COUNT"
        ]
        .max()
    )


    frequency_range = int(
        maximum_frequency
        - minimum_frequency
    )


    frequency_dispersion_table = pd.DataFrame({

        "METRIC": [
            "Mean day frequency",
            "Median day frequency",
            "Standard deviation of day frequencies",
            "Variance of day frequencies",
            "Coefficient of variation of day frequencies",
            "Minimum day frequency",
            "Maximum day frequency",
            "Frequency range"
        ],

        "VALUE": [
            mean_day_frequency,
            median_day_frequency,
            standard_deviation_day_frequency,
            variance_day_frequency,
            coefficient_of_variation_day_frequency,
            minimum_frequency,
            maximum_frequency,
            frequency_range
        ]
    })


    # ========================================================
    # 21. CREATE DISTRIBUTION CHART
    # ========================================================

    if not distribution_chart_exists:

        fig, ax = plt.subplots(
            figsize=(
                14,
                6
            )
        )


        ax.bar(
            day_distribution[
                "DAY"
            ],
            day_distribution[
                "COUNT"
            ]
        )


        ax.set_title(
            "Transaction distribution by day of the month"
        )


        ax.set_xlabel(
            "Day of the month"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.set_xticks(
            EXPECTED_DAYS
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(y)
                )
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            DISTRIBUTION_CHART_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nDistribution chart created:"
        )


        print(
            DISTRIBUTION_CHART_PATH
        )


    else:

        print(
            "\nDistribution chart already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 22. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 23. PREPARE DESCRIPTIVE STATISTICS TABLE FOR HTML
    # ========================================================

    descriptive_statistics_html = (
        descriptive_statistics
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.6f}"
            }
        )
    )


    # ========================================================
    # 24. PREPARE DAY DISTRIBUTION TABLE FOR HTML
    # ========================================================

    day_distribution_html = (
        day_distribution
        .to_html(
            index=False,
            border=0,
            formatters={
                "PERCENTAGE":
                    lambda x:
                    f"{x:.6f}%"
            }
        )
    )


    # ========================================================
    # 25. PREPARE FREQUENCY DISPERSION TABLE FOR HTML
    # ========================================================

    frequency_dispersion_html = (
        frequency_dispersion_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.6f}"
            }
        )
    )


    # ========================================================
    # 26. FORMAT DAY LISTS
    # ========================================================

    observed_days_text = (
        ", ".join(
            str(
                day
            )
            for day in observed_days
        )
    )


    missing_expected_days_text = (
        ", ".join(
            str(
                day
            )
            for day in missing_expected_days
        )
        if missing_expected_days
        else "None"
    )


    unexpected_days_text = (
        ", ".join(
            str(
                day
            )
            for day in unexpected_days
        )
        if unexpected_days
        else "None"
    )


    most_represented_days_text = (
        ", ".join(
            str(
                day
            )
            for day in most_represented_days
        )
    )


    least_represented_days_text = (
        ", ".join(
            str(
                day
            )
            for day in least_represented_days
        )
    )


    # ========================================================
    # 27. CREATE HTML REPORT
    # ========================================================

    if not html_exists:

        distribution_chart_base64 = (
            image_to_base64(
                DISTRIBUTION_CHART_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Individual Exploratory Analysis —
{FEATURE_COLUMN}
</h1>


<p>

The variable
<strong>{FEATURE_COLUMN}</strong>
represents the day of the month on which
the transaction occurred.

It is a bounded discrete numerical variable
with possible integer values from
<strong>1 to 31</strong>.

</p>


<p class="result">

1 ≤ TRANS_DAY ≤ 31

</p>


<!-- ========================================================
     1. FEATURE OVERVIEW
========================================================= -->


<h2>
1. Feature overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>Valid observations</td>
<td>{valid_observations}</td>
</tr>

<tr>
<td>Missing values</td>
<td>{missing_values}</td>
</tr>

<tr>
<td>Missing percentage</td>
<td>{missing_percentage:.6f}%</td>
</tr>

<tr>
<td>Unique observed values</td>
<td>{unique_values}</td>
</tr>

<tr>
<td>Unique-value percentage</td>
<td>{unique_percentage:.6f}%</td>
</tr>

<tr>
<td>Expected valid day values</td>
<td>{number_expected_days}</td>
</tr>

<tr>
<td>Observed valid day values</td>
<td>{number_observed_valid_days}</td>
</tr>

<tr>
<td>Valid-day coverage</td>
<td>{day_coverage_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     2. DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
2. Descriptive statistics
</h2>


{descriptive_statistics_html}


<div class="note">

<strong>Important:</strong>

<br><br>

These statistics summarize the numerical
distribution of day-of-month values.

Because the variable is discrete and bounded
between 1 and 31, its descriptive statistics
should be interpreted together with the
frequency distribution.

</div>


<!-- ========================================================
     3. DAY-VARIABLE VALIDATION
========================================================= -->


<h2>
3. Day-variable validation
</h2>


<p>

A valid day-of-month value must be an integer
within the following interval:

</p>


<p class="result">

1 ≤ day ≤ 31

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Values below 1</td>
<td>{values_below_valid_range}</td>
</tr>

<tr>
<td>Values above 31</td>
<td>{values_above_valid_range}</td>
</tr>

<tr>
<td>Values outside valid range</td>
<td>{invalid_range_values}</td>
</tr>

<tr>
<td>Non-integer values</td>
<td>{non_integer_values}</td>
</tr>

<tr>
<td>Total invalid day values</td>
<td>{invalid_day_values}</td>
</tr>

<tr>
<td>Invalid day percentage</td>
<td>{invalid_day_percentage:.6f}%</td>
</tr>

<tr>
<td>Valid day values</td>
<td>{valid_day_values}</td>
</tr>

<tr>
<td>Valid day percentage</td>
<td>{valid_day_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     4. DAY COVERAGE
========================================================= -->


<h2>
4. Day coverage
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Observed days</td>
<td>{observed_days_text}</td>
</tr>

<tr>
<td>Missing expected days</td>
<td>{missing_expected_days_text}</td>
</tr>

<tr>
<td>Unexpected days</td>
<td>{unexpected_days_text}</td>
</tr>

<tr>
<td>Coverage of days 1–31</td>
<td>{day_coverage_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     5. DAY FREQUENCY DISTRIBUTION
========================================================= -->


<h2>
5. Day frequency distribution
</h2>


<p>

The following table shows the absolute and
relative frequency of transactions for each
day of the month.

</p>


{day_distribution_html}


<!-- ========================================================
     6. MOST AND LEAST REPRESENTED DAYS
========================================================= -->


<h2>
6. Most and least represented days
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Most represented day(s)</td>
<td>{most_represented_days_text}</td>
</tr>

<tr>
<td>Maximum frequency</td>
<td>{maximum_day_count}</td>
</tr>

<tr>
<td>Maximum frequency percentage</td>
<td>{most_represented_percentage:.6f}%</td>
</tr>

<tr>
<td>Least represented observed day(s)</td>
<td>{least_represented_days_text}</td>
</tr>

<tr>
<td>Minimum observed frequency</td>
<td>{minimum_day_count}</td>
</tr>

<tr>
<td>Minimum observed frequency percentage</td>
<td>{least_represented_percentage:.6f}%</td>
</tr>

</table>


<div class="note">

<strong>Important:</strong>

<br><br>

Lower frequencies near the end of the month
are not automatically anomalous.

Days 29, 30, and 31 do not occur with the same
calendar frequency across all months.

Therefore, frequency differences should be
interpreted descriptively rather than assuming
that all 31 days must have identical counts.

</div>


<!-- ========================================================
     7. FREQUENCY DISPERSION
========================================================= -->


<h2>
7. Frequency dispersion
</h2>


<p>

These statistics describe how transaction counts
vary across the 31 possible day-of-month values.

</p>


{frequency_dispersion_html}


<div class="note">

<strong>Coefficient of variation:</strong>

<br><br>

The coefficient of variation summarizes the
relative dispersion of transaction counts
across days.

It is calculated as:

<br><br>

<strong>
standard deviation / mean frequency × 100
</strong>

</div>


<!-- ========================================================
     8. DISTRIBUTION CHART
========================================================= -->


<h2>
8. Transaction distribution by day
</h2>


<p>

The bar chart presents the number of transactions
observed on each day of the month.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{distribution_chart_base64}"
    alt="Transaction distribution by day of the month"
>

</div>


<!-- ========================================================
     9. SUMMARY
========================================================= -->


<h2>
9. Summary of results
</h2>


<ul>

<li>
<strong>Total observations:</strong>
{total_observations}
</li>

<li>
<strong>Valid observations:</strong>
{valid_observations}
</li>

<li>
<strong>Missing values:</strong>
{missing_values}
</li>

<li>
<strong>Unique observed days:</strong>
{unique_values}
</li>

<li>
<strong>Minimum observed day:</strong>
{minimum_value:.0f}
</li>

<li>
<strong>Maximum observed day:</strong>
{maximum_value:.0f}
</li>

<li>
<strong>Mean day:</strong>
{mean_value:.6f}
</li>

<li>
<strong>Median day:</strong>
{median_value:.6f}
</li>

<li>
<strong>Standard deviation:</strong>
{standard_deviation:.6f}
</li>

<li>
<strong>Skewness:</strong>
{skewness:.12f}
</li>

<li>
<strong>Kurtosis:</strong>
{kurtosis:.12f}
</li>

<li>
<strong>Values below 1:</strong>
{values_below_valid_range}
</li>

<li>
<strong>Values above 31:</strong>
{values_above_valid_range}
</li>

<li>
<strong>Non-integer values:</strong>
{non_integer_values}
</li>

<li>
<strong>Valid day percentage:</strong>
{valid_day_percentage:.6f}%
</li>

<li>
<strong>Coverage of days 1–31:</strong>
{day_coverage_percentage:.6f}%
</li>

<li>
<strong>Most represented day(s):</strong>
{most_represented_days_text}
</li>

<li>
<strong>Least represented observed day(s):</strong>
{least_represented_days_text}
</li>

<li>
<strong>Coefficient of variation of day frequencies:</strong>
{coefficient_of_variation_day_frequency:.6f}%
</li>

</ul>


</body>

</html>
"""


        # ====================================================
        # 28. SAVE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )


        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 29. DISPLAY FEATURE OVERVIEW
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "TRANS_DAY SUMMARY"
    )


    print(
        "=" * 100
    )


    print(
        "Total observations:",
        total_observations
    )


    print(
        "Valid observations:",
        valid_observations
    )


    print(
        "Missing values:",
        missing_values
    )


    print(
        "Missing percentage:",
        f"{missing_percentage:.6f}%"
    )


    print(
        "Unique observed days:",
        unique_values
    )


    print(
        "Day coverage:",
        f"{day_coverage_percentage:.6f}%"
    )


    # ========================================================
    # 30. DISPLAY DESCRIPTIVE STATISTICS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DESCRIPTIVE STATISTICS"
    )


    print(
        "=" * 100
    )


    display(
        descriptive_statistics
    )


    # ========================================================
    # 31. DISPLAY VALIDATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DAY-VARIABLE VALIDATION"
    )


    print(
        "=" * 100
    )


    print(
        "Valid interval:",
        f"[{VALID_MINIMUM}, {VALID_MAXIMUM}]"
    )


    print(
        "Values below 1:",
        values_below_valid_range
    )


    print(
        "Values above 31:",
        values_above_valid_range
    )


    print(
        "Non-integer values:",
        non_integer_values
    )


    print(
        "Total invalid values:",
        invalid_day_values
    )


    print(
        "Valid day percentage:",
        f"{valid_day_percentage:.6f}%"
    )


    # ========================================================
    # 32. DISPLAY DAY DISTRIBUTION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DAY FREQUENCY DISTRIBUTION"
    )


    print(
        "=" * 100
    )


    display(
        day_distribution
    )


    # ========================================================
    # 33. DISPLAY MOST AND LEAST REPRESENTED DAYS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "MOST AND LEAST REPRESENTED DAYS"
    )


    print(
        "=" * 100
    )


    print(
        "Most represented day(s):",
        most_represented_days_text
    )


    print(
        "Maximum frequency:",
        maximum_day_count
    )


    print(
        "Maximum frequency percentage:",
        f"{most_represented_percentage:.6f}%"
    )


    print(
        "\nLeast represented observed day(s):",
        least_represented_days_text
    )


    print(
        "Minimum observed frequency:",
        minimum_day_count
    )


    print(
        "Minimum observed frequency percentage:",
        f"{least_represented_percentage:.6f}%"
    )


    # ========================================================
    # 34. DISPLAY FREQUENCY DISPERSION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "FREQUENCY DISPERSION"
    )


    print(
        "=" * 100
    )


    display(
        frequency_dispersion_table
    )


    # ========================================================
    # 35. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature
    del valid_feature
    del day_counts

    gc.collect()


    # ========================================================
    # 36. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "ANALYSIS COMPLETED"
    )


    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )


    print(
        HTML_PATH
    )


    print(
        "\nPNG:"
    )


    print(
        DISTRIBUTION_CHART_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Distribution chart: Will be created

Distribution chart created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/discrete_numeral/trans_day/trans_day_distribution.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/discrete_numeral/trans_day/analysis_trans_day.html

TRANS_DAY SUMMARY
Total observations: 1852394
Valid observations: 1852394
Missing values: 0
Missing percentage: 0.000000%
Unique observed days: 31
Day coverage: 100.000000%

DESCRIPTIVE STATISTICS


,STATISTIC,VALUE
0,Minimum,1.000000
1,Q1,8.000000
2,Median,16.000000
3,Mean,15.850756
4,Q3,24.000000
5,Maximum,31.000000
6,Standard deviation,8.876245
7,Variance,78.787727
8,Range,30.000000
9,Skewness,-0.003649



DAY-VARIABLE VALIDATION
Valid interval: [1, 31]
Values below 1: 0
Values above 31: 0
Non-integer values: 0
Total invalid values: 0
Valid day percentage: 100.000000%

DAY FREQUENCY DISTRIBUTION


,DAY,COUNT,PERCENTAGE
0,1,65691,3.546276
1,2,59762,3.226203
2,3,58271,3.145713
3,4,57546,3.106575
4,5,57655,3.112459
5,6,60653,3.274303
6,7,63665,3.436904
7,8,63907,3.449968
8,9,60072,3.242939
9,10,58651,3.166227



MOST AND LEAST REPRESENTED DAYS
Most represented day(s): 28
Maximum frequency: 65910
Maximum frequency percentage: 3.558098%

Least represented observed day(s): 31
Minimum observed frequency: 36311
Minimum observed frequency percentage: 1.960220%

FREQUENCY DISPERSION


,METRIC,VALUE
0,Mean day frequency,5.975465e+04
1,Median day frequency,5.984000e+04
2,Standard deviation of day frequencies,5.062215e+03
3,Variance of day frequencies,2.562602e+07
4,Coefficient of variation of day frequencies,8.471667e+00
5,Minimum day frequency,3.631100e+04
6,Maximum day frequency,6.591000e+04
7,Frequency range,2.959900e+04



ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/discrete_numeral/trans_day

HTML:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/discrete_numeral/trans_day/analysis_trans_day.html

PNG:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/discrete_numeral/trans_day/trans_day_distribution.png


### <span style="color:maroon"> SEND_POP_REGISTER </span> ###

In [15]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

FEATURE_GROUP = "comum_features"

FEATURE_TYPE = "discrete_numeral"

FEATURE_NAME = "send_pop_register"

FEATURE_COLUMN = "SEND_POP_REGISTER"

VALID_MINIMUM = 0

ALPHA = 0.05

TOP_FREQUENT_VALUES = 10


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / FEATURE_GROUP
    / FEATURE_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


HISTOGRAM_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_histogram.png"
)


LOG_HISTOGRAM_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_log_histogram.png"
)


BOXPLOT_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_boxplot.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


histogram_exists = (
    HISTOGRAM_PATH.exists()
)


log_histogram_exists = (
    LOG_HISTOGRAM_PATH.exists()
)


boxplot_exists = (
    BOXPLOT_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and histogram_exists
    and log_histogram_exists
    and boxplot_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )


    print(
        "No analysis or file creation is required."
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nExisting files:"
    )


    print(
        HTML_PATH
    )


    print(
        HISTOGRAM_PATH
    )


    print(
        LOG_HISTOGRAM_PATH
    )


    print(
        BOXPLOT_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )


    print(
        "=" * 100
    )


    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )


    print(
        "Histogram:",
        "Already exists"
        if histogram_exists
        else "Will be created"
    )


    print(
        "Log histogram:",
        "Already exists"
        if log_histogram_exists
        else "Will be created"
    )


    print(
        "Boxplot:",
        "Already exists"
        if boxplot_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. BASIC FEATURE OVERVIEW
    # ========================================================

    total_observations = int(
        len(
            feature
        )
    )


    if total_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    missing_values = int(
        feature
        .isna()
        .sum()
    )


    missing_percentage = (
        missing_values
        / total_observations
        * 100
    )


    valid_feature = (
        feature
        .dropna()
        .astype(
            "float64"
        )
    )


    valid_observations = int(
        len(
            valid_feature
        )
    )


    if valid_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no valid observations."
        )


    unique_values = int(
        valid_feature
        .nunique()
    )


    unique_percentage = (
        unique_values
        / valid_observations
        * 100
    )


    # ========================================================
    # 13. DESCRIPTIVE STATISTICS
    # ========================================================

    minimum_value = float(
        valid_feature.min()
    )


    q1_value = float(
        valid_feature.quantile(
            0.25
        )
    )


    median_value = float(
        valid_feature.median()
    )


    mean_value = float(
        valid_feature.mean()
    )


    q3_value = float(
        valid_feature.quantile(
            0.75
        )
    )


    maximum_value = float(
        valid_feature.max()
    )


    standard_deviation = float(
        valid_feature.std()
    )


    variance = float(
        valid_feature.var()
    )


    data_range = float(
        maximum_value
        - minimum_value
    )


    coefficient_of_variation = (
        standard_deviation
        / mean_value
        * 100
        if mean_value != 0
        else np.nan
    )


    descriptive_statistics = pd.DataFrame({

        "STATISTIC": [
            "Minimum",
            "Q1",
            "Median",
            "Mean",
            "Q3",
            "Maximum",
            "Standard deviation",
            "Variance",
            "Range",
            "Coefficient of variation"
        ],

        "VALUE": [
            minimum_value,
            q1_value,
            median_value,
            mean_value,
            q3_value,
            maximum_value,
            standard_deviation,
            variance,
            data_range,
            coefficient_of_variation
        ]
    })


    # ========================================================
    # 14. PERCENTILES
    # ========================================================

    percentile_levels = [
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]


    percentile_values = (
        valid_feature
        .quantile(
            percentile_levels
        )
    )


    percentiles_table = pd.DataFrame({

        "PERCENTILE": [
            "P1",
            "P5",
            "P10",
            "P25",
            "P50",
            "P75",
            "P90",
            "P95",
            "P99"
        ],

        "VALUE": [
            float(
                percentile_values.loc[
                    percentile
                ]
            )
            for percentile in percentile_levels
        ]
    })


    # ========================================================
    # 15. COUNT-VARIABLE VALIDATION
    #
    # Population is a count variable.
    #
    # Therefore:
    #
    # value >= 0
    #
    # and values should be integer counts.
    # ========================================================

    negative_values = int(
        (
            valid_feature
            < VALID_MINIMUM
        )
        .sum()
    )


    zero_values = int(
        (
            valid_feature
            == 0
        )
        .sum()
    )


    positive_values = int(
        (
            valid_feature
            > 0
        )
        .sum()
    )


    non_integer_values = int(
        (
            ~np.isclose(
                valid_feature.to_numpy(),
                np.rint(
                    valid_feature.to_numpy()
                )
            )
        )
        .sum()
    )


    negative_percentage = (
        negative_values
        / valid_observations
        * 100
    )


    zero_percentage = (
        zero_values
        / valid_observations
        * 100
    )


    positive_percentage = (
        positive_values
        / valid_observations
        * 100
    )


    non_integer_percentage = (
        non_integer_values
        / valid_observations
        * 100
    )


    valid_count_values = int(
        valid_observations
        - negative_values
        - non_integer_values
    )


    valid_count_percentage = (
        valid_count_values
        / valid_observations
        * 100
    )


    # ========================================================
    # 16. MOST FREQUENT POPULATION VALUES
    # ========================================================

    value_counts = (
        valid_feature
        .value_counts()
        .head(
            TOP_FREQUENT_VALUES
        )
    )


    most_frequent_values_table = pd.DataFrame({

        "POPULATION_VALUE":
            value_counts.index.astype(
                "float64"
            ),

        "COUNT":
            value_counts.values.astype(
                "int64"
            )
    })


    most_frequent_values_table[
        "PERCENTAGE"
    ] = (
        most_frequent_values_table[
            "COUNT"
        ]
        / valid_observations
        * 100
    )


    mode_value = float(
        value_counts.index[
            0
        ]
    )


    mode_count = int(
        value_counts.iloc[
            0
        ]
    )


    mode_percentage = (
        mode_count
        / valid_observations
        * 100
    )


    # ========================================================
    # 17. DISTRIBUTION SHAPE
    # ========================================================

    skewness = float(
        valid_feature.skew()
    )


    kurtosis = float(
        valid_feature.kurt()
    )


    distribution_shape_table = pd.DataFrame({

        "METRIC": [
            "Skewness",
            "Kurtosis"
        ],

        "VALUE": [
            skewness,
            kurtosis
        ]
    })


    # ========================================================
    # 18. SHAPIRO-WILK NORMALITY TEST
    #
    # H0:
    # The feature follows a normal distribution.
    #
    # H1:
    # The feature does not follow a normal distribution.
    #
    # The complete valid feature is used.
    # No random sampling is performed.
    # ========================================================

    (
        shapiro_statistic,
        shapiro_p_value
    ) = stats.shapiro(
        valid_feature.to_numpy()
    )


    shapiro_statistic = float(
        shapiro_statistic
    )


    shapiro_p_value = float(
        shapiro_p_value
    )


    # ========================================================
    # 19. SHAPIRO-WILK DECISION
    # ========================================================

    if shapiro_p_value < ALPHA:

        shapiro_decision = (
            "Reject H0"
        )


        shapiro_interpretation = (
            "The data provide statistical evidence "
            "against a normal distribution."
        )


    else:

        shapiro_decision = (
            "Fail to reject H0"
        )


        shapiro_interpretation = (
            "The data do not provide sufficient "
            "statistical evidence against a normal distribution."
        )


    # ========================================================
    # 20. JARQUE-BERA NORMALITY TEST
    #
    # H0:
    # The feature follows a normal distribution.
    #
    # H1:
    # The feature does not follow a normal distribution.
    # ========================================================

    jarque_bera_result = stats.jarque_bera(
        valid_feature.to_numpy()
    )


    jarque_bera_statistic = float(
        jarque_bera_result.statistic
    )


    jarque_bera_p_value = float(
        jarque_bera_result.pvalue
    )


    # ========================================================
    # 21. JARQUE-BERA DECISION
    # ========================================================

    if jarque_bera_p_value < ALPHA:

        jarque_bera_decision = (
            "Reject H0"
        )


        jarque_bera_interpretation = (
            "The data provide statistical evidence "
            "against a normal distribution."
        )


    else:

        jarque_bera_decision = (
            "Fail to reject H0"
        )


        jarque_bera_interpretation = (
            "The data do not provide sufficient "
            "statistical evidence against a normal distribution."
        )


    # ========================================================
    # 22. NORMALITY TEST RESULTS TABLE
    # ========================================================

    normality_tests_table = pd.DataFrame({

        "TEST": [
            "Shapiro-Wilk",
            "Jarque-Bera"
        ],

        "STATISTIC": [
            shapiro_statistic,
            jarque_bera_statistic
        ],

        "P_VALUE": [
            shapiro_p_value,
            jarque_bera_p_value
        ],

        "ALPHA": [
            ALPHA,
            ALPHA
        ],

        "DECISION": [
            shapiro_decision,
            jarque_bera_decision
        ]
    })


    # ========================================================
    # 23. IQR OUTLIER ANALYSIS
    #
    # Outliers indicate statistical extremes.
    #
    # They do not automatically indicate invalid
    # population values.
    # ========================================================

    iqr = (
        q3_value
        - q1_value
    )


    lower_iqr_bound = (
        q1_value
        - 1.5
        * iqr
    )


    upper_iqr_bound = (
        q3_value
        + 1.5
        * iqr
    )


    lower_iqr_outliers = int(
        (
            valid_feature
            < lower_iqr_bound
        )
        .sum()
    )


    upper_iqr_outliers = int(
        (
            valid_feature
            > upper_iqr_bound
        )
        .sum()
    )


    total_iqr_outliers = (
        lower_iqr_outliers
        + upper_iqr_outliers
    )


    iqr_outlier_percentage = (
        total_iqr_outliers
        / valid_observations
        * 100
    )


    observations_inside_iqr_limits = (
        valid_observations
        - total_iqr_outliers
    )


    observations_inside_iqr_percentage = (
        observations_inside_iqr_limits
        / valid_observations
        * 100
    )


    # ========================================================
    # 24. CREATE REGULAR HISTOGRAM
    # ========================================================

    if not histogram_exists:

        fig, ax = plt.subplots(
            figsize=(
                11,
                6
            )
        )


        ax.hist(
            valid_feature,
            bins=60
        )


        ax.set_title(
            "Distribution of sender registered population"
        )


        ax.set_xlabel(
            "Population"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.xaxis.set_major_formatter(
            FuncFormatter(
                lambda x, pos:
                str(
                    int(x)
                )
            )
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(y)
                )
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            HISTOGRAM_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nHistogram created:"
        )


        print(
            HISTOGRAM_PATH
        )


    else:

        print(
            "\nHistogram already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 25. CREATE LOG10 HISTOGRAM
    #
    # Only strictly positive population values are used.
    #
    # This visualization helps inspect a strongly
    # right-skewed count distribution.
    # ========================================================

    if not log_histogram_exists:

        positive_feature = (
            valid_feature[
                valid_feature
                > 0
            ]
        )


        if len(
            positive_feature
        ) > 0:

            log_feature = np.log10(
                positive_feature
            )


            fig, ax = plt.subplots(
                figsize=(
                    11,
                    6
                )
            )


            ax.hist(
                log_feature,
                bins=60
            )


            ax.set_title(
                "Log10 distribution of sender registered population"
            )


            ax.set_xlabel(
                "log10(Population)"
            )


            ax.set_ylabel(
                "Number of observations"
            )


            ax.yaxis.set_major_formatter(
                FuncFormatter(
                    lambda y, pos:
                    str(
                        int(y)
                    )
                )
            )


            ax.grid(
                axis="y",
                alpha=0.3
            )


            fig.tight_layout()


            fig.savefig(
                LOG_HISTOGRAM_PATH,
                format="png",
                dpi=600,
                bbox_inches="tight"
            )


            plt.close(
                fig
            )


            del positive_feature
            del log_feature


            print(
                "\nLog histogram created:"
            )


            print(
                LOG_HISTOGRAM_PATH
            )


        else:

            raise ValueError(
                "No positive population values are available "
                "for the logarithmic histogram."
            )


    else:

        print(
            "\nLog histogram already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 26. CREATE BOXPLOT
    # ========================================================

    if not boxplot_exists:

        fig, ax = plt.subplots(
            figsize=(
                11,
                4
            )
        )


        ax.boxplot(
            valid_feature,
            vert=False
        )


        ax.set_title(
            "Boxplot of sender registered population"
        )


        ax.set_xlabel(
            "Population"
        )


        ax.set_yticks(
            []
        )


        ax.xaxis.set_major_formatter(
            FuncFormatter(
                lambda x, pos:
                str(
                    int(x)
                )
            )
        )


        ax.grid(
            axis="x",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            BOXPLOT_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nBoxplot created:"
        )


        print(
            BOXPLOT_PATH
        )


    else:

        print(
            "\nBoxplot already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 27. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 28. PREPARE DESCRIPTIVE STATISTICS TABLE FOR HTML
    # ========================================================

    descriptive_statistics_html = (
        descriptive_statistics
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.6f}"
            }
        )
    )


    # ========================================================
    # 29. PREPARE PERCENTILES TABLE FOR HTML
    # ========================================================

    percentiles_html = (
        percentiles_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.6f}"
            }
        )
    )


    # ========================================================
    # 30. PREPARE MOST FREQUENT VALUES TABLE FOR HTML
    # ========================================================

    most_frequent_values_html = (
        most_frequent_values_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "POPULATION_VALUE":
                    lambda x:
                    f"{x:.0f}",

                "PERCENTAGE":
                    lambda x:
                    f"{x:.6f}%"
            }
        )
    )


    # ========================================================
    # 31. PREPARE DISTRIBUTION SHAPE TABLE FOR HTML
    # ========================================================

    distribution_shape_html = (
        distribution_shape_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    # ========================================================
    # 32. PREPARE NORMALITY TEST TABLE FOR HTML
    # ========================================================

    normality_tests_html = (
        normality_tests_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "STATISTIC":
                    lambda x:
                    f"{x:.12f}",

                "P_VALUE":
                    lambda x:
                    f"{x:.12e}",

                "ALPHA":
                    lambda x:
                    f"{x:.2f}"
            }
        )
    )


    # ========================================================
    # 33. CREATE HTML REPORT
    # ========================================================

    if not html_exists:

        histogram_base64 = (
            image_to_base64(
                HISTOGRAM_PATH
            )
        )


        log_histogram_base64 = (
            image_to_base64(
                LOG_HISTOGRAM_PATH
            )
        )


        boxplot_base64 = (
            image_to_base64(
                BOXPLOT_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Individual Exploratory Analysis —
{FEATURE_COLUMN}
</h1>


<p>

The variable
<strong>{FEATURE_COLUMN}</strong>
represents the registered population associated
with the transaction sender location.

It is treated as a
<strong>discrete numerical count variable</strong>.

</p>


<!-- ========================================================
     1. FEATURE OVERVIEW
========================================================= -->


<h2>
1. Feature overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>Valid observations</td>
<td>{valid_observations}</td>
</tr>

<tr>
<td>Missing values</td>
<td>{missing_values}</td>
</tr>

<tr>
<td>Missing percentage</td>
<td>{missing_percentage:.6f}%</td>
</tr>

<tr>
<td>Unique values</td>
<td>{unique_values}</td>
</tr>

<tr>
<td>Unique-value percentage</td>
<td>{unique_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     2. DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
2. Descriptive statistics
</h2>


{descriptive_statistics_html}


<!-- ========================================================
     3. PERCENTILES
========================================================= -->


<h2>
3. Percentiles
</h2>


<p>

Percentiles describe the distribution of registered
population values without assuming a particular
probability distribution.

</p>


{percentiles_html}


<!-- ========================================================
     4. COUNT-VARIABLE VALIDATION
========================================================= -->


<h2>
4. Count-variable validation
</h2>


<p>

Population represents a count.

Therefore, valid population values are expected
to be non-negative integer values.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Negative values</td>
<td>{negative_values}</td>
</tr>

<tr>
<td>Negative-value percentage</td>
<td>{negative_percentage:.6f}%</td>
</tr>

<tr>
<td>Zero values</td>
<td>{zero_values}</td>
</tr>

<tr>
<td>Zero-value percentage</td>
<td>{zero_percentage:.6f}%</td>
</tr>

<tr>
<td>Positive values</td>
<td>{positive_values}</td>
</tr>

<tr>
<td>Positive-value percentage</td>
<td>{positive_percentage:.6f}%</td>
</tr>

<tr>
<td>Non-integer values</td>
<td>{non_integer_values}</td>
</tr>

<tr>
<td>Non-integer percentage</td>
<td>{non_integer_percentage:.6f}%</td>
</tr>

<tr>
<td>Valid count values</td>
<td>{valid_count_values}</td>
</tr>

<tr>
<td>Valid count percentage</td>
<td>{valid_count_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     5. MOST FREQUENT POPULATION VALUES
========================================================= -->


<h2>
5. Most frequent population values
</h2>


<p>

The following table presents the
{TOP_FREQUENT_VALUES}
population values with the highest frequency
in the dataset.

</p>


{most_frequent_values_html}


<p>

<strong>Mode:</strong>
{mode_value:.0f}

<br>

<strong>Mode frequency:</strong>
{mode_count}

<br>

<strong>Mode percentage:</strong>
{mode_percentage:.6f}%

</p>


<!-- ========================================================
     6. DISTRIBUTION SHAPE
========================================================= -->


<h2>
6. Distribution shape
</h2>


<p>

Skewness and kurtosis characterize the empirical
shape of the population distribution.

</p>


{distribution_shape_html}


<div class="note">

<strong>Interpretation:</strong>

<br><br>

Positive skewness indicates a longer right tail.

Negative skewness indicates a longer left tail.

<br><br>

Kurtosis provides information about the relative
tail weight and shape of the distribution.

</div>


<!-- ========================================================
     7. NORMALITY TESTS
========================================================= -->


<h2>
7. Normality tests
</h2>


<p>

The Shapiro-Wilk and Jarque-Bera tests were applied
to evaluate whether the empirical distribution of
<strong>{FEATURE_COLUMN}</strong>
is statistically compatible with a normal
distribution.

</p>


<p>

The complete set of valid observations was used.

No random sampling was performed.

</p>


<p class="result">

H0: The feature follows a normal distribution.

<br><br>

H1: The feature does not follow a normal distribution.

<br><br>

Significance level: α = {ALPHA}

</p>


{normality_tests_html}


<h3>
Shapiro-Wilk interpretation
</h3>


<p>

<strong>Decision:</strong>
{shapiro_decision}

</p>


<p>

{shapiro_interpretation}

</p>


<h3>
Jarque-Bera interpretation
</h3>


<p>

<strong>Decision:</strong>
{jarque_bera_decision}

</p>


<p>

{jarque_bera_interpretation}

</p>


<div class="note">

<strong>Important:</strong>

<br><br>

{FEATURE_COLUMN} is a discrete count variable
and is not required to follow a normal distribution.

The tests are used here only to characterize
its empirical distribution.

<br><br>

Because the dataset contains a very large number
of observations, statistical tests may detect
even small departures from theoretical normality.

Therefore, normality results should be interpreted
together with the histogram, skewness, kurtosis,
percentiles, and descriptive statistics.

</div>


<!-- ========================================================
     8. IQR OUTLIER ANALYSIS
========================================================= -->


<h2>
8. IQR outlier analysis
</h2>


<p>

The interquartile range method identifies
population values statistically distant from
the central portion of the observed distribution.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Q1</td>
<td>{q1_value:.6f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{q3_value:.6f}</td>
</tr>

<tr>
<td>IQR</td>
<td>{iqr:.6f}</td>
</tr>

<tr>
<td>Lower IQR bound</td>
<td>{lower_iqr_bound:.6f}</td>
</tr>

<tr>
<td>Upper IQR bound</td>
<td>{upper_iqr_bound:.6f}</td>
</tr>

<tr>
<td>Lower statistical outliers</td>
<td>{lower_iqr_outliers}</td>
</tr>

<tr>
<td>Upper statistical outliers</td>
<td>{upper_iqr_outliers}</td>
</tr>

<tr>
<td>Total statistical outliers</td>
<td>{total_iqr_outliers}</td>
</tr>

<tr>
<td>Statistical outlier percentage</td>
<td>{iqr_outlier_percentage:.6f}%</td>
</tr>

<tr>
<td>Observations inside IQR limits</td>
<td>{observations_inside_iqr_limits}</td>
</tr>

<tr>
<td>Percentage inside IQR limits</td>
<td>{observations_inside_iqr_percentage:.6f}%</td>
</tr>

</table>


<div class="note">

<strong>Important:</strong>

<br><br>

Population values identified as IQR outliers
are not automatically invalid observations.

A highly populated location can legitimately
produce a population value far above the
central portion of the distribution.

</div>


<!-- ========================================================
     9. HISTOGRAM
========================================================= -->


<h2>
9. Population distribution
</h2>


<p>

The histogram presents the empirical distribution
of sender registered population values.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{histogram_base64}"
    alt="Distribution of sender registered population"
>

</div>


<!-- ========================================================
     10. LOG10 HISTOGRAM
========================================================= -->


<h2>
10. Logarithmic population distribution
</h2>


<p>

Because population values can span a wide numerical
range, the logarithmic representation helps reveal
distributional structure that may be compressed
in the original scale.

</p>


<p class="result">

log10(Population)

</p>


<div class="chart">

<img
    src="data:image/png;base64,{log_histogram_base64}"
    alt="Log10 distribution of sender registered population"
>

</div>


<!-- ========================================================
     11. BOXPLOT
========================================================= -->


<h2>
11. Population boxplot
</h2>


<p>

The boxplot summarizes the central distribution
of registered population and highlights observations
classified as statistical extremes according
to the IQR rule.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{boxplot_base64}"
    alt="Boxplot of sender registered population"
>

</div>


<!-- ========================================================
     12. SUMMARY
========================================================= -->


<h2>
12. Summary of results
</h2>


<ul>

<li>
<strong>Total observations:</strong>
{total_observations}
</li>

<li>
<strong>Valid observations:</strong>
{valid_observations}
</li>

<li>
<strong>Missing values:</strong>
{missing_values}
</li>

<li>
<strong>Unique population values:</strong>
{unique_values}
</li>

<li>
<strong>Minimum population:</strong>
{minimum_value:.0f}
</li>

<li>
<strong>Median population:</strong>
{median_value:.6f}
</li>

<li>
<strong>Mean population:</strong>
{mean_value:.6f}
</li>

<li>
<strong>Maximum population:</strong>
{maximum_value:.0f}
</li>

<li>
<strong>Standard deviation:</strong>
{standard_deviation:.6f}
</li>

<li>
<strong>Coefficient of variation:</strong>
{coefficient_of_variation:.6f}%
</li>

<li>
<strong>Mode:</strong>
{mode_value:.0f}
</li>

<li>
<strong>Negative values:</strong>
{negative_values}
</li>

<li>
<strong>Non-integer values:</strong>
{non_integer_values}
</li>

<li>
<strong>Skewness:</strong>
{skewness:.12f}
</li>

<li>
<strong>Kurtosis:</strong>
{kurtosis:.12f}
</li>

<li>
<strong>Shapiro-Wilk statistic:</strong>
{shapiro_statistic:.12f}
</li>

<li>
<strong>Shapiro-Wilk p-value:</strong>
{shapiro_p_value:.12e}
</li>

<li>
<strong>Shapiro-Wilk decision:</strong>
{shapiro_decision}
</li>

<li>
<strong>Jarque-Bera statistic:</strong>
{jarque_bera_statistic:.12f}
</li>

<li>
<strong>Jarque-Bera p-value:</strong>
{jarque_bera_p_value:.12e}
</li>

<li>
<strong>Jarque-Bera decision:</strong>
{jarque_bera_decision}
</li>

<li>
<strong>IQR statistical outliers:</strong>
{total_iqr_outliers}
</li>

<li>
<strong>IQR outlier percentage:</strong>
{iqr_outlier_percentage:.6f}%
</li>

</ul>


</body>

</html>
"""


        # ====================================================
        # 34. SAVE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )


        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 35. DISPLAY FEATURE OVERVIEW
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "SEND_POP_REGISTER SUMMARY"
    )


    print(
        "=" * 100
    )


    print(
        "Total observations:",
        total_observations
    )


    print(
        "Valid observations:",
        valid_observations
    )


    print(
        "Missing values:",
        missing_values
    )


    print(
        "Missing percentage:",
        f"{missing_percentage:.6f}%"
    )


    print(
        "Unique values:",
        unique_values
    )


    print(
        "Unique-value percentage:",
        f"{unique_percentage:.6f}%"
    )


    # ========================================================
    # 36. DISPLAY DESCRIPTIVE STATISTICS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DESCRIPTIVE STATISTICS"
    )


    print(
        "=" * 100
    )


    display(
        descriptive_statistics
    )


    # ========================================================
    # 37. DISPLAY PERCENTILES
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "PERCENTILES"
    )


    print(
        "=" * 100
    )


    display(
        percentiles_table
    )


    # ========================================================
    # 38. DISPLAY COUNT-VARIABLE VALIDATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "COUNT-VARIABLE VALIDATION"
    )


    print(
        "=" * 100
    )


    print(
        "Negative values:",
        negative_values
    )


    print(
        "Zero values:",
        zero_values
    )


    print(
        "Positive values:",
        positive_values
    )


    print(
        "Non-integer values:",
        non_integer_values
    )


    print(
        "Valid count percentage:",
        f"{valid_count_percentage:.6f}%"
    )


    # ========================================================
    # 39. DISPLAY MOST FREQUENT VALUES
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "MOST FREQUENT POPULATION VALUES"
    )


    print(
        "=" * 100
    )


    display(
        most_frequent_values_table
    )


    # ========================================================
    # 40. DISPLAY DISTRIBUTION SHAPE
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DISTRIBUTION SHAPE"
    )


    print(
        "=" * 100
    )


    print(
        "Skewness:",
        f"{skewness:.12f}"
    )


    print(
        "Kurtosis:",
        f"{kurtosis:.12f}"
    )


    # ========================================================
    # 41. DISPLAY NORMALITY TESTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "NORMALITY TESTS"
    )


    print(
        "=" * 100
    )


    print(
        "Significance level:",
        ALPHA
    )


    print(
        "\nShapiro-Wilk statistic:",
        f"{shapiro_statistic:.12f}"
    )


    print(
        "Shapiro-Wilk p-value:",
        f"{shapiro_p_value:.12e}"
    )


    print(
        "Shapiro-Wilk decision:",
        shapiro_decision
    )


    print(
        "Shapiro-Wilk interpretation:",
        shapiro_interpretation
    )


    print(
        "\nJarque-Bera statistic:",
        f"{jarque_bera_statistic:.12f}"
    )


    print(
        "Jarque-Bera p-value:",
        f"{jarque_bera_p_value:.12e}"
    )


    print(
        "Jarque-Bera decision:",
        jarque_bera_decision
    )


    print(
        "Jarque-Bera interpretation:",
        jarque_bera_interpretation
    )


    # ========================================================
    # 42. DISPLAY IQR OUTLIER ANALYSIS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "IQR OUTLIER ANALYSIS"
    )


    print(
        "=" * 100
    )


    print(
        "Q1:",
        f"{q1_value:.6f}"
    )


    print(
        "Q3:",
        f"{q3_value:.6f}"
    )


    print(
        "IQR:",
        f"{iqr:.6f}"
    )


    print(
        "Lower bound:",
        f"{lower_iqr_bound:.6f}"
    )


    print(
        "Upper bound:",
        f"{upper_iqr_bound:.6f}"
    )


    print(
        "Lower statistical outliers:",
        lower_iqr_outliers
    )


    print(
        "Upper statistical outliers:",
        upper_iqr_outliers
    )


    print(
        "Total statistical outliers:",
        total_iqr_outliers
    )


    print(
        "Outlier percentage:",
        f"{iqr_outlier_percentage:.6f}%"
    )


    # ========================================================
    # 43. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature
    del valid_feature
    del percentile_values
    del value_counts

    gc.collect()


    # ========================================================
    # 44. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "ANALYSIS COMPLETED"
    )


    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )


    print(
        HTML_PATH
    )


    print(
        "\nPNGs:"
    )


    print(
        HISTOGRAM_PATH
    )


    print(
        LOG_HISTOGRAM_PATH
    )


    print(
        BOXPLOT_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Histogram: Will be created
Log histogram: Will be created
Boxplot: Will be created


/usr/local/lib/python3.14/site-packages/scipy/stats/_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 1852394.
  res = hypotest_fun_out(*samples, **kwds)



Histogram created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/discrete_numeral/send_pop_register/send_pop_register_histogram.png

Log histogram created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/discrete_numeral/send_pop_register/send_pop_register_log_histogram.png


/tmp/ipykernel_2304/3179489233.py:1105: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(



Boxplot created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/discrete_numeral/send_pop_register/send_pop_register_boxplot.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/discrete_numeral/send_pop_register/analysis_send_pop_register.html

SEND_POP_REGISTER SUMMARY
Total observations: 1852394
Valid observations: 1852394
Missing values: 0
Missing percentage: 0.000000%
Unique values: 891
Unique-value percentage: 0.048100%

DESCRIPTIVE STATISTICS


...


PERCENTILES


,PERCENTILE,VALUE
0,P1,53.0
1,P5,139.0
2,P10,260.0
3,P25,741.0
4,P50,2443.0
5,P75,20328.0
6,P90,186140.0
7,P95,525713.0
8,P99,1577385.0



COUNT-VARIABLE VALIDATION
Negative values: 0
Zero values: 0
Positive values: 1852394
Non-integer values: 0
Valid count percentage: 100.000000%

MOST FREQUENT POPULATION VALUES


,POPULATION_VALUE,COUNT,PERCENTAGE
0,606.0,8049,0.434519
1,1595797.0,7312,0.394732
2,1312922.0,7297,0.393923
3,241.0,6578,0.355108
4,1766.0,6556,0.353920
5,2906700.0,5865,0.316617
6,302.0,5853,0.315969
7,198.0,5850,0.315808
8,276002.0,5849,0.315754
9,1126.0,5841,0.315322



DISTRIBUTION SHAPE
Skewness: 5.590804561529
Kurtosis: 37.572846094267

NORMALITY TESTS
Significance level: 0.05

Shapiro-Wilk statistic: 0.318950569802
Shapiro-Wilk p-value: 6.067120353792e-231
Shapiro-Wilk decision: Reject H0
Shapiro-Wilk interpretation: The data provide statistical evidence against a normal distribution.

Jarque-Bera statistic: 118610260.417933195829
Jarque-Bera p-value: 0.000000000000e+00
Jarque-Bera decision: Reject H0
Jarque-Bera interpretation: The data provide statistical evidence against a normal distribution.

IQR OUTLIER ANALYSIS
Q1: 741.000000
Q3: 20328.000000
IQR: 19587.000000
Lower bound: -28639.500000
Upper bound: 49708.500000
Lower statistical outliers: 0
Upper statistical outliers: 346191
Total statistical outliers: 346191
Outlier percentage: 18.688843%

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/discrete_numeral/send_pop_register

HTML:
/projeto_tcc_2026/results/explorat

### <span style="color:teal"> CONTINUOS NUMERICAL </span> ###

### <span style="color:teal"> SEND_AGE </span> ###

In [16]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

FEATURE_GROUP = "comum_features"

FEATURE_TYPE = "continuous_numeral"

FEATURE_NAME = "send_age"

FEATURE_COLUMN = "SEND_AGE"

PLAUSIBLE_MINIMUM_AGE = 0.0

PLAUSIBLE_MAXIMUM_AGE = 100.0

EXPECTED_DECIMAL_PLACES = 2

DECIMAL_TOLERANCE = 1e-4

ALPHA = 0.05


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / FEATURE_GROUP
    / FEATURE_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


HISTOGRAM_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_histogram.png"
)


BOXPLOT_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_boxplot.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


histogram_exists = (
    HISTOGRAM_PATH.exists()
)


boxplot_exists = (
    BOXPLOT_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and histogram_exists
    and boxplot_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )


    print(
        "No analysis or file creation is required."
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nExisting files:"
    )


    print(
        HTML_PATH
    )


    print(
        HISTOGRAM_PATH
    )


    print(
        BOXPLOT_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )


    print(
        "=" * 100
    )


    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )


    print(
        "Histogram:",
        "Already exists"
        if histogram_exists
        else "Will be created"
    )


    print(
        "Boxplot:",
        "Already exists"
        if boxplot_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. BASIC FEATURE OVERVIEW
    # ========================================================

    total_observations = int(
        len(
            feature
        )
    )


    if total_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    missing_values = int(
        feature
        .isna()
        .sum()
    )


    missing_percentage = (
        missing_values
        / total_observations
        * 100
    )


    valid_feature = (
        feature
        .dropna()
        .astype(
            "float64"
        )
    )


    valid_observations = int(
        len(
            valid_feature
        )
    )


    if valid_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no valid observations."
        )


    unique_values = int(
        valid_feature
        .nunique()
    )


    unique_percentage = (
        unique_values
        / valid_observations
        * 100
    )


    # ========================================================
    # 13. DESCRIPTIVE STATISTICS
    # ========================================================

    minimum_value = float(
        valid_feature.min()
    )


    q1_value = float(
        valid_feature.quantile(
            0.25
        )
    )


    median_value = float(
        valid_feature.median()
    )


    mean_value = float(
        valid_feature.mean()
    )


    q3_value = float(
        valid_feature.quantile(
            0.75
        )
    )


    maximum_value = float(
        valid_feature.max()
    )


    standard_deviation = float(
        valid_feature.std()
    )


    variance = float(
        valid_feature.var()
    )


    data_range = float(
        maximum_value
        - minimum_value
    )


    coefficient_of_variation = (
        standard_deviation
        / mean_value
        * 100
        if mean_value != 0
        else np.nan
    )


    descriptive_statistics = pd.DataFrame({

        "STATISTIC": [
            "Minimum",
            "Q1",
            "Median",
            "Mean",
            "Q3",
            "Maximum",
            "Standard deviation",
            "Variance",
            "Range",
            "Coefficient of variation"
        ],

        "VALUE": [
            minimum_value,
            q1_value,
            median_value,
            mean_value,
            q3_value,
            maximum_value,
            standard_deviation,
            variance,
            data_range,
            coefficient_of_variation
        ]
    })


    # ========================================================
    # 14. PERCENTILES
    # ========================================================

    percentile_levels = [
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]


    percentile_values = (
        valid_feature
        .quantile(
            percentile_levels
        )
    )


    percentiles_table = pd.DataFrame({

        "PERCENTILE": [
            "P1",
            "P5",
            "P10",
            "P25",
            "P50",
            "P75",
            "P90",
            "P95",
            "P99"
        ],

        "VALUE": [
            float(
                percentile_values.loc[
                    percentile
                ]
            )
            for percentile in percentile_levels
        ]
    })


    # ========================================================
    # 15. AGE PLAUSIBILITY VALIDATION
    #
    # The project adopts the following plausibility interval:
    #
    # 0 <= age <= 100
    #
    # Values outside this interval are flagged for review.
    # They are not automatically removed.
    # ========================================================

    ages_below_plausible_minimum = int(
        (
            valid_feature
            < PLAUSIBLE_MINIMUM_AGE
        )
        .sum()
    )


    ages_above_plausible_maximum = int(
        (
            valid_feature
            > PLAUSIBLE_MAXIMUM_AGE
        )
        .sum()
    )


    ages_outside_plausible_range = (
        ages_below_plausible_minimum
        + ages_above_plausible_maximum
    )


    ages_outside_plausible_percentage = (
        ages_outside_plausible_range
        / valid_observations
        * 100
    )


    ages_inside_plausible_range = (
        valid_observations
        - ages_outside_plausible_range
    )


    ages_inside_plausible_percentage = (
        ages_inside_plausible_range
        / valid_observations
        * 100
    )


    # ========================================================
    # 16. DECIMAL PRECISION VALIDATION
    #
    # SEND_AGE was generated with two decimal places.
    #
    # Float storage can introduce very small representation
    # differences, so a numerical tolerance is used.
    # ========================================================

    rounded_to_expected_precision = np.round(
        valid_feature.to_numpy(),
        EXPECTED_DECIMAL_PLACES
    )


    decimal_precision_matches = np.isclose(
        valid_feature.to_numpy(),
        rounded_to_expected_precision,
        atol=DECIMAL_TOLERANCE,
        rtol=0
    )


    values_matching_expected_precision = int(
        decimal_precision_matches.sum()
    )


    values_exceeding_expected_precision = int(
        (
            ~decimal_precision_matches
        )
        .sum()
    )


    expected_precision_percentage = (
        values_matching_expected_precision
        / valid_observations
        * 100
    )


    unexpected_precision_percentage = (
        values_exceeding_expected_precision
        / valid_observations
        * 100
    )


    # ========================================================
    # 17. INTEGER AND DECIMAL AGE VALUES
    #
    # This is descriptive only.
    #
    # SEND_AGE remains a continuous numerical feature.
    # ========================================================

    integer_age_mask = np.isclose(
        valid_feature.to_numpy(),
        np.rint(
            valid_feature.to_numpy()
        ),
        atol=DECIMAL_TOLERANCE,
        rtol=0
    )


    integer_age_values = int(
        integer_age_mask.sum()
    )


    decimal_age_values = int(
        (
            ~integer_age_mask
        )
        .sum()
    )


    integer_age_percentage = (
        integer_age_values
        / valid_observations
        * 100
    )


    decimal_age_percentage = (
        decimal_age_values
        / valid_observations
        * 100
    )


    # ========================================================
    # 18. DISTRIBUTION SHAPE
    # ========================================================

    skewness = float(
        valid_feature.skew()
    )


    kurtosis = float(
        valid_feature.kurt()
    )


    distribution_shape_table = pd.DataFrame({

        "METRIC": [
            "Skewness",
            "Kurtosis"
        ],

        "VALUE": [
            skewness,
            kurtosis
        ]
    })


    # ========================================================
    # 19. SHAPIRO-WILK NORMALITY TEST
    #
    # H0:
    # The feature follows a normal distribution.
    #
    # H1:
    # The feature does not follow a normal distribution.
    #
    # The complete valid feature is used.
    # No random sampling is performed.
    # ========================================================

    (
        shapiro_statistic,
        shapiro_p_value
    ) = stats.shapiro(
        valid_feature.to_numpy()
    )


    shapiro_statistic = float(
        shapiro_statistic
    )


    shapiro_p_value = float(
        shapiro_p_value
    )


    # ========================================================
    # 20. SHAPIRO-WILK DECISION
    # ========================================================

    if shapiro_p_value < ALPHA:

        shapiro_decision = (
            "Reject H0"
        )


        shapiro_interpretation = (
            "The data provide statistical evidence "
            "against a normal distribution."
        )


    else:

        shapiro_decision = (
            "Fail to reject H0"
        )


        shapiro_interpretation = (
            "The data do not provide sufficient "
            "statistical evidence against a normal distribution."
        )


    # ========================================================
    # 21. JARQUE-BERA NORMALITY TEST
    #
    # H0:
    # The feature follows a normal distribution.
    #
    # H1:
    # The feature does not follow a normal distribution.
    #
    # The complete valid feature is used.
    # ========================================================

    jarque_bera_result = stats.jarque_bera(
        valid_feature.to_numpy()
    )


    jarque_bera_statistic = float(
        jarque_bera_result.statistic
    )


    jarque_bera_p_value = float(
        jarque_bera_result.pvalue
    )


    # ========================================================
    # 22. JARQUE-BERA DECISION
    # ========================================================

    if jarque_bera_p_value < ALPHA:

        jarque_bera_decision = (
            "Reject H0"
        )


        jarque_bera_interpretation = (
            "The data provide statistical evidence "
            "against a normal distribution."
        )


    else:

        jarque_bera_decision = (
            "Fail to reject H0"
        )


        jarque_bera_interpretation = (
            "The data do not provide sufficient "
            "statistical evidence against a normal distribution."
        )


    # ========================================================
    # 23. NORMALITY TEST RESULTS TABLE
    # ========================================================

    normality_tests_table = pd.DataFrame({

        "TEST": [
            "Shapiro-Wilk",
            "Jarque-Bera"
        ],

        "STATISTIC": [
            shapiro_statistic,
            jarque_bera_statistic
        ],

        "P_VALUE": [
            shapiro_p_value,
            jarque_bera_p_value
        ],

        "ALPHA": [
            ALPHA,
            ALPHA
        ],

        "DECISION": [
            shapiro_decision,
            jarque_bera_decision
        ]
    })


    # ========================================================
    # 24. IQR OUTLIER ANALYSIS
    #
    # IQR outliers are statistical extremes.
    #
    # They are not automatically data errors.
    # ========================================================

    iqr = (
        q3_value
        - q1_value
    )


    lower_iqr_bound = (
        q1_value
        - 1.5
        * iqr
    )


    upper_iqr_bound = (
        q3_value
        + 1.5
        * iqr
    )


    lower_iqr_outliers = int(
        (
            valid_feature
            < lower_iqr_bound
        )
        .sum()
    )


    upper_iqr_outliers = int(
        (
            valid_feature
            > upper_iqr_bound
        )
        .sum()
    )


    total_iqr_outliers = (
        lower_iqr_outliers
        + upper_iqr_outliers
    )


    iqr_outlier_percentage = (
        total_iqr_outliers
        / valid_observations
        * 100
    )


    observations_inside_iqr_limits = (
        valid_observations
        - total_iqr_outliers
    )


    observations_inside_iqr_percentage = (
        observations_inside_iqr_limits
        / valid_observations
        * 100
    )


    # ========================================================
    # 25. CREATE HISTOGRAM
    # ========================================================

    if not histogram_exists:

        fig, ax = plt.subplots(
            figsize=(
                11,
                6
            )
        )


        ax.hist(
            valid_feature,
            bins=60
        )


        ax.set_title(
            "Distribution of sender age"
        )


        ax.set_xlabel(
            "Age in years"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(y)
                )
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            HISTOGRAM_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nHistogram created:"
        )


        print(
            HISTOGRAM_PATH
        )


    else:

        print(
            "\nHistogram already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 26. CREATE BOXPLOT
    # ========================================================

    if not boxplot_exists:

        fig, ax = plt.subplots(
            figsize=(
                11,
                4
            )
        )


        ax.boxplot(
            valid_feature,
            vert=False
        )


        ax.set_title(
            "Boxplot of sender age"
        )


        ax.set_xlabel(
            "Age in years"
        )


        ax.set_yticks(
            []
        )


        ax.grid(
            axis="x",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            BOXPLOT_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nBoxplot created:"
        )


        print(
            BOXPLOT_PATH
        )


    else:

        print(
            "\nBoxplot already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 27. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 28. PREPARE DESCRIPTIVE STATISTICS TABLE FOR HTML
    # ========================================================

    descriptive_statistics_html = (
        descriptive_statistics
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.6f}"
            }
        )
    )


    # ========================================================
    # 29. PREPARE PERCENTILES TABLE FOR HTML
    # ========================================================

    percentiles_html = (
        percentiles_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.6f}"
            }
        )
    )


    # ========================================================
    # 30. PREPARE DISTRIBUTION SHAPE TABLE FOR HTML
    # ========================================================

    distribution_shape_html = (
        distribution_shape_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    # ========================================================
    # 31. PREPARE NORMALITY TEST TABLE FOR HTML
    # ========================================================

    normality_tests_html = (
        normality_tests_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "STATISTIC":
                    lambda x:
                    f"{x:.12f}",

                "P_VALUE":
                    lambda x:
                    f"{x:.12e}",

                "ALPHA":
                    lambda x:
                    f"{x:.2f}"
            }
        )
    )


    # ========================================================
    # 32. CREATE HTML REPORT
    # ========================================================

    if not html_exists:

        histogram_base64 = (
            image_to_base64(
                HISTOGRAM_PATH
            )
        )


        boxplot_base64 = (
            image_to_base64(
                BOXPLOT_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Individual Exploratory Analysis —
{FEATURE_COLUMN}
</h1>


<p>

The variable
<strong>{FEATURE_COLUMN}</strong>
represents the estimated age of the transaction
sender in years.

The feature is represented using decimal years
and is treated as a
<strong>continuous numerical variable</strong>.

</p>


<!-- ========================================================
     1. FEATURE OVERVIEW
========================================================= -->


<h2>
1. Feature overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>Valid observations</td>
<td>{valid_observations}</td>
</tr>

<tr>
<td>Missing values</td>
<td>{missing_values}</td>
</tr>

<tr>
<td>Missing percentage</td>
<td>{missing_percentage:.6f}%</td>
</tr>

<tr>
<td>Unique values</td>
<td>{unique_values}</td>
</tr>

<tr>
<td>Unique-value percentage</td>
<td>{unique_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     2. DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
2. Descriptive statistics
</h2>


{descriptive_statistics_html}


<!-- ========================================================
     3. PERCENTILES
========================================================= -->


<h2>
3. Percentiles
</h2>


<p>

Percentiles provide a detailed description
of the empirical age distribution.

</p>


{percentiles_html}


<!-- ========================================================
     4. AGE PLAUSIBILITY VALIDATION
========================================================= -->


<h2>
4. Age plausibility validation
</h2>


<p>

For this analysis, the following interval is used
as a project-defined plausibility criterion:

</p>


<p class="result">

{PLAUSIBLE_MINIMUM_AGE:.0f}
≤ age ≤
{PLAUSIBLE_MAXIMUM_AGE:.0f}

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Age values below {PLAUSIBLE_MINIMUM_AGE:.0f}</td>
<td>{ages_below_plausible_minimum}</td>
</tr>

<tr>
<td>Age values above {PLAUSIBLE_MAXIMUM_AGE:.0f}</td>
<td>{ages_above_plausible_maximum}</td>
</tr>

<tr>
<td>Total values outside plausibility interval</td>
<td>{ages_outside_plausible_range}</td>
</tr>

<tr>
<td>Percentage outside plausibility interval</td>
<td>{ages_outside_plausible_percentage:.6f}%</td>
</tr>

<tr>
<td>Values inside plausibility interval</td>
<td>{ages_inside_plausible_range}</td>
</tr>

<tr>
<td>Percentage inside plausibility interval</td>
<td>{ages_inside_plausible_percentage:.6f}%</td>
</tr>

</table>


<div class="note">

<strong>Important:</strong>

<br><br>

The upper value of
{PLAUSIBLE_MAXIMUM_AGE:.0f}
years is a project-defined plausibility criterion
used for data-quality inspection.

Values above this threshold are flagged for review
and are not automatically removed from the dataset.

</div>


<!-- ========================================================
     5. DECIMAL PRECISION VALIDATION
========================================================= -->


<h2>
5. Decimal precision validation
</h2>


<p>

The age feature is expected to contain values
rounded to
<strong>{EXPECTED_DECIMAL_PLACES} decimal places</strong>.

A small numerical tolerance is used because
floating-point storage can introduce very small
representation differences.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Values matching expected precision</td>
<td>{values_matching_expected_precision}</td>
</tr>

<tr>
<td>Expected-precision percentage</td>
<td>{expected_precision_percentage:.6f}%</td>
</tr>

<tr>
<td>Values exceeding expected precision</td>
<td>{values_exceeding_expected_precision}</td>
</tr>

<tr>
<td>Unexpected-precision percentage</td>
<td>{unexpected_precision_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     6. INTEGER AND DECIMAL AGE VALUES
========================================================= -->


<h2>
6. Integer and decimal age values
</h2>


<p>

This section describes how many observations
happen to fall exactly on an integer age and
how many contain a decimal component.

This does not change the classification of
<strong>{FEATURE_COLUMN}</strong>
as a continuous numerical feature.

</p>


<table>

<tr>
<th>Age representation</th>
<th>Observations</th>
<th>Percentage</th>
</tr>

<tr>
<td>Integer-valued ages</td>
<td>{integer_age_values}</td>
<td>{integer_age_percentage:.6f}%</td>
</tr>

<tr>
<td>Decimal-valued ages</td>
<td>{decimal_age_values}</td>
<td>{decimal_age_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     7. DISTRIBUTION SHAPE
========================================================= -->


<h2>
7. Distribution shape
</h2>


<p>

Skewness and kurtosis characterize the empirical
shape of the age distribution.

</p>


{distribution_shape_html}


<div class="note">

<strong>Interpretation:</strong>

<br><br>

Skewness measures asymmetry in the observed
distribution.

<br><br>

Kurtosis provides information about the shape
and relative weight of the distribution tails.

</div>


<!-- ========================================================
     8. NORMALITY TESTS
========================================================= -->


<h2>
8. Normality tests
</h2>


<p>

The Shapiro-Wilk and Jarque-Bera tests were
applied to evaluate whether the empirical
distribution of
<strong>{FEATURE_COLUMN}</strong>
is statistically compatible with a normal
distribution.

</p>


<p>

The complete set of valid observations was used.

No random sampling was performed.

</p>


<p class="result">

H0: The feature follows a normal distribution.

<br><br>

H1: The feature does not follow a normal distribution.

<br><br>

Significance level: α = {ALPHA}

</p>


{normality_tests_html}


<h3>
Shapiro-Wilk interpretation
</h3>


<p>

<strong>Decision:</strong>
{shapiro_decision}

</p>


<p>

{shapiro_interpretation}

</p>


<h3>
Jarque-Bera interpretation
</h3>


<p>

<strong>Decision:</strong>
{jarque_bera_decision}

</p>


<p>

{jarque_bera_interpretation}

</p>


<div class="note">

<strong>Important:</strong>

<br><br>

Because the dataset contains a very large number
of observations, statistical normality tests can
detect relatively small departures from a theoretical
normal distribution.

Therefore, the test results should be interpreted
together with skewness, kurtosis, the histogram,
the boxplot, and descriptive statistics.

</div>


<!-- ========================================================
     9. IQR OUTLIER ANALYSIS
========================================================= -->


<h2>
9. IQR outlier analysis
</h2>


<p>

The interquartile range method identifies age
values statistically distant from the central
portion of the observed distribution.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Q1</td>
<td>{q1_value:.6f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{q3_value:.6f}</td>
</tr>

<tr>
<td>IQR</td>
<td>{iqr:.6f}</td>
</tr>

<tr>
<td>Lower IQR bound</td>
<td>{lower_iqr_bound:.6f}</td>
</tr>

<tr>
<td>Upper IQR bound</td>
<td>{upper_iqr_bound:.6f}</td>
</tr>

<tr>
<td>Lower statistical outliers</td>
<td>{lower_iqr_outliers}</td>
</tr>

<tr>
<td>Upper statistical outliers</td>
<td>{upper_iqr_outliers}</td>
</tr>

<tr>
<td>Total statistical outliers</td>
<td>{total_iqr_outliers}</td>
</tr>

<tr>
<td>Statistical outlier percentage</td>
<td>{iqr_outlier_percentage:.6f}%</td>
</tr>

<tr>
<td>Observations inside IQR limits</td>
<td>{observations_inside_iqr_limits}</td>
</tr>

<tr>
<td>Percentage inside IQR limits</td>
<td>{observations_inside_iqr_percentage:.6f}%</td>
</tr>

</table>


<div class="note">

<strong>Important:</strong>

<br><br>

An observation classified as an IQR outlier
is not automatically an invalid age.

The IQR method identifies statistical extremes,
while the plausibility analysis evaluates the
project-defined age interval separately.

</div>


<!-- ========================================================
     10. HISTOGRAM
========================================================= -->


<h2>
10. Age distribution
</h2>


<p>

The histogram presents the empirical distribution
of sender age values.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{histogram_base64}"
    alt="Distribution of sender age"
>

</div>


<!-- ========================================================
     11. BOXPLOT
========================================================= -->


<h2>
11. Age boxplot
</h2>


<p>

The boxplot summarizes the central distribution
of age and highlights observations classified
as statistical extremes according to the IQR rule.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{boxplot_base64}"
    alt="Boxplot of sender age"
>

</div>


<!-- ========================================================
     12. SUMMARY
========================================================= -->


<h2>
12. Summary of results
</h2>


<ul>

<li>
<strong>Total observations:</strong>
{total_observations}
</li>

<li>
<strong>Valid observations:</strong>
{valid_observations}
</li>

<li>
<strong>Missing values:</strong>
{missing_values}
</li>

<li>
<strong>Unique age values:</strong>
{unique_values}
</li>

<li>
<strong>Minimum age:</strong>
{minimum_value:.6f}
</li>

<li>
<strong>Median age:</strong>
{median_value:.6f}
</li>

<li>
<strong>Mean age:</strong>
{mean_value:.6f}
</li>

<li>
<strong>Maximum age:</strong>
{maximum_value:.6f}
</li>

<li>
<strong>Standard deviation:</strong>
{standard_deviation:.6f}
</li>

<li>
<strong>Coefficient of variation:</strong>
{coefficient_of_variation:.6f}%
</li>

<li>
<strong>Values below {PLAUSIBLE_MINIMUM_AGE:.0f} years:</strong>
{ages_below_plausible_minimum}
</li>

<li>
<strong>Values above {PLAUSIBLE_MAXIMUM_AGE:.0f} years:</strong>
{ages_above_plausible_maximum}
</li>

<li>
<strong>Values matching two-decimal precision:</strong>
{expected_precision_percentage:.6f}%
</li>

<li>
<strong>Decimal-valued ages:</strong>
{decimal_age_percentage:.6f}%
</li>

<li>
<strong>Skewness:</strong>
{skewness:.12f}
</li>

<li>
<strong>Kurtosis:</strong>
{kurtosis:.12f}
</li>

<li>
<strong>Shapiro-Wilk statistic:</strong>
{shapiro_statistic:.12f}
</li>

<li>
<strong>Shapiro-Wilk p-value:</strong>
{shapiro_p_value:.12e}
</li>

<li>
<strong>Shapiro-Wilk decision:</strong>
{shapiro_decision}
</li>

<li>
<strong>Jarque-Bera statistic:</strong>
{jarque_bera_statistic:.12f}
</li>

<li>
<strong>Jarque-Bera p-value:</strong>
{jarque_bera_p_value:.12e}
</li>

<li>
<strong>Jarque-Bera decision:</strong>
{jarque_bera_decision}
</li>

<li>
<strong>IQR statistical outliers:</strong>
{total_iqr_outliers}
</li>

<li>
<strong>IQR outlier percentage:</strong>
{iqr_outlier_percentage:.6f}%
</li>

</ul>


</body>

</html>
"""


        # ====================================================
        # 33. SAVE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )


        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 34. DISPLAY FEATURE OVERVIEW
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "SEND_AGE SUMMARY"
    )


    print(
        "=" * 100
    )


    print(
        "Total observations:",
        total_observations
    )


    print(
        "Valid observations:",
        valid_observations
    )


    print(
        "Missing values:",
        missing_values
    )


    print(
        "Missing percentage:",
        f"{missing_percentage:.6f}%"
    )


    print(
        "Unique values:",
        unique_values
    )


    print(
        "Unique-value percentage:",
        f"{unique_percentage:.6f}%"
    )


    # ========================================================
    # 35. DISPLAY DESCRIPTIVE STATISTICS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DESCRIPTIVE STATISTICS"
    )


    print(
        "=" * 100
    )


    display(
        descriptive_statistics
    )


    # ========================================================
    # 36. DISPLAY PERCENTILES
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "PERCENTILES"
    )


    print(
        "=" * 100
    )


    display(
        percentiles_table
    )


    # ========================================================
    # 37. DISPLAY AGE PLAUSIBILITY VALIDATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "AGE PLAUSIBILITY VALIDATION"
    )


    print(
        "=" * 100
    )


    print(
        "Plausibility interval:",
        f"[{PLAUSIBLE_MINIMUM_AGE}, {PLAUSIBLE_MAXIMUM_AGE}]"
    )


    print(
        "Ages below minimum:",
        ages_below_plausible_minimum
    )


    print(
        "Ages above maximum:",
        ages_above_plausible_maximum
    )


    print(
        "Total outside plausibility interval:",
        ages_outside_plausible_range
    )


    print(
        "Percentage inside plausibility interval:",
        f"{ages_inside_plausible_percentage:.6f}%"
    )


    # ========================================================
    # 38. DISPLAY DECIMAL PRECISION VALIDATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DECIMAL PRECISION VALIDATION"
    )


    print(
        "=" * 100
    )


    print(
        "Expected decimal places:",
        EXPECTED_DECIMAL_PLACES
    )


    print(
        "Values matching expected precision:",
        values_matching_expected_precision
    )


    print(
        "Values exceeding expected precision:",
        values_exceeding_expected_precision
    )


    print(
        "Expected-precision percentage:",
        f"{expected_precision_percentage:.6f}%"
    )


    # ========================================================
    # 39. DISPLAY AGE VALUE REPRESENTATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "INTEGER AND DECIMAL AGE VALUES"
    )


    print(
        "=" * 100
    )


    print(
        "Integer-valued ages:",
        integer_age_values,
        f"({integer_age_percentage:.6f}%)"
    )


    print(
        "Decimal-valued ages:",
        decimal_age_values,
        f"({decimal_age_percentage:.6f}%)"
    )


    # ========================================================
    # 40. DISPLAY DISTRIBUTION SHAPE
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DISTRIBUTION SHAPE"
    )


    print(
        "=" * 100
    )


    print(
        "Skewness:",
        f"{skewness:.12f}"
    )


    print(
        "Kurtosis:",
        f"{kurtosis:.12f}"
    )


    # ========================================================
    # 41. DISPLAY NORMALITY TESTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "NORMALITY TESTS"
    )


    print(
        "=" * 100
    )


    print(
        "Significance level:",
        ALPHA
    )


    print(
        "\nShapiro-Wilk statistic:",
        f"{shapiro_statistic:.12f}"
    )


    print(
        "Shapiro-Wilk p-value:",
        f"{shapiro_p_value:.12e}"
    )


    print(
        "Shapiro-Wilk decision:",
        shapiro_decision
    )


    print(
        "Shapiro-Wilk interpretation:",
        shapiro_interpretation
    )


    print(
        "\nJarque-Bera statistic:",
        f"{jarque_bera_statistic:.12f}"
    )


    print(
        "Jarque-Bera p-value:",
        f"{jarque_bera_p_value:.12e}"
    )


    print(
        "Jarque-Bera decision:",
        jarque_bera_decision
    )


    print(
        "Jarque-Bera interpretation:",
        jarque_bera_interpretation
    )


    # ========================================================
    # 42. DISPLAY IQR OUTLIER ANALYSIS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "IQR OUTLIER ANALYSIS"
    )


    print(
        "=" * 100
    )


    print(
        "Q1:",
        f"{q1_value:.6f}"
    )


    print(
        "Q3:",
        f"{q3_value:.6f}"
    )


    print(
        "IQR:",
        f"{iqr:.6f}"
    )


    print(
        "Lower bound:",
        f"{lower_iqr_bound:.6f}"
    )


    print(
        "Upper bound:",
        f"{upper_iqr_bound:.6f}"
    )


    print(
        "Lower statistical outliers:",
        lower_iqr_outliers
    )


    print(
        "Upper statistical outliers:",
        upper_iqr_outliers
    )


    print(
        "Total statistical outliers:",
        total_iqr_outliers
    )


    print(
        "Outlier percentage:",
        f"{iqr_outlier_percentage:.6f}%"
    )


    # ========================================================
    # 43. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature
    del valid_feature
    del percentile_values
    del rounded_to_expected_precision
    del decimal_precision_matches
    del integer_age_mask

    gc.collect()


    # ========================================================
    # 44. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "ANALYSIS COMPLETED"
    )


    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )


    print(
        HTML_PATH
    )


    print(
        "\nPNGs:"
    )


    print(
        HISTOGRAM_PATH
    )


    print(
        BOXPLOT_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Histogram: Will be created
Boxplot: Will be created


/usr/local/lib/python3.14/site-packages/scipy/stats/_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 1852394.
  res = hypotest_fun_out(*samples, **kwds)



Histogram created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_numeral/send_age/send_age_histogram.png


/tmp/ipykernel_2304/3289857506.py:949: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(



Boxplot created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_numeral/send_age/send_age_boxplot.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_numeral/send_age/analysis_send_age.html

SEND_AGE SUMMARY
Total observations: 1852394
Valid observations: 1852394
Missing values: 0
Missing percentage: 0.000000%
Unique values: 927
Unique-value percentage: 0.050043%

DESCRIPTIVE STATISTICS


,STATISTIC,VALUE
0,Minimum,21.590000
1,Q1,39.369999
2,Median,50.759998
3,Mean,52.884141
4,Q3,64.059998
5,Maximum,101.839996
6,Standard deviation,17.402901
7,Variance,302.860967
8,Range,80.249996
9,Coefficient of variation,32.907599



PERCENTILES


,PERCENTILE,VALUE
0,P1,23.330000
1,P5,28.690001
2,P10,32.369999
3,P25,39.369999
4,P50,50.759998
5,P75,64.059998
6,P90,77.459999
7,P95,86.820000
8,P99,98.190002



AGE PLAUSIBILITY VALIDATION
Plausibility interval: [0.0, 100.0]
Ages below minimum: 0
Ages above maximum: 8791
Total outside plausibility interval: 8791
Percentage inside plausibility interval: 99.525425%

DECIMAL PRECISION VALIDATION
Expected decimal places: 2
Values matching expected precision: 1852394
Values exceeding expected precision: 0
Expected-precision percentage: 100.000000%

INTEGER AND DECIMAL AGE VALUES
Integer-valued ages: 19697 (1.063327%)
Decimal-valued ages: 1832697 (98.936673%)

DISTRIBUTION SHAPE
Skewness: 0.610409093551
Kurtosis: -0.178327604258

NORMALITY TESTS
Significance level: 0.05

Shapiro-Wilk statistic: 0.964535263114
Shapiro-Wilk p-value: 4.714214495008e-128
Shapiro-Wilk decision: Reject H0
Shapiro-Wilk interpretation: The data provide statistical evidence against a normal distribution.

Jarque-Bera statistic: 117487.807748903913
Jarque-Bera p-value: 0.000000000000e+00
Jarque-Bera decision: Reject H0
Jarque-Bera interpretation: The data provide statistical

### <span style="color:teal"> TRANS_VALUE </span> ###

In [17]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

FEATURE_GROUP = "comum_features"

FEATURE_TYPE = "continuous_numeral"

FEATURE_NAME = "trans_value"

FEATURE_COLUMN = "TRANS_VALUE"

VALID_MINIMUM = 0.0

ALPHA = 0.05


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_individual_variables"
    / FEATURE_GROUP
    / FEATURE_TYPE
    / FEATURE_NAME
)


# ============================================================
# 05. CREATE OR USE THE RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 06. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / f"analysis_{FEATURE_NAME}.html"
)


HISTOGRAM_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_histogram.png"
)


LOG_HISTOGRAM_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_log1p_histogram.png"
)


BOXPLOT_PATH = (
    RESULTS_DIRECTORY
    / f"{FEATURE_NAME}_boxplot.png"
)


# ============================================================
# 07. CHECK WHICH OUTPUT FILES ALREADY EXIST
# ============================================================

html_exists = (
    HTML_PATH.exists()
)


histogram_exists = (
    HISTOGRAM_PATH.exists()
)


log_histogram_exists = (
    LOG_HISTOGRAM_PATH.exists()
)


boxplot_exists = (
    BOXPLOT_PATH.exists()
)


all_output_files_exist = (
    html_exists
    and histogram_exists
    and log_histogram_exists
    and boxplot_exists
)


# ============================================================
# 08. STOP IF ALL OUTPUT FILES ALREADY EXIST
# ============================================================

if all_output_files_exist:

    print(
        "All analysis files already exist."
    )


    print(
        "No analysis or file creation is required."
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nExisting files:"
    )


    print(
        HTML_PATH
    )


    print(
        HISTOGRAM_PATH
    )


    print(
        LOG_HISTOGRAM_PATH
    )


    print(
        BOXPLOT_PATH
    )


else:

    # ========================================================
    # 09. CHECK THE DATASET
    # ========================================================

    if not DATASET_PATH.exists():

        raise FileNotFoundError(
            f"Dataset not found:\n"
            f"{DATASET_PATH}"
        )


    # ========================================================
    # 10. DISPLAY OUTPUT FILE STATUS
    # ========================================================

    print(
        "\nOUTPUT FILE STATUS"
    )


    print(
        "=" * 100
    )


    print(
        "HTML:",
        "Already exists"
        if html_exists
        else "Will be created"
    )


    print(
        "Histogram:",
        "Already exists"
        if histogram_exists
        else "Will be created"
    )


    print(
        "Log1p histogram:",
        "Already exists"
        if log_histogram_exists
        else "Will be created"
    )


    print(
        "Boxplot:",
        "Already exists"
        if boxplot_exists
        else "Will be created"
    )


    # ========================================================
    # 11. LOAD ONLY THE FEATURE BEING ANALYZED
    # ========================================================

    dataset_feature = pd.read_parquet(
        DATASET_PATH,
        columns=[
            FEATURE_COLUMN
        ]
    )


    feature = (
        dataset_feature[
            FEATURE_COLUMN
        ]
    )


    # ========================================================
    # 12. BASIC FEATURE OVERVIEW
    # ========================================================

    total_observations = int(
        len(
            feature
        )
    )


    if total_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no observations."
        )


    missing_values = int(
        feature
        .isna()
        .sum()
    )


    missing_percentage = (
        missing_values
        / total_observations
        * 100
    )


    valid_feature = (
        feature
        .dropna()
        .astype(
            "float64"
        )
    )


    valid_observations = int(
        len(
            valid_feature
        )
    )


    if valid_observations == 0:

        raise ValueError(
            f"{FEATURE_COLUMN} contains no valid observations."
        )


    unique_values = int(
        valid_feature
        .nunique()
    )


    unique_percentage = (
        unique_values
        / valid_observations
        * 100
    )


    # ========================================================
    # 13. DESCRIPTIVE STATISTICS
    # ========================================================

    minimum_value = float(
        valid_feature.min()
    )


    q1_value = float(
        valid_feature.quantile(
            0.25
        )
    )


    median_value = float(
        valid_feature.median()
    )


    mean_value = float(
        valid_feature.mean()
    )


    q3_value = float(
        valid_feature.quantile(
            0.75
        )
    )


    maximum_value = float(
        valid_feature.max()
    )


    standard_deviation = float(
        valid_feature.std()
    )


    variance = float(
        valid_feature.var()
    )


    data_range = float(
        maximum_value
        - minimum_value
    )


    coefficient_of_variation = (
        standard_deviation
        / mean_value
        * 100
        if mean_value != 0
        else np.nan
    )


    descriptive_statistics = pd.DataFrame({

        "STATISTIC": [
            "Minimum",
            "Q1",
            "Median",
            "Mean",
            "Q3",
            "Maximum",
            "Standard deviation",
            "Variance",
            "Range",
            "Coefficient of variation"
        ],

        "VALUE": [
            minimum_value,
            q1_value,
            median_value,
            mean_value,
            q3_value,
            maximum_value,
            standard_deviation,
            variance,
            data_range,
            coefficient_of_variation
        ]
    })


    # ========================================================
    # 14. PERCENTILES
    # ========================================================

    percentile_levels = [
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]


    percentile_values = (
        valid_feature
        .quantile(
            percentile_levels
        )
    )


    percentiles_table = pd.DataFrame({

        "PERCENTILE": [
            "P1",
            "P5",
            "P10",
            "P25",
            "P50",
            "P75",
            "P90",
            "P95",
            "P99"
        ],

        "VALUE": [
            float(
                percentile_values.loc[
                    percentile
                ]
            )
            for percentile in percentile_levels
        ]
    })


    # ========================================================
    # 15. TRANSACTION VALUE VALIDATION
    #
    # Transaction monetary values are expected to be
    # non-negative.
    #
    # Negative values are flagged for review.
    # ========================================================

    negative_values = int(
        (
            valid_feature
            < VALID_MINIMUM
        )
        .sum()
    )


    zero_values = int(
        (
            valid_feature
            == 0
        )
        .sum()
    )


    positive_values = int(
        (
            valid_feature
            > 0
        )
        .sum()
    )


    negative_percentage = (
        negative_values
        / valid_observations
        * 100
    )


    zero_percentage = (
        zero_values
        / valid_observations
        * 100
    )


    positive_percentage = (
        positive_values
        / valid_observations
        * 100
    )


    non_negative_values = (
        valid_observations
        - negative_values
    )


    non_negative_percentage = (
        non_negative_values
        / valid_observations
        * 100
    )


    transaction_validation_table = pd.DataFrame({

        "CATEGORY": [
            "Negative values",
            "Zero values",
            "Positive values",
            "Non-negative values"
        ],

        "COUNT": [
            negative_values,
            zero_values,
            positive_values,
            non_negative_values
        ],

        "PERCENTAGE": [
            negative_percentage,
            zero_percentage,
            positive_percentage,
            non_negative_percentage
        ]
    })


    # ========================================================
    # 16. DISTRIBUTION SHAPE
    # ========================================================

    skewness = float(
        valid_feature.skew()
    )


    kurtosis = float(
        valid_feature.kurt()
    )


    distribution_shape_table = pd.DataFrame({

        "METRIC": [
            "Skewness",
            "Kurtosis"
        ],

        "VALUE": [
            skewness,
            kurtosis
        ]
    })


    # ========================================================
    # 17. SHAPIRO-WILK NORMALITY TEST
    #
    # H0:
    # The feature follows a normal distribution.
    #
    # H1:
    # The feature does not follow a normal distribution.
    #
    # The complete valid feature is used.
    # No random sampling is performed.
    # ========================================================

    (
        shapiro_statistic,
        shapiro_p_value
    ) = stats.shapiro(
        valid_feature.to_numpy()
    )


    shapiro_statistic = float(
        shapiro_statistic
    )


    shapiro_p_value = float(
        shapiro_p_value
    )


    # ========================================================
    # 18. SHAPIRO-WILK DECISION
    # ========================================================

    if shapiro_p_value < ALPHA:

        shapiro_decision = (
            "Reject H0"
        )


        shapiro_interpretation = (
            "The data provide statistical evidence "
            "against a normal distribution."
        )


    else:

        shapiro_decision = (
            "Fail to reject H0"
        )


        shapiro_interpretation = (
            "The data do not provide sufficient "
            "statistical evidence against a normal distribution."
        )


    # ========================================================
    # 19. JARQUE-BERA NORMALITY TEST
    #
    # H0:
    # The feature follows a normal distribution.
    #
    # H1:
    # The feature does not follow a normal distribution.
    #
    # The complete valid feature is used.
    # ========================================================

    jarque_bera_result = stats.jarque_bera(
        valid_feature.to_numpy()
    )


    jarque_bera_statistic = float(
        jarque_bera_result.statistic
    )


    jarque_bera_p_value = float(
        jarque_bera_result.pvalue
    )


    # ========================================================
    # 20. JARQUE-BERA DECISION
    # ========================================================

    if jarque_bera_p_value < ALPHA:

        jarque_bera_decision = (
            "Reject H0"
        )


        jarque_bera_interpretation = (
            "The data provide statistical evidence "
            "against a normal distribution."
        )


    else:

        jarque_bera_decision = (
            "Fail to reject H0"
        )


        jarque_bera_interpretation = (
            "The data do not provide sufficient "
            "statistical evidence against a normal distribution."
        )


    # ========================================================
    # 21. NORMALITY TEST RESULTS TABLE
    # ========================================================

    normality_tests_table = pd.DataFrame({

        "TEST": [
            "Shapiro-Wilk",
            "Jarque-Bera"
        ],

        "STATISTIC": [
            shapiro_statistic,
            jarque_bera_statistic
        ],

        "P_VALUE": [
            shapiro_p_value,
            jarque_bera_p_value
        ],

        "ALPHA": [
            ALPHA,
            ALPHA
        ],

        "DECISION": [
            shapiro_decision,
            jarque_bera_decision
        ]
    })


    # ========================================================
    # 22. IQR OUTLIER ANALYSIS
    #
    # IQR outliers are statistical extremes.
    #
    # A high transaction value is not automatically an
    # invalid observation.
    # ========================================================

    iqr = (
        q3_value
        - q1_value
    )


    lower_iqr_bound = (
        q1_value
        - 1.5
        * iqr
    )


    upper_iqr_bound = (
        q3_value
        + 1.5
        * iqr
    )


    lower_iqr_outliers = int(
        (
            valid_feature
            < lower_iqr_bound
        )
        .sum()
    )


    upper_iqr_outliers = int(
        (
            valid_feature
            > upper_iqr_bound
        )
        .sum()
    )


    total_iqr_outliers = (
        lower_iqr_outliers
        + upper_iqr_outliers
    )


    iqr_outlier_percentage = (
        total_iqr_outliers
        / valid_observations
        * 100
    )


    observations_inside_iqr_limits = (
        valid_observations
        - total_iqr_outliers
    )


    observations_inside_iqr_percentage = (
        observations_inside_iqr_limits
        / valid_observations
        * 100
    )


    # ========================================================
    # 23. CREATE HISTOGRAM
    # ========================================================

    if not histogram_exists:

        fig, ax = plt.subplots(
            figsize=(
                11,
                6
            )
        )


        ax.hist(
            valid_feature,
            bins=60
        )


        ax.set_title(
            "Distribution of transaction value"
        )


        ax.set_xlabel(
            "Transaction value"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(y)
                )
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            HISTOGRAM_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nHistogram created:"
        )


        print(
            HISTOGRAM_PATH
        )


    else:

        print(
            "\nHistogram already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 24. CREATE LOG1P HISTOGRAM
    #
    # The logarithmic view is created only from
    # non-negative values because log1p(x) requires
    # x > -1.
    #
    # This does not modify TRANS_VALUE.
    # ========================================================

    if not log_histogram_exists:

        non_negative_feature = (
            valid_feature[
                valid_feature
                >= 0
            ]
        )


        if len(
            non_negative_feature
        ) == 0:

            raise ValueError(
                "No non-negative transaction values are available "
                "for the logarithmic histogram."
            )


        log_feature = np.log1p(
            non_negative_feature
        )


        fig, ax = plt.subplots(
            figsize=(
                11,
                6
            )
        )


        ax.hist(
            log_feature,
            bins=60
        )


        ax.set_title(
            "Log1p distribution of transaction value"
        )


        ax.set_xlabel(
            "log1p(Transaction value)"
        )


        ax.set_ylabel(
            "Number of observations"
        )


        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda y, pos:
                str(
                    int(y)
                )
            )
        )


        ax.grid(
            axis="y",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            LOG_HISTOGRAM_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        del non_negative_feature
        del log_feature


        print(
            "\nLog1p histogram created:"
        )


        print(
            LOG_HISTOGRAM_PATH
        )


    else:

        print(
            "\nLog1p histogram already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 25. CREATE BOXPLOT
    # ========================================================

    if not boxplot_exists:

        fig, ax = plt.subplots(
            figsize=(
                11,
                4
            )
        )


        ax.boxplot(
            valid_feature,
            vert=False
        )


        ax.set_title(
            "Boxplot of transaction value"
        )


        ax.set_xlabel(
            "Transaction value"
        )


        ax.set_yticks(
            []
        )


        ax.grid(
            axis="x",
            alpha=0.3
        )


        fig.tight_layout()


        fig.savefig(
            BOXPLOT_PATH,
            format="png",
            dpi=600,
            bbox_inches="tight"
        )


        plt.close(
            fig
        )


        print(
            "\nBoxplot created:"
        )


        print(
            BOXPLOT_PATH
        )


    else:

        print(
            "\nBoxplot already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 26. FUNCTION TO CONVERT PNG TO BASE64
    # ========================================================

    def image_to_base64(
        image_path
    ):

        with open(
            image_path,
            "rb"
        ) as image_file:

            return (
                base64.b64encode(
                    image_file.read()
                )
                .decode(
                    "utf-8"
                )
            )


    # ========================================================
    # 27. PREPARE DESCRIPTIVE STATISTICS TABLE FOR HTML
    # ========================================================

    descriptive_statistics_html = (
        descriptive_statistics
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.6f}"
            }
        )
    )


    # ========================================================
    # 28. PREPARE PERCENTILES TABLE FOR HTML
    # ========================================================

    percentiles_html = (
        percentiles_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.6f}"
            }
        )
    )


    # ========================================================
    # 29. PREPARE TRANSACTION VALIDATION TABLE FOR HTML
    # ========================================================

    transaction_validation_html = (
        transaction_validation_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "PERCENTAGE":
                    lambda x:
                    f"{x:.6f}%"
            }
        )
    )


    # ========================================================
    # 30. PREPARE DISTRIBUTION SHAPE TABLE FOR HTML
    # ========================================================

    distribution_shape_html = (
        distribution_shape_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "VALUE":
                    lambda x:
                    f"{x:.12f}"
            }
        )
    )


    # ========================================================
    # 31. PREPARE NORMALITY TEST TABLE FOR HTML
    # ========================================================

    normality_tests_html = (
        normality_tests_table
        .to_html(
            index=False,
            border=0,
            formatters={
                "STATISTIC":
                    lambda x:
                    f"{x:.12f}",

                "P_VALUE":
                    lambda x:
                    f"{x:.12e}",

                "ALPHA":
                    lambda x:
                    f"{x:.2f}"
            }
        )
    )


    # ========================================================
    # 32. CREATE HTML REPORT
    # ========================================================

    if not html_exists:

        histogram_base64 = (
            image_to_base64(
                HISTOGRAM_PATH
            )
        )


        log_histogram_base64 = (
            image_to_base64(
                LOG_HISTOGRAM_PATH
            )
        )


        boxplot_base64 = (
            image_to_base64(
                BOXPLOT_PATH
            )
        )


        html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Individual Exploratory Analysis - {FEATURE_COLUMN}
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Individual Exploratory Analysis —
{FEATURE_COLUMN}
</h1>


<p>

The variable
<strong>{FEATURE_COLUMN}</strong>
represents the monetary value associated with
each transaction.

For exploratory and modeling purposes, it is
treated as a
<strong>continuous numerical variable</strong>.

</p>


<!-- ========================================================
     1. FEATURE OVERVIEW
========================================================= -->


<h2>
1. Feature overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>Valid observations</td>
<td>{valid_observations}</td>
</tr>

<tr>
<td>Missing values</td>
<td>{missing_values}</td>
</tr>

<tr>
<td>Missing percentage</td>
<td>{missing_percentage:.6f}%</td>
</tr>

<tr>
<td>Unique values</td>
<td>{unique_values}</td>
</tr>

<tr>
<td>Unique-value percentage</td>
<td>{unique_percentage:.6f}%</td>
</tr>

</table>


<!-- ========================================================
     2. DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
2. Descriptive statistics
</h2>


{descriptive_statistics_html}


<!-- ========================================================
     3. PERCENTILES
========================================================= -->


<h2>
3. Percentiles
</h2>


<p>

Percentiles describe the empirical distribution
of transaction values and help identify the
concentration of observations across different
parts of the distribution.

</p>


{percentiles_html}


<!-- ========================================================
     4. TRANSACTION VALUE VALIDATION
========================================================= -->


<h2>
4. Transaction value validation
</h2>


<p>

Transaction values are expected to be
non-negative.

Negative values are therefore flagged for
data-quality review.

Zero values are reported separately and are
not automatically considered invalid.

</p>


{transaction_validation_html}


<div class="note">

<strong>Important:</strong>

<br><br>

No theoretical upper limit is imposed on
transaction value.

Large transaction values are therefore evaluated
statistically rather than being automatically
classified as invalid.

</div>


<!-- ========================================================
     5. DISTRIBUTION SHAPE
========================================================= -->


<h2>
5. Distribution shape
</h2>


<p>

Skewness and kurtosis characterize the empirical
shape of the transaction-value distribution.

</p>


{distribution_shape_html}


<div class="note">

<strong>Interpretation:</strong>

<br><br>

Positive skewness indicates a longer right tail,
which is common in monetary transaction data.

<br><br>

Kurtosis provides information about the shape
and relative weight of the distribution tails.

</div>


<!-- ========================================================
     6. NORMALITY TESTS
========================================================= -->


<h2>
6. Normality tests
</h2>


<p>

The Shapiro-Wilk and Jarque-Bera tests were
applied to evaluate whether the empirical
distribution of
<strong>{FEATURE_COLUMN}</strong>
is statistically compatible with a normal
distribution.

</p>


<p>

The complete set of valid observations was used.

No random sampling was performed.

</p>


<p class="result">

H0: The feature follows a normal distribution.

<br><br>

H1: The feature does not follow a normal distribution.

<br><br>

Significance level: α = {ALPHA}

</p>


{normality_tests_html}


<h3>
Shapiro-Wilk interpretation
</h3>


<p>

<strong>Decision:</strong>
{shapiro_decision}

</p>


<p>

{shapiro_interpretation}

</p>


<h3>
Jarque-Bera interpretation
</h3>


<p>

<strong>Decision:</strong>
{jarque_bera_decision}

</p>


<p>

{jarque_bera_interpretation}

</p>


<div class="note">

<strong>Important:</strong>

<br><br>

Because the dataset contains a very large number
of observations, statistical normality tests can
detect relatively small deviations from a
theoretical normal distribution.

Therefore, the tests should be interpreted
together with skewness, kurtosis, percentiles,
the histogram, and the boxplot.

</div>


<!-- ========================================================
     7. IQR OUTLIER ANALYSIS
========================================================= -->


<h2>
7. IQR outlier analysis
</h2>


<p>

The interquartile range method identifies
transaction values statistically distant from
the central portion of the observed distribution.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Q1</td>
<td>{q1_value:.6f}</td>
</tr>

<tr>
<td>Q3</td>
<td>{q3_value:.6f}</td>
</tr>

<tr>
<td>IQR</td>
<td>{iqr:.6f}</td>
</tr>

<tr>
<td>Lower IQR bound</td>
<td>{lower_iqr_bound:.6f}</td>
</tr>

<tr>
<td>Upper IQR bound</td>
<td>{upper_iqr_bound:.6f}</td>
</tr>

<tr>
<td>Lower statistical outliers</td>
<td>{lower_iqr_outliers}</td>
</tr>

<tr>
<td>Upper statistical outliers</td>
<td>{upper_iqr_outliers}</td>
</tr>

<tr>
<td>Total statistical outliers</td>
<td>{total_iqr_outliers}</td>
</tr>

<tr>
<td>Statistical outlier percentage</td>
<td>{iqr_outlier_percentage:.6f}%</td>
</tr>

<tr>
<td>Observations inside IQR limits</td>
<td>{observations_inside_iqr_limits}</td>
</tr>

<tr>
<td>Percentage inside IQR limits</td>
<td>{observations_inside_iqr_percentage:.6f}%</td>
</tr>

</table>


<div class="note">

<strong>Important:</strong>

<br><br>

A transaction value classified as an IQR outlier
is not automatically an invalid transaction.

The IQR method identifies statistical extremes.

In fraud analysis, unusually high transaction
values may themselves contain relevant information
and should not be removed solely because they are
statistical outliers.

</div>


<!-- ========================================================
     8. HISTOGRAM
========================================================= -->


<h2>
8. Transaction-value distribution
</h2>


<p>

The histogram presents the empirical distribution
of transaction values on the original numerical
scale.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{histogram_base64}"
    alt="Distribution of transaction value"
>

</div>


<!-- ========================================================
     9. LOG1P HISTOGRAM
========================================================= -->


<h2>
9. Logarithmic visualization
</h2>


<p>

The following visualization applies the
transformation:

</p>


<p class="result">

log1p(TRANS_VALUE) = ln(1 + TRANS_VALUE)

</p>


<p>

This transformation is used only for visualization.

The original
<strong>{FEATURE_COLUMN}</strong>
values are not modified.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{log_histogram_base64}"
    alt="Log1p distribution of transaction value"
>

</div>


<div class="note">

<strong>Why use log1p?</strong>

<br><br>

Monetary values often present strong right
skewness, with many low or moderate values and
a smaller number of very high transactions.

The logarithmic representation compresses the
right tail and can reveal distributional
structure that is difficult to observe on the
original scale.

</div>


<!-- ========================================================
     10. BOXPLOT
========================================================= -->


<h2>
10. Transaction-value boxplot
</h2>


<p>

The boxplot summarizes the central distribution
of transaction values and highlights observations
classified as statistical extremes according
to the IQR rule.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{boxplot_base64}"
    alt="Boxplot of transaction value"
>

</div>


<!-- ========================================================
     11. SUMMARY
========================================================= -->


<h2>
11. Summary of results
</h2>


<ul>

<li>
<strong>Total observations:</strong>
{total_observations}
</li>

<li>
<strong>Valid observations:</strong>
{valid_observations}
</li>

<li>
<strong>Missing values:</strong>
{missing_values}
</li>

<li>
<strong>Unique transaction values:</strong>
{unique_values}
</li>

<li>
<strong>Minimum transaction value:</strong>
{minimum_value:.6f}
</li>

<li>
<strong>Median transaction value:</strong>
{median_value:.6f}
</li>

<li>
<strong>Mean transaction value:</strong>
{mean_value:.6f}
</li>

<li>
<strong>Maximum transaction value:</strong>
{maximum_value:.6f}
</li>

<li>
<strong>Standard deviation:</strong>
{standard_deviation:.6f}
</li>

<li>
<strong>Coefficient of variation:</strong>
{coefficient_of_variation:.6f}%
</li>

<li>
<strong>Negative values:</strong>
{negative_values}
</li>

<li>
<strong>Zero values:</strong>
{zero_values}
</li>

<li>
<strong>Positive values:</strong>
{positive_values}
</li>

<li>
<strong>Skewness:</strong>
{skewness:.12f}
</li>

<li>
<strong>Kurtosis:</strong>
{kurtosis:.12f}
</li>

<li>
<strong>Shapiro-Wilk statistic:</strong>
{shapiro_statistic:.12f}
</li>

<li>
<strong>Shapiro-Wilk p-value:</strong>
{shapiro_p_value:.12e}
</li>

<li>
<strong>Shapiro-Wilk decision:</strong>
{shapiro_decision}
</li>

<li>
<strong>Jarque-Bera statistic:</strong>
{jarque_bera_statistic:.12f}
</li>

<li>
<strong>Jarque-Bera p-value:</strong>
{jarque_bera_p_value:.12e}
</li>

<li>
<strong>Jarque-Bera decision:</strong>
{jarque_bera_decision}
</li>

<li>
<strong>IQR statistical outliers:</strong>
{total_iqr_outliers}
</li>

<li>
<strong>IQR outlier percentage:</strong>
{iqr_outlier_percentage:.6f}%
</li>

</ul>


</body>

</html>
"""


        # ====================================================
        # 33. SAVE HTML REPORT
        # ====================================================

        HTML_PATH.write_text(
            html_content,
            encoding="utf-8"
        )


        print(
            "\nHTML report created:"
        )


        print(
            HTML_PATH
        )


    else:

        print(
            "\nHTML report already exists."
        )


        print(
            "The existing file was not overwritten."
        )


    # ========================================================
    # 34. DISPLAY FEATURE OVERVIEW
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "TRANS_VALUE SUMMARY"
    )


    print(
        "=" * 100
    )


    print(
        "Total observations:",
        total_observations
    )


    print(
        "Valid observations:",
        valid_observations
    )


    print(
        "Missing values:",
        missing_values
    )


    print(
        "Missing percentage:",
        f"{missing_percentage:.6f}%"
    )


    print(
        "Unique values:",
        unique_values
    )


    print(
        "Unique-value percentage:",
        f"{unique_percentage:.6f}%"
    )


    # ========================================================
    # 35. DISPLAY DESCRIPTIVE STATISTICS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DESCRIPTIVE STATISTICS"
    )


    print(
        "=" * 100
    )


    display(
        descriptive_statistics
    )


    # ========================================================
    # 36. DISPLAY PERCENTILES
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "PERCENTILES"
    )


    print(
        "=" * 100
    )


    display(
        percentiles_table
    )


    # ========================================================
    # 37. DISPLAY TRANSACTION VALUE VALIDATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "TRANSACTION VALUE VALIDATION"
    )


    print(
        "=" * 100
    )


    print(
        "Negative values:",
        negative_values,
        f"({negative_percentage:.6f}%)"
    )


    print(
        "Zero values:",
        zero_values,
        f"({zero_percentage:.6f}%)"
    )


    print(
        "Positive values:",
        positive_values,
        f"({positive_percentage:.6f}%)"
    )


    print(
        "Non-negative values:",
        non_negative_values,
        f"({non_negative_percentage:.6f}%)"
    )


    # ========================================================
    # 38. DISPLAY DISTRIBUTION SHAPE
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "DISTRIBUTION SHAPE"
    )


    print(
        "=" * 100
    )


    print(
        "Skewness:",
        f"{skewness:.12f}"
    )


    print(
        "Kurtosis:",
        f"{kurtosis:.12f}"
    )


    # ========================================================
    # 39. DISPLAY NORMALITY TESTS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "NORMALITY TESTS"
    )


    print(
        "=" * 100
    )


    print(
        "Significance level:",
        ALPHA
    )


    print(
        "\nShapiro-Wilk statistic:",
        f"{shapiro_statistic:.12f}"
    )


    print(
        "Shapiro-Wilk p-value:",
        f"{shapiro_p_value:.12e}"
    )


    print(
        "Shapiro-Wilk decision:",
        shapiro_decision
    )


    print(
        "Shapiro-Wilk interpretation:",
        shapiro_interpretation
    )


    print(
        "\nJarque-Bera statistic:",
        f"{jarque_bera_statistic:.12f}"
    )


    print(
        "Jarque-Bera p-value:",
        f"{jarque_bera_p_value:.12e}"
    )


    print(
        "Jarque-Bera decision:",
        jarque_bera_decision
    )


    print(
        "Jarque-Bera interpretation:",
        jarque_bera_interpretation
    )


    # ========================================================
    # 40. DISPLAY IQR OUTLIER ANALYSIS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "IQR OUTLIER ANALYSIS"
    )


    print(
        "=" * 100
    )


    print(
        "Q1:",
        f"{q1_value:.6f}"
    )


    print(
        "Q3:",
        f"{q3_value:.6f}"
    )


    print(
        "IQR:",
        f"{iqr:.6f}"
    )


    print(
        "Lower bound:",
        f"{lower_iqr_bound:.6f}"
    )


    print(
        "Upper bound:",
        f"{upper_iqr_bound:.6f}"
    )


    print(
        "Lower statistical outliers:",
        lower_iqr_outliers
    )


    print(
        "Upper statistical outliers:",
        upper_iqr_outliers
    )


    print(
        "Total statistical outliers:",
        total_iqr_outliers
    )


    print(
        "Outlier percentage:",
        f"{iqr_outlier_percentage:.6f}%"
    )


    # ========================================================
    # 41. RELEASE MEMORY
    # ========================================================

    del dataset_feature
    del feature
    del valid_feature
    del percentile_values

    gc.collect()


    # ========================================================
    # 42. FINAL CONFIRMATION
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )


    print(
        "ANALYSIS COMPLETED"
    )


    print(
        "=" * 100
    )


    print(
        "\nResults directory:"
    )


    print(
        RESULTS_DIRECTORY
    )


    print(
        "\nHTML:"
    )


    print(
        HTML_PATH
    )


    print(
        "\nPNGs:"
    )


    print(
        HISTOGRAM_PATH
    )


    print(
        LOG_HISTOGRAM_PATH
    )


    print(
        BOXPLOT_PATH
    )


OUTPUT FILE STATUS
HTML: Will be created
Histogram: Will be created
Log1p histogram: Will be created
Boxplot: Will be created


/usr/local/lib/python3.14/site-packages/scipy/stats/_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 1852394.
  res = hypotest_fun_out(*samples, **kwds)



Histogram created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_numeral/trans_value/trans_value_histogram.png

Log1p histogram created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_numeral/trans_value/trans_value_log1p_histogram.png


/tmp/ipykernel_2304/2355129642.py:1036: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(



Boxplot created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_numeral/trans_value/trans_value_boxplot.png

HTML report created:
/projeto_tcc_2026/results/exploratory_analysis_of_individual_variables/comum_features/continuous_numeral/trans_value/analysis_trans_value.html

TRANS_VALUE SUMMARY
Total observations: 1852394
Valid observations: 1852394
Missing values: 0
Missing percentage: 0.000000%
Unique values: 60616
Unique-value percentage: 3.272306%

DESCRIPTIVE STATISTICS


,STATISTIC,VALUE
0,Minimum,1.000000
1,Q1,9.640000
2,Median,47.450000
3,Mean,70.063567
4,Q3,83.100000
5,Maximum,28948.900000
6,Standard deviation,159.253975
7,Variance,25361.828481
8,Range,28947.900000
9,Coefficient of variation,227.299266



PERCENTILES


,PERCENTILE,VALUE
0,P1,1.26
1,P5,2.44
2,P10,4.10
3,P25,9.64
4,P50,47.45
5,P75,83.10
6,P90,136.33
7,P95,195.34
8,P99,537.90



TRANSACTION VALUE VALIDATION
Negative values: 0 (0.000000%)
Zero values: 0 (0.000000%)
Positive values: 1852394 (100.000000%)
Non-negative values: 1852394 (100.000000%)

DISTRIBUTION SHAPE
Skewness: 40.812809176296
Kurtosis: 4181.907343887258

NORMALITY TESTS
Significance level: 0.05

Shapiro-Wilk statistic: 0.270064818997
Shapiro-Wilk p-value: 1.008480089547e-233
Shapiro-Wilk decision: Reject H0
Shapiro-Wilk interpretation: The data provide statistical evidence against a normal distribution.

Jarque-Bera statistic: 1350311661982.904541015625
Jarque-Bera p-value: 0.000000000000e+00
Jarque-Bera decision: Reject H0
Jarque-Bera interpretation: The data provide statistical evidence against a normal distribution.

IQR OUTLIER ANALYSIS
Q1: 9.640000
Q3: 83.100000
IQR: 73.460000
Lower bound: -100.550000
Upper bound: 193.290000
Lower statistical outliers: 0
Upper statistical outliers: 95054
Total statistical outliers: 95054
Outlier percentage: 5.131414%

ANALYSIS COMPLETED

Results directory:
